# ARRGO: Algorithm Design

This notebook translates the theoretical foundations of ARRGO into a
concrete deterministic optimization procedure.

The objective is to define the internal state, region analysis,
candidate-generation mechanisms, refinement decisions, global selection
strategy, and execution cycle of ARRGO before implementation.

The design must preserve the theoretical properties established in the
previous notebook while remaining practical for numerical computation.

ARRGO is designed as an adaptive refinement framework in which optimization
progress is achieved through the repeated acquisition and organization of
information about the objective function.

The algorithm distinguishes between two fundamental refinement operations:

- **Sampling:** acquiring new function evaluations inside an existing region.
- **Splitting:** introducing new spatial structure by dividing a region into
  smaller regions.

The central design question is therefore:

> Given the current global information state, which region should be refined,
> which unresolved objective should be addressed, and whether sampling or
> splitting provides the more appropriate refinement action?

The algorithm will be developed progressively from these principles.

No implementation-specific decision is introduced before its mathematical
role and justification have been established.

## ARRGO Global State

ARRGO operates as an iterative adaptive optimization process.

At iteration $t$, the complete optimization state is represented by

$$
G_t
=
\left(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t
\right).
$$

Each component represents a different aspect of the information available to
the algorithm.

### Region Structure

The set

$$
\mathcal{R}_t
$$

contains all regions represented in the current hierarchical search
structure.

The region structure describes:

- the geometry of each region;
- parent-child relationships;
- the current state of each region;
- locally available information;
- regional optimization potential;
- refinement history.

ARRGO does not remove previously created regions from the hierarchy.

Therefore, the region structure is persistent throughout the optimization
process.

### Global Evaluation History

The set

$$
D_t
=
\{(x_i,f(x_i))\}_{i=1}^{N_t}
$$

contains all function evaluations performed up to iteration $t$.

This history is persistent.

Once a function value has been evaluated, it remains available to ARRGO for
later analysis and refinement decisions.

Therefore,

$$
D_t\subseteq D_{t+1}.
$$

### Incumbent Solution

The current best evaluated point is

$$
x_{\mathrm{best}}^{(t)}
=
\arg\max_{x_i\in D_t}f(x_i).
$$

Its corresponding objective value is

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in D_t}f(x_i).
$$

The incumbent represents the best solution that ARRGO has actually observed.

It must not automatically be interpreted as the true global optimizer.

The distinction is

$$
f_{\mathrm{best}}^{(t)}
\leq
f^*.
$$

Equality is obtained only when the global optimum has been identified in
objective value.

### Evaluation Counter

The variable

$$
N_t=|D_t|
$$

denotes the number of function evaluations performed up to iteration $t$.

This quantity is used to control the evaluation budget.

If a maximum budget

$$
N_{\max}
$$

is specified, budget-limited termination occurs when

$$
N_t\geq N_{\max}.
$$

### Global State Update

After every refinement action, ARRGO updates the global state.

Conceptually,

$$
G_t
\longrightarrow
G_{t+1}.
$$

The update may modify:

- the region hierarchy;
- the evaluation history;
- regional information;
- regional priorities;
- the incumbent;
- the evaluation counter.

The global state must always remain consistent with the actual information
available to the algorithm.

### Persistent Information

ARRGO follows a persistent-information principle:

$$
\boxed{
\text{New Refinement}
\neq
\text{Loss of Previous Information}.
}
$$

New evaluations and structural refinements extend the existing information
state rather than replacing it.

This is particularly important because an observation that is not immediately
useful may become relevant after another region has been refined.

### Separation of Local and Global State

ARRGO distinguishes between global information and region-local information.

The global state contains information required to coordinate the entire
optimization process.

A region contains information required to analyze and refine that particular
portion of the search space.

Therefore,

$$
\boxed{
\text{Global State}
\neq
\text{Region State}.
}
$$

The two levels interact continuously during region selection and refinement.

### Global State Invariant

At every iteration, ARRGO must maintain the following consistency conditions:

$$
D_t
=
\text{all evaluations performed up to }t,
$$

$$
N_t=|D_t|,
$$

and

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in D_t}f(x_i).
$$

In Certified Mode, the global state must additionally maintain a valid global
potential

$$
P_{\mathrm{global}}^{(t)}
$$

and therefore a valid certified gap

$$
\Delta_{\mathrm{global}}^{(t)}
=
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}.
$$

These invariants ensure that every subsequent algorithmic decision is based
on a consistent representation of the current optimization state.

### Role of the Global State

The global state acts as the coordination layer of ARRGO.

At each iteration, it provides the information required to answer three
fundamental questions:

1. **Where should ARRGO focus next?**
2. **What unresolved information should be addressed?**
3. **Should the next refinement be sampling or splitting?**

The mechanisms for answering these questions are defined in the following
sections.

## ARRGO Region State

A region is the fundamental spatial unit used by ARRGO to organize,
analyze, and refine the search space.

Each region represents a bounded subset of the original optimization
domain together with the information currently available about that subset.

### Region Definition

For the one-dimensional formulation,

$$
R=[l,r],
\qquad
l<r.
$$

The geometric properties of the region are defined by its boundaries:

$$
l=\text{left boundary},
\qquad
r=\text{right boundary}.
$$

Its diameter is

$$
\operatorname{diam}(R)=r-l.
$$

The diameter represents the current spatial resolution of the region.

### Region State

ARRGO represents a region conceptually as

$$
R=
\left(
[l,r],
D_R,
B_R,
Q_R,
U_R,
P_R,
state,
parent,
children
\right).
$$

Each component has a specific role.

### Geometry

The interval

$$
[l,r]
$$

defines the portion of the search space represented by the region.

All region-local samples and candidate points must respect this geometry.

### Local Evaluation Set

The region contains the function evaluations relevant to its current
analysis:

$$
D_R
=
\{(x_i,f(x_i)):x_i\in R\}.
$$

These evaluations provide the empirical information used to analyze the
objective within the region.

The local dataset may contain both inherited observations and newly acquired
observations.

### Behavioral Information

The variable

$$
B_R
$$

represents the observed behavioral information of the region.

Depending on the available samples, this may include:

- observed function variation;
- directional behavior;
- local slope estimates;
- candidate extrema;
- changes in slope;
- local structural patterns.

These quantities are observational descriptors rather than assumptions about
the unseen function.

### Unresolved Information Profile

The vector

$$
Q_R
=
\left(
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
\right)
$$

represents unresolved information relevant to optimization.

Each component answers a different question.

#### Coverage

$$
Q_{\mathrm{coverage}}
$$

measures whether the region has sufficient spatial representation.

#### Behavior

$$
Q_{\mathrm{behavior}}
$$

represents unresolved local behavioral structure.

#### Uncertainty

$$
Q_{\mathrm{uncertainty}}
$$

represents information that remains unknown about the objective.

In Certified Mode, this can be related to the width of valid Lipschitz-based
bounds.

#### Optimization Potential

$$
Q_{\mathrm{potential}}
$$

represents unresolved information about whether the region may contain a
solution that improves the current incumbent.

These components must not be collapsed into a single arbitrary weighted
score unless a mathematically justified common scale is introduced.

### Certified Upper Bound

In Certified Mode, the region maintains a valid upper envelope

$$
U_R(x).
$$

For a valid Lipschitz constant $L$ and local observations,

$$
U_R(x)
=
\min_{x_i\in D_R}
\left[
f(x_i)+L|x-x_i|
\right].
$$

The validity condition is

$$
f(x)\leq U_R(x),
\qquad
\forall x\in R.
$$

In Empirical Mode, $U_R$ may be unavailable because no rigorous unseen-value
bound is assumed.

### Regional Optimization Potential

When a valid upper bound exists, the regional optimization potential is

$$
P_R
=
\max_{x\in R}U_R(x).
$$

Therefore,

$$
\max_{x\in R}f(x)
\leq
P_R.
$$

The potential measures how good the region could still be according to the
available certified information.

This differs from uncertainty.

A region may have high uncertainty without having high optimization
potential, and it may have high potential even when its uncertainty is
relatively small.

### Region State

ARRGO assigns each region a structural state.

The main conceptual states are:

$$
\text{ACTIVE},
\qquad
\text{STABLE},
\qquad
\text{REFINED}.
$$

#### ACTIVE

An active region currently contains unresolved information that may justify
further refinement.

#### STABLE

A stable region has reached a state where additional refinement is not
currently justified by the selected resolution criteria.

A stable region remains represented in the hierarchy.

#### REFINED

A refined region has undergone structural subdivision and therefore has
children in the region hierarchy.

A refined parent is not deleted from the global structure.

### Parent Relationship

Each non-root region may have a parent:

$$
parent(R_c)=R_p.
$$

The child satisfies

$$
R_c\subseteq R_p.
$$

This relationship allows ARRGO to preserve the history of how a region was
created.

### Child Relationship

A region may contain a set of children

$$
children(R)
=
\{R_1,\ldots,R_k\}.
$$

For a valid complete split,

$$
\bigcup_{i=1}^{k}R_i=R.
$$

Thus, structural refinement preserves the represented search space.

### Region Priority

ARRGO assigns each region information required for global prioritization.

In Certified Mode, regional potential provides a mathematically meaningful
component:

$$
P_R.
$$

In Empirical Mode, prioritization may rely on observed optimization
relevance and unresolved information.

The exact priority mechanism is defined separately and must not be confused
with the region state itself.

### Region State Invariant

Every region maintained by ARRGO must satisfy:

$$
\boxed{
\text{Geometry}
+
\text{Valid Local Information}
+
\text{Structural Identity}
}
$$

and, when operating in Certified Mode,

$$
\boxed{
\text{Valid Certified Bound}
+
\text{Valid Regional Potential}.
}
$$

### Region as the Decision Unit

The region is the unit at which ARRGO combines:

$$
\text{Spatial Information},
$$

$$
\text{Observed Behavior},
$$

$$
\text{Uncertainty},
$$

and

$$
\text{Optimization Potential}.
$$

Therefore, the region is not merely an interval.

It is the smallest structural unit that connects function evaluations to
adaptive refinement decisions.

The next design stage defines how ARRGO analyzes this region state and
determines what information remains unresolved.

## Region Analysis

After a region is selected for consideration, ARRGO analyzes the information
currently available inside that region.

The purpose of region analysis is not to predict the complete behavior of the
unknown function.

Instead, it is to determine which aspects of the region remain unresolved
and therefore require further refinement.

### Analysis Input

For a region

$$
R=[l,r],
$$

the primary input is its local evaluation set

$$
D_R
=
\{(x_i,f(x_i))\}.
$$

The analysis may also use:

- the geometry of the region;
- inherited information;
- neighboring regions;
- the current global incumbent;
- certified bounds, when available;
- the region's refinement history.

### Analysis Output

ARRGO produces a region behavior profile

$$
A_R
=
\left(
C_R,
B_R,
U_R,
P_R,
E_R
\right),
$$

where each component describes a different aspect of the current state.

These components are analyzed separately rather than being combined through
arbitrary fixed weights.

### Coverage Analysis

Coverage analysis determines how well the available samples represent the
region spatially.

For ordered samples

$$
x_1<x_2<\cdots<x_n,
$$

the internal sampling gaps are

$$
g_i=x_{i+1}-x_i.
$$

The largest unresolved spatial gap is

$$
g_{\max}
=
\max_i g_i.
$$

Large gaps indicate that significant portions of the region have not yet
received direct evaluations.

Coverage analysis therefore answers:

$$
\text{Where is the region spatially under-observed?}
$$

### Boundary Coverage

The region boundaries are also important.

For

$$
R=[l,r],
$$

ARRGO considers the distances between the region boundaries and nearby
evaluated points.

This prevents a region from appearing well sampled in its interior while
remaining poorly represented near one of its boundaries.

### Behavioral Analysis

ARRGO analyzes the observed changes in the objective function.

For neighboring samples,

$$
(x_i,f_i)
\quad\text{and}\quad
(x_{i+1},f_{i+1}),
$$

the observed secant slope is

$$
s_i
=
\frac{f_{i+1}-f_i}{x_{i+1}-x_i}.
$$

These slopes provide local directional information.

If

$$
s_i>0,
$$

the observed function is increasing across that interval.

If

$$
s_i<0,
$$

the observed function is decreasing across that interval.

If

$$
s_i\approx0,
$$

the observed change is small relative to the selected numerical tolerance.

These statements describe observed behavior only.

They do not constitute a claim about every unseen point inside the interval.

### Directional Changes

Changes between neighboring observed slopes can reveal changes in local
behavior.

For example,

$$
s_i\cdot s_{i+1}<0
$$

indicates an observed directional reversal between consecutive intervals.

Such a reversal may indicate a candidate extremum near the transition.

However, the candidate is an observational hypothesis rather than a
guaranteed global or local extremum.

### Candidate Extrema

When neighboring slopes exhibit a directional reversal, ARRGO may generate a
candidate location for additional sampling.

For example,

$$
s_i>0
\quad\text{and}\quad
s_{i+1}<0
$$

suggests an observed transition from increasing to decreasing behavior.

This is evidence for a possible local maximum.

Conversely,

$$
s_i<0
\quad\text{and}\quad
s_{i+1}>0
$$

suggests a possible local minimum.

The exact candidate-generation mechanism is defined later.

### Behavioral Variation

ARRGO also analyzes how rapidly observed slopes change.

For neighboring slopes,

$$
\Delta s_i
=
s_{i+1}-s_i.
$$

Large values of

$$
|\Delta s_i|
$$

indicate stronger observed changes in local behavior.

This information can be used to distinguish a relatively smooth region from a
region whose observed behavior changes rapidly.

### Local Behavioral Complexity

A region may contain:

- approximately monotone behavior;
- a single observed directional transition;
- multiple directional transitions;
- rapidly changing observed slopes;
- insufficient samples to determine the local structure.

ARRGO records these observations as behavioral information rather than
assuming a particular analytical function class.

### Certified Uncertainty Analysis

In Certified Mode, ARRGO additionally evaluates the valid upper and lower
envelopes.

The lower envelope is

$$
L_R(x)
=
\max_i
\left[
f(x_i)-L|x-x_i|
\right],
$$

and the upper envelope is

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

The pointwise uncertainty is

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

The maximum regional uncertainty is

$$
u_{\max}(R)
=
\max_{x\in R}u_R(x).
$$

This quantity provides a certified measure of unresolved function-value
information under the Lipschitz assumption.

### Optimization Potential Analysis

In Certified Mode, ARRGO computes

$$
P_R
=
\max_{x\in R}U_R(x).
$$

The current incumbent is

$$
f_{\mathrm{best}}.
$$

Therefore, the region remains potentially competitive when

$$
P_R>f_{\mathrm{best}}.
$$

For a target tolerance $\epsilon$, the stronger condition is

$$
P_R>f_{\mathrm{best}}+\epsilon.
$$

Such a region cannot yet be certified as irrelevant to the requested
objective accuracy.

### Empirical Optimization Relevance

In Empirical Mode, no rigorous upper bound is assumed.

ARRGO therefore evaluates optimization relevance using observable evidence,
such as:

- the best function value observed in the region;
- proximity to promising observed values;
- candidate extrema;
- unresolved behavioral patterns;
- sampling gaps around promising locations.

These quantities guide search decisions but do not provide a mathematical
optimality certificate.

### Information Sufficiency

The final purpose of region analysis is to determine whether the current
information is sufficient for the next decision.

ARRGO asks:

$$
\boxed{
\text{What is the most important unresolved information in }R?
}
$$

The answer may be:

$$
\text{Coverage},
$$

$$
\text{Behavior},
$$

$$
\text{Uncertainty},
$$

or

$$
\text{Optimization Potential}.
$$

Multiple objectives may remain unresolved simultaneously.

ARRGO therefore does not assume that every region requires the same type of
refinement.

### Analysis Does Not Automatically Refine

Region analysis and region refinement are separate operations.

The analysis stage observes and organizes the current information:

$$
D_R
\longrightarrow
A_R.
$$

Only after this analysis does ARRGO decide whether refinement is necessary.

Thus,

$$
\boxed{
\text{Analyze}
\neq
\text{Refine}.
}
$$

This separation prevents the algorithm from performing unnecessary function
evaluations or structural splits.

### Region Analysis Principle

The central principle of ARRGO region analysis is:

$$
\boxed{
\text{Use Observations to Identify Unresolved Information,
Not to Invent Unverified Function Behavior}.
}
$$

The resulting analysis profile becomes the input to the next stage:

$$
\boxed{
\text{Region Analysis}
\longrightarrow
\text{Dominant Unresolved Objective}.
}
$$

The mechanism for selecting that dominant objective is defined next.

## Dominant Unresolved Objective

After analyzing a region, ARRGO may identify several unresolved information
objectives simultaneously.

For a region $R$, define the unresolved information profile as

$$
Q_R
=
\left(
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
\right).
$$

These components describe different forms of incomplete information.

### Multiple Unresolved Objectives

A region may simultaneously exhibit:

- insufficient spatial coverage;
- unresolved local behavior;
- large certified uncertainty;
- significant optimization potential.

Therefore, ARRGO must not assume that only one source of uncertainty exists.

For example, a region may have both a large sampling gap and a promising
observed extremum.

In such a case, either objective may influence the next refinement decision.

### Dominant Objective

The dominant unresolved objective is the objective whose resolution is
currently most important for determining the next useful refinement action.

Conceptually,

$$
O_R^{*}
=
\operatorname{Dominant}
\left(
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
\right).
$$

The operator $\operatorname{Dominant}$ is not defined as an arbitrary
weighted sum.

Instead, dominance is determined from the current decision context and the
actual unresolved information represented by the region.

### Why Fixed Weights Are Avoided

A fixed score such as

$$
S_R
=
w_1Q_{\mathrm{coverage}}
+
w_2Q_{\mathrm{behavior}}
+
w_3Q_{\mathrm{uncertainty}}
+
w_4Q_{\mathrm{potential}}
$$

would require the quantities to have compatible scales and would introduce
arbitrary design parameters.

ARRGO therefore does not assume universal weights.

Different objectives may also have different natural units and meanings.

Consequently, they are analyzed separately.

### Decision Relevance

Dominance is determined relative to the current optimization decision.

The important question is not:

$$
\text{Which quantity is numerically largest?}
$$

but rather:

$$
\boxed{
\text{Which unresolved objective can most affect the correctness or quality
of the next refinement decision?}
}
$$

This distinction is essential because numerical magnitude alone does not imply
optimization importance.

### Coverage Dominance

Coverage may become dominant when the available samples leave substantial
spatial gaps.

For example,

$$
g_{\max}
$$

may be large relative to the current region resolution.

In this case, ARRGO may need to acquire information from an under-sampled
portion of the region before making stronger behavioral conclusions.

### Behavior Dominance

Behavior may become dominant when the region contains enough samples for
basic coverage but the observed function behavior remains unresolved.

Examples include:

$$
s_i\cdot s_{i+1}<0
$$

or strong observed changes in neighboring slopes.

Such evidence may justify targeted sampling near a suspected behavioral
transition.

### Uncertainty Dominance

In Certified Mode, uncertainty may become dominant when the valid envelope
remains too wide to distinguish plausible objective behavior.

For example,

$$
u_{\max}(R)
$$

may remain large compared with the desired optimization resolution.

Additional samples can then tighten the certified information available
inside the region.

### Potential Dominance

Optimization potential becomes dominant when a region remains capable of
containing an objective value significantly better than the current
incumbent.

In Certified Mode, this can be expressed through

$$
P_R>f_{\mathrm{best}}+\epsilon.
$$

Such a region remains globally competitive.

In Empirical Mode, potential is represented through observed evidence rather
than a rigorous unseen-value bound.

### Objective Interactions

The unresolved objectives are not independent.

For example:

$$
\text{Coverage}
\rightarrow
\text{Behavior Resolution}
$$

because additional spatial observations may reveal previously unseen
behavior.

Likewise,

$$
\text{Sampling}
\rightarrow
\text{Uncertainty Reduction}
$$

in Certified Mode because new evaluations can tighten the Lipschitz-based
envelope.

Similarly,

$$
\text{Splitting}
\rightarrow
\text{Potential Separation}
$$

because a single region may contain subregions with substantially different
optimization relevance.

Therefore, dominance concerns the most useful current refinement target, not
an immutable property of the region.

### Dominance Is Dynamic

After every refinement,

$$
Q_R^{(t)}
\longrightarrow
Q_R^{(t+1)}.
$$

Consequently, the dominant objective may change.

For example,

$$
\text{Coverage}
\rightarrow
\text{Behavior}
\rightarrow
\text{Potential}
$$

may occur as a region receives progressively more information.

ARRGO therefore recomputes the unresolved objective after refinement rather
than permanently assigning a refinement type to a region.

### Ties Between Objectives

Two or more objectives may be equally important.

ARRGO must not force an artificial ordering when the available information
does not justify one.

Instead, a tie may produce a set of dominant objectives:

$$
\mathcal{O}_R^{*}
\subseteq
\{
\text{coverage},
\text{behavior},
\text{uncertainty},
\text{potential}
\}.
$$

The subsequent candidate-generation stage can then determine which refinement
action best addresses the competing objectives.

### Dominant Objective and Action Selection

The dominant unresolved objective does not directly determine the action.

Instead, the relationship is:

$$
\boxed{
\text{Region Analysis}
\rightarrow
\text{Dominant Objective}
\rightarrow
\text{Candidate Generation}
\rightarrow
\text{Action Selection}.
}
$$

For example, unresolved behavior may lead to sampling candidates, while a
structural separation of competing subregions may justify splitting.

The final action must depend on the available candidates and their expected
refinement effects.

### Principle of Dominant Resolution

ARRGO follows the principle:

$$
\boxed{
\text{Resolve the Most Decision-Relevant Unresolved Information First}.
}
$$

This allows refinement to remain adaptive without imposing a fixed global
priority between all information objectives.

### Role in ARRGO

The dominant unresolved objective acts as the bridge between analysis and
action.

It converts the descriptive region profile

$$
A_R
$$

into a refinement question:

$$
\boxed{
\text{What information should ARRGO acquire or expose next?}
}
$$

The next stage defines how ARRGO generates concrete candidate sampling points
and split locations to answer that question.

## Candidate Generation for Sampling

Once the dominant unresolved objective has been identified, ARRGO generates
candidate locations at which additional function evaluations may provide
useful information.

The purpose of candidate generation is not to select the final sampling point.

Instead, it constructs a set of mathematically motivated candidate locations
that represent different possible sources of information.

The selection among these candidates is performed in a later stage.

### Sampling Candidate Set

For a region

$$
R=[l,r],
$$

define the sampling candidate set as

$$
C_{\mathrm{sample}}(R)
=
\{x_1^{c},x_2^{c},\ldots,x_m^{c}\}
\subseteq R.
$$

Every candidate must satisfy

$$
l\leq x_j^{c}\leq r.
$$

Candidates that are already evaluated within the numerical duplicate
tolerance are excluded.

### Candidate Generation Principles

ARRGO generates candidates from the information already available inside the
region.

The candidate-generation mechanism may consider:

1. spatial coverage;
2. observed behavioral transitions;
3. certified uncertainty;
4. optimization relevance;
5. region boundaries and structural locations.

No candidate is generated solely because it is convenient for the
implementation.

Each candidate must have an identifiable information role.

### Coverage Candidates

When spatial coverage is insufficient, candidates are generated inside
important sampling gaps.

Let the ordered samples be

$$
x_1<x_2<\cdots<x_n.
$$

For every adjacent pair,

$$
g_i=x_{i+1}-x_i.
$$

A coverage candidate may be generated inside a gap

$$
(x_i,x_{i+1}).
$$

The midpoint

$$
x_i^c=\frac{x_i+x_{i+1}}{2}
$$

is one possible deterministic candidate.

However, the midpoint is not inherently optimal.

Other locations may become preferable when additional information indicates
that a non-central location contains greater refinement value.

### Behavioral Candidates

Behavioral analysis may identify transitions between neighboring observed
slopes.

Given

$$
s_i=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i},
$$

a directional reversal can occur when

$$
s_i s_{i+1}<0.
$$

Such a transition defines an interval in which the local behavior may change.

ARRGO may therefore generate candidates inside the corresponding interval.

The candidate is intended to investigate the unresolved behavior rather than
assume that an extremum actually exists there.

### Slope-Variation Candidates

Large changes between neighboring observed slopes may indicate that the
current sampling resolution is insufficient to characterize local behavior.

Define

$$
\Delta s_i=s_{i+1}-s_i.
$$

A candidate may therefore be generated near regions where

$$
|\Delta s_i|
$$

is large relative to the surrounding observed slope variation.

The comparison must remain scale-aware and must not interpret a raw numerical
magnitude as universally significant.

### Certified Uncertainty Candidates

In Certified Mode, the regional uncertainty is

$$
u_R(x)=U_R(x)-L_R(x).
$$

Locations where

$$
u_R(x)
$$

is large represent areas where the current valid information leaves a wide
range of possible objective values.

ARRGO may therefore generate candidates near regions of high certified
uncertainty.

These candidates have a direct mathematical role: obtaining an evaluation
there may tighten the regional enclosure.

### Optimization-Relevance Candidates

A candidate may also be generated in a region that appears particularly
relevant to the current optimization objective.

In Certified Mode, this includes locations associated with a large regional
upper bound

$$
U_R(x).
$$

The purpose is to investigate locations that could influence the regional
optimization potential

$$
P_R=\max_{x\in R}U_R(x).
$$

Importantly, a large upper bound does not imply that the function actually
attains a large value there.

It only indicates that the current valid information does not exclude that
possibility.

### Boundary Candidates

Region boundaries can contain important structural information.

For a region

$$
R=[l,r],
$$

the boundary points

$$
l,\qquad r
$$

may therefore be considered as candidates when they have not already been
evaluated.

Boundary evaluations are especially useful because they provide information
about the behavior of the objective near the limits of the current region.

### Candidate Deduplication

Different generation mechanisms may produce the same candidate.

Therefore ARRGO constructs the candidate set as a unique numerical set:

$$
C_{\mathrm{sample}}(R)
=
\operatorname{Unique}
\left(
C_{\mathrm{coverage}}
\cup
C_{\mathrm{behavior}}
\cup
C_{\mathrm{uncertainty}}
\cup
C_{\mathrm{potential}}
\cup
C_{\mathrm{boundary}}
\right).
$$

Two candidates are considered duplicates when

$$
|x_i^c-x_j^c|\leq\tau_x.
$$

The tolerance $\tau_x$ is a numerical-resolution parameter rather than an
optimization preference.

### Candidate Validity

Every generated candidate must satisfy

$$
x^c\in R
$$

and must not duplicate an existing evaluation within the numerical tolerance.

If no valid candidate can be generated from the current information state,
ARRGO does not invent an arbitrary location.

Instead, the region requires a different refinement mechanism or a structural
decision.

### Candidate Generation Is Not Candidate Selection

Candidate generation answers:

$$
\boxed{
\text{Where could ARRGO acquire useful information?}
}
$$

Candidate selection answers a different question:

$$
\boxed{
\text{Which candidate provides the most useful refinement?}
}
$$

Therefore candidate generation must remain broader than the final selection
process.

Keeping these stages separate prevents the algorithm from prematurely
committing to a single heuristic location.

### Deterministic Candidate Generation

Given the same:

- region state;
- evaluation history;
- numerical tolerances;
- certified assumptions;
- algorithm configuration;

ARRGO must generate the same candidate set.

Thus,

$$
S_t
\longrightarrow
C_{\mathrm{sample}}(R)
$$

is deterministic.

No random candidate generation is required by the framework.

### Information-Source Traceability

Each candidate should retain information about why it was generated.

Conceptually, a candidate can be associated with one or more sources:

$$
\operatorname{Source}(x^c)
\subseteq
\{
\mathrm{coverage},
\mathrm{behavior},
\mathrm{uncertainty},
\mathrm{potential},
\mathrm{boundary}
\}.
$$

This allows the later selection mechanism to determine not only where a
candidate lies, but also which unresolved objective it addresses.

### Role in ARRGO

The complete sampling pipeline is therefore

$$
\boxed{
\text{Region Analysis}
\rightarrow
\text{Dominant Objective}
\rightarrow
\text{Candidate Generation}
\rightarrow
\text{Candidate Evaluation}
\rightarrow
\text{Sampling}.
}
$$

Candidate generation expands the available refinement possibilities.

The next stage determines how these candidates are compared without
introducing arbitrary fixed weights.

## Sampling Candidate Evaluation

After generating sampling candidates, ARRGO evaluates the potential value of
each candidate with respect to the current unresolved information state.

The purpose of this stage is not to predict the unknown objective value at a
candidate.

Instead, it evaluates the expected **refinement effect** of acquiring an
observation at that location.

### Candidate Information Profile

For each sampling candidate $x^c$, define an information profile

$$
I(x^c)
=
\left(
I_{\mathrm{coverage}},
I_{\mathrm{behavior}},
I_{\mathrm{uncertainty}},
I_{\mathrm{potential}}
\right).
$$

Each component represents a different reason why evaluating $x^c$ may be
useful.

The components are kept separate rather than combined into an arbitrary
weighted scalar.

### Coverage Value

A candidate can provide spatial information by reducing an existing sampling
gap.

Suppose

$$
x_i<x^c<x_{i+1}.
$$

Before sampling, the gap is

$$
g=x_{i+1}-x_i.
$$

After evaluating $x^c$, the gap is divided into

$$
g_L=x^c-x_i
$$

and

$$
g_R=x_{i+1}-x^c.
$$

The resulting maximum local gap is

$$
g_{\mathrm{new}}
=
\max(g_L,g_R).
$$

Therefore the coverage improvement can be represented by the reduction

$$
\Delta g
=
g-g_{\mathrm{new}}.
$$

A candidate that substantially reduces a large unresolved gap has high
coverage value.

### Behavioral Value

A candidate may also provide information about local function behavior.

Suppose the candidate lies between two evaluated points:

$$
x_i<x^c<x_{i+1}.
$$

Before evaluation, the behavior inside this interval is represented only by
the secant slope

$$
s_i=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}.
$$

After evaluating $x^c$, two local slopes become available:

$$
s_L=
\frac{f(x^c)-f(x_i)}
{x^c-x_i}
$$

and

$$
s_R=
\frac{f(x_{i+1})-f(x^c)}
{x_{i+1}-x^c}.
$$

These quantities can reveal information that was unavailable before the
evaluation.

In particular, the new observation can determine whether the previous
apparent behavior was consistent with the newly observed local structure.

The exact behavioral value cannot be known before evaluating the objective,
because it depends on the unknown value $f(x^c)$.

Therefore ARRGO must distinguish between:

$$
\text{potential behavioral value}
$$

and

$$
\text{observed behavioral information}.
$$

The former is an acquisition criterion; the latter becomes available only
after evaluation.

### Uncertainty Value in Certified Mode

In Certified Mode, the candidate can be evaluated according to its ability to
tighten the valid regional enclosure.

Before evaluating $x^c$,

$$
L_R(x)\leq f(x)\leq U_R(x).
$$

After observing

$$
y^c=f(x^c),
$$

the new envelopes become

$$
L_R^{\mathrm{new}}(x)
=
\max
\left(
L_R(x),
y^c-L|x-x^c|
\right),
$$

and

$$
U_R^{\mathrm{new}}(x)
=
\min
\left(
U_R(x),
y^c+L|x-x^c|
\right).
$$

The new uncertainty is therefore

$$
u_R^{\mathrm{new}}(x)
=
U_R^{\mathrm{new}}(x)
-
L_R^{\mathrm{new}}(x).
$$

The exact uncertainty reduction depends on the unknown observation $y^c$.

Consequently, ARRGO must not claim a guaranteed numerical uncertainty
reduction before the function is evaluated.

### Potential Value

In Certified Mode, the candidate may influence the regional optimization
potential

$$
P_R=\max_{x\in R}U_R(x).
$$

After observing $y^c$, the updated potential becomes

$$
P_R^{\mathrm{new}}
=
\max_{x\in R}
U_R^{\mathrm{new}}(x).
$$

Since

$$
U_R^{\mathrm{new}}(x)\leq U_R(x),
$$

we have

$$
P_R^{\mathrm{new}}\leq P_R.
$$

Thus, additional valid observations can never make the certified regional
potential larger.

However, the exact reduction

$$
P_R-P_R^{\mathrm{new}}
$$

cannot be known before evaluating the candidate.

### Pre-Evaluation Value Versus Post-Evaluation Effect

This distinction is fundamental.

Before evaluating a candidate, ARRGO knows:

- its spatial position;
- the existing sampling geometry;
- the current behavioral information;
- the current certified envelope, if available;
- the current optimization relevance.

ARRGO does **not** know the actual objective value at the candidate.

Therefore candidate evaluation must use information available before the new
function evaluation.

After the evaluation, ARRGO measures the actual information obtained.

This gives two different concepts:

$$
\boxed{
\text{Pre-Evaluation Candidate Value}
}
$$

and

$$
\boxed{
\text{Observed Post-Evaluation Information Gain}.
}
$$

They must not be conflated.

### Multi-Criteria Candidate Comparison

Because candidate value consists of several dimensions, ARRGO represents each
candidate by its information profile rather than a weighted scalar.

For candidates $x_i^c$ and $x_j^c$, candidate $x_i^c$ dominates $x_j^c$ when
it is at least as useful in every relevant criterion and strictly more useful
in at least one criterion.

Formally,

$$
I_k(x_i^c)\geq I_k(x_j^c)
\qquad
\forall k,
$$

and

$$
I_m(x_i^c)>I_m(x_j^c)
$$

for at least one criterion $m$.

The dominated candidate can then be excluded from the final comparison set.

### Non-Dominated Candidate Set

Define

$$
C_{\mathrm{ND}}
=
\operatorname{NonDominated}
\left(
C_{\mathrm{sample}}
\right).
$$

The resulting set contains candidates for which no other candidate is
strictly better across all relevant information dimensions.

This avoids imposing arbitrary weights between fundamentally different types
of information.

### Relevance-Conditioned Comparison

Not every information criterion is relevant in every region.

For example, certified uncertainty is not a certified criterion in Empirical
Mode.

Similarly, behavioral information may have little decision value when the
region contains too few samples to establish meaningful local structure.

Therefore the active comparison profile is determined by the current region
state:

$$
I_{\mathrm{active}}(x^c)
=
I(x^c)
\big|_{\mathcal{O}_R^{*}},
$$

where $\mathcal{O}_R^{*}$ is the current set of dominant unresolved
objectives.

This prevents irrelevant criteria from affecting the candidate comparison.

### Tie Preservation

Several candidates may remain non-dominated.

ARRGO does not artificially introduce a preference merely to force a unique
winner.

Instead,

$$
|C_{\mathrm{ND}}|>1
$$

means that the available information does not yet establish a strict
preference between those candidates.

A later deterministic tie-breaking rule may be required for implementation,
but such a rule must not be presented as an information-theoretic advantage.

### Deterministic Selection

Once the candidate profiles and dominance relations are known, the selection
procedure must be deterministic.

Given the same:

- region state;
- evaluation history;
- candidate-generation rules;
- numerical tolerances;

ARRGO must produce the same non-dominated candidate set and the same final
selection result.

### Important Limitation

The information profile does not predict whether a candidate contains a
better objective value.

For example, a candidate may have high optimization relevance because it lies
inside a region with a large certified potential.

This does not mean

$$
f(x^c)
$$

will actually be large.

The candidate is valuable because evaluating it may resolve an important
optimization uncertainty.

### Principle

ARRGO therefore follows:

$$
\boxed{
\text{Select Evaluations for Their Potential to Resolve Important Information,
Not Because Their Unknown Function Value Is Assumed to Be Good.}
}
$$

This preserves the black-box nature of the objective function while allowing
the refinement strategy to remain adaptive and information-driven.

### Role in the Algorithm

The sampling decision now has the structure

$$
\boxed{
\text{Region Analysis}
\rightarrow
\text{Dominant Objective}
\rightarrow
\text{Candidate Generation}
\rightarrow
\text{Candidate Evaluation}
\rightarrow
\text{Non-Dominated Comparison}.
}
$$

The next stage defines the final deterministic mechanism that chooses one
sampling candidate from the remaining non-dominated alternatives.

## Final Sampling Candidate Selection

After candidate generation and multi-criteria evaluation, ARRGO may still have
multiple non-dominated sampling candidates.

The purpose of this stage is to select one candidate for the next function
evaluation.

The selection mechanism must remain deterministic and must not introduce
arbitrary numerical weights between unrelated information objectives.

### Input to Final Selection

The selection procedure receives:

- the current region $R$;
- the non-dominated candidate set $C_{\mathrm{ND}}$;
- the dominant unresolved objective set $\mathcal{O}_R^{*}$;
- the current evaluation history;
- the current numerical tolerances;
- the current certified information, when Certified Mode is active.

The selection procedure does not use the unknown value $f(x)$ at any
unevaluated candidate.

### Primary Selection Principle

The first requirement is that the selected candidate addresses the currently
dominant unresolved information.

Therefore, candidates that do not contribute to any currently relevant
objective are not preferred over candidates that do.

Conceptually,

$$
x_{\mathrm{selected}}
\in
\operatorname*{arg\,max}_{x\in C_{\mathrm{ND}}}
\operatorname{Relevance}
\left(
x,\mathcal{O}_R^{*}
\right).
$$

The relevance relation is defined from the information profile rather than
from an arbitrary weighted score.

### Objective-Consistent Selection

If only one unresolved objective remains dominant, selection is performed using
the corresponding information criterion.

For example:

$$
\mathcal{O}_R^{*}
=
\{
\mathrm{coverage}
\}
$$

causes the comparison to focus on spatial refinement.

Similarly:

$$
\mathcal{O}_R^{*}
=
\{
\mathrm{behavior}
\}
$$

causes the comparison to focus on behavioral resolution.

In Certified Mode:

$$
\mathcal{O}_R^{*}
=
\{
\mathrm{uncertainty}
\}
$$

may cause the comparison to focus on certified uncertainty reduction.

### Multiple Dominant Objectives

When

$$
|\mathcal{O}_R^{*}|>1,
$$

there is no mathematically justified reason to collapse the objectives into a
single arbitrary scalar.

Instead, ARRGO retains the multi-objective structure.

A candidate that is simultaneously useful for several dominant objectives may
therefore be preferred because it resolves multiple unresolved information
sources with one evaluation.

Conceptually, define the objective-support set

$$
S(x^c)
\subseteq
\mathcal{O}_R^{*}.
$$

A candidate with

$$
|S(x_i^c)|>|S(x_j^c)|
$$

addresses more currently dominant unresolved objectives.

This comparison is based on structural relevance rather than arbitrary
weights.

### Dominance Before Tie-Breaking

The dominance relation remains the primary comparison mechanism.

If one candidate dominates another according to all active criteria, the
dominated candidate is not selected.

Thus,

$$
x_i^c \succ x_j^c
\quad\Rightarrow\quad
x_j^c
\notin
C_{\mathrm{preferred}}.
$$

This preserves the information-based ordering established during candidate
evaluation.

### Deterministic Tie-Breaking

It is possible that multiple candidates remain equivalent under all
information criteria.

In that case, ARRGO requires a deterministic tie-breaking rule.

The tie-breaker must not be interpreted as evidence that one candidate is
mathematically more informative.

A suitable deterministic ordering can use intrinsic geometric properties,
such as:

1. smaller distance to the relevant unresolved interval;
2. smaller numerical representation under a fixed ordering;
3. a deterministic coordinate ordering.

The exact implementation rule may be selected later.

Its purpose is only reproducibility.

### Why Tie-Breaking Is Not a Heuristic Preference

Suppose

$$
x_1^c,\;x_2^c\in C_{\mathrm{preferred}}
$$

and neither dominates the other.

The available information does not establish that

$$
x_1^c
$$

is intrinsically better than

$$
x_2^c.
$$

A deterministic tie-breaker simply ensures that the algorithm can continue
without randomness.

Therefore:

$$
\boxed{
\text{Deterministic}
\neq
\text{Mathematically Superior}.
}
$$

This distinction is important for interpreting experimental results.

### Selection Under Certified Competition

In Certified Mode, optimization relevance may impose an additional constraint.

If a region is competitive according to

$$
P_R>f_{\mathrm{best}}+\epsilon,
$$

then sampling decisions inside that region should preserve the ability to
resolve its certified potential.

A candidate that can provide information relevant to the competitive region
therefore remains eligible even when another candidate appears attractive
according to empirical behavioral evidence.

Certified relevance cannot be replaced by an empirical heuristic.

### Selection Must Respect Region Validity

Every selected candidate must satisfy

$$
x_{\mathrm{selected}}\in R.
$$

It must also satisfy the duplicate constraint

$$
|x_{\mathrm{selected}}-x_i|>\tau_x
$$

for every previously evaluated point $x_i$ that would make the evaluation
numerically redundant.

If no candidate satisfies the validity requirements, sampling is not
performed at that stage.

The algorithm must instead reconsider the refinement mechanism.

### Post-Selection Evaluation

After selecting

$$
x_{\mathrm{selected}},
$$

ARRGO evaluates the black-box objective:

$$
y_{\mathrm{selected}}
=
f(x_{\mathrm{selected}}).
$$

The new observation is then added to the persistent global evaluation history:

$$
D_{t+1}
=
D_t
\cup
\left\{
(x_{\mathrm{selected}},y_{\mathrm{selected}})
\right\}.
$$

The corresponding region state is updated using the new observation.

### Information Update After Evaluation

The newly observed value can now be used to compute information that was
previously unavailable.

For example:

$$
s_L,\qquad s_R
$$

can be computed for the intervals created by the new sample.

In Certified Mode, the regional envelopes can also be recomputed:

$$
L_R^{\mathrm{new}}(x)
=
\max_i
\left[
f(x_i)-L|x-x_i|
\right],
$$

and

$$
U_R^{\mathrm{new}}(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

Consequently, the actual post-evaluation refinement effect becomes observable.

### Incumbent Update

If the new observation improves the current best value,

$$
y_{\mathrm{selected}}>f_{\mathrm{best}},
$$

then the incumbent is updated:

$$
x_{\mathrm{best}}^{(t+1)}
=
x_{\mathrm{selected}},
$$

and

$$
f_{\mathrm{best}}^{(t+1)}
=
y_{\mathrm{selected}}.
$$

Otherwise, the previous incumbent remains valid.

Therefore,

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

### Sampling Selection Cycle

The complete sampling-selection cycle is

$$
\boxed{
\begin{aligned}
&\text{Analyze Region}\\
&\rightarrow
\text{Identify Dominant Objectives}\\
&\rightarrow
\text{Generate Candidates}\\
&\rightarrow
\text{Evaluate Candidate Profiles}\\
&\rightarrow
\text{Remove Dominated Candidates}\\
&\rightarrow
\text{Apply Deterministic Selection}\\
&\rightarrow
\text{Evaluate Objective}\\
&\rightarrow
\text{Update Information}.
\end{aligned}
}
$$

### Separation of Concerns

ARRGO therefore separates four logically different decisions:

$$
\boxed{
\begin{array}{ll}
\text{Region Analysis} & \text{What is unresolved?}\\
\text{Candidate Generation} & \text{Where could information be acquired?}\\
\text{Candidate Evaluation} & \text{Why is each candidate useful?}\\
\text{Candidate Selection} & \text{Which valid candidate is selected?}
\end{array}
}
$$

This separation prevents the implementation from hiding arbitrary heuristics
inside a single scoring function.

### Principle

The final sampling rule follows:

$$
\boxed{
\text{Choose the Deterministically Best-Supported Candidate Under the
Currently Relevant Information Objectives.}
}
$$

The selected point is therefore the result of the current information state,
not an assumed prediction of the unknown objective function.

The next part of ARRGO addresses the second fundamental refinement operation:
**splitting a region into smaller structural regions**.

## Splitting as Structural Refinement

Sampling increases the amount of information available inside a region.

Splitting performs a different operation: it increases the spatial resolution of
the region hierarchy.

For a region

$$
R=[l,r],
$$

a split introduces a point

$$
s\in(l,r)
$$

and creates two child regions:

$$
R_L=[l,s]
$$

and

$$
R_R=[s,r].
$$

The parent region remains part of the persistent region hierarchy.

Therefore splitting does not remove the parent or discard its information.

### Purpose of Splitting

The primary purpose of splitting is to expose spatial differences that cannot
be represented adequately by the current region.

A region may contain:

- multiple behavioral patterns;
- multiple candidate extrema;
- substantially different sampling densities;
- different levels of certified uncertainty;
- different optimization potentials.

Representing all of these phenomena using one region may make the current
analysis too coarse.

Splitting creates separate structural units in which these properties can be
analyzed independently.

### Structural Resolution

Before splitting,

$$
R=[l,r]
$$

is treated as one spatial unit.

After splitting,

$$
R
\rightarrow
\{R_L,R_R\}.
$$

The child regions satisfy

$$
R_L\cup R_R=R
$$

and

$$
R_L\cap R_R=\{s\}.
$$

Thus the domain represented by the parent is preserved.

### Valid Split Point

A valid split point must satisfy

$$
l<s<r.
$$

A split at a boundary would produce an empty child and therefore does not
provide valid structural refinement.

The split point must also satisfy the numerical tolerance requirements of the
implementation.

### Contraction Requirement

ARRGO requires every valid split to provide spatial contraction.

For the parent diameter,

$$
\operatorname{diam}(R)=r-l.
$$

The child diameters are

$$
\operatorname{diam}(R_L)=s-l
$$

and

$$
\operatorname{diam}(R_R)=r-s.
$$

A valid contraction condition is

$$
\max
\left(
s-l,\;
r-s
\right)
\leq
\rho(r-l),
$$

where

$$
0<\rho<1.
$$

Therefore,

$$
\operatorname{diam}(R_L),
\operatorname{diam}(R_R)
<
\operatorname{diam}(R).
$$

### Why Contraction Matters

Repeated splitting along a nested region sequence produces

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Consequently,

$$
\lim_{k\to\infty}
\operatorname{diam}(R_k)
=
0.
$$

Under a valid Lipschitz assumption,

$$
|f(x)-f(y)|
\leq
L|x-y|,
$$

this also implies

$$
\operatorname{osc}_{R_k}(f)
\leq
L\operatorname{diam}(R_k)
\rightarrow
0.
$$

Thus structural refinement can progressively localize the objective.

### Splitting Does Not Assume a Function Model

ARRGO does not require the objective to be polynomial, quadratic, convex,
concave, or differentiable in order to split a region.

The split decision is based on the information available from the region and
its surrounding global state.

Observed slopes, behavioral transitions, sampling gaps, uncertainty, and
optimization potential may all contribute to the decision.

None of these observations is treated as a guaranteed representation of the
unobserved function unless an explicit mathematical assumption supports such
a claim.

### Splitting and Behavioral Structure

Suppose the region contains observed slopes

$$
s_1,s_2,\ldots,s_m.
$$

A substantial change in the observed slope sequence may indicate that the
current region contains different local behaviors.

For example,

$$
s_i>0,\qquad s_{i+1}<0
$$

may indicate a possible local maximum between the corresponding samples.

However, the observation does not prove the existence or uniqueness of such a
maximum.

Splitting can expose this interval as an independent structural region so that
its behavior can be analyzed at higher resolution.

### Splitting and Optimization Potential

In Certified Mode, a region may have a large potential

$$
P_R
=
\max_{x\in R}U_R(x).
$$

A single region may contain areas with substantially different certified
potentials.

Splitting allows the children to obtain separate potentials:

$$
P_{R_L}
=
\max_{x\in R_L}U_{R_L}(x),
$$

and

$$
P_{R_R}
=
\max_{x\in R_R}U_{R_R}(x).
$$

Because the children cover the parent and their bounds remain valid,

$$
\max_{x\in R}f(x)
\leq
\max(P_{R_L},P_{R_R}).
$$

Thus splitting can transform one coarse optimization unit into several more
localized optimization units.

### Splitting and Information Inheritance

When a region is split, the child regions inherit all previously evaluated
points that lie inside them.

For example,

$$
D_{R_L}
=
\{(x_i,f(x_i))\in D_R:x_i\in R_L\}
$$

and similarly,

$$
D_{R_R}
=
\{(x_i,f(x_i))\in D_R:x_i\in R_R\}.
$$

No previously acquired function evaluation is discarded.

The parent region continues to preserve its historical information.

### Splitting Does Not Equal Pruning

ARRGO explicitly separates splitting from pruning.

Splitting:

$$
R\rightarrow\{R_L,R_R\}
$$

adds structural information.

Pruning would remove a region from consideration.

ARRGO does not delete regions from the hierarchy.

A region may become inactive or stable according to its state, but it remains
represented and can be reconsidered when new global information changes its
relative relevance.

### Structural Refinement Versus Sampling

The two operations have different mathematical roles:

$$
\boxed{
\begin{array}{ll}
\text{Sampling}
&
\text{adds objective evaluations}
\\[4pt]
\text{Splitting}
&
\text{adds spatial structure}.
\end{array}
}
$$

Sampling answers:

$$
\text{What is the objective doing at another point?}
$$

Splitting answers:

$$
\text{Should the current spatial unit be represented by smaller units?}
$$

Neither operation universally dominates the other.

The appropriate operation depends on the unresolved information state.

### Splitting as Information Exposure

A split does not itself reveal a new function value.

Instead, it exposes structural boundaries that allow existing information to be
analyzed separately.

Therefore the immediate information effect of splitting is different from the
information effect of sampling.

This distinction will be important when ARRGO compares the expected refinement
effect of the two operations.

### Structural Validity

A split is valid only when it satisfies all structural requirements:

$$
\boxed{
\begin{aligned}
&l<s<r,\\
&\text{both children are non-empty},\\
&\text{children cover the parent},\\
&\text{child regions satisfy contraction},\\
&\text{existing information remains accessible}.
\end{aligned}
}
$$

These are validity conditions, not optimization preferences.

### Deterministic Splitting

Given the same:

- region state;
- evaluation history;
- neighboring information;
- numerical tolerances;
- algorithm configuration;

the generated split candidates must be deterministic.

Random splitting is not required by ARRGO.

The algorithm should instead derive structural candidates from the current
information state.

### Principle

ARRGO follows the structural refinement principle:

$$
\boxed{
\text{Split a Region When Its Current Spatial Representation Is Too Coarse
to Resolve Important Optimization-Relevant Structure.}
}
$$

The next stage defines how ARRGO identifies candidate split locations inside a
region.

## Candidate Split Locations

After determining that a region may benefit from structural refinement, ARRGO
generates candidate split locations inside the region.

The purpose of this stage is to identify spatial locations at which dividing
the region may expose useful structural differences.

Candidate generation does not itself determine whether splitting is the final
refinement action.

### Split Candidate Set

For a region

$$
R=[l,r],
$$

define the set of candidate split locations as

$$
C_{\mathrm{split}}(R)
=
\{s_1^c,s_2^c,\ldots,s_m^c\}.
$$

Every candidate must satisfy

$$
l<s_j^c<r.
$$

Each candidate therefore defines a potential pair of child regions:

$$
R_L^j=[l,s_j^c]
$$

and

$$
R_R^j=[s_j^c,r].
$$

### Candidate Generation Principle

A split candidate should be generated because the current information suggests
that a particular spatial division may provide useful structural resolution.

ARRGO therefore considers locations associated with observable differences in
the current region.

Possible sources include:

- changes in observed behavior;
- candidate extrema;
- changes in sampling density;
- changes in certified uncertainty;
- changes in optimization potential;
- structurally important gaps.

These sources are treated as evidence for candidate generation, not as proofs
about the unobserved function.

### Behavioral Transition Candidates

Observed slope changes can identify intervals in which the local behavior may
change.

Given consecutive secant slopes

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i},
$$

and

$$
s_{i+1}
=
\frac{f(x_{i+2})-f(x_{i+1})}
{x_{i+2}-x_{i+1}},
$$

a directional reversal occurs when

$$
s_i s_{i+1}<0.
$$

The interval containing the corresponding transition can therefore provide a
candidate split location.

The split does not assert that a true extremum exists at the transition.

It only creates a structural boundary around an area whose observed behavior
may require higher resolution.

### Candidate Extrema

Observed slope reversals may also identify possible local extrema.

For example,

$$
s_i>0,\qquad s_{i+1}<0
$$

is consistent with a possible local maximum.

A candidate split can therefore be generated near the corresponding spatial
transition.

Similarly,

$$
s_i<0,\qquad s_{i+1}>0
$$

is consistent with a possible local minimum.

Because ARRGO solves a maximization problem, candidate regions surrounding
possible maxima may have greater optimization relevance.

However, this remains an empirical observation unless additional assumptions
provide a formal guarantee.

### Sampling-Gap Candidates

A large sampling gap can also provide a structural split candidate.

For adjacent samples,

$$
g_i=x_{i+1}-x_i.
$$

A split inside a large gap can divide an under-resolved spatial interval into
smaller structural units.

A natural geometric candidate is the midpoint:

$$
s_i^c
=
\frac{x_i+x_{i+1}}{2}.
$$

The midpoint is used as a deterministic geometric reference, not because it is
universally optimal.

Other locations may be generated when behavioral or certified information
provides stronger structural evidence.

### Density-Transition Candidates

Sampling density may differ substantially across a region.

Suppose neighboring intervals have lengths

$$
g_i
\qquad\text{and}\qquad
g_{i+1}.
$$

A substantial spatial-density transition may indicate that one side of the
region is more resolved than the other.

Such transitions can provide candidate locations for structural refinement.

The purpose is to make the spatial representation more balanced where this
improves the ability to analyze the objective.

### Uncertainty-Transition Candidates

In Certified Mode, the uncertainty function is

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

Spatial changes in the uncertainty profile may indicate that different parts
of the region have substantially different levels of unresolved information.

A split candidate can therefore be generated near a meaningful transition in
the certified uncertainty structure.

This allows subsequent child regions to maintain separate uncertainty
representations.

### Potential-Transition Candidates

Similarly, the certified upper envelope

$$
U_R(x)
$$

may contain spatially distinct areas with different optimization potential.

A candidate split can be generated near a transition between areas with
different levels of regional potential.

The purpose is not to assume that the upper envelope equals the objective.

It is to expose spatial structure in the current valid bound.

### Structural Candidate Set

The complete split candidate set can therefore be represented conceptually as

$$
C_{\mathrm{split}}(R)
=
C_{\mathrm{behavior}}
\cup
C_{\mathrm{extrema}}
\cup
C_{\mathrm{gap}}
\cup
C_{\mathrm{density}}
\cup
C_{\mathrm{uncertainty}}
\cup
C_{\mathrm{potential}}.
$$

Candidates outside the valid interior of the region are removed.

Duplicate candidates are merged using the numerical tolerance

$$
|s_i^c-s_j^c|\leq\tau_x.
$$

### Contraction Validation

Candidate generation must also respect the contraction requirement.

For a candidate $s^c$, the child diameters are

$$
d_L=s^c-l
$$

and

$$
d_R=r-s^c.
$$

The candidate is structurally valid only if

$$
\max(d_L,d_R)
\leq
\rho(r-l),
$$

where

$$
0<\rho<1.
$$

This prevents the algorithm from generating a split that produces an
arbitrarily small child and an almost unchanged parent-sized child.

### Boundary Exclusion

Candidates satisfying

$$
s^c=l
$$

or

$$
s^c=r
$$

are invalid.

Such a candidate would fail to produce two non-empty child regions.

Therefore,

$$
C_{\mathrm{split}}(R)
\subset
(l,r).
$$

### Existing Structural Boundaries

A candidate split should not unnecessarily reproduce an existing structural
boundary.

If a candidate is numerically indistinguishable from a previously established
split location, it is treated as a duplicate.

This prevents the region hierarchy from repeatedly generating the same
structural division.

### Candidate Traceability

Each split candidate should retain information about its generation source.

Conceptually,

$$
\operatorname{Source}(s^c)
\subseteq
\{
\mathrm{behavior},
\mathrm{extrema},
\mathrm{gap},
\mathrm{density},
\mathrm{uncertainty},
\mathrm{potential}
\}.
$$

A candidate may have more than one source.

For example, a location may simultaneously correspond to:

- a large sampling gap;
- a behavioral transition;
- a change in certified potential.

Such overlap is itself useful information for the later comparison stage.

### Deterministic Candidate Generation

Given the same:

- region state;
- evaluation history;
- numerical tolerances;
- certified assumptions;
- candidate-generation configuration;

ARRGO must generate the same split candidate set.

Thus,

$$
S_R
\longrightarrow
C_{\mathrm{split}}(R)
$$

is deterministic.

Random split generation is not required.

### Candidate Generation Does Not Imply Splitting

The existence of a split candidate does not mean that ARRGO must split the
region.

The candidate set only represents structurally plausible locations.

The algorithm must still compare the structural value of the alternatives and
determine whether splitting provides sufficient refinement benefit.

Therefore,

$$
\boxed{
\text{Candidate Generation}
\neq
\text{Split Decision}.
}
$$

### Principle

ARRGO follows the structural candidate principle:

$$
\boxed{
\text{Generate Split Locations Where the Current Information Suggests That
Spatial Separation May Resolve Important Structure.}
}
$$

The next stage evaluates the structural value of these candidate splits and
compares them without introducing arbitrary fixed weights.

## Structural Value of a Split

After generating candidate split locations, ARRGO evaluates the structural
value of each candidate.

The purpose of this stage is to determine how much useful spatial resolution
may be obtained by replacing one region with two child regions.

A split is valuable when it exposes distinctions that are difficult to resolve
while the region is represented as a single spatial unit.

### Candidate Split

Let the current region be

$$
R=[l,r]
$$

and let a candidate split location be

$$
s^c\in(l,r).
$$

The candidate produces two child regions:

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

The structural value of the candidate is evaluated by comparing the current
representation of $R$ with the information that would be available after
creating $R_L$ and $R_R$.

### Structural Value Profile

For each candidate split $s^c$, define a structural value profile

$$
V_{\mathrm{split}}(s^c)
=
\left(
V_{\mathrm{coverage}},
V_{\mathrm{behavior}},
V_{\mathrm{uncertainty}},
V_{\mathrm{potential}}
\right).
$$

Each component represents a different structural effect of the split.

As with sampling candidates, these quantities are not combined through an
arbitrary fixed weighted sum.

### Coverage Resolution

Splitting changes the spatial resolution of the region.

The parent diameter is

$$
d_R=r-l.
$$

The child diameters are

$$
d_L=s^c-l
$$

and

$$
d_R^{c}=r-s^c.
$$

A split is structurally useful when it reduces the maximum spatial scale that
must be analyzed as one unit.

Define

$$
d_{\max}(s^c)
=
\max(d_L,d_R^{c}).
$$

The contraction improvement can be represented by

$$
\Delta d
=
d_R-d_{\max}(s^c).
$$

A larger $\Delta d$ indicates greater geometric contraction from the
candidate split.

This quantity describes structural resolution only; it does not by itself
establish optimization value.

### Behavioral Separation

A split may also separate regions with different observed behavior.

Suppose the observed secant slopes near the candidate include values

$$
s_i,\qquad s_{i+1}.
$$

If their signs or magnitudes differ substantially, splitting may expose
different local behavioral structures to subsequent analysis.

The corresponding structural effect is therefore related to the difference
between the behavior represented on the two sides of the candidate.

For example, a transition

$$
s_i>0,\qquad s_{i+1}<0
$$

may indicate that the candidate lies near a possible local maximum.

Splitting around such a transition can make the two sides independently
analyzable.

The split does not prove the existence of the extremum.

### Behavioral Separation Is Observational

ARRGO does not infer a complete function model from observed slopes.

The behavioral value of a split is based only on the structure already visible
in the sampled data.

Therefore,

$$
\text{Observed Behavioral Difference}
\neq
\text{Guaranteed Functional Difference}.
$$

This distinction prevents the structural evaluation from becoming an
unverified prediction mechanism.

### Certified Uncertainty Separation

In Certified Mode, the parent region has a valid uncertainty profile

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

A split can expose portions of the region having different uncertainty
characteristics.

For the two children, define

$$
u_{R_L}(x)
=
U_{R_L}(x)-L_{R_L}(x)
$$

and

$$
u_{R_R}(x)
=
U_{R_R}(x)-L_{R_R}(x).
$$

The structural value is related to how differently the two children are
resolved compared with the parent representation.

This is particularly relevant when one part of the parent is already well
resolved while another part remains highly uncertain.

### Certified Potential Separation

In Certified Mode, each child obtains its own regional potential:

$$
P_{R_L}
=
\max_{x\in R_L}U_{R_L}(x)
$$

and

$$
P_{R_R}
=
\max_{x\in R_R}U_{R_R}(x).
$$

The parent has potential

$$
P_R
=
\max_{x\in R}U_R(x).
$$

The split is structurally useful when it allows the optimization potential to
be represented at a finer spatial resolution.

The purpose is not necessarily to decrease the maximum potential immediately.

Instead, the important effect is that potentially competitive areas become
separate optimization units.

### Potential Separation

Suppose the parent contains two spatially separated areas with different
levels of certified potential.

Representing them as one region may hide this distinction.

After splitting,

$$
R\rightarrow\{R_L,R_R\},
$$

the algorithm can maintain separate values

$$
P_{R_L}
\qquad\text{and}\qquad
P_{R_R}.
$$

This allows global selection to distinguish between the children.

### Structural Difference

The split can therefore be viewed as creating a difference between two child
states:

$$
S_{R_L}
\qquad\text{and}\qquad
S_{R_R}.
$$

The greater the meaningful difference between these states, the more
potentially useful the split is for subsequent adaptive refinement.

However, this difference must be based on measurable information rather than
an arbitrary heuristic score.

### Structural Value Is Context-Dependent

A split location does not have an intrinsic universal value.

Its value depends on:

- the current samples;
- the current region geometry;
- observed behavior;
- neighboring information;
- the current incumbent;
- certified information, when available;
- the unresolved objectives of the region.

Therefore,

$$
V_{\mathrm{split}}^{(t)}(s^c)
$$

may change after every new observation or structural refinement.

### Comparison Between Split Candidates

For two candidates

$$
s_1^c,\qquad s_2^c,
$$

ARRGO compares their structural value profiles rather than collapsing them
into a single arbitrary scalar.

Candidate $s_1^c$ dominates $s_2^c$ when it is at least as useful according
to every active structural criterion and strictly better according to at
least one.

Formally,

$$
V_k(s_1^c)\geq V_k(s_2^c)
\qquad
\forall k,
$$

and

$$
V_m(s_1^c)>V_m(s_2^c)
$$

for at least one active criterion $m$.

The dominated candidate can then be excluded from further comparison.

### Non-Dominated Split Candidates

Define

$$
C_{\mathrm{split,ND}}
=
\operatorname{NonDominated}
\left(
C_{\mathrm{split}}
\right).
$$

These candidates represent the currently strongest structural alternatives
according to the available information.

Several candidates may remain non-dominated.

This is not a failure of the method.

It indicates that the available information does not establish a strict
preference between them.

### Structural Value Does Not Predict the Objective

A candidate split can have high structural value without implying that one of
its child regions contains the global optimum.

For example, a strong behavioral transition may justify finer structural
resolution even if the corresponding region ultimately contains no global
maximum.

Therefore,

$$
\boxed{
\text{Structural Value}
\neq
\text{Predicted Objective Value}.
}
$$

The purpose of structural value is to determine where finer representation
may be useful.

### Splitting and Global Relevance

A structurally valuable split becomes particularly important when its parent
region remains globally competitive.

In Certified Mode, this can be expressed through

$$
P_R>f_{\mathrm{best}}+\epsilon.
$$

In such a case, splitting may help separate potentially competitive and
non-competitive portions of the region without deleting either portion.

### Structural Value and No-Pruning Principle

Even if one child appears substantially less promising than the other, ARRGO
does not delete it.

After splitting,

$$
R\rightarrow\{R_L,R_R\},
$$

both children remain represented in the region hierarchy.

Their relative relevance may change as global information improves.

### Principle

ARRGO follows the structural-value principle:

$$
\boxed{
\text{Prefer Structural Refinement When It Meaningfully Separates
Optimization-Relevant Information That Is Currently Represented Too Coarsely.}
}
$$

The next stage decomposes structural value into explicit measurable
differences between the parent and its candidate children.

## Structural Difference Across a Split

The structural value of a split depends on the differences that become
observable between the resulting child regions.

Let the current region be

$$
R=[l,r]
$$

and let a candidate split location be

$$
s^c\in(l,r).
$$

The candidate creates

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

The structural difference measures how differently these two child regions are
represented by the current information.

### Structural Difference Profile

Define the structural difference profile as

$$
D_{\mathrm{split}}(s^c)
=
\left(
D_{\mathrm{coverage}},
D_{\mathrm{behavior}},
D_{\mathrm{uncertainty}},
D_{\mathrm{potential}}
\right).
$$

Each component describes a different form of spatial separation.

These components remain separate and are not combined through arbitrary fixed
weights.

### Coverage Difference

The first source of structural difference is sampling density.

Let the samples belonging to the left and right children be represented by

$$
D_{R_L}
$$

and

$$
D_{R_R}.
$$

Their spatial resolutions can be characterized using their maximum internal
sampling gaps.

Define

$$
g_{\max}(R_L)
$$

and

$$
g_{\max}(R_R).
$$

A large difference

$$
\left|
g_{\max}(R_L)-g_{\max}(R_R)
\right|
$$

indicates that the two sides of the proposed split have different levels of
sampling resolution.

Such a difference may justify representing the two sides as separate
structural units.

### Density Interpretation

Suppose one side of the candidate contains densely distributed observations
while the other side contains a large unresolved gap.

The current region then combines areas with substantially different spatial
resolution.

Splitting can expose this difference and allow each child to be refined
independently.

The density difference is therefore an indicator of structural
heterogeneity, not an optimization guarantee.

### Behavioral Difference

The second source of structural difference is observed function behavior.

Let the observed slope information on the two sides be represented by

$$
B_L
\qquad\text{and}\qquad
B_R.
$$

These may contain:

- secant slope signs;
- slope magnitudes;
- slope changes;
- directional reversals;
- observed local variation.

A split is behaviorally meaningful when the two sides exhibit distinguishable
observed behavior.

For example, if

$$
s_L>0
$$

on one side while

$$
s_R<0
$$

on the other side, the candidate may separate two different directional
behaviors.

### Directional Difference

A particularly important behavioral distinction is directional change.

Define the observed directional state of an interval as belonging to the
categories

$$
\{
\mathrm{increasing},
\mathrm{decreasing},
\mathrm{approximately\ flat}
\}.
$$

A difference between the directional states on the two sides indicates a
possible behavioral transition.

Such a transition can make the split structurally informative.

### Behavioral Magnitude Difference

Two regions may have the same directional state but substantially different
slope magnitudes.

For example,

$$
|s_L|\ll |s_R|.
$$

This indicates that the rate of observed change differs between the two
regions.

Such a difference may also justify structural separation.

### Slope-Variation Difference

The rate at which the observed behavior changes can also differ.

For the left and right regions, define representative observed slope-change
information as

$$
\Delta s_L
\qquad\text{and}\qquad
\Delta s_R.
$$

A large difference

$$
|\Delta s_L-\Delta s_R|
$$

indicates that the local behavioral complexity may differ across the proposed
split.

This allows ARRGO to distinguish between regions that have similar average
behavior but different behavioral variability.

### Certified Uncertainty Difference

In Certified Mode, the two child regions have their own uncertainty profiles:

$$
u_{R_L}(x)
=
U_{R_L}(x)-L_{R_L}(x)
$$

and

$$
u_{R_R}(x)
=
U_{R_R}(x)-L_{R_R}(x).
$$

Their maximum uncertainties are

$$
u_{\max}(R_L)
=
\max_{x\in R_L}u_{R_L}(x)
$$

and

$$
u_{\max}(R_R)
=
\max_{x\in R_R}u_{R_R}(x).
$$

A difference between these quantities,

$$
\left|
u_{\max}(R_L)-u_{\max}(R_R)
\right|,
$$

indicates that the two sides have different levels of certified unresolved
information.

### Potential Difference

In Certified Mode, the children also have separate optimization potentials:

$$
P_{R_L}
=
\max_{x\in R_L}U_{R_L}(x)
$$

and

$$
P_{R_R}
=
\max_{x\in R_R}U_{R_R}(x).
$$

The potential difference is

$$
D_{\mathrm{potential}}
=
|P_{R_L}-P_{R_R}|.
$$

A substantial difference indicates that the two sides have different levels
of certified optimization relevance.

The split can therefore expose which child currently carries the larger
certified potential.

### Potential Difference Is Not Objective Difference

The quantity

$$
|P_{R_L}-P_{R_R}|
$$

does not imply

$$
\left|
\max_{x\in R_L}f(x)
-
\max_{x\in R_R}f(x)
\right|
$$

has the same value.

The former concerns valid upper bounds.

The latter concerns the unknown true regional optima.

ARRGO therefore uses potential difference only as certified structural
information.

### Combined Structural Profile

The complete structural difference profile can be represented as

$$
D_{\mathrm{split}}(s^c)
=
\left(
D_{\mathrm{coverage}},
D_{\mathrm{behavior}},
D_{\mathrm{uncertainty}},
D_{\mathrm{potential}}
\right).
$$

The profile describes how differently the two child regions would be
represented according to the current information state.

### Structural Difference and Homogeneity

If the two sides of a candidate split are highly similar across the active
criteria, the split may provide limited additional structural information.

Conversely, strong differences across multiple active criteria indicate that
the parent region may be hiding meaningful spatial heterogeneity.

Conceptually,

$$
\text{High Structural Difference}
\Rightarrow
\text{Potentially Valuable Spatial Separation}.
$$

This is an indication for refinement, not a proof of optimization benefit.

### No Arbitrary Scalarization

ARRGO does not define a universal scalar such as

$$
D_{\mathrm{total}}
=
w_1D_{\mathrm{coverage}}
+
w_2D_{\mathrm{behavior}}
+
w_3D_{\mathrm{uncertainty}}
+
w_4D_{\mathrm{potential}}.
$$

Such a formulation would introduce arbitrary weights and require incompatible
information dimensions to be placed on a common scale.

Instead, the structural difference profile remains multi-dimensional.

### Non-Dominance Between Split Candidates

For two split candidates

$$
s_1^c
\qquad\text{and}\qquad
s_2^c,
$$

candidate $s_1^c$ dominates $s_2^c$ when it is at least as informative across
every active structural criterion and strictly more informative in at least
one.

Formally,

$$
D_k(s_1^c)\geq D_k(s_2^c)
\qquad
\forall k,
$$

and

$$
D_m(s_1^c)>D_m(s_2^c)
$$

for at least one active criterion $m$.

The dominated candidate can then be removed from the comparison set.

### Multiple Non-Dominated Splits

After dominance filtering, multiple split candidates may remain.

This means that the available information does not establish a unique
structural preference.

ARRGO preserves these alternatives rather than inventing an artificial
ranking.

The final choice can then use the current refinement objective and a
deterministic tie-breaking mechanism.

### Structural Difference Is Dynamic

After a new function evaluation,

$$
D_t
\rightarrow
D_{t+1},
$$

the observed slopes, sampling density, uncertainty, and certified potentials
may all change.

Therefore,

$$
D_{\mathrm{split}}^{(t)}(s^c)
\neq
D_{\mathrm{split}}^{(t+1)}(s^c)
$$

in general.

Split candidates must therefore be reevaluated from the current information
state.

### Principle

ARRGO follows the structural-difference principle:

$$
\boxed{
\text{A Split Is Structurally Valuable When It Separates Child Regions That
Are Meaningfully Different Under the Current Information Representation.}
}
$$

The next stage focuses specifically on directional differences across a
candidate split.

## Directional Difference Across a Split

One important source of structural information is the difference in observed
directional behavior on the two sides of a candidate split.

Let the current region be

$$
R=[l,r]
$$

and let a candidate split location be

$$
s^c\in(l,r).
$$

The candidate creates

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

The objective of this analysis is to determine whether the available
observations indicate different directional behavior on the two sides.

### Observed Directional State

For an interval between two evaluated points

$$
x_i<x_{i+1},
$$

the observed secant slope is

$$
s_i=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}.
$$

Because

$$
x_{i+1}-x_i>0,
$$

the sign of $s_i$ determines the observed direction of change.

The directional state can therefore be represented conceptually as

$$
\operatorname{Dir}(s_i)
=
\begin{cases}
\mathrm{increasing}, & s_i>0,\\
\mathrm{decreasing}, & s_i<0,\\
\mathrm{flat}, & s_i=0.
\end{cases}
$$

In numerical computation, an approximately flat state may later be defined
using an appropriate numerical tolerance.

### Directional Difference

For the left and right sides of a candidate split, let their representative
observed directional states be

$$
D_L
\qquad\text{and}\qquad
D_R.
$$

A directional difference exists when

$$
D_L\neq D_R.
$$

For example,

$$
D_L=\mathrm{increasing},
\qquad
D_R=\mathrm{decreasing}
$$

represents a change from increasing to decreasing behavior.

Similarly,

$$
D_L=\mathrm{decreasing},
\qquad
D_R=\mathrm{increasing}
$$

represents the opposite directional transition.

### Directional Reversal

A directional reversal is particularly important for maximization.

Suppose the observed behavior changes from increasing to decreasing:

$$
s_L>0,
\qquad
s_R<0.
$$

This pattern is consistent with the presence of a possible local maximum near
the transition.

However, the observed slopes do not prove that a true local maximum exists.

The correct interpretation is:

$$
\boxed{
\text{Observed Directional Reversal}
\Rightarrow
\text{Evidence for Local Structural Change}.
}
$$

It does not imply

$$
\text{Observed Directional Reversal}
\Rightarrow
\text{Guaranteed Local Maximum}.
$$

### Relevance to Maximization

ARRGO is formulated as a maximization framework.

Therefore an increasing-to-decreasing transition is particularly relevant
because it is consistent with a locally favorable region.

The opposite transition,

$$
s_L<0,
\qquad
s_R>0,
$$

is consistent with a possible local minimum.

Although such a minimum is not directly desirable for maximization, it may
still provide useful structural information about the surrounding regions.

### Magnitude of Directional Difference

Direction alone may not capture the complete difference between two regions.

Two regions may both be increasing while exhibiting substantially different
rates of change.

For example,

$$
0<s_L\ll s_R.
$$

Thus ARRGO distinguishes between:

$$
\text{directional state}
$$

and

$$
\text{directional magnitude}.
$$

The first describes the sign of observed change.

The second describes the magnitude of the observed rate of change.

### Directional Magnitude Difference

For two representative slopes, the magnitude difference may be represented as

$$
D_{\mathrm{magnitude}}
=
\left|
|s_L|-|s_R|
\right|.
$$

A large value indicates that the observed rates of change differ substantially.

This can identify regions in which the local behavior is becoming more or less
rapid across the candidate split.

The numerical significance of this difference remains dependent on the scale of
the objective and the current region.

### Directional Transition as Structural Evidence

A candidate split near a directional transition may be useful because it
separates two different observed behaviors.

For example,

$$
\mathrm{increasing}
\rightarrow
\mathrm{decreasing}
$$

may indicate that a single region is representing two qualitatively different
local behaviors.

Splitting can then allow the child regions to be analyzed independently.

### No Derivative Assumption

The directional analysis is based on secant slopes between evaluated points.

Therefore ARRGO does not require analytical derivatives.

The quantity

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}
$$

is an observed finite-difference quantity.

It should not automatically be interpreted as the exact derivative

$$
f'(x).
$$

If differentiability is later assumed, secant slopes may provide numerical
approximations to derivatives, but that is an additional interpretation.

### Sparse Observations

Directional difference becomes less reliable when very few observations are
available.

For example, with only one sample inside a region, no internal secant slope
can be computed.

Therefore ARRGO must distinguish between:

$$
\text{No Evidence of Directional Difference}
$$

and

$$
\text{Insufficient Information to Detect Directional Difference}.
$$

The second state is an unresolved information condition rather than evidence
that the region is behaviorally homogeneous.

### Boundary Considerations

If the candidate split lies close to a region boundary, one side may contain
too few observations to establish a meaningful directional state.

Such a candidate should not receive artificial behavioral evidence merely
because the available data are sparse.

Instead, the lack of directional information should remain explicit in the
region state.

### Directional Difference Profile

For a candidate split, ARRGO may therefore represent directional information
through

$$
D_{\mathrm{direction}}
=
\left(
D_L,
D_R,
D_{\mathrm{magnitude}}
\right).
$$

This profile distinguishes:

- the observed direction on each side;
- whether the directions differ;
- whether their observed rates differ.

### Comparison Between Split Candidates

For two candidates

$$
s_1^c
\qquad\text{and}\qquad
s_2^c,
$$

their directional profiles can be compared only over criteria that are
meaningful for both candidates.

A candidate that provides a clearer observed directional separation may be
structurally preferable when behavioral resolution is the active refinement
objective.

However, directional difference alone does not establish global or local
optimality.

### Interaction With Other Structural Information

Directional difference should be interpreted together with other available
information.

For example:

$$
\text{Directional Reversal}
+
\text{High Sampling Resolution}
$$

provides stronger observational support for a localized behavioral transition
than a directional reversal inferred from extremely sparse samples.

Similarly,

$$
\text{Directional Difference}
+
\text{High Certified Potential}
$$

may make a structural split particularly relevant in Certified Mode.

These combinations remain evidence-based and do not constitute guarantees
about unobserved function behavior.

### Role in Structural Refinement

Directional difference helps answer:

$$
\boxed{
\text{Does the candidate split separate meaningfully different observed
directions of function change?}
}
$$

If the answer is yes, the split may provide useful structural resolution.

If the answer cannot be determined because of insufficient observations, the
region remains information-limited rather than being classified as
behaviorally uniform.

### Principle

ARRGO follows the directional-difference principle:

$$
\boxed{
\text{Use Observed Directional Changes to Identify Potential Structural
Transitions, Without Treating Them as Proof of Unobserved Extrema.}
}
$$

The next structural criterion examines differences in the rate at which the
observed slopes themselves change.

## Slope Variation Difference Across a Split

Directional behavior describes whether the observed function values increase,
decrease, or remain approximately unchanged.

However, direction alone does not describe how the rate of change itself
evolves across a region.

Two regions may both be increasing while one has nearly constant slope and
the other has rapidly changing slopes.

ARRGO therefore analyzes the variation of observed secant slopes as a separate
behavioral criterion.

### Observed Secant Slopes

For consecutive evaluated points

$$
x_i<x_{i+1},
$$

the observed secant slope is

$$
s_i=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}.
$$

For three consecutive evaluated points

$$
x_i<x_{i+1}<x_{i+2},
$$

two consecutive secant slopes are available:

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i},
$$

and

$$
s_{i+1}
=
\frac{f(x_{i+2})-f(x_{i+1})}
{x_{i+2}-x_{i+1}}.
$$

### Slope Variation

The observed change in slope can be represented as

$$
\Delta s_i=s_{i+1}-s_i.
$$

Its magnitude is

$$
|\Delta s_i|.
$$

A small value indicates that the observed secant slopes are similar.

A larger value indicates that the observed rate of change is changing more
strongly across the corresponding sample intervals.

This is an observational quantity and does not require analytical
differentiability.

### Interpretation of Slope Variation

The sign of

$$
\Delta s_i
$$

provides additional behavioral information.

If

$$
\Delta s_i>0,
$$

the observed slope is increasing.

If

$$
\Delta s_i<0,
$$

the observed slope is decreasing.

If

$$
\Delta s_i\approx0,
$$

the observed slopes are approximately stable over the corresponding
observations.

These patterns describe observed changes in secant slopes rather than exact
changes in the derivative.

### Candidate Split and Child Regions

Consider a candidate split

$$
s^c\in(l,r)
$$

with child regions

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

Let the available slope-variation information in the two children be
represented by

$$
V_L
\qquad\text{and}\qquad
V_R.
$$

Each quantity summarizes only the slope variations supported by the available
observations in its corresponding region.

### Left-Right Slope Variation Difference

A basic comparison between the two sides is

$$
D_{\Delta s}
=
\left|
V_L-V_R
\right|.
$$

The exact definition of $V_L$ and $V_R$ may depend on the available
observations and the chosen regional representation.

For example, a regional representation may retain the observed set

$$
\{
\Delta s_i
\}
$$

rather than collapsing all behavior into a single statistic.

This preserves more information and avoids introducing an arbitrary summary.

### Why a Single Variation Statistic May Be Insufficient

Consider two regions with slope-variation sets

$$
\{\Delta s_i\}_{L}
$$

and

$$
\{\Delta s_i\}_{R}.
$$

Two regions may have the same maximum variation but very different internal
patterns.

For example, one region may contain a single sharp transition while the
other contains several moderate transitions.

Therefore ARRGO should retain the underlying variation profile whenever
possible.

A scalar summary is used only when it has a clearly defined role in a specific
comparison.

### Structural Interpretation

A large difference between the slope-variation profiles may indicate that the
candidate split separates regions with different behavioral complexity.

For example:

$$
|\Delta s_i|_{\!L}
\ll
|\Delta s_j|_{\!R}
$$

suggests that the right side exhibits stronger observed changes in its rate of
variation.

The split may therefore improve the representation of local behavior by
allowing the two regions to be analyzed separately.

### Relation to Curvature

Slope variation is related conceptually to curvature.

For sufficiently smooth functions, changes in derivative values are related
to second-order behavior.

However, ARRGO does not assume that

$$
\Delta s_i
$$

is an exact measurement of

$$
f''(x).
$$

The algorithm uses finite differences because the objective is treated as a
black-box function.

Therefore the correct interpretation is:

$$
\boxed{
\text{Slope Variation}
=
\text{Observed Change in Secant Slopes}.
}
$$

It is not automatically an analytical second derivative.

### Directional Change Versus Slope Variation

Directional difference and slope variation provide different information.

Directional difference asks:

$$
\text{Do the two sides move in different observed directions?}
$$

Slope variation asks:

$$
\text{Does the rate of observed change evolve differently on the two sides?}
$$

Consequently, two regions can have

$$
D_L=D_R
$$

while still having substantially different slope-variation profiles.

For example, both regions may be increasing while one has nearly constant
slope and the other has rapidly increasing slope.

### Behavioral Complexity

Large and irregular slope variations may indicate that the current region has
not yet resolved its local behavior adequately.

This does not mean that the underlying function is necessarily globally
complex.

It only means that the available observations exhibit stronger changes in
their local rate of variation.

Thus ARRGO uses slope variation as evidence for possible behavioral
complexity.

### Sparse-Data Limitation

Slope variation requires at least two adjacent secant slopes.

Therefore it requires at least three appropriately ordered evaluated points.

If this information is unavailable, ARRGO must represent the corresponding
behavioral criterion as unresolved.

In particular,

$$
\text{No Observed Slope Variation}
$$

must not automatically be interpreted as

$$
\text{Constant Underlying Behavior}.
$$

The distinction is between lack of evidence and evidence of stability.

### Uneven Sampling

If the sample intervals have different widths,

$$
x_{i+1}-x_i
\neq
x_{i+2}-x_{i+1},
$$

the secant slopes remain valid because each is normalized by its own spatial
distance.

However, comparisons of slope variation should retain the underlying sample
locations because the same value of

$$
|\Delta s_i|
$$

can arise from different spatial configurations.

Therefore ARRGO keeps geometry and behavioral information separate.

### Candidate Split Near a Behavioral Transition

A candidate split is particularly interesting when it lies near a location at
which slope variation changes substantially.

For example,

$$
|\Delta s_i|_{\!L}
\ll
|\Delta s_j|_{\!R}
$$

may indicate that the right side contains a more rapidly changing observed
behavior.

Similarly, a transition from

$$
\Delta s_i>0
$$

to

$$
\Delta s_j<0
$$

may indicate that the rate of observed change itself changes direction.

Such a pattern can motivate structural refinement.

It does not prove the presence of a particular analytical feature of the
unknown function.

### Interaction With Directional Reversal

Slope variation can provide additional information when combined with a
directional reversal.

For example,

$$
s_L>0,
\qquad
s_R<0
$$

indicates an observed increasing-to-decreasing transition.

If the corresponding slope changes are also substantial, the available
observations provide stronger evidence that the current spatial representation
contains a localized behavioral transition.

The interpretation remains observational.

### Structural Difference Profile

The behavioral component of a split can therefore include both directional and
slope-variation information:

$$
D_{\mathrm{behavior}}
=
\left(
D_{\mathrm{direction}},
D_{\Delta s}
\right).
$$

This preserves the distinction between:

- direction of observed change;
- magnitude of observed change;
- variation of observed slopes;
- transition patterns across the candidate split.

### Comparison Without Arbitrary Weighting

These behavioral criteria should not be combined using an arbitrary expression
such as

$$
\alpha D_{\mathrm{direction}}
+
\beta D_{\Delta s}.
$$

Unless a mathematically justified common scale and weighting rule are
introduced, such a scalar score would impose an unsupported preference.

Instead, ARRGO retains the criteria separately and uses dominance or
objective-specific comparison when selecting among candidate splits.

### Role in Split Selection

Slope-variation difference contributes to the question:

$$
\boxed{
\text{Does the candidate split separate regions with meaningfully different
observed rates of behavioral change?}
}
$$

If the available evidence supports such a difference, the candidate may provide
valuable structural refinement.

If the available data are insufficient, ARRGO preserves that uncertainty
rather than assuming behavioral uniformity.

### Principle

ARRGO follows the slope-variation principle:

$$
\boxed{
\text{Use Changes in Observed Secant Slopes to Detect Potential Differences
in Local Behavioral Complexity, Without Treating Them as Exact Derivative
Information.}
}
$$

The next structural criterion examines how sampling density and spatial
coverage differ across the two child regions.

## Sampling Density and Coverage Difference Across a Split

Behavioral differences are not the only source of structural information.

A candidate split can also separate regions with substantially different levels of
sampling density and spatial coverage.

This distinction is important because an apparent behavioral difference may be
caused by uneven sampling rather than by a genuine difference in the observed
function behavior.

ARRGO therefore evaluates sampling density and coverage as separate structural
criteria.

### Spatial Coverage

Let a region be

$$
R=[l,r].
$$

Suppose its evaluated points are ordered as

$$
x_1<x_2<\cdots<x_n.
$$

The internal sampling gaps are

$$
g_i=x_{i+1}-x_i,
\qquad
i=1,\ldots,n-1.
$$

The largest internal gap is

$$
g_{\max}(R)
=
\max_i g_i.
$$

A large value of

$$
g_{\max}(R)
$$

indicates that a relatively large portion of the region is not directly
represented by nearby evaluations.

### Boundary Coverage

Internal gaps do not completely describe spatial coverage.

The distance from the left boundary to the first observation is

$$
g_{\mathrm{left}}
=
x_1-l,
$$

and the distance from the last observation to the right boundary is

$$
g_{\mathrm{right}}
=
r-x_n.
$$

Therefore a complete coverage profile can be represented as

$$
G(R)
=
\left(
g_{\mathrm{left}},
\{g_i\},
g_{\mathrm{right}}
\right).
$$

This prevents ARRGO from treating a region as well covered merely because its
internal samples are dense while one of its boundaries remains poorly
represented.

### Coverage Resolution

The largest spatial gap provides a simple measure of the coarsest observed
coverage:

$$
G_{\max}(R)
=
\max
\left\{
g_{\mathrm{left}},
g_{\max}(R),
g_{\mathrm{right}}
\right\}.
$$

A smaller value indicates finer spatial coverage under the current sampling
configuration.

This quantity describes the sampling structure and does not by itself provide
a bound on the unknown function.

### Sampling Density

Sampling density describes how many observations are available relative to the
spatial extent of a region.

For

$$
R=[l,r],
$$

the region diameter is

$$
d(R)=r-l.
$$

A simple descriptive density measure is

$$
\rho_{\mathrm{sample}}(R)
=
\frac{n}{d(R)},
$$

when

$$
d(R)>0.
$$

This quantity is useful for describing the distribution of observations, but
density alone does not guarantee uniform coverage.

A region may contain many observations concentrated in a small subinterval
while leaving another part poorly sampled.

Therefore ARRGO treats density and coverage as distinct pieces of information.

### Split-Induced Child Regions

For a candidate split

$$
s^c\in(l,r),
$$

define

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

The sampling configurations of the two children are inherited from the
available observations in the parent region.

The resulting coverage profiles are

$$
G(R_L)
\qquad\text{and}\qquad
G(R_R).
$$

Their density measures are

$$
\rho_{\mathrm{sample}}(R_L)
\qquad\text{and}\qquad
\rho_{\mathrm{sample}}(R_R).
$$

### Coverage Difference Across the Split

A basic measure of coverage imbalance is

$$
D_{\mathrm{coverage}}
=
\left|
G_{\max}(R_L)-G_{\max}(R_R)
\right|.
$$

A large value indicates that the two child regions have substantially different
coarsest spatial resolutions.

However, this scalar does not preserve the complete gap structure.

Therefore the underlying profiles

$$
G(R_L)
\qquad\text{and}\qquad
G(R_R)
$$

remain part of the regional information state.

### Density Difference Across the Split

The corresponding difference in descriptive sampling density is

$$
D_{\mathrm{density}}
=
\left|
\rho_{\mathrm{sample}}(R_L)
-
\rho_{\mathrm{sample}}(R_R)
\right|.
$$

A large value indicates that the observations are distributed at different
densities on the two sides of the split.

This should not be interpreted as evidence that one side is more important for
optimization.

It only describes the current information distribution.

### Coverage Versus Density

Coverage and density answer different questions.

Coverage asks:

$$
\text{How large are the unresolved spatial gaps?}
$$

Density asks:

$$
\text{How many observations are available relative to spatial extent?}
$$

For example, one child may have a high sampling density but still contain one
large unresolved gap.

Conversely, another child may have fewer observations but reasonably uniform
coverage.

Therefore ARRGO does not replace the coverage profile with a density measure.

### Structural Difference Caused by Sampling

A candidate split may expose a substantial information imbalance:

$$
G_{\max}(R_L)
\gg
G_{\max}(R_R).
$$

This means that the left side is spatially less resolved than the right side.

Such a split may be useful because subsequent sampling can then focus on the
less-resolved structure without requiring the two children to share the same
sampling representation.

### Midpoint and Gap-Based Splits

Large sampling gaps naturally generate candidate split locations.

For an internal gap

$$
[x_i,x_{i+1}],
$$

a deterministic candidate may be placed at its midpoint:

$$
s^c=
\frac{x_i+x_{i+1}}{2}.
$$

This creates a direct relationship between coverage analysis and structural
refinement.

The purpose of such a split is not to assume that the function behaves
linearly inside the gap.

It simply introduces additional spatial structure into an insufficiently
resolved interval.

### Avoiding Artificial Information

A split itself does not create a new function evaluation.

Therefore splitting a region with sparse observations does not automatically
increase the amount of objective information.

It only creates smaller spatial units in which the existing information can be
analyzed separately.

Consequently,

$$
\boxed{
\text{Structural Resolution}
\neq
\text{New Function Information}.
}
$$

New function information requires sampling.

### Interaction With Behavioral Information

Sampling density and coverage also affect the reliability of behavioral
analysis.

For example, suppose

$$
R_L
$$

has many closely spaced observations while

$$
R_R
$$

has only a few widely separated observations.

A stronger apparent behavioral pattern on the left may simply reflect better
observability.

Therefore ARRGO should not interpret behavioral differences independently of
their spatial information context.

### Interaction With Certified Uncertainty

In Certified Mode, sampling coverage has a direct relationship with the
tightness of Lipschitz-based envelopes.

More informative spatial placement of observations can reduce the distance
between an arbitrary point and nearby evaluated points.

Since the envelope terms contain

$$
L|x-x_i|,
$$

better spatial coverage can contribute to tighter regional bounds.

However, the exact reduction depends on the observed function values and the
configuration of all available samples.

Therefore coverage is an input to certified uncertainty analysis, not a
replacement for it.

### Coverage Difference and Structural Refinement

A candidate split may therefore be structurally useful when it separates
regions with different spatial resolutions.

For example,

$$
G_{\max}(R_L)
>
G_{\max}(R_R)
$$

indicates that the left child currently contains a larger unresolved spatial
gap.

This may support further sampling or refinement on the left side.

The split itself does not determine which child contains the global optimum.

### Sparse-Data State

If one child contains too few observations to construct a meaningful internal
coverage profile, ARRGO records the corresponding information as unresolved.

For example, with zero or one observation in a child, internal gaps cannot be
defined.

This must not be interpreted as perfect coverage.

Instead:

$$
\boxed{
\text{Insufficient Sampling Information}
\neq
\text{Sufficient Spatial Coverage}.
}
$$

### Structural Coverage Profile

For a candidate split, ARRGO can therefore represent the spatial information
through

$$
D_{\mathrm{spatial}}
=
\left(
G(R_L),
G(R_R),
D_{\mathrm{coverage}},
D_{\mathrm{density}}
\right).
$$

This profile preserves both absolute spatial resolution and the difference
between the two child regions.

### Comparison Between Split Candidates

Two candidate split locations may produce different spatial structures.

A candidate may create:

$$
\text{balanced coverage}
$$

while another may create:

$$
\text{strongly asymmetric coverage}.
$$

Neither structure is universally superior.

The appropriate choice depends on the currently unresolved information
objective.

For example, when coverage resolution is dominant, candidates that address
large unresolved gaps may be preferred.

When behavioral resolution is dominant, the spatial profile is considered
together with the behavioral evidence rather than overriding it automatically.

### No Arbitrary Scalarization

Coverage and density should not be combined into an arbitrary score such as

$$
\alpha D_{\mathrm{coverage}}
+
\beta D_{\mathrm{density}}.
$$

Without a mathematically justified weighting scheme, such a score would encode
an unsupported preference between different information dimensions.

ARRGO therefore retains these quantities separately and uses
objective-consistent comparison and dominance.

### Role in Split Evaluation

Sampling density and coverage help answer:

$$
\boxed{
\text{Does this split separate regions with substantially different spatial
information resolution?}
}
$$

If so, the split may expose an information imbalance that can guide later
sampling and refinement.

### Principle

ARRGO follows the spatial-information principle:

$$
\boxed{
\text{Use Sampling Density and Spatial Coverage to Measure Where the Current
Information Representation Is Strong or Weak, Without Confusing Sampling
Structure With Objective Behavior.}
}
$$

The next structural criterion examines how the certified uncertainty changes
across the two child regions.

## Uncertainty Difference Across a Split

In Certified Mode, ARRGO has access to a valid Lipschitz constant $L$ and can
construct rigorous lower and upper envelopes from the available function
evaluations.

A candidate split can therefore be evaluated according to how it separates
regions with different levels of certified uncertainty.

### Certified Regional Envelopes

For a region

$$
R=[l,r],
$$

with evaluated points

$$
D_R=
\left\{
(x_i,f(x_i))
\right\}_{i=1}^{n},
$$

the Lipschitz-based lower envelope is

$$
L_R(x)
=
\max_i
\left[
f(x_i)-L|x-x_i|
\right],
$$

and the upper envelope is

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

Under the valid Lipschitz assumption,

$$
L_R(x)
\leq
f(x)
\leq
U_R(x)
$$

for every

$$
x\in R.
$$

### Pointwise Certified Uncertainty

The pointwise uncertainty is

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

Therefore,

$$
u_R(x)\geq0.
$$

A small value means that the available observations impose a relatively tight
certified enclosure at that location.

A larger value means that the current observations still allow a wider range
of function values consistent with the Lipschitz bound.

This quantity is a rigorous enclosure width under the stated assumption.

It is not a probability, confidence level, or prediction interval.

### Regional Certified Uncertainty

A regional summary can be defined as

$$
u_{\max}(R)
=
\max_{x\in R}u_R(x).
$$

This represents the largest remaining certified enclosure width in the region.

A smaller value corresponds to stronger worst-case information resolution.

The complete uncertainty profile

$$
u_R(x)
$$

should nevertheless be retained whenever possible because a single maximum
does not describe where the uncertainty occurs.

### Candidate Split

Consider a candidate split

$$
s^c\in(l,r)
$$

with children

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

Each child inherits all parent observations that lie inside that child.

The corresponding certified envelopes are constructed from the inherited
observations.

Thus the children obtain their own uncertainty profiles:

$$
u_{R_L}(x)
$$

and

$$
u_{R_R}(x).
$$

### Uncertainty Difference Between Children

The simplest worst-case comparison is

$$
D_{\mathrm{uncertainty}}
=
\left|
u_{\max}(R_L)
-
u_{\max}(R_R)
\right|.
$$

A large value indicates that the two child regions have substantially
different worst-case certified uncertainty.

For example,

$$
u_{\max}(R_L)
\gg
u_{\max}(R_R)
$$

means that the left child currently has a much wider certified uncertainty
range.

This can identify the left child as a potentially more informative target for
future sampling.

### Why Uncertainty Difference Matters

Suppose two regions have similar observed objective behavior but very different
certified uncertainty.

Then behavioral similarity alone does not imply equal information quality.

For example,

$$
u_{\max}(R_L)
<
u_{\max}(R_R)
$$

indicates that the available observations provide a tighter certified
representation on the left.

The right side may therefore require additional information before its
optimization status can be resolved.

### Split Does Not Automatically Reduce Uncertainty

A crucial distinction is that splitting itself does not create a new function
evaluation.

Therefore a split does not automatically guarantee that uncertainty decreases
in every child.

The split changes the spatial organization of the existing information.

Any additional reduction in uncertainty must follow from the new regional
geometry and the observations available to the child.

New objective information requires sampling.

Thus:

$$
\boxed{
\text{Split}
\neq
\text{Automatic Uncertainty Reduction}.
}
$$

### Relationship With Spatial Coverage

Certified uncertainty is strongly related to spatial coverage.

The envelopes contain terms of the form

$$
L|x-x_i|.
$$

Therefore the spatial arrangement of evaluated points directly affects the
tightness of the certified enclosure.

Poorly covered regions can contain locations that are far from nearby
observations and may therefore retain larger uncertainty.

However, the uncertainty profile depends on all observations and their
function values.

Consequently, coverage analysis and certified uncertainty remain separate
criteria.

### Relationship With Region Diameter

Under the Lipschitz assumption, any two points $x,y\in R$ satisfy

$$
|f(x)-f(y)|
\leq
L|x-y|
\leq
L\,\operatorname{diam}(R).
$$

Therefore,

$$
\operatorname{osc}_R(f)
\leq
L\,\operatorname{diam}(R).
$$

As a region contracts,

$$
\operatorname{diam}(R)\rightarrow0,
$$

the maximum possible function variation allowed by the Lipschitz assumption
also approaches zero.

This provides the theoretical connection between spatial refinement and
certified resolution.

### Localized Uncertainty

A split may also separate a region into areas where uncertainty is concentrated
differently.

For example, the parent region may have

$$
u_R(x)
$$

small near one part of the interval and large near another part.

A split near the transition can produce children with more distinct
uncertainty profiles.

This can make subsequent sampling decisions more targeted.

### Uncertainty Profile Rather Than Only a Scalar

For a child region, ARRGO should retain:

$$
U_R(x),
\qquad
L_R(x),
\qquad
u_R(x).
$$

The scalar

$$
u_{\max}(R)
$$

is useful as a summary for comparison, but it should not replace the underlying
functions.

This preserves information about both the magnitude and spatial location of
the remaining uncertainty.

### Split Candidate Comparison

For candidate split locations

$$
s_1^c,s_2^c,\ldots,s_m^c,
$$

each candidate can be associated with an uncertainty-difference profile.

A candidate may produce:

$$
\left(
u_{\max}(R_L),
u_{\max}(R_R)
\right).
$$

Another candidate may produce a different pair.

These candidates should not automatically be ranked using an arbitrary weighted
sum.

Instead, uncertainty-related criteria are compared according to the currently
dominant unresolved objectives.

### Uncertainty and Optimization Potential

Certified uncertainty and optimization potential are related but distinct.

Uncertainty asks:

$$
\text{How wide is the currently certified enclosure?}
$$

Potential asks:

$$
\text{How good could the region still be according to its certified upper
bound?}
$$

A region can have relatively low uncertainty while still having high
optimization potential.

Conversely, a region can have high uncertainty but a potential already below
the incumbent.

Therefore ARRGO must not replace one criterion with the other.

### Competitive Regions

In Certified Mode, a region is optimization-competitive when

$$
P_R
>
f_{\mathrm{best}}+\epsilon.
$$

If such a region also has large certified uncertainty, additional information
may be particularly important because the current certificate cannot yet rule
out that region.

The combination

$$
P_R>f_{\mathrm{best}}+\epsilon
$$

and

$$
u_{\max}(R)\text{ large}
$$

therefore identifies a region whose optimization status remains both relevant
and incompletely resolved.

This does not imply that the region contains the global optimum.

It only means that the current certified information has not ruled out that
possibility.

### Uncertainty Difference and Structural Refinement

A candidate split is structurally informative when it creates child regions
whose certified information can be analyzed more distinctly.

For example,

$$
u_{\max}(R_L)
\gg
u_{\max}(R_R)
$$

reveals an information imbalance.

The less-resolved child can then become a natural target for future sampling.

The split therefore provides a structural mechanism for localizing where
certified information remains weak.

### Validity Requirement

All uncertainty calculations in this section depend on the validity of the
Lipschitz constant.

If the supplied value $L$ is not a valid global upper bound, then

$$
L_R(x)
$$

and

$$
U_R(x)
$$

cannot be treated as rigorous certified bounds.

An estimated Lipschitz constant obtained only from observed data is therefore
not sufficient for Certified Mode unless its validity is independently
established.

### No Probabilistic Interpretation

ARRGO does not interpret

$$
u_R(x)
$$

as the probability that the function lies near the center of the envelope.

There is no probability distribution involved.

The correct interpretation is deterministic:

$$
\boxed{
f(x)\in
[L_R(x),U_R(x)]
}
$$

under the stated Lipschitz assumption.

### Structural Uncertainty Profile

The uncertainty-related structural information for a candidate split can be
represented as

$$
D_{\mathrm{uncertainty}}
=
\left(
u_{R_L}(x),
u_{R_R}(x),
u_{\max}(R_L),
u_{\max}(R_R)
\right).
$$

This profile describes how certified uncertainty is distributed across the two
children.

### Role in Split Evaluation

The uncertainty criterion answers:

$$
\boxed{
\text{Does this split separate regions with meaningfully different levels or
patterns of certified uncertainty?}
}
$$

If so, the split may improve the structural organization of information and
help ARRGO identify where additional certified refinement is required.

### Principle

ARRGO follows the certified-uncertainty principle:

$$
\boxed{
\text{Use Valid Lipschitz Envelopes to Identify Spatial Differences in
Certified Uncertainty, Without Interpreting Enclosure Width as Probabilistic
Uncertainty.}
}
$$

The next criterion examines how the certified optimization potential differs
between the two child regions.

## Optimization Potential Difference Across a Split

Structural refinement is not useful only because it creates smaller regions.

A split is particularly relevant to global optimization when it separates child
regions according to their remaining certified ability to contain a better
objective value.

In Certified Mode, this information is represented by the regional
optimization potential.

### Regional Optimization Potential

For a region

$$
R=[l,r],
$$

with a valid certified upper envelope

$$
U_R(x),
$$

the regional optimization potential is

$$
P_R
=
\max_{x\in R}U_R(x).
$$

Because the upper envelope is valid,

$$
f(x)\leq U_R(x)
$$

for every

$$
x\in R.
$$

Therefore,

$$
\max_{x\in R}f(x)
\leq
P_R.
$$

The potential is consequently an upper bound on the best objective value that
may still be achievable inside the region.

### Candidate Split

Consider a candidate split

$$
s^c\in(l,r)
$$

with children

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

The two children have their own certified upper envelopes and corresponding
potentials:

$$
P_{R_L}
=
\max_{x\in R_L}U_{R_L}(x),
$$

and

$$
P_{R_R}
=
\max_{x\in R_R}U_{R_R}(x).
$$

These quantities describe the remaining certified optimization potential on
each side of the split.

### Potential Difference Between Children

A basic structural comparison is

$$
D_{\mathrm{potential}}
=
\left|
P_{R_L}
-
P_{R_R}
\right|.
$$

A large value indicates that the two child regions have substantially
different certified upper potentials.

For example,

$$
P_{R_L}
\gg
P_{R_R}
$$

means that the left child currently has a substantially larger remaining
certified potential.

This can make the two children behave differently from the perspective of
global optimization.

### Potential Is Not a Prediction

The value

$$
P_R
$$

must not be interpreted as a prediction of the objective value.

It represents the largest value allowed by the current certified upper
envelope.

Thus,

$$
P_R
\neq
\text{Predicted Maximum}.
$$

Instead,

$$
\boxed{
P_R
=
\text{Certified Upper Potential Under the Current Information}.
}
$$

The actual function maximum inside the region may be substantially smaller.

### Competitive Potential

Given the current incumbent

$$
f_{\mathrm{best}},
$$

a region is potentially competitive when

$$
P_R
>
f_{\mathrm{best}}+\epsilon.
$$

This means that the current certified information cannot yet rule out an
improvement of more than $\epsilon$ over the incumbent.

If instead

$$
P_R
\leq
f_{\mathrm{best}}+\epsilon,
$$

then the region is not capable of producing an objective value that would
improve the current certified result by more than $\epsilon$.

This is a certificate-based statement, not a heuristic judgment.

### Potential Difference and Global Search

Suppose a split produces

$$
P_{R_L}
>
P_{R_R}.
$$

The left child has greater remaining certified potential.

This does not prove that the optimum lies in the left child.

It only means that the current information provides a larger upper bound there.

Therefore potential difference is useful for prioritization, but must not be
confused with a localization proof.

### Parent-Child Consistency

Because the child regions cover the parent,

$$
R_L\cup R_R=R,
$$

the global maximum over the parent satisfies

$$
\max_{x\in R}f(x)
=
\max
\left\{
\max_{x\in R_L}f(x),
\max_{x\in R_R}f(x)
\right\}.
$$

Since each child has a valid upper bound,

$$
\max_{x\in R_L}f(x)
\leq
P_{R_L},
$$

and

$$
\max_{x\in R_R}f(x)
\leq
P_{R_R}.
$$

Therefore,

$$
\max_{x\in R}f(x)
\leq
\max
\left\{
P_{R_L},
P_{R_R}
\right\}.
$$

This preserves certified consistency after structural refinement.

### Parent Region Persistence

ARRGO does not delete the parent region after splitting.

The parent remains part of the persistent region hierarchy.

However, its children provide a more detailed spatial representation of the
same domain.

Thus the split changes the structural resolution without destroying the
historical representation.

The hierarchy maintains:

$$
R_{\mathrm{parent}}
\supseteq
R_L,R_R.
$$

### Potential and Uncertainty Are Different

Optimization potential and certified uncertainty should not be conflated.

Potential asks:

$$
\text{How good could the region still be?}
$$

Uncertainty asks:

$$
\text{How wide is the current certified enclosure?}
$$

A region may have

$$
P_R
>
f_{\mathrm{best}}+\epsilon
$$

while also having relatively small uncertainty.

In this situation, the region remains competitive even though its current
certified enclosure may already be relatively tight.

Conversely, a region may have large uncertainty but

$$
P_R
\leq
f_{\mathrm{best}}+\epsilon.
$$

In that case, its optimization relevance has already been ruled out to the
specified tolerance.

Therefore both criteria must remain available.

### Potential Difference Without New Evaluations

A structural split can change how existing information is organized.

Consequently, child potentials can be computed from inherited information even
without evaluating the objective at the split point.

However, this does not mean that the split has discovered a new function value.

The information remains derived from existing observations.

Thus:

$$
\boxed{
\text{Potential Reorganization}
\neq
\text{New Objective Information}.
}
$$

### Effect of Sampling

When a new point

$$
x^c
$$

is evaluated inside a region, its observed value

$$
f(x^c)
$$

can tighten the certified upper envelope:

$$
U_R^{\mathrm{new}}(x)
=
\min
\left\{
U_R(x),
f(x^c)+L|x-x^c|
\right\}.
$$

Therefore,

$$
U_R^{\mathrm{new}}(x)
\leq
U_R(x).
$$

Consequently,

$$
P_R^{\mathrm{new}}
\leq
P_R.
$$

This is one of the key mechanisms through which actual information acquisition
can reduce certified optimization potential.

### Potential Difference as a Structural Criterion

A candidate split can therefore be evaluated according to the potential
structure it produces:

$$
D_{\mathrm{potential}}
=
\left(
P_{R_L},
P_{R_R},
|P_{R_L}-P_{R_R}|
\right).
$$

This profile preserves both the absolute potential of each child and their
difference.

The absolute values are important because a large difference between two very
low-potential regions may be less relevant than a smaller difference involving
a highly competitive region.

Therefore the difference should not be interpreted independently of the
absolute potential levels.

### Competitive-Region Structure

A particularly important situation occurs when

$$
P_{R_L}
>
f_{\mathrm{best}}+\epsilon
$$

while

$$
P_{R_R}
\leq
f_{\mathrm{best}}+\epsilon.
$$

The split then separates a potentially competitive child from a child that is
already certified as non-competitive at the current tolerance.

This can significantly improve the organization of the global search state.

The interpretation is still based on certified upper bounds.

### Both Children Competitive

It is also possible that

$$
P_{R_L}
>
f_{\mathrm{best}}+\epsilon
$$

and

$$
P_{R_R}
>
f_{\mathrm{best}}+\epsilon.
$$

In this case both children remain competitive.

The split has increased structural resolution, but it has not yet resolved
which side can be ruled out.

ARRGO therefore retains both children.

This is consistent with the no-pruning principle.

### Neither Child Competitive

If

$$
P_{R_L}
\leq
f_{\mathrm{best}}+\epsilon
$$

and

$$
P_{R_R}
\leq
f_{\mathrm{best}}+\epsilon,
$$

then both children are certified non-competitive at the current tolerance.

The parent and children remain represented in the hierarchy, but none of them
requires competitive refinement under this criterion.

### Comparison Between Split Candidates

For multiple candidate split locations

$$
s_1^c,s_2^c,\ldots,s_m^c,
$$

each candidate produces a corresponding potential profile.

A candidate may provide stronger separation between competitive and
non-competitive regions.

Another candidate may preserve two competitive regions but create a more
balanced structural decomposition.

There is no universally correct scalar ranking between these alternatives.

The appropriate comparison depends on the current unresolved optimization
objective.

### No Arbitrary Weighting

Potential difference must not be combined with unrelated criteria using
unsupported weights such as

$$
\alpha D_{\mathrm{potential}}
+
\beta D_{\mathrm{uncertainty}}
+
\gamma D_{\mathrm{behavior}}.
$$

Such a scalarization would impose a preference that has not been mathematically
justified.

ARRGO therefore retains:

$$
D_{\mathrm{potential}},
\qquad
D_{\mathrm{uncertainty}},
\qquad
D_{\mathrm{behavior}},
\qquad
D_{\mathrm{spatial}}
$$

as separate components of the structural information state.

### Relationship With Global Potential

The global certified potential is

$$
P_{\mathrm{global}}
=
\max_R P_R.
$$

Therefore the child with the larger potential may become the dominant
contributor to the current global certificate.

However, the global potential considers all represented regions.

A local split must therefore be interpreted within the global hierarchy rather
than in isolation.

### Role in Global Region Selection

Potential information becomes especially important when ARRGO later selects
which region should receive refinement.

A competitive region with large potential remains relevant to the global
optimization problem.

A region whose potential is already below the incumbent tolerance has no
remaining certified ability to improve the result beyond that tolerance.

This provides a principled connection between local structural refinement and
global search.

### Principle

ARRGO follows the optimization-potential principle:

$$
\boxed{
\text{Use Certified Upper Potential to Distinguish the Remaining Optimization
Relevance of Child Regions, Without Treating Potential as a Prediction of the
Unknown Objective.}
}
$$

The next step combines coverage, behavior, uncertainty, and optimization
potential into the complete structural difference profile of a candidate split.

## Structural Difference Profile

A candidate split is not characterized by a single type of information.

A useful structural refinement may separate child regions according to several
different dimensions of the currently available information.

ARRGO therefore represents the structural effect of a candidate split through a
multi-dimensional structural difference profile.

### Candidate Split

Let

$$
R=[l,r]
$$

be the current region and let

$$
s^c\in(l,r)
$$

be a candidate split point.

The resulting child regions are

$$
R_L=[l,s^c]
$$

and

$$
R_R=[s^c,r].
$$

The candidate split creates a structural comparison between these two children.

### Main Structural Information Dimensions

The structural profile contains four principal information dimensions:

$$
\boxed{
D_{\mathrm{split}}
=
\left(
D_{\mathrm{spatial}},
D_{\mathrm{behavior}},
D_{\mathrm{uncertainty}},
D_{\mathrm{potential}}
\right).
}
$$

These dimensions correspond to:

- spatial coverage and sampling density;
- observed directional and slope-variation behavior;
- certified uncertainty;
- certified optimization potential.

The last two components are available only in Certified Mode.

### Spatial Difference

The spatial component describes how the current sampling information is
distributed across the two children.

It includes the coverage profiles

$$
G(R_L)
\qquad\text{and}\qquad
G(R_R),
$$

as well as descriptive sampling densities

$$
\rho_{\mathrm{sample}}(R_L)
\qquad\text{and}\qquad
\rho_{\mathrm{sample}}(R_R).
$$

Useful summaries include

$$
D_{\mathrm{coverage}}
=
\left|
G_{\max}(R_L)-G_{\max}(R_R)
\right|,
$$

and

$$
D_{\mathrm{density}}
=
\left|
\rho_{\mathrm{sample}}(R_L)
-
\rho_{\mathrm{sample}}(R_R)
\right|.
$$

These quantities describe differences in the current information
distribution.

They do not describe differences in the unknown objective function itself.

### Behavioral Difference

The behavioral component describes differences in the observed function
behavior on the two sides.

It includes directional information such as

$$
D_L
\qquad\text{and}\qquad
D_R,
$$

where each represents the observed direction of change.

It also includes slope-variation information such as

$$
\Delta s_i=s_{i+1}-s_i.
$$

Therefore the behavioral profile can be represented as

$$
D_{\mathrm{behavior}}
=
\left(
D_{\mathrm{direction}},
D_{\Delta s}
\right).
$$

This preserves the distinction between:

- observed direction;
- observed slope magnitude;
- change in observed slope;
- directional transitions.

### Behavioral Evidence Is Not a Proof

A strong behavioral difference may indicate that the candidate split separates
different local patterns.

For example,

$$
s_L>0
\qquad\text{and}\qquad
s_R<0
$$

provides evidence of an observed increasing-to-decreasing transition.

However, it does not prove that an exact local maximum exists at the split.

Similarly, a large value of

$$
|\Delta s_i|
$$

indicates a strong observed change in secant slope, but it is not automatically
an exact second derivative.

Therefore:

$$
\boxed{
\text{Observed Behavioral Difference}
\neq
\text{Guaranteed Function Structure}.
}
$$

### Certified Uncertainty Difference

In Certified Mode, each child has its own certified uncertainty profile:

$$
u_{R_L}(x)
$$

and

$$
u_{R_R}(x).
$$

Their worst-case uncertainties are

$$
u_{\max}(R_L)
=
\max_{x\in R_L}u_{R_L}(x),
$$

and

$$
u_{\max}(R_R)
=
\max_{x\in R_R}u_{R_R}(x).
$$

A simple comparison is

$$
D_{\mathrm{uncertainty}}
=
\left|
u_{\max}(R_L)
-
u_{\max}(R_R)
\right|.
$$

The complete uncertainty profiles should remain available because the maximum
alone does not indicate where the uncertainty is concentrated.

### Certified Potential Difference

Each child also has a certified optimization potential:

$$
P_{R_L}
=
\max_{x\in R_L}U_{R_L}(x),
$$

and

$$
P_{R_R}
=
\max_{x\in R_R}U_{R_R}(x).
$$

Their difference can be represented as

$$
D_{\mathrm{potential}}
=
\left|
P_{R_L}
-
P_{R_R}
\right|.
$$

The absolute potential levels must also be retained because the difference
alone does not indicate whether either child is globally competitive.

### Competitive Structure

Using the incumbent value

$$
f_{\mathrm{best}},
$$

a child is competitive at tolerance $\epsilon$ when

$$
P_R
>
f_{\mathrm{best}}+\epsilon.
$$

Therefore the structural profile should also record whether each child is:

- competitive;
- non-competitive;
- unresolved because the required certified information is unavailable.

This provides a discrete structural property in addition to the numerical
potential values.

### Complete Candidate Profile

For a candidate split $s^c$, ARRGO can therefore maintain the profile

$$
\boxed{
\begin{aligned}
D_{\mathrm{split}}(s^c)
=
\big(
&
D_{\mathrm{spatial}},
D_{\mathrm{behavior}},
\\
&
D_{\mathrm{uncertainty}},
D_{\mathrm{potential}}
\big).
\end{aligned}
}
$$

The profile should be interpreted as a structured collection of evidence
rather than as a single score.

### Why a Vector Is Preferable to a Scalar

The components of

$$
D_{\mathrm{split}}
$$

have different meanings and generally different numerical scales.

For example, spatial gaps may be measured in units of $x$, while slope
variation depends on the scale of $f$ relative to $x$.

Certified uncertainty and potential are measured in objective-value units.

Therefore directly adding these quantities would generally have no principled
meaning.

ARRGO consequently does not define a universal scalar expression such as

$$
\alpha D_{\mathrm{spatial}}
+
\beta D_{\mathrm{behavior}}
+
\gamma D_{\mathrm{uncertainty}}
+
\delta D_{\mathrm{potential}}.
$$

No arbitrary weights are introduced.

### Candidate Comparison

Suppose there are several candidate split locations:

$$
C_{\mathrm{split}}
=
\left\{
s_1^c,s_2^c,\ldots,s_m^c
\right\}.
$$

Each candidate produces its own structural profile:

$$
D_{\mathrm{split}}(s_j^c).
$$

The candidates are compared according to the objectives that are currently
unresolved in the parent region.

Thus a candidate is not considered universally superior simply because it has
a larger value in one structural component.

### Pareto-Style Non-Dominance

When multiple structural criteria are simultaneously relevant, ARRGO uses
non-dominance rather than arbitrary scalarization.

A candidate

$$
s_i^c
$$

dominates another candidate

$$
s_j^c
$$

only when it is at least as favorable under every currently active criterion
and strictly more favorable under at least one.

Conceptually,

$$
s_i^c\succ s_j^c
$$

means that the available structural evidence provides no active criterion under
which $s_j^c$ is better.

The exact direction of each comparison depends on whether the criterion
represents a quantity to maximize or minimize.

### Context-Dependent Direction of Preference

Not every structural difference should be maximized.

For example:

- larger coverage improvement is generally preferable;
- smaller unresolved uncertainty is preferable;
- larger optimization potential identifies greater remaining relevance;
- stronger behavioral separation may be useful when behavioral resolution is
  dominant.

Therefore ARRGO must define the optimization direction of each criterion
before applying dominance.

The numerical value itself is not sufficient to determine preference.

### Dominant Unresolved Objective

The parent region already has an unresolved-information profile

$$
\mathcal{O}_R^*.
$$

Only criteria relevant to

$$
\mathcal{O}_R^*
$$

should influence the current structural comparison.

For example, if behavioral resolution is dominant, behavioral differences receive
primary consideration.

If certified potential is unresolved, potential-related criteria become
important.

This keeps structural selection consistent with the earlier region-analysis
stage.

### Structural Difference Does Not Equal Refinement Benefit

A split may produce a large difference between its children without necessarily
providing the best next action.

For example, a candidate may create two highly different children, but both may
already have sufficient information for the current optimization objective.

Therefore

$$
\boxed{
\text{Structural Difference}
\neq
\text{Automatically Optimal Refinement}.
}
$$

The structural profile is an input to the refinement decision, not the final
decision itself.

### Relationship With Sampling

Splitting and sampling remain distinct operations.

A split:

$$
R\rightarrow R_L\cup R_R
$$

creates structural resolution.

Sampling:

$$
x^c\rightarrow f(x^c)
$$

creates new objective information.

A candidate split may therefore reveal where information differs without
actually acquiring a new function value.

This distinction is preserved throughout the algorithm.

### Relationship With the Global State

The structural profile is computed locally for a candidate split, but its
interpretation depends on the global state.

In particular, certified potential must be compared with

$$
f_{\mathrm{best}},
$$

and region relevance depends on the current global optimization state.

Therefore a split candidate cannot be evaluated completely independently of the
global incumbent and other represented regions.

### No-Pruning Principle

After a split,

$$
R\rightarrow\{R_L,R_R\},
$$

the parent remains in the persistent region hierarchy.

Neither child is deleted merely because it currently has lower potential,
lower behavioral relevance, or weaker information value.

The structural profile supports refinement decisions but does not authorize
destructive removal of regions.

### Information Persistence

All observations remain part of the global history:

$$
D_t
\subseteq
D_{t+1}.
$$

Child regions inherit the relevant observations from their parent.

Therefore the structural profile can be recomputed whenever new information
becomes available.

This is important because a split that is weakly informative at one point in
the search may become informative after additional sampling.

### Dynamic Nature of the Profile

The structural profile is therefore time-dependent:

$$
D_{\mathrm{split}}^{(t)}(s^c).
$$

After a new evaluation,

$$
D_{t+1}
=
D_t
\cup
\{(x_{\mathrm{new}},f(x_{\mathrm{new}}))\},
$$

the behavioral, spatial, uncertainty, and potential components may all change.

Consequently, ARRGO does not treat a candidate's structural value as a
permanent property.

It is a property of the candidate under the current information state.

### Structural Refinement Principle

The complete structural profile provides the following decision perspective:

$$
\boxed{
\text{Where does a candidate split create the most useful separation of
currently unresolved optimization information?}
}
$$

This question combines the spatial, behavioral, certified-uncertainty, and
certified-potential views without collapsing them into an unsupported scalar.

### Principle

ARRGO follows the structural-profile principle:

$$
\boxed{
\text{Represent the Effect of a Split as a Multi-Dimensional Information
Profile, and Compare Candidate Splits According to the Currently Relevant
Objectives Rather Than an Arbitrary Global Score.}
}
$$

The next section examines how candidate split locations can be compared using
dominance when several structural objectives are simultaneously active.

## Dominance Between Candidate Splits

When several candidate split locations are available, ARRGO must compare them
without introducing arbitrary weights between unrelated structural objectives.

The structural difference profile provides a multi-dimensional representation
for this comparison.

### Candidate Set

For a region

$$
R=[l,r],
$$

let the candidate split set be

$$
C_{\mathrm{split}}(R)
=
\left\{
s_1^c,s_2^c,\ldots,s_m^c
\right\}.
$$

Each candidate produces a structural profile

$$
D_{\mathrm{split}}(s_j^c).
$$

The purpose of dominance analysis is to eliminate candidates that are
demonstrably inferior under all currently relevant structural criteria.

### Active Structural Objectives

The comparison must first identify the currently relevant objective set

$$
\mathcal{O}_R^*.
$$

Possible active objectives include:

$$
\mathrm{coverage},
\qquad
\mathrm{behavior},
\qquad
\mathrm{uncertainty},
\qquad
\mathrm{potential}.
$$

Not every objective is active at every stage.

This prevents ARRGO from treating every available measurement as equally
important regardless of the current information state.

### Criterion Orientation

Different structural quantities have different preferred directions.

For example:

- greater reduction of an unresolved spatial gap is preferable;
- smaller remaining certified uncertainty is preferable;
- stronger separation of currently unresolved behavior may be preferable;
- larger certified potential indicates greater optimization relevance.

Therefore each criterion must have a clearly defined interpretation before it
participates in dominance.

The numerical magnitude alone does not determine whether larger or smaller is
better.

### Candidate Comparison

Consider two candidate split locations

$$
s_i^c
\qquad\text{and}\qquad
s_j^c.
$$

Let their active structural criteria be represented by

$$
V_k(s_i^c)
$$

and

$$
V_k(s_j^c),
$$

where $k$ indexes the currently active objectives.

Candidate $s_i^c$ dominates $s_j^c$ when:

1. $s_i^c$ is at least as favorable as $s_j^c$ under every active criterion;
2. $s_i^c$ is strictly more favorable under at least one active criterion.

Conceptually,

$$
s_i^c\succ s_j^c
$$

when

$$
V_k(s_i^c)
\succeq
V_k(s_j^c)
$$

for every active criterion $k$, and

$$
V_q(s_i^c)
\succ
V_q(s_j^c)
$$

for at least one active criterion $q$.

### Interpretation of Dominance

If

$$
s_i^c\succ s_j^c,
$$

then the currently available structural information provides no active
objective under which $s_j^c$ is better.

Therefore $s_j^c$ can be removed from the preferred candidate set for the
current decision.

This is not pruning of a region.

It is only elimination of a candidate split that is demonstrably inferior under
the current comparison.

### Candidate Non-Dominance

A candidate is non-dominated when no other candidate dominates it.

The non-dominated set is

$$
C_{\mathrm{ND}}
=
\left\{
s^c\in C_{\mathrm{split}}(R)
:
\nexists\,z^c\in C_{\mathrm{split}}(R)
\text{ such that }
z^c\succ s^c
\right\}.
$$

This set contains the candidates that remain legitimate alternatives under the
current multi-objective information state.

### Why Non-Dominance Is Important

Suppose one candidate provides better behavioral separation while another
provides better coverage resolution.

Neither candidate may dominate the other.

In that situation, choosing one using an arbitrary weighted score would impose
an unsupported preference.

ARRGO therefore retains both candidates as non-dominated alternatives.

### Example of Non-Dominance

Suppose two candidates have the following qualitative properties:

$$
s_1^c:
\quad
\text{strong behavior separation},
\quad
\text{moderate coverage improvement},
$$

and

$$
s_2^c:
\quad
\text{moderate behavior separation},
\quad
\text{strong coverage improvement}.
$$

Neither is uniformly better.

Therefore,

$$
s_1^c\nsucc s_2^c
$$

and

$$
s_2^c\nsucc s_1^c.
$$

Both remain in

$$
C_{\mathrm{ND}}.
$$

### Dominance Is Context-Dependent

Dominance depends on the active objective set

$$
\mathcal{O}_R^*.
$$

A candidate that is non-dominated when behavior and coverage are both active
may become dominated when only coverage remains unresolved.

Thus:

$$
\boxed{
\text{Dominance Is a Property of the Current Information State.}
}
$$

It is not a permanent property of a split location.

### Certified Mode

In Certified Mode, uncertainty and potential can participate in dominance.

For example, consider two candidates with child potential profiles

$$
(P_{L}^{(1)},P_{R}^{(1)})
$$

and

$$
(P_{L}^{(2)},P_{R}^{(2)}).
$$

Their comparison must respect the current certified optimization objective.

If one candidate creates a child structure that is at least as informative for
the active certified criteria and strictly better for one criterion, it can
dominate the alternative.

### Potential Requires Careful Interpretation

A larger potential is not automatically better in every context.

A high potential means that a region remains capable, according to the current
certified bound, of containing a high objective value.

If the current objective is to reduce the remaining global certificate, then
the useful refinement may instead be the one that more effectively resolves or
reduces that potential.

Therefore ARRGO must distinguish between:

$$
\text{optimization relevance}
$$

and

$$
\text{potential reduction}.
$$

The active decision objective determines which interpretation applies.

### Uncertainty Requires Careful Interpretation

Similarly, smaller certified uncertainty is generally preferable when the
objective is information resolution.

However, a candidate that produces very different uncertainty levels between
its children may be structurally informative even though the larger uncertainty
itself is not desirable.

Therefore ARRGO distinguishes:

$$
\text{uncertainty magnitude}
$$

from

$$
\text{uncertainty separation}.
$$

This distinction prevents the comparison from confusing “more uncertainty” with
“more useful structure.”

### Structural Difference Versus Refinement Value

Dominance operates on the structural information profile.

It does not directly prove that a candidate will produce the largest future
objective improvement.

Therefore:

$$
\boxed{
\text{Dominance of Structural Profiles}
\neq
\text{Guaranteed Optimization Improvement}.
}
$$

The comparison only establishes that one candidate is no worse under all
currently active structural criteria and better under at least one.

### No Arbitrary Weighted Score

ARRGO explicitly avoids a universal scalar such as

$$
S(s^c)
=
\sum_k w_kV_k(s^c),
$$

when the weights

$$
w_k
$$

have no mathematical justification.

Different objectives can have different units, scales, and meanings.

Dominance avoids requiring an arbitrary conversion into a single numerical
preference.

### Deterministic Candidate Set

The candidate generation process itself remains deterministic.

After generation and validation,

$$
C_{\mathrm{split}}(R)
$$

is fixed for the current information state.

Dominance is then applied deterministically to obtain

$$
C_{\mathrm{ND}}.
$$

If multiple candidates remain non-dominated, a later deterministic selection
stage is required.

### Relation to Tie-Breaking

Dominance does not necessarily produce a unique candidate.

It is possible that

$$
|C_{\mathrm{ND}}|>1.
$$

In this case, ARRGO must not falsely claim that one candidate is mathematically
superior.

Instead, the remaining candidates represent genuinely different trade-offs
under the available information.

A deterministic tie-breaking rule may then select one candidate for execution,
but the tie-breaker is only a reproducibility mechanism.

### No-Pruning of Regions

Candidate dominance must not be confused with region pruning.

If a candidate split is dominated, only that candidate action is discarded.

The underlying region remains fully represented in the global hierarchy.

Likewise, a child region with lower potential remains represented even if it is
not selected for immediate refinement.

Thus:

$$
\boxed{
\text{Candidate Elimination}
\neq
\text{Region Elimination}.
}
$$

### Re-Evaluation After New Information

Because the structural profile depends on the current observations, dominance
must be recomputed whenever relevant information changes.

After a new evaluation,

$$
D_{t+1}
=
D_t
\cup
\{(x_{\mathrm{new}},f(x_{\mathrm{new}}))\},
$$

the candidate profiles may change.

Consequently, a candidate that was previously dominated may later become
non-dominated, and vice versa.

ARRGO therefore does not permanently discard a structural possibility merely
because it was dominated under an earlier information state.

### Global Context

Although dominance is evaluated locally within a region, its active objectives
can depend on global information.

For example, certified potential is interpreted relative to

$$
f_{\mathrm{best}}
$$

and the global certified state.

Therefore local candidate comparison must remain consistent with the global
optimization state.

### Dominance Principle

The dominance mechanism follows:

$$
\boxed{
\text{Discard Only Candidates That Are Demonstrably Inferior Under Every
Currently Active Structural Objective.}
}
$$

All non-dominated alternatives remain available for deterministic selection.

### Summary

For each region, ARRGO performs:

$$
\boxed{
C_{\mathrm{split}}
\rightarrow
\text{Validate}
\rightarrow
\text{Evaluate Structural Profiles}
\rightarrow
\text{Apply Dominance}
\rightarrow
C_{\mathrm{ND}}.
}
$$

This produces a principled set of candidate split locations without requiring
arbitrary weights.

The next section defines how ARRGO selects one split from the remaining
non-dominated candidates.

## Final Selection Among Non-Dominated Splits

After dominance analysis, ARRGO may still have multiple non-dominated split
candidates.

The purpose of this stage is to select exactly one candidate for the next
structural refinement operation.

This selection must remain deterministic while preserving the distinction
between mathematical evidence and implementation-level tie-breaking.

### Input

The selection procedure receives:

- the current region $R$;
- the non-dominated split set $C_{\mathrm{ND}}$;
- the active unresolved objective set $\mathcal{O}_R^*$;
- the structural profiles of the candidates;
- the current global optimization state;
- the numerical tolerances;
- the certified state when Certified Mode is active.

The selection procedure does not evaluate the unknown objective function at an
unevaluated split point.

### First Selection Principle

The selected split should address the currently dominant unresolved structural
objective.

Therefore ARRGO does not treat all possible structural properties as equally
important at every stage.

The active objective set

$$
\mathcal{O}_R^*
$$

determines which information dimensions are relevant to the current decision.

### Single Active Objective

If exactly one objective is dominant,

$$
\mathcal{O}_R^*
=
\{O\},
$$

the selection can focus directly on the corresponding structural criterion.

For example, if

$$
O=\mathrm{coverage},
$$

candidate splits are compared according to their ability to improve spatial
resolution.

If

$$
O=\mathrm{behavior},
$$

the comparison focuses on observed behavioral separation.

In Certified Mode, if

$$
O=\mathrm{uncertainty},
$$

the comparison focuses on the certified uncertainty structure.

Similarly, when certified optimization relevance is dominant, the potential
structure becomes central to the comparison.

### Multiple Active Objectives

If

$$
|\mathcal{O}_R^*|>1,
$$

several structural objectives remain simultaneously unresolved.

In this situation ARRGO does not collapse the objectives into an arbitrary
weighted scalar.

Instead, the non-dominated candidate set is retained as the mathematically
supported set of alternatives.

### Objective Support

For a candidate split $s^c$, define its active objective-support set as

$$
S(s^c)
\subseteq
\mathcal{O}_R^*.
$$

This identifies which currently unresolved objectives are meaningfully
addressed by the candidate.

A candidate satisfying

$$
S(s_i^c)
\supset
S(s_j^c)
$$

has broader structural support across the active unresolved objectives.

This relation may be useful when the available candidates differ in the number
of unresolved objectives they address.

### Multi-Objective Preference

If one candidate is demonstrably at least as useful across all active
objectives and better on at least one, it would already have been dominated.

Therefore, after the dominance stage, the remaining candidates represent
different trade-offs or equivalent structural alternatives.

This is why ARRGO does not claim that one non-dominated candidate is universally
superior.

### Deterministic Selection Requirement

Despite the absence of a universal mathematical ranking, the algorithm must
still select one candidate to continue execution.

ARRGO therefore uses a deterministic selection hierarchy.

The hierarchy is applied only after dominance has been established.

Conceptually:

$$
\boxed{
\text{Dominance}
\rightarrow
\text{Objective Relevance}
\rightarrow
\text{Structural Tie-Breaking}
}
$$

The first stages preserve mathematical information.

The final stage provides reproducibility when the available information does
not establish a unique preference.

### Structural Tie-Breaking

When multiple candidates remain equivalent with respect to the active
objectives, ARRGO may use intrinsic geometric properties.

Possible deterministic properties include:

1. distance to the most relevant unresolved spatial location;
2. alignment with a detected behavioral transition;
3. location within the largest unresolved coverage gap;
4. a fixed numerical ordering of the split coordinate.

These properties must only be used when they have an explicitly defined role in
the current structural decision.

They must not be presented as universal evidence of optimization superiority.

### Largest-Gap Tie-Breaking

When spatial coverage is the active unresolved objective, a natural geometric
criterion is the size of the unresolved gap associated with the candidate.

Suppose candidate $s^c$ lies inside an interval

$$
[x_i,x_{i+1}].
$$

Its associated gap is

$$
g_i=x_{i+1}-x_i.
$$

A candidate associated with a larger unresolved gap provides a direct response
to the coverage objective.

Thus, when coverage is dominant, candidates may be ordered according to the
coverage deficiency they address.

### Behavioral-Transition Tie-Breaking

When behavioral resolution is dominant, a candidate located near a strong
observed behavioral transition may receive priority.

Relevant evidence may include:

$$
s_i s_{i+1}<0
$$

or a substantial observed slope variation

$$
|\Delta s_i|.
$$

The candidate is then connected directly to an observed transition in the
current information state.

This remains observational evidence rather than a proof of an underlying
extremum.

### Certified Tie-Breaking

In Certified Mode, the selection hierarchy must preserve certified relevance.

If a candidate is associated with a child region satisfying

$$
P_R
>
f_{\mathrm{best}}+\epsilon,
$$

the candidate remains relevant to the certified optimization state.

Certified information must not be overridden by an empirical heuristic that
claims the region is unimportant.

### Potential Reduction Versus Potential Magnitude

When certified potential is involved, ARRGO distinguishes between two different
questions.

The first is:

$$
\text{Which child has larger remaining potential?}
$$

The second is:

$$
\text{Which split provides a stronger opportunity to resolve the current
potential structure?}
$$

These are not identical.

A candidate producing a high-potential child is not automatically the best split
if another candidate provides more useful structural resolution of the
remaining certified uncertainty.

Therefore the active refinement objective must determine which quantity is
being compared.

### No Arbitrary Weighted Score

ARRGO explicitly avoids expressions such as

$$
S(s^c)
=
w_1D_{\mathrm{coverage}}
+
w_2D_{\mathrm{behavior}}
+
w_3D_{\mathrm{uncertainty}}
+
w_4D_{\mathrm{potential}}.
$$

Unless these weights are derived from a justified mathematical model, such a
score would introduce an unsupported preference.

The final selection therefore remains objective-consistent rather than
globally weighted.

### Deterministic Coordinate Ordering

If all meaningful structural criteria remain tied, ARRGO can use a fixed
coordinate ordering.

For example,

$$
s_i^c<s_j^c
\quad\Rightarrow\quad
s_i^c
\prec
s_j^c.
$$

The smallest valid coordinate may then be selected.

This rule has no claim of optimization superiority.

Its only purpose is to guarantee deterministic execution.

### Why Determinism Matters

A deterministic selection rule ensures that identical inputs and identical
information states produce the same refinement decision.

Therefore repeated executions can be reproduced without relying on random
selection.

This is particularly important for experimental evaluation because changes in
performance can then be attributed to algorithmic changes rather than random
candidate selection.

### Selection Does Not Create New Information

Selecting a split point does not evaluate the objective there.

The selected point only determines the structural partition:

$$
R
\rightarrow
\{R_L,R_R\}.
$$

The objective information available immediately after the split is therefore
still derived from the existing evaluation history.

New function information requires later sampling.

### Split Execution

Once

$$
s_{\mathrm{selected}}
$$

has been determined, ARRGO creates

$$
R_L=[l,s_{\mathrm{selected}}],
$$

and

$$
R_R=[s_{\mathrm{selected}},r].
$$

The contraction condition must hold:

$$
\max
\left\{
s_{\mathrm{selected}}-l,
r-s_{\mathrm{selected}}
\right\}
\leq
\rho(r-l),
$$

where

$$
0<\rho<1.
$$

If the candidate does not satisfy the required contraction condition, it is
invalid and cannot be executed.

### Persistent Hierarchy Update

The parent region remains stored:

$$
R_{\mathrm{parent}}
\rightarrow
\{R_L,R_R\}.
$$

The children become new structural units in the region hierarchy.

No region is deleted as a consequence of the split.

This preserves ARRGO's no-pruning principle.

### Post-Split Reanalysis

After the split, both children receive their corresponding inherited
information.

ARRGO then recomputes the relevant regional information:

$$
\text{Coverage},
\qquad
\text{Behavior},
\qquad
\text{Certified Uncertainty},
\qquad
\text{Certified Potential}.
$$

The new information state may change the dominant unresolved objectives.

Therefore the next refinement decision is not required to follow the same
criterion as the previous split.

### Split Selection Cycle

The complete split-selection process is:

$$
\boxed{
\begin{aligned}
&\text{Analyze Region}\\
&\rightarrow
\text{Identify Active Objectives}\\
&\rightarrow
\text{Generate Split Candidates}\\
&\rightarrow
\text{Construct Structural Profiles}\\
&\rightarrow
\text{Apply Dominance}\\
&\rightarrow
\text{Retain Non-Dominated Candidates}\\
&\rightarrow
\text{Apply Objective-Consistent Selection}\\
&\rightarrow
\text{Apply Deterministic Tie-Breaking}\\
&\rightarrow
\text{Execute Split}\\
&\rightarrow
\text{Reanalyze Children}.
\end{aligned}
}
$$

### Sampling and Splitting Selection

ARRGO now has two complete local refinement mechanisms:

$$
\text{Sampling Selection}
$$

and

$$
\text{Split Selection}.
$$

Sampling selects a location at which new objective information will be acquired.

Splitting selects a location at which the current spatial representation will be
refined.

The two decisions must remain conceptually separate.

### Final Structural Selection Principle

The split-selection mechanism follows:

$$
\boxed{
\text{Select a Non-Dominated Split That Best Addresses the Currently
Relevant Structural Information, Using Deterministic Tie-Breaking Only When
the Available Evidence Does Not Establish a Unique Preference.}
}
$$

This completes the candidate-level design of structural refinement.

The next stage defines the higher-level objective that determines whether a
region should be sampled, split, or temporarily considered sufficiently
resolved.

## Region Refinement Objective

The purpose of region refinement is to reduce unresolved optimization-relevant
information inside a region while preserving the global search structure.

For a region

$$
R=[l,r]
$$

refinement is not defined as simply reducing its geometric size or increasing
the number of function evaluations.

Instead, refinement should improve the resolution of the information that is
currently relevant to the optimization decision.

The main unresolved information dimensions are:

$$
Q_R =
\left(
Q_{\text{coverage}},
Q_{\text{behavior}},
Q_{\text{uncertainty}},
Q_{\text{potential}}
\right)
$$

These components represent different aspects of the current information state:

- \(Q_{\text{coverage}}\): unresolved spatial coverage inside the region.
- \(Q_{\text{behavior}}\): unresolved local behavioral structure.
- \(Q_{\text{uncertainty}}\): unresolved uncertainty in the function representation.
- \(Q_{\text{potential}}\): unresolved information about the region's remaining
  ability to contain a globally competitive solution.

The exact interpretation of these objectives depends on the available
information and the execution mode.

In Empirical Mode, coverage and behavioral information are based on observed
samples and therefore do not constitute rigorous bounds on the unknown
function.

In Certified Mode, valid Lipschitz-based bounds provide additional rigorous
information about uncertainty and optimization potential.

### Refinement as Information Resolution

The refinement objective can be expressed conceptually as:

$$
\text{Refine}(R)
\quad\Longrightarrow\quad
\text{reduce the currently important unresolved information in }R
$$

The objective is therefore not to maximize an arbitrary numerical gain.

Different unresolved objectives may have different units, scales, and meanings.
Combining them into a fixed weighted scalar would introduce arbitrary
preferences that are not justified by the mathematical model.

Instead, ARRGO maintains the objectives separately and determines which
objectives are currently relevant to the refinement decision.

Let

$$
O_R^* \subseteq
\left\{
\text{coverage},
\text{behavior},
\text{uncertainty},
\text{potential}
\right\}
$$

denote the set of currently dominant or optimization-relevant unresolved
objectives in region \(R\).

The region-level refinement objective is then:

$$
\boxed{
\text{Resolve the currently relevant objectives in }O_R^*
}
$$

rather than:

$$
\boxed{
\text{maximize a fixed scalar refinement score}
}
$$

### Refinement Actions

ARRGO has two fundamental refinement actions:

$$
A_R=
\left\{
\text{sample},
\text{split}
\right\}
$$

A third outcome is possible when the available information is sufficiently
resolved:

$$
\text{stable}
$$

Thus, the region-level decision can be represented as:

$$
\text{Decision}(R)
\in
\left\{
\text{Sample},
\text{Split},
\text{Stable}
\right\}
$$

These outcomes have different meanings.

#### Sample

Sampling is selected when additional function information is required inside
the existing spatial representation.

Sampling primarily addresses unresolved:

- coverage,
- local behavior,
- certified uncertainty,
- optimization-relevant potential.

Sampling introduces a new function evaluation and therefore adds new information
to the persistent evaluation history.

#### Split

Splitting is selected when the current spatial representation is too coarse to
adequately distinguish important structural differences inside the region.

Splitting primarily addresses unresolved:

- spatial structure,
- behavioral transitions,
- differences between subregions,
- regional uncertainty structure,
- regional optimization potential.

Splitting does not itself evaluate the objective function at a new point.

Its primary role is to increase spatial resolution.

#### Stable

A region may be considered stable when the currently relevant unresolved
objectives have reached their required resolution for the current decision
context.

Stable does not mean that the region is globally proven irrelevant.

It means that no refinement action is currently required according to the
defined resolution criteria.

A stable region remains part of the persistent region hierarchy and may be
reconsidered if new global information changes its relevance.

### Feasibility and Preference

ARRGO separates two different questions:

1. **Is an action feasible?**
2. **Which feasible action is preferable?**

An action is feasible only if it satisfies the structural and numerical
requirements defined for that action.

For sampling, this includes:

- the candidate lies inside the region,
- the candidate is not a duplicate within the spatial tolerance,
- the candidate is generated from the available information,
- the candidate addresses at least one relevant unresolved objective.

For splitting, this includes:

- the split point lies strictly inside the region,
- the resulting children satisfy the contraction condition,
- the split point is not structurally redundant,
- the split addresses at least one relevant unresolved objective.

Feasibility does not imply superiority.

Among feasible actions, ARRGO compares their ability to resolve the currently
relevant information without introducing arbitrary fixed weights.

### Action-Dependent Resolution

The effect of an action depends on the unresolved objective.

For example, a large coverage gap may indicate that sampling can provide
additional spatial information, while a strong structural transition may
indicate that splitting can provide more appropriate spatial resolution.

Similarly, in Certified Mode, a large certified uncertainty may indicate that
additional sampling can tighten the available bounds, whereas different
certified potentials across subregions may indicate that splitting can better
organize the remaining search structure.

These relationships are decision principles rather than universal guarantees.

The actual effect of an action must be determined from the resulting
information after the action is executed.

The relationship can be summarized as:

$$
\begin{aligned}
\text{Large coverage gap}
&\;\Rightarrow\;
\text{sampling may provide additional spatial information},\\
\text{Strong structural transition}
&\;\Rightarrow\;
\text{splitting may provide more appropriate spatial resolution},\\
\text{Large certified uncertainty}
&\;\Rightarrow\;
\text{additional sampling may tighten the available bounds},\\
\text{Different certified potentials across subregions}
&\;\Rightarrow\;
\text{splitting may better organize the remaining search structure}.
\end{aligned}
$$

These implications describe how current information can guide the refinement
decision. They do not constitute mathematical guarantees that a particular
action will produce a specific amount of improvement.

### No-Pruning Principle

Region refinement never means deleting a region from the global representation.

If a region is split,

$$
R \rightarrow \left\{R_L,R_R\right\}
$$

the parent region remains stored in the hierarchy.

If a region becomes stable, it also remains stored.

Therefore, refinement changes the structure and information state of the region
hierarchy without destroying historical information.

This persistence is essential for maintaining:

- evaluation history,
- parent-child relationships,
- inherited information,
- reproducibility,
- global analysis,
- later reconsideration of previously stable regions.

### Region-Level Refinement Principle

The central principle of region refinement is:

$$
\boxed{
\text{Choose the refinement action that most appropriately resolves the
currently important optimization-relevant information in the region.}
}
$$

This principle deliberately avoids assuming that sampling is always superior to
splitting, that splitting is always superior to sampling, or that a single
scalar score can universally determine the correct action.

The next sections decompose the region refinement objective into explicit
resolution objectives for coverage, behavior, certified uncertainty, and
optimization potential.

## Coverage Resolution Objective

Coverage resolution describes how completely the currently available function
evaluations represent the spatial extent of a region.

For a region

$$
R=[l,r]
$$

suppose the evaluated points inside the region are ordered as

$$
x_1 < x_2 < \cdots < x_n
$$

with corresponding function values

$$
f(x_1),f(x_2),\ldots,f(x_n).
$$

The purpose of coverage analysis is not to determine the unknown function
between these points.

Instead, it measures where the current spatial representation is sparse and
where additional evaluation may provide useful information.

### Coverage Profile

The complete coverage profile of a region consists of the boundary gaps and
the internal gaps between consecutive evaluations.

The left boundary gap is

$$
g_{\mathrm{left}} = x_1-l
$$

and the right boundary gap is

$$
g_{\mathrm{right}} = r-x_n.
$$

The internal gaps are

$$
g_i=x_{i+1}-x_i,
\qquad
i=1,\ldots,n-1.
$$

Therefore, the coverage profile can be represented as

$$
G(R)=
\left(
g_{\mathrm{left}},
g_1,\ldots,g_{n-1},
g_{\mathrm{right}}
\right).
$$

This profile preserves the spatial distribution of the available evaluations
rather than reducing coverage immediately to a single number.

### Maximum Unresolved Spatial Gap

A useful summary of the current spatial coverage is the largest gap:

$$
G_{\max}(R)
=
\max
\left\{
g_{\mathrm{left}},
g_1,\ldots,g_{n-1},
g_{\mathrm{right}}
\right\}.
$$

A large value of \(G_{\max}(R)\) indicates that a relatively large portion of
the region is not directly represented by nearby evaluations.

This does not imply that the objective function behaves badly inside that gap.

It only indicates that the current evaluation distribution provides limited
direct spatial information there.

Therefore,

$$
\text{Large spatial gap}
\;\not\Rightarrow\;
\text{large function variation}.
$$

Instead,

$$
\text{Large spatial gap}
\;\Rightarrow\;
\text{potentially insufficient spatial information}.
$$

### Coverage Resolution Through Sampling

Suppose a new candidate point \(x^c\) is inserted into an existing gap

$$
[x_i,x_{i+1}].
$$

Before sampling, the gap size is

$$
g=x_{i+1}-x_i.
$$

After evaluating \(x^c\), the original gap is replaced by two smaller gaps:

$$
g_{\mathrm{left}}=x^c-x_i
$$

and

$$
g_{\mathrm{right}}=x_{i+1}-x^c.
$$

The largest remaining gap in this interval becomes

$$
g_{\mathrm{new}}
=
\max
\left\{
g_{\mathrm{left}},
g_{\mathrm{right}}
\right\}.
$$

The geometric coverage improvement associated with the candidate is therefore

$$
\Delta G
=
g-g_{\mathrm{new}}.
$$

This quantity measures the reduction in the largest local spatial gap caused
by inserting the candidate.

For example, choosing the midpoint

$$
x^c=\frac{x_i+x_{i+1}}{2}
$$

produces two equal sub-gaps and maximizes the immediate reduction of the
largest gap within that interval.

However, midpoint sampling is not universally optimal for ARRGO.

If another candidate provides stronger information about behavior, uncertainty,
or optimization potential, the multi-criteria decision mechanism may prefer
that candidate.

### Coverage and Sampling Density

The number of samples alone does not determine coverage quality.

For a region of diameter

$$
d(R)=r-l,
$$

a simple sample-density measure can be written as

$$
\rho_{\mathrm{sample}}(R)
=
\frac{n}{d(R)},
\qquad
d(R)>0.
$$

However, high sample density does not necessarily imply uniform coverage.

For example, many evaluations concentrated in a small part of the region may
leave another part almost completely unexplored.

Therefore ARRGO distinguishes between:

- **Sample density:** how many evaluations exist relative to region size.
- **Coverage:** how evenly those evaluations represent the spatial extent.

Coverage resolution is therefore determined from the spatial distribution of
evaluations rather than sample count alone.

### Coverage in Sparse Regions

Special cases must be handled explicitly.

If a region contains no evaluation,

$$
n=0,
$$

there is no direct function information inside the region.

If a region contains only one evaluation,

$$
n=1,
$$

there are no internal gaps, but the two boundary gaps still describe how far
the available information is from the region boundaries.

These cases should be treated as unresolved coverage rather than as evidence of
good spatial resolution.

### Coverage and Structural Splitting

Sampling and splitting affect coverage in different ways.

Sampling introduces a new function evaluation and therefore increases the
available spatial information.

Splitting introduces new region boundaries without evaluating the objective at
the split point.

Consequently,

$$
\text{Sampling}
\;\Rightarrow\;
\text{new spatial function information},
$$

whereas

$$
\text{Splitting}
\;\Rightarrow\;
\text{new spatial structure}.
$$

A split may make coverage statistics more localized because each child region
has its own coverage profile.

However, splitting alone does not create new function evaluations and therefore
does not create new objective information.

### Coverage and Spatial Contraction

Coverage resolution should also be distinguished from geometric contraction.

Spatial contraction guarantees that a refined region becomes smaller:

$$
\operatorname{diam}(R_{k+1})
\le
\rho\operatorname{diam}(R_k),
\qquad
0<\rho<1.
$$

Coverage resolution instead concerns how well the available evaluations
represent the interior and boundaries of the current region.

Therefore,

$$
\text{Small region}
\;\not\Rightarrow\;
\text{well-covered region}.
$$

A region may be geometrically small while still containing insufficient
evaluation information.

ARRGO therefore treats spatial contraction and coverage resolution as related
but distinct refinement objectives.

### Coverage Resolution Objective

The coverage objective can be summarized as:

$$
\boxed{
\text{Reduce unresolved spatial gaps where additional information is
optimization-relevant.}
}
$$

Coverage is therefore not an independent goal of uniformly filling the entire
domain.

The purpose of improving coverage is to provide sufficient spatial information
for reliable behavioral analysis, uncertainty estimation, and optimization
decisions.

Coverage resolution must consequently be considered together with the other
refinement objectives:

$$
\left(
\text{coverage},
\text{behavior},
\text{uncertainty},
\text{potential}
\right).
$$

No fixed weighted combination of these objectives is assumed.

The relative importance of coverage depends on the current information state
and on which unresolved objective is currently relevant to the refinement
decision.

The next section defines the corresponding objective for resolving local
behavioral information inside a region.

## Behavior Resolution Objective

Behavior resolution describes how well the available function evaluations allow
ARRGO to characterize the observed local behavior of the objective function
inside a region.

For a region

$$
R=[l,r]
$$

with ordered evaluations

$$
x_1<x_2<\cdots<x_n,
$$

and corresponding function values

$$
f(x_1),f(x_2),\ldots,f(x_n),
$$

ARRGO analyzes the changes between neighboring observations rather than
assuming an analytical form for the unknown function.

The purpose of behavioral analysis is to identify where the currently observed
information is insufficient to distinguish between different local behaviors.

### Observed Secant Slopes

For two consecutive evaluations, the observed secant slope is

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i},
\qquad
i=1,\ldots,n-1.
$$

Because the evaluation points are ordered,

$$
x_{i+1}-x_i>0.
$$

Therefore, the sign of \(s_i\) describes the observed directional behavior
between the two evaluations.

The basic directional interpretation is:

- \(s_i>0\): the observed function value increases across the interval.
- \(s_i<0\): the observed function value decreases across the interval.
- \(s_i\approx0\): the observed function value changes only slightly.

These statements describe only the observed intervals.

They do not establish the behavior of the unknown function at unsampled points.

### Directional Behavior

ARRGO compares neighboring observed slopes to detect possible changes in
direction.

For two consecutive secant slopes,

$$
s_i
\quad\text{and}\quad
s_{i+1},
$$

a directional reversal occurs when

$$
s_i s_{i+1}<0.
$$

A transition from positive to negative slope,

$$
s_i>0,
\qquad
s_{i+1}<0,
$$

is consistent with a possible local maximum between the observations.

Similarly, a transition from negative to positive slope,

$$
s_i<0,
\qquad
s_{i+1}>0,
$$

is consistent with a possible local minimum.

These observations are candidates for further investigation.

They are not proofs of the existence or location of an extremum because the
function between sampled points remains unknown.

Therefore,

$$
\text{Observed directional reversal}
\;\not\Rightarrow\;
\text{proven extremum}.
$$

Instead,

$$
\text{Observed directional reversal}
\;\Rightarrow\;
\text{potential structural transition}.
$$

### Slope Variation

Directional behavior alone does not describe how rapidly the observed behavior
changes.

ARRGO therefore also examines the variation between neighboring secant
slopes.

For consecutive slopes, define

$$
\Delta s_i
=
s_{i+1}-s_i.
$$

The magnitude

$$
|\Delta s_i|
$$

measures the observed change in slope between the two intervals.

A large value of \( |\Delta s_i| \) indicates that the observed directional
behavior changes substantially across the corresponding spatial region.

A small value indicates that the observed slope values are relatively similar.

However, secant-slope variation should not be interpreted as an exact second
derivative.

In particular,

$$
\Delta s_i
\neq
f''(x)
$$

in general.

It is an observation-based measure of local behavioral change.

### Behavioral Complexity

The available observations can reveal different levels of behavioral
resolution.

A region may exhibit:

- approximately consistent increasing behavior,
- approximately consistent decreasing behavior,
- approximately flat behavior,
- a directional transition,
- strong slope variation,
- insufficient observations to determine the local pattern.

These categories describe the current information state rather than a permanent
classification of the underlying function.

For example, a region containing only two evaluations provides one secant slope.

It can reveal an observed direction, but it cannot establish whether the
function changes direction inside the interval.

With three or more ordered evaluations, neighboring slopes can be compared and
possible directional transitions can be detected.

Therefore, increasing the number of evaluations can improve behavioral
resolution when the new evaluations are informative.

### Behavioral Resolution and Sampling

Sampling can directly improve behavioral resolution because a new evaluation
creates additional neighboring intervals.

Suppose a new evaluation \(x^c\) is inserted between \(x_i\) and \(x_{i+1}\).

The original secant slope

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}
$$

is replaced by two newly observed slopes:

$$
s_{\mathrm{left}}
=
\frac{f(x^c)-f(x_i)}
{x^c-x_i}
$$

and

$$
s_{\mathrm{right}}
=
\frac{f(x_{i+1})-f(x^c)}
{x_{i+1}-x^c}.
$$

This creates new information about how the objective behaves within the
original interval.

The exact behavioral information gained cannot be known before evaluating
\(f(x^c)\).

Therefore, ARRGO can identify candidates that are expected to be informative
from the current geometry and observed behavior, but the actual behavioral
effect is determined only after evaluation.

### Behavioral Resolution and Splitting

Splitting and behavioral sampling have different roles.

Sampling introduces a new function value and can therefore reveal new local
behavior.

Splitting does not introduce a new function value.

Instead, it creates separate spatial representations for the left and right
parts of the region.

This allows ARRGO to represent potentially different behavioral states in the
two children.

Thus,

$$
\text{Sampling}
\;\Rightarrow\;
\text{new observed behavioral information},
$$

whereas

$$
\text{Splitting}
\;\Rightarrow\;
\text{finer spatial organization of observed behavior}.
$$

A split motivated by a detected behavioral transition does not prove that the
transition corresponds to a true extremum.

Its role is to provide a more appropriate spatial representation for subsequent
analysis.

### Behavioral Resolution and Coverage

Behavioral conclusions depend on the quality of the underlying spatial
coverage.

If a region contains a large unresolved spatial gap, the absence of an
observed directional transition does not imply that no transition exists.

Therefore,

$$
\text{No observed transition}
\;\not\Rightarrow\;
\text{No transition in the function}.
$$

Instead, the algorithm must distinguish between:

- evidence that behavior is currently stable under the available observations,
- and insufficient information to determine the behavior.

This distinction prevents ARRGO from treating sparse observations as evidence
of simple function structure.

### Behavioral Resolution Objective

The behavioral objective can be summarized as:

$$
\boxed{
\text{Resolve currently important changes in observed local behavior.}
}
$$

The objective is not to reconstruct the complete unknown function.

Instead, ARRGO seeks enough behavioral information to support decisions about
sampling, splitting, and global region selection.

The behavioral resolution state therefore depends on:

$$
B_R
=
\left(
\text{direction},
\text{slope variation},
\text{candidate transitions},
\text{behavioral complexity}
\right).
$$

This state is updated whenever new function evaluations are added or when the
region structure changes.

Behavioral information remains observation-based unless additional assumptions
provide stronger guarantees.

In Certified Mode, behavioral analysis may be combined with valid
Lipschitz-based bounds, but observed slope behavior itself does not become a
formal certificate.

### Role in the Refinement Decision

Behavioral resolution is one of the objectives considered by ARRGO when
determining whether a region requires further refinement.

A region with unresolved behavioral structure may benefit from additional
sampling when new function evaluations are needed to distinguish local
behavior.

A region containing a meaningful structural transition may benefit from
splitting when the current spatial representation is too coarse to represent
the observed difference between its subregions.

Therefore,

$$
\text{Behavioral Information}
\;\longrightarrow\;
\text{Refinement Decision}.
$$

The direction of this decision depends on the current information state and
does not follow a fixed rule.

Behavioral resolution must consequently be considered together with coverage,
uncertainty, and optimization potential.

The next section defines the corresponding objective for reducing unresolved
certified uncertainty.

## Uncertainty Resolution Objective

Uncertainty resolution describes how tightly ARRGO can bound the unknown
objective function using the information currently available inside a region.

This objective is relevant to Certified Mode, where a valid Lipschitz constant
\(L\) is available.

For a region

$$
R=[l,r]
$$

with evaluated points

$$
D_R=
\left\{
(x_i,f(x_i))
\right\}_{i=1}^{n},
$$

the Lipschitz assumption gives

$$
|f(x)-f(y)|
\le
L|x-y|.
$$

Therefore, every observed function value provides a valid constraint on the
possible value of the function at another point.

### Lower and Upper Envelopes

For every \(x\in R\), the lower envelope is defined as

$$
L_R(x)
=
\max_i
\left\{
f(x_i)-L|x-x_i|
\right\},
$$

and the upper envelope is

$$
U_R(x)
=
\min_i
\left\{
f(x_i)+L|x-x_i|
\right\}.
$$

Under the valid Lipschitz assumption,

$$
L_R(x)
\le
f(x)
\le
U_R(x).
$$

Therefore, the interval

$$
\left[
L_R(x),U_R(x)
\right]
$$

contains every function value that is consistent with the available
evaluations and the assumed Lipschitz bound.

These envelopes are certificates only when the Lipschitz constant is valid.

An estimated Lipschitz value obtained from observed samples is not automatically
a valid certificate.

### Pointwise Uncertainty

The pointwise certified uncertainty is defined as

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

This quantity measures the width of the currently valid interval of possible
function values at \(x\).

A small value of \(u_R(x)\) means that the available information tightly
constrains the function value at that location.

A large value means that multiple function values remain consistent with the
current observations and the Lipschitz assumption.

Thus,

$$
u_R(x)\ge0
$$

for every \(x\in R\).

The uncertainty profile is therefore a function over the region rather than a
single scalar.

### Regional Uncertainty

For decision-making, ARRGO may summarize the uncertainty profile using its
maximum value:

$$
u_{\max}(R)
=
\max_{x\in R}
u_R(x).
$$

This identifies the point at which the current certified representation is
least resolved.

However, \(u_{\max}(R)\) does not contain all information about the uncertainty
distribution.

Two regions may have the same maximum uncertainty while having very different
uncertainty profiles.

Therefore, ARRGO retains the underlying uncertainty profile and uses the
maximum only as a summary when appropriate.

### Effect of Additional Sampling

Suppose a new point \(x^c\in R\) is evaluated and produces the value

$$
y^c=f(x^c).
$$

The new lower envelope becomes

$$
L_R^{\mathrm{new}}(x)
=
\max
\left\{
L_R(x),
y^c-L|x-x^c|
\right\},
$$

while the new upper envelope becomes

$$
U_R^{\mathrm{new}}(x)
=
\min
\left\{
U_R(x),
y^c+L|x-x^c|
\right\}.
$$

Consequently,

$$
L_R^{\mathrm{new}}(x)
\ge
L_R(x)
$$

and

$$
U_R^{\mathrm{new}}(x)
\le
U_R(x).
$$

Therefore, the certified uncertainty cannot increase as a result of adding a
valid new evaluation:

$$
u_R^{\mathrm{new}}(x)
\le
u_R(x).
$$

This monotonicity is one of the key properties that makes additional sampling
useful in Certified Mode.

The exact amount of uncertainty reduction is not known before evaluating the
candidate because it depends on the unknown value \(f(x^c)\).

### Effect of Spatial Contraction

Suppose a region is refined into smaller regions.

For any child region \(R_c\subseteq R\), its diameter satisfies the contraction
condition

$$
\operatorname{diam}(R_c)
\le
\rho\operatorname{diam}(R),
\qquad
0<\rho<1.
$$

Under the Lipschitz assumption, the maximum possible variation of the function
inside the child is bounded by

$$
\operatorname{osc}_{R_c}(f)
\le
L\operatorname{diam}(R_c).
$$

Therefore, as spatial contraction continues,

$$
\operatorname{diam}(R_k)\rightarrow0
$$

and consequently,

$$
L\operatorname{diam}(R_k)\rightarrow0.
$$

This establishes that spatial contraction reduces the maximum function
variation that can remain possible inside sufficiently small regions.

However, this does not mean that the exact envelope uncertainty automatically
equals \(L\operatorname{diam}(R)\).

The envelope depends on the locations and values of the available evaluations.

### Uncertainty and Coverage

Coverage and certified uncertainty are closely related but distinct.

A large spatial gap can lead to a large uncertainty interval because the nearest
observations may be far from the unsampled location.

However, coverage alone does not determine uncertainty.

The actual uncertainty depends on both:

- the geometry of the evaluations,
- and their observed function values under the Lipschitz constraint.

Therefore,

$$
\text{Coverage Resolution}
\neq
\text{Uncertainty Resolution}.
$$

Coverage describes where observations exist.

Uncertainty describes how tightly the unknown function is bounded by those
observations.

### Uncertainty and Splitting

Splitting creates smaller spatial regions but does not introduce a new function
evaluation.

Therefore, splitting alone does not necessarily tighten the certified
envelopes.

Its role is instead to represent the existing uncertainty separately across
smaller spatial regions.

If additional evaluations are subsequently acquired inside the children, the
regional envelopes can become tighter.

Thus,

$$
\text{Splitting}
\;\Rightarrow\;
\text{finer spatial organization of uncertainty},
$$

whereas

$$
\text{Sampling}
\;\Rightarrow\;
\text{additional information that may tighten uncertainty}.
$$

### Uncertainty Resolution Objective

The uncertainty objective can be summarized as:

$$
\boxed{
\text{Reduce unresolved certified uncertainty where it is relevant to the
optimization decision.}
}
$$

This objective is meaningful only in Certified Mode with a valid Lipschitz
bound.

In Empirical Mode, ARRGO may still identify sparse or poorly resolved regions,
but it must not interpret those observations as rigorous uncertainty bounds.

### Role in the Refinement Decision

A region with large unresolved certified uncertainty may require additional
sampling when new function evaluations can meaningfully tighten its envelopes.

A region whose uncertainty is structurally distributed across different
subregions may benefit from splitting so that the uncertainty can be represented
more locally.

The decision remains context-dependent.

ARRGO therefore does not define a universal rule such as:

$$
\text{large uncertainty}
\;\Rightarrow\;
\text{always sample}.
$$

Instead, uncertainty is considered together with coverage, behavior, and
optimization potential.

The next section defines the objective for resolving the remaining
optimization potential of a region.

## Optimization-Potential Resolution Objective

Optimization-potential resolution describes how well ARRGO can determine the
remaining ability of a region to contain a solution that can improve upon the
current global incumbent.

This objective is especially important in Certified Mode, where valid
Lipschitz-based upper bounds provide a rigorous estimate of the best function
value that may still be attainable inside a region.

For a region

$$
R=[l,r],
$$

let the current incumbent value be

$$
f_{\mathrm{best}}
=
\max_{x_i\in D_t} f(x_i).
$$

### Regional Optimization Potential

Using the valid upper envelope

$$
U_R(x)
=
\min_i
\left\{
f(x_i)+L|x-x_i|
\right\},
$$

the certified optimization potential of the region is defined as

$$
P(R)
=
\max_{x\in R}
U_R(x).
$$

Because the upper envelope is valid,

$$
f(x)\le U_R(x)
$$

for every \(x\in R\).

Therefore,

$$
\max_{x\in R} f(x)
\le
P(R).
$$

The value \(P(R)\) represents the largest objective value that remains
consistent with the available information and the valid Lipschitz assumption.

It is therefore an upper potential, not a prediction of where the optimizer
will be or what value the function will actually attain.

### Competitive Regions

A region is potentially competitive when its certified potential is greater
than the current incumbent.

The basic competitive condition is

$$
P(R)>f_{\mathrm{best}}.
$$

For an epsilon-based decision, the competitive condition can instead be written
as

$$
P(R)>f_{\mathrm{best}}+\varepsilon.
$$

If

$$
P(R)\le f_{\mathrm{best}}+\varepsilon,
$$

then the region cannot contain a point whose function value is certified to
exceed the incumbent by more than the allowed tolerance.

This does not mean that the region contains no good solutions.

It means that, according to the available certified information, its remaining
potential is not sufficient to justify improving the current objective-value
certificate beyond the specified tolerance.

### Global Optimization Potential

ARRGO maintains the potential of all persistent regions.

The global certified potential is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}
P(R),
$$

where \(\mathcal{R}_t\) denotes the regions currently used in the global
coverage representation.

Under valid regional bounds and complete domain coverage,

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}}.
$$

Therefore, the global certified gap is

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

The potential-resolution objective is closely connected to reducing this gap.

### Effect of Additional Sampling

Suppose a new evaluation is obtained at \(x^c\in R\), with

$$
y^c=f(x^c).
$$

The new upper envelope becomes

$$
U_R^{\mathrm{new}}(x)
=
\min
\left\{
U_R(x),
y^c+L|x-x^c|
\right\}.
$$

Therefore,

$$
U_R^{\mathrm{new}}(x)
\le
U_R(x).
$$

Consequently, the regional potential cannot increase:

$$
P_{\mathrm{new}}(R)
\le
P(R).
$$

Thus, valid additional evaluations can only preserve or reduce the certified
optimization potential.

However, the exact reduction cannot be known before evaluating the candidate
because it depends on the unknown value \(f(x^c)\).

### Effect of Splitting

Suppose

$$
R
\rightarrow
\left\{
R_L,R_R
\right\}.
$$

The child regions provide a more localized spatial representation of the
available information.

Their potentials are

$$
P(R_L)
=
\max_{x\in R_L}
U_{R_L}(x)
$$

and

$$
P(R_R)
=
\max_{x\in R_R}
U_{R_R}(x).
$$

Because the child regions cover the parent region,

$$
R_L\cup R_R=R,
$$

the true optimum of the parent satisfies

$$
\max_{x\in R}f(x)
=
\max
\left\{
\max_{x\in R_L}f(x),
\max_{x\in R_R}f(x)
\right\}.
$$

Therefore, valid child bounds provide the consistency relation

$$
\max_{x\in R}f(x)
\le
\max
\left\{
P(R_L),P(R_R)
\right\}.
$$

Splitting itself does not create a new function evaluation.

Therefore, it should not be interpreted as automatically improving the
underlying function information.

Its main role is to organize the remaining potential across smaller spatial
regions.

### Potential and Uncertainty

Optimization potential and uncertainty represent different concepts.

Uncertainty asks:

> How widely can the unknown function value still vary?

Optimization potential asks:

> How good could the function still be inside this region?

These quantities are related but are not interchangeable.

For example, a region may have substantial uncertainty but a low certified
potential.

Another region may have a relatively narrow uncertainty range while still
having a high potential because its observed values are already strong.

Therefore,

$$
\text{Uncertainty}
\neq
\text{Optimization Potential}.
$$

ARRGO keeps these objectives separate.

### Potential and the Global Incumbent

Optimization potential is meaningful only relative to the current global
incumbent.

Suppose a region has

$$
P(R)\le f_{\mathrm{best}}.
$$

Then its certified potential does not exceed the best observed objective value.

If instead

$$
P(R)>f_{\mathrm{best}},
$$

the region still contains unresolved optimization potential according to the
current certificate.

As the incumbent improves, the set of competitive regions may therefore
change.

This means that region relevance is dynamic rather than permanently assigned.

A region that is currently non-competitive may become relevant only if the
global representation or certification state changes.

### Potential Resolution and Refinement

Potential resolution concerns whether ARRGO has enough information to determine
which regions can still contribute meaningfully to improving the global
objective.

A region with high unresolved potential may require:

- additional sampling to tighten its certified upper envelope,
- splitting to organize the potential across smaller spatial regions,
- or both.

The appropriate action depends on the current unresolved information state.

ARRGO therefore does not assume:

$$
\text{High Potential}
\;\Rightarrow\;
\text{Always Sample}
$$

or

$$
\text{High Potential}
\;\Rightarrow\;
\text{Always Split}.
$$

Instead, potential is considered together with coverage, behavior, and
uncertainty.

### Empirical Mode

Optimization potential must be interpreted differently in Empirical Mode.

Without a valid upper bound on the unknown function, ARRGO cannot claim that an
observed or estimated quantity is a rigorous upper potential.

Therefore, Empirical Mode may use observed objective values and behavioral
evidence to prioritize regions, but it must not describe such a heuristic
quantity as a certified bound.

The distinction is:

$$
\text{Empirical relevance}
\neq
\text{Certified optimization potential}.
$$

This separation prevents heuristic search information from being presented as a
mathematical guarantee.

### Optimization-Potential Resolution Objective

The optimization-potential objective can be summarized as:

$$
\boxed{
\text{Resolve which regions retain meaningful potential to improve the global
objective.}
}
$$

In Certified Mode, this objective is directly connected to the certified global
gap:

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

Reducing the certified potential or increasing the incumbent can therefore
reduce the remaining certified gap.

The objective is not to maximize regional potential.

Instead, ARRGO uses potential to determine where unresolved optimization value
may still exist and whether further refinement is justified.

The next section combines coverage, behavior, uncertainty, and optimization
potential into a unified unresolved-information state for each region.

## Unresolved Information State

ARRGO must maintain an explicit representation of what remains unresolved
inside each region.

The purpose of this state is to prevent the refinement mechanism from treating
all regions in the same way regardless of their current information content.

For a region \(R\), the unresolved information state is represented by

$$
Q_R=
\left(
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
\right).
$$

Each component describes a different type of unresolved information.

### Coverage Resolution State

The coverage component represents unresolved spatial sampling information.

It is derived from the spatial distribution of evaluations inside the region.

The coverage state may depend on quantities such as

$$
G(R)=
\left(
g_{\mathrm{left}},
g_1,\ldots,g_{n-1},
g_{\mathrm{right}}
\right)
$$

and

$$
G_{\max}(R)
=
\max G(R).
$$

A large unresolved gap indicates that the current evaluations provide limited
direct spatial information over part of the region.

Coverage resolution therefore asks:

> Where is the current evaluation distribution spatially insufficient?

Coverage information is observation-based and does not by itself provide a
bound on the unknown function.

### Behavioral Resolution State

The behavioral component represents unresolved information about observed
local changes in the objective function.

It is derived from quantities such as the secant slopes

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}
$$

and their variations

$$
\Delta s_i=s_{i+1}-s_i.
$$

Directional reversals, strong slope changes, and insufficient numbers of
observations may all contribute to the behavioral unresolved state.

The behavioral state therefore asks:

> What important local behavior cannot yet be distinguished from the available
> observations?

Behavioral information remains observational unless additional mathematical
assumptions provide stronger guarantees.

### Certified Uncertainty State

The uncertainty component is active in Certified Mode when a valid Lipschitz
constant is available.

The pointwise certified uncertainty is

$$
u_R(x)
=
U_R(x)-L_R(x),
$$

where

$$
L_R(x)
=
\max_i
\left\{
f(x_i)-L|x-x_i|
\right\}
$$

and

$$
U_R(x)
=
\min_i
\left\{
f(x_i)+L|x-x_i|
\right\}.
$$

A regional summary may be obtained from

$$
u_{\max}(R)
=
\max_{x\in R}u_R(x).
$$

The uncertainty state asks:

> Where does the current certified representation remain too wide to support
> the required optimization decision?

This state is not probabilistic.

It represents the width of a valid deterministic enclosure under the assumed
Lipschitz condition.

In Empirical Mode, this component is not interpreted as a rigorous uncertainty
bound.

### Optimization-Potential State

The potential component represents unresolved information about the region's
ability to improve the current global objective.

In Certified Mode, the regional potential is

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

Relative to the current incumbent,

$$
f_{\mathrm{best}}
=
\max_{x_i\in D_t}f(x_i),
$$

the region remains competitive when

$$
P(R)>f_{\mathrm{best}}+\varepsilon.
$$

The potential state therefore asks:

> Can this region still contain a solution that is relevant to the current
> optimization objective?

In Empirical Mode, observed objective values and behavioral evidence may guide
region relevance, but they must not be presented as certified upper potential.

### Joint Information State

The four components should not be collapsed into a single fixed numerical
score.

Instead, ARRGO maintains their identities separately:

$$
Q_R=
\left(
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
\right).
$$

This preserves the different meanings, units, and levels of evidence
associated with each objective.

For example, two regions may have similar coverage resolution but very different
behavioral resolution.

Similarly, two regions may have similar certified uncertainty while having
very different optimization potentials.

Therefore,

$$
Q_{R_1}=Q_{R_2}
$$

cannot be inferred merely from equality of one component.

The complete information state must be considered.

### Active Unresolved Objectives

Not every component must be relevant at every iteration.

ARRGO therefore identifies an active objective set

$$
O_R^*
\subseteq
\left\{
\mathrm{coverage},
\mathrm{behavior},
\mathrm{uncertainty},
\mathrm{potential}
\right\}.
$$

The active set contains the objectives whose current unresolved information is
important enough to influence the refinement decision.

For example,

$$
O_R^*=
\left\{
\mathrm{coverage}
\right\}
$$

indicates that spatial coverage is currently the dominant unresolved concern.

Another region may require simultaneous consideration of

$$
O_R^*=
\left\{
\mathrm{behavior},
\mathrm{coverage}
\right\}.
$$

In Certified Mode, the set may additionally contain uncertainty or potential.

The active objective set is therefore dynamic and region-dependent.

### Information State Update

The information state changes whenever ARRGO acquires new information or changes
the spatial representation.

A sampling action produces a new function evaluation:

$$
(x^c,f(x^c))
$$

which may change:

- coverage,
- observed behavior,
- certified uncertainty,
- certified optimization potential.

A splitting action changes the spatial organization of the information and
creates child-specific states.

Therefore,

$$
Q_R^{(t)}
\rightarrow
Q_R^{(t+1)}
$$

after refinement.

The new state must be recomputed from the updated observations and region
structure rather than assuming a predetermined improvement.

### Information Resolution and Stability

A region is sufficiently resolved only relative to the current decision
requirements.

This means that a region may be considered stable when all currently relevant
objectives satisfy their required resolution criteria.

Conceptually,

$$
O_R^*
=
\varnothing
$$

represents a state in which no unresolved objective currently requires further
refinement.

However, stability is not permanent.

A change in the global incumbent, global potential, neighboring information, or
other relevant global state may cause a previously stable region to become
relevant again.

Therefore,

$$
\text{Stable}
\neq
\text{Permanently Resolved}.
$$

### No-Pruning and Information Persistence

The unresolved-information state does not determine whether a region remains
stored in the hierarchy.

Even when

$$
O_R^*=\varnothing,
$$

the region remains part of the persistent region structure.

Similarly, after a split,

$$
R
\rightarrow
\left\{
R_L,R_R
\right\},
$$

the parent remains stored.

This separation between information state and structural existence ensures that
ARRGO can preserve historical information and reconsider earlier decisions.

### Unified Refinement Principle

The unresolved-information state provides the bridge between region analysis and
refinement action.

The overall decision flow is therefore:

$$
\text{Region Analysis}
\rightarrow
\text{Unresolved Information State}
\rightarrow
\text{Active Objectives}
\rightarrow
\text{Refinement Decision}.
$$

The state does not itself determine a universal action.

Instead, it identifies what remains unresolved so that the next stage can
compare feasible sampling and splitting actions against the currently relevant
objectives.

The next section defines explicit resolution criteria for determining when an
individual objective is sufficiently resolved.

## Resolution Criteria

ARRGO requires explicit criteria for determining whether an unresolved
information objective has reached a sufficient level of resolution.

Without such criteria, the algorithm could identify unresolved information
indefinitely without being able to determine when further refinement is
necessary.

The resolution criteria therefore connect the information state of a region to
the decision of whether additional refinement is required.

### Resolution as a Decision Requirement

For a region \(R\), let the active objective set be

$$
O_R^*
\subseteq
\left\{
\mathrm{coverage},
\mathrm{behavior},
\mathrm{uncertainty},
\mathrm{potential}
\right\}.
$$

Each active objective must be evaluated according to criteria appropriate to
its meaning.

ARRGO does not assume that all objectives become resolved at the same rate or
according to the same numerical condition.

Therefore, resolution is objective-specific.

Conceptually,

$$
\text{Resolved}(O)
\iff
\text{current information satisfies the required resolution condition for }O.
$$

### Coverage Resolution Criterion

Coverage resolution is based on the spatial distribution of the available
evaluations.

The largest unresolved spatial gap is

$$
G_{\max}(R)
=
\max
\left\{
g_{\mathrm{left}},
g_1,\ldots,g_{n-1},
g_{\mathrm{right}}
\right\}.
$$

A coverage criterion may require this quantity to fall below an appropriate
spatial resolution threshold:

$$
G_{\max}(R)\le\varepsilon_{\mathrm{coverage}}.
$$

The threshold represents the spatial resolution required for the current
optimization task.

The criterion should be interpreted relative to the scale of the region and
the numerical precision of the implementation.

A small number of samples does not automatically satisfy the coverage
criterion.

Likewise, a large number of concentrated samples may still leave a large
unresolved spatial gap.

### Behavioral Resolution Criterion

Behavioral resolution cannot be determined from sample count alone.

ARRGO evaluates the currently observed directional and slope-variation
information.

Relevant indicators include:

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}
$$

and

$$
\Delta s_i=s_{i+1}-s_i.
$$

A behavioral state may be considered sufficiently resolved when the available
observations provide enough local information for the current refinement
decision.

For example, a region may have sufficiently resolved behavior when:

- relevant directional transitions have been investigated,
- observed slope variation is below the required resolution,
- and remaining spatial gaps do not prevent meaningful behavioral
  interpretation.

No universal numerical threshold is imposed by the mathematical framework.

The actual behavioral criterion must therefore be defined by the numerical
implementation and its intended resolution.

This distinction is important because

$$
\text{No observed behavioral change}
\;\not\Rightarrow\;
\text{No behavioral change exists}.
$$

A region with insufficient spatial information cannot be declared behaviorally
resolved merely because its currently observed slopes appear simple.

### Certified Uncertainty Criterion

In Certified Mode, uncertainty can be evaluated using the valid Lipschitz-based
envelopes.

The pointwise uncertainty is

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

A regional summary is

$$
u_{\max}(R)
=
\max_{x\in R}u_R(x).
$$

A direct uncertainty-resolution criterion is

$$
u_{\max}(R)
\le
\varepsilon_{\mathrm{uncertainty}}.
$$

When this condition holds, the certified representation has reached the
required maximum uncertainty resolution inside the region.

The criterion is rigorous only when the Lipschitz constant is valid and the
regional envelopes are computed correctly.

### Optimization-Potential Resolution Criterion

In Certified Mode, optimization-potential resolution is directly related to the
current incumbent.

A region is no longer competitive at tolerance \(\varepsilon\) when

$$
P(R)
\le
f_{\mathrm{best}}+\varepsilon.
$$

This means that its certified upper potential is not sufficient to improve the
current objective-value certificate beyond the allowed tolerance.

For the entire domain, certified termination requires

$$
P_{\mathrm{global}}
\le
f_{\mathrm{best}}+\varepsilon.
$$

Equivalently,

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
\le
\varepsilon.
$$

This is stronger than declaring a region stable based only on local behavior.

### Objective-Specific Resolution

The four objectives therefore have different resolution criteria.

They can be summarized as:

$$
\begin{aligned}
\text{Coverage}
&\rightarrow
G_{\max}(R)
\le
\varepsilon_{\mathrm{coverage}},\\
\text{Behavior}
&\rightarrow
\text{behavioral information sufficient for the current decision},\\
\text{Uncertainty}
&\rightarrow
u_{\max}(R)
\le
\varepsilon_{\mathrm{uncertainty}},\\
\text{Potential}
&\rightarrow
P(R)
\le
f_{\mathrm{best}}+\varepsilon.
\end{aligned}
$$

These criteria are not combined into a fixed weighted score.

Each objective retains its own interpretation.

### Resolution Is Context-Dependent

The required resolution depends on the purpose of the current decision.

For example, a region may have sufficiently good coverage for behavioral
analysis but still have excessive certified uncertainty.

Another region may have well-resolved local behavior but retain a high
optimization potential.

Therefore,

$$
\text{Resolved Coverage}
\;\not\Rightarrow\;
\text{Resolved Region}.
$$

Likewise,

$$
\text{Resolved Behavior}
\;\not\Rightarrow\;
\text{Resolved Potential}.
$$

A region is sufficiently resolved only when the currently active objectives have
reached their required conditions.

### Active Objective Resolution

Let

$$
O_R^*
=
\left\{
O_1,O_2,\ldots,O_k
\right\}
$$

be the currently active objective set.

The region can be considered sufficiently resolved when every active objective
satisfies its corresponding criterion:

$$
\forall O\in O_R^*,
\qquad
\mathrm{Resolved}(O)=\mathrm{True}.
$$

Equivalently, the unresolved objective set becomes empty:

$$
O_R^*
=
\varnothing.
$$

This does not mean that every possible property of the unknown function has
been determined.

It means only that the information currently required for the refinement
decision has reached its specified resolution.

### Resolution and Empirical Mode

Empirical Mode requires additional caution.

Coverage resolution can still be measured from observed evaluation geometry.

Behavioral resolution can still be assessed from observed slopes and
transitions.

However, uncertainty and optimization-potential quantities cannot be treated
as rigorous certificates without valid mathematical bounds.

Therefore, empirical resolution criteria must be interpreted as decision
heuristics rather than formal optimality guarantees.

The distinction is:

$$
\text{Empirical Resolution}
\neq
\text{Certified Resolution}.
$$

### Resolution and Refinement

If an active objective fails its resolution criterion, additional refinement
may be required.

The refinement mechanism then determines whether sampling or splitting is more
appropriate for resolving that objective.

Conceptually,

$$
\text{Unresolved Objective}
\rightarrow
\text{Candidate Refinement Actions}
\rightarrow
\text{Action Selection}.
$$

The resolution criteria therefore do not directly select the next candidate.

Their role is to determine what information remains insufficient.

### No-Pruning Principle

Resolution does not determine whether a region is deleted.

If a region satisfies all currently relevant resolution criteria, it may become
stable:

$$
\text{Resolved}
\rightarrow
\text{Stable}.
$$

The region nevertheless remains part of the persistent hierarchy.

If global information changes later, the region may become relevant again.

Therefore,

$$
\text{Stable}
\neq
\text{Deleted}.
$$

### Resolution Criteria Principle

The central principle is:

$$
\boxed{
\text{Declare an objective resolved only when its information satisfies the
criterion appropriate to its mathematical meaning and decision role.}
}
$$

This prevents ARRGO from using a single arbitrary threshold or scalar score to
represent fundamentally different types of information.

The next section examines how ARRGO determines whether the refinement decision
itself has become stable, rather than merely determining whether an individual
information objective has been resolved.

## Decision Stability

Resolution criteria determine whether an individual information objective has
reached its required level of resolution.

Decision stability addresses a different question:

> Has the current information state become sufficiently stable that the same
> refinement decision is likely to remain appropriate under the available
> information?

This distinction is important because a region may satisfy one resolution
criterion while its overall refinement decision remains uncertain.

### Stability of a Refinement Decision

For a region \(R\), let the available refinement actions be

$$
A_R=
\left\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\right\}.
$$

The decision is stable when the currently available information provides a
sufficiently consistent basis for selecting the same action.

Conceptually,

$$
\text{Decision Stability}
=
\text{Consistency of the refinement decision under current information}.
$$

Decision stability is therefore different from mathematical certainty about
the unknown function.

A stable decision does not imply that the complete behavior of the objective
function is known.

### Why Stability Is Necessary

New evaluations can change the observed behavior of a region.

For example, a new sample may:

- reveal a previously unseen directional transition,
- reduce a large coverage gap,
- tighten a certified uncertainty interval,
- reduce the regional optimization potential,
- or change the global incumbent.

Therefore, a decision that appears appropriate before sampling may no longer be
appropriate afterward.

ARRGO must consequently reevaluate the region after information changes.

The refinement cycle is therefore:

$$
\text{Analyze}
\rightarrow
\text{Decide}
\rightarrow
\text{Refine}
\rightarrow
\text{Reanalyze}.
$$

### Coverage Decision Stability

Coverage stability concerns whether the spatial evaluation distribution provides
a sufficiently consistent basis for the current refinement decision.

The primary spatial quantity is

$$
G_{\max}(R)
=
\max G(R).
$$

If the unresolved coverage gaps remain large, sampling may continue to be
relevant.

If the coverage criterion is satisfied,

$$
G_{\max}(R)
\le
\varepsilon_{\mathrm{coverage}},
$$

coverage is considered resolved for the current decision context.

However, coverage resolution alone does not establish stability of the entire
region.

Another objective may still require refinement.

### Behavioral Decision Stability

Behavioral stability concerns whether the observed local behavior is sufficiently
resolved for the current decision.

Relevant information includes:

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}
$$

and

$$
\Delta s_i=s_{i+1}-s_i.
$$

A region may be considered behaviorally stable when additional observations are
not currently required to distinguish between refinement actions.

This does not imply that the unknown function has no additional behavior.

It means that the current evidence is sufficient for the present decision.

Therefore,

$$
\text{Behaviorally Stable}
\neq
\text{Functionally Simple}.
$$

### Certified Uncertainty Stability

In Certified Mode, uncertainty stability can be evaluated from the certified
envelope.

The regional uncertainty is

$$
u_{\max}(R)
=
\max_{x\in R}
\left(
U_R(x)-L_R(x)
\right).
$$

If

$$
u_{\max}(R)
\le
\varepsilon_{\mathrm{uncertainty}},
$$

the certified uncertainty objective is resolved to the required tolerance.

A stable uncertainty state means that additional sampling is not currently
required solely to reduce certified uncertainty below the specified threshold.

### Optimization-Potential Stability

Optimization-potential stability concerns whether the region remains relevant
to improving the global objective.

The regional potential is

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

A region is non-competitive at tolerance \(\varepsilon\) when

$$
P(R)
\le
f_{\mathrm{best}}+\varepsilon.
$$

If the inequality is reversed,

$$
P(R)
>
f_{\mathrm{best}}+\varepsilon,
$$

the region retains certified optimization relevance.

Therefore, potential stability is directly related to the current global
incumbent.

When \(f_{\mathrm{best}}\) changes, the competitive status of regions must be
reconsidered.

### Stability of the Combined Decision

Let the active objective set be

$$
O_R^*.
$$

A region may be considered decision-stable when:

1. all currently active objectives satisfy their resolution criteria, and
2. the available feasible refinement actions do not reveal a clearly more
   appropriate action under the current information.

Conceptually,

$$
\text{Stable}(R)
\iff
\text{Resolved}(O_R^*)
\land
\text{No clearly preferred refinement action}.
$$

The second condition is important because an objective can be close to its
resolution threshold while the available candidate actions still provide
meaningful differences.

### Stability Is Not Permanent

Decision stability is always relative to the current information state.

Suppose a region is marked stable at iteration \(t\):

$$
\mathrm{State}_t(R)=\mathrm{STABLE}.
$$

A later global update may change the incumbent:

$$
f_{\mathrm{best}}^{(t+1)}
>
f_{\mathrm{best}}^{(t)}.
$$

This can change the region's optimization relevance.

Likewise, new information from neighboring or parent-child regions may change
the interpretation of the current region.

Therefore,

$$
\mathrm{STABLE}_t(R)
\not\Rightarrow
\mathrm{STABLE}_{t+1}(R).
$$

A stable region must remain eligible for reconsideration.

### Stability and No-Pruning

Decision stability does not justify removing a region from the global
representation.

If

$$
\mathrm{State}(R)=\mathrm{STABLE},
$$

the region remains stored in the persistent hierarchy.

Its information, parent relationship, and historical evaluations remain
available.

This allows ARRGO to revisit the region when its global relevance changes.

Thus,

$$
\text{Stable}
\rightarrow
\text{temporarily inactive},
$$

rather than

$$
\text{Stable}
\rightarrow
\text{deleted}.
$$

### Stability and Global Information

ARRGO decisions are not completely local.

A region's refinement state can depend on global information such as:

- the current incumbent,
- the global certified potential,
- the evaluation budget,
- neighboring region information,
- and the global refinement history.

Therefore, the same local region state can lead to different refinement
decisions at different iterations.

This dynamic behavior is intentional.

ARRGO is a global optimization framework rather than a collection of independent
local optimizers.

### Stability Versus Termination

Decision stability must also be distinguished from global termination.

A region can be stable while other regions remain competitive.

Therefore,

$$
\text{Local Stability}
\not\Rightarrow
\text{Global Termination}.
$$

Certified termination requires the global condition

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
\le
\varepsilon.
$$

Thus, ARRGO may have many stable regions while continuing to refine other
regions.

### Stability Principle

The central principle is:

$$
\boxed{
\text{Treat stability as a current decision state, not as a permanent claim
about the unknown function.}
}
$$

A stable region remains part of the global representation and may be
reconsidered whenever new information changes its relevance.

The next section defines how ARRGO chooses between sampling, splitting, and
stability once the current unresolved objectives and their resolution states
have been established.

## Refinement Action Selection: Sampling vs Splitting vs Stable

After analyzing a region and identifying its unresolved information objectives,
ARRGO must determine the appropriate refinement action.

The available outcomes are:

$$
A_R=
\left\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\right\}.
$$

The purpose of this decision is not to maximize the number of evaluations or
the number of regions.

Instead, ARRGO selects the action that most appropriately addresses the
currently unresolved optimization-relevant information.

### Decision Inputs

The refinement decision for a region is based on its current information state:

$$
Q_R=
\left(
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
\right).
$$

The active objective set is

$$
O_R^*
\subseteq
\left\{
\mathrm{coverage},
\mathrm{behavior},
\mathrm{uncertainty},
\mathrm{potential}
\right\}.
$$

The decision also considers:

- the available sampling candidates,
- the available split candidates,
- the structural validity of each action,
- the current global incumbent,
- the certified potential when available,
- numerical tolerances,
- and the remaining evaluation budget.

### Stable Decision

The first question is whether further refinement is currently required.

If all active objectives satisfy their resolution criteria and no feasible action
provides a clearly necessary refinement, the region may enter the stable state:

$$
\mathrm{Decision}(R)
=
\mathrm{Stable}.
$$

This does not mean that the region has been proven globally irrelevant.

It means that no refinement is currently required under the defined decision
criteria.

The region remains stored in the hierarchy.

### Sampling Decision

Sampling is appropriate when additional function evaluations are required to
resolve information within the existing spatial representation.

Sampling is particularly relevant when the unresolved information concerns:

- large spatial gaps,
- insufficient behavioral observations,
- certified uncertainty,
- or unresolved optimization potential that can be investigated through
  additional evaluations.

Conceptually,

$$
\text{Need for New Function Information}
\;\Rightarrow\;
\mathrm{Sample}.
$$

A sampling decision must have at least one valid candidate that addresses an
active unresolved objective.

The selected point is then evaluated by the black-box objective function.

### Splitting Decision

Splitting is appropriate when the current spatial representation is too coarse
to distinguish important structural information.

Splitting is particularly relevant when the region contains:

- meaningful observed directional differences,
- strong behavioral transitions,
- spatially separated information,
- or optimization-relevant differences that should be represented by separate
  subregions.

Conceptually,

$$
\text{Need for Finer Spatial Representation}
\;\Rightarrow\;
\mathrm{Split}.
$$

A valid split must satisfy

$$
l<s<r
$$

and the contraction condition

$$
\max
\left\{
s-l,
r-s
\right\}
\le
\rho(r-l),
\qquad
0<\rho<1.
$$

Splitting does not introduce a new function evaluation.

Its primary purpose is to reorganize the current information into smaller
spatial units.

### Action Feasibility

Before comparing actions, ARRGO verifies whether each action is feasible.

For sampling, feasibility requires:

- at least one valid candidate,
- candidate inside the region,
- candidate not duplicated within \(\tau_x\),
- candidate associated with an active unresolved objective.

For splitting, feasibility requires:

- at least one valid split candidate,
- split point strictly inside the region,
- contraction condition satisfied,
- split not structurally redundant,
- split associated with an active unresolved objective.

Therefore,

$$
\text{Feasible Actions}
\subseteq
\left\{
\mathrm{Sample},
\mathrm{Split}
\right\}.
$$

### Action Relevance

Feasibility alone does not determine which action should be selected.

ARRGO first identifies which unresolved objectives are currently important.

For example, if

$$
O_R^*
=
\left\{
\mathrm{coverage}
\right\},
$$

sampling candidates that reduce unresolved coverage are naturally relevant.

If instead

$$
O_R^*
=
\left\{
\mathrm{behavior}
\right\},
$$

candidates associated with unresolved behavioral transitions become more
relevant.

In Certified Mode, uncertainty and potential may also be active objectives.

### Comparing Sampling and Splitting

Sampling and splitting provide different types of refinement.

Sampling:

$$
\text{existing spatial structure}
+
\text{new function evaluation}
\rightarrow
\text{new information}.
$$

Splitting:

$$
\text{existing information}
+
\text{new spatial structure}
\rightarrow
\text{finer representation}.
$$

Therefore, the two actions cannot be compared only by counting evaluations or
regions.

They must be compared according to how well they address the active unresolved
objectives.

### Action Support

For a candidate action \(a\), define its objective-support set as

$$
S(a)
\subseteq
O_R^*.
$$

The set \(S(a)\) contains the active objectives that the action is relevant to
addressing.

For example,

$$
S(a_{\mathrm{sample}})
=
\left\{
\mathrm{coverage},
\mathrm{behavior}
\right\}
$$

indicates that the sampling candidate is relevant to both objectives.

Likewise,

$$
S(a_{\mathrm{split}})
=
\left\{
\mathrm{behavior},
\mathrm{potential}
\right\}
$$

indicates that the split addresses both currently active objectives.

A broader support set can be useful when the actions are otherwise comparable,
but support alone does not establish that one action is mathematically
superior.

### Multi-Criteria Action Comparison

ARRGO does not use a fixed weighted sum such as

$$
w_1Q_{\mathrm{coverage}}
+
w_2Q_{\mathrm{behavior}}
+
w_3Q_{\mathrm{uncertainty}}
+
w_4Q_{\mathrm{potential}}.
$$

Such a score would introduce arbitrary preferences between quantities with
different meanings and scales.

Instead, action comparison follows the same principle used for candidate
selection:

$$
\text{Active Objectives}
\rightarrow
\text{Feasible Actions}
\rightarrow
\text{Multi-Criteria Comparison}.
$$

Dominance may be used when the action profiles are directly comparable.

If no action dominates another, deterministic tie-breaking is applied only
after the meaningful objective comparison has been exhausted.

### Decision Priority

The refinement decision follows this conceptual hierarchy:

$$
\boxed{
\text{Resolution Requirement}
\rightarrow
\text{Action Feasibility}
\rightarrow
\text{Objective Relevance}
\rightarrow
\text{Action Comparison}
\rightarrow
\text{Deterministic Tie-Breaking}
}
$$

This hierarchy prevents implementation convenience from replacing the
information-driven objective of ARRGO.

### Post-Action Reanalysis

After an action is executed, the region information must be updated.

For sampling,

$$
R
\xrightarrow{\mathrm{Sample}}
R'
$$

where \(R'\) contains the new function evaluation.

For splitting,

$$
R
\xrightarrow{\mathrm{Split}}
\left\{
R_L,R_R
\right\}.
$$

The resulting regions are then reanalyzed.

Therefore, the action selected at one iteration is not assumed to remain optimal
after new information becomes available.

The refinement cycle is:

$$
\text{Analyze}
\rightarrow
\text{Select Action}
\rightarrow
\text{Execute}
\rightarrow
\text{Update}
\rightarrow
\text{Reanalyze}.
$$

### Global Context

The refinement action is selected using both local and global information.

Local information includes:

- coverage,
- behavior,
- uncertainty,
- potential,
- and candidate action profiles.

Global information includes:

- the current incumbent,
- global potential,
- global certified gap,
- evaluation budget,
- and the persistent region hierarchy.

Thus,

$$
\text{Local Information}
+
\text{Global Information}
\rightarrow
\text{Refinement Decision}.
$$

This prevents ARRGO from optimizing a region independently of the global
objective.

### Stable Regions Remain Persistent

If the decision is

$$
\mathrm{Decision}(R)
=
\mathrm{Stable},
$$

the region remains in the global hierarchy.

It is not deleted or permanently excluded.

A later global update may change its relevance and cause it to be reconsidered.

Therefore,

$$
\mathrm{Stable}
\neq
\mathrm{Removed}.
$$

### Refinement Action Principle

The central decision principle is:

$$
\boxed{
\text{Select Sample, Split, or Stable according to which outcome best resolves
the currently relevant information under the available feasible actions.}
}
$$

This principle preserves the distinction between acquiring new function
information, increasing spatial resolution, and temporarily requiring no further
refinement.

The next section moves from local refinement decisions to the global question:
which region should ARRGO refine next?

## Global Region Selection

ARRGO performs refinement inside a global collection of persistent regions.

After the local information state of each region has been analyzed, the
algorithm must determine which region should receive the next refinement
opportunity.

This is a global decision because improving one region can change the relevance
of other regions through the global incumbent, certified potential, and overall
search state.

### Global Region Set

Let the persistent region hierarchy at iteration \(t\) be represented by

$$
\mathcal{R}_t.
$$

The hierarchy contains all regions created during the execution of ARRGO.

Regions are never deleted from this structure.

Each region has a current state such as:

$$
\mathrm{ACTIVE},
\qquad
\mathrm{STABLE},
\qquad
\mathrm{REFINED}.
$$

The region state describes its current role in the refinement process, while the
hierarchy preserves its historical existence and information.

### Eligible Regions

Not every stored region must be refined at every iteration.

ARRGO first identifies regions that are currently eligible for a refinement
opportunity.

Let

$$
\mathcal{E}_t
\subseteq
\mathcal{R}_t
$$

denote the set of eligible regions.

Eligibility may depend on:

- whether the region has unresolved objectives,
- whether a valid refinement action exists,
- whether the region remains relevant to the global objective,
- the current certified potential when available,
- and the numerical constraints of the current iteration.

A stable region may still remain in the persistent hierarchy while temporarily
being excluded from the current eligible set.

### Empirical Mode

In Empirical Mode, region relevance is determined from observed information.

Relevant evidence may include:

- observed objective values,
- behavioral transitions,
- unresolved coverage,
- local sampling density,
- and other observation-based refinement information.

These quantities can guide the global search but do not provide a rigorous
upper bound on the unknown objective.

Therefore, empirical region selection is a deterministic search heuristic.

It must not be interpreted as a proof that an excluded region cannot contain a
better solution.

### Certified Mode

In Certified Mode, region selection can additionally use the valid regional
optimization potential

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

A region remains potentially competitive at tolerance \(\varepsilon\) when

$$
P(R)
>
f_{\mathrm{best}}+\varepsilon.
$$

The competitive region set is therefore

$$
\mathcal{C}_t(\varepsilon)
=
\left\{
R\in\mathcal{R}_t:
P(R)>f_{\mathrm{best}}+\varepsilon
\right\}.
$$

These regions contain certified unresolved potential that may still improve the
current objective-value certificate.

Regions outside this set may still be stored and analyzed, but they are not
required for further refinement solely for improving the current certified
objective-value bound.

### Global Relevance

A region's relevance is dynamic.

Suppose the incumbent improves:

$$
f_{\mathrm{best}}^{(t+1)}
>
f_{\mathrm{best}}^{(t)}.
$$

A region that was previously competitive may become non-competitive.

Similarly, a region that was previously non-competitive may become relevant if
the global representation or certification state changes.

Therefore, global region selection must be recomputed from the current global
state rather than permanently assigning a priority to each region.

### Global Objective

The purpose of global region selection is:

$$
\boxed{
\text{Allocate the next refinement opportunity to the most
optimization-relevant eligible region under the current information state.}
}
$$

This does not necessarily mean selecting the region with the largest raw
function value.

A region may have a moderate observed value but significant unresolved
optimization potential.

Conversely, a region with a high observed value may already be sufficiently
resolved.

Therefore, observed value and unresolved potential are distinct concepts.

### Region Refinement State

For each eligible region \(R\), ARRGO considers its local refinement state:

$$
\mathrm{Decision}(R)
\in
\left\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\right\}.
$$

A region whose decision is Stable does not require an immediate refinement
operation.

A region requiring Sample or Split remains a candidate for global selection.

Thus, the global candidate set can be represented conceptually as

$$
\mathcal{G}_t
=
\left\{
R\in\mathcal{E}_t:
\mathrm{Decision}(R)
\in
\left\{
\mathrm{Sample},
\mathrm{Split}
\right\}
\right\}.
$$

### Global Comparison of Regions

Regions may differ in several dimensions:

- unresolved coverage,
- unresolved behavior,
- certified uncertainty,
- certified potential,
- spatial scale,
- and refinement feasibility.

These quantities have different meanings and scales.

Therefore, ARRGO does not define a universal fixed weighted score such as

$$
w_1Q_{\mathrm{coverage}}
+
w_2Q_{\mathrm{behavior}}
+
w_3Q_{\mathrm{uncertainty}}
+
w_4Q_{\mathrm{potential}}
$$

for global region selection.

Instead, region comparison is based on the currently relevant objectives and
the global optimization context.

### Dominance Between Regions

When multiple regions are comparable under the same active objectives, ARRGO
may use dominance.

A region \(R_i\) dominates another region \(R_j\) when it is at least as
relevant under every currently active comparison criterion and strictly more
relevant under at least one criterion.

The non-dominated region set is then

$$
\mathcal{R}_{\mathrm{ND}}
=
\left\{
R\in\mathcal{G}_t:
\text{no eligible region dominates }R
\right\}.
$$

This preserves trade-offs between different types of unresolved information.

For example, one region may have stronger optimization potential while another
has substantially more unresolved behavioral information.

Neither should be discarded solely because these objectives cannot be reduced
to a justified scalar score.

### Certified Potential Priority

In Certified Mode, optimization potential has a particularly important global
role.

The global potential is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P(R).
$$

A region achieving this maximum is directly relevant to the current certified
global bound.

However, equal or nearly equal potential values may occur across multiple
regions.

ARRGO therefore does not assume that the region with the numerically largest
potential is uniquely superior.

Other active objectives may distinguish between the candidates.

### Fair Global Selection

Global convergence requires that relevant regions cannot be permanently
ignored.

Therefore, the global selection mechanism must satisfy a fairness condition.

Conceptually,

$$
\text{Persistently Relevant Region}
\;\Rightarrow\;
\text{Eventual Refinement Opportunity}.
$$

This does not require uniform refinement.

It means that a region whose relevance persists must not be starved indefinitely
by repeatedly selecting other regions.

Fairness is therefore a property of the selection mechanism rather than a
numerical score.

### Deterministic Tie-Breaking

After objective-based comparison, multiple regions may remain equally relevant.

ARRGO then applies deterministic tie-breaking.

Possible tie-breakers include:

- region depth,
- region diameter,
- deterministic spatial ordering,
- creation order,
- or another explicitly defined deterministic rule.

A tie-breaker is used only after meaningful objective comparisons fail to
establish a unique preference.

Therefore,

$$
\text{Deterministic Tie-Breaking}
\neq
\text{Mathematical Superiority}.
$$

Its purpose is reproducibility.

### Global Selection Cycle

The global region-selection process can be summarized as:

$$
\text{Persistent Regions}
\rightarrow
\text{Eligibility}
\rightarrow
\text{Active Objectives}
\rightarrow
\text{Global Comparison}
\rightarrow
\text{Non-Dominated Regions}
\rightarrow
\text{Deterministic Selection}.
$$

The selected region is then passed to the local refinement mechanism.

### Interaction with Local Refinement

Once a region \(R^*\) is selected globally, its local decision is executed:

$$
R^*
\rightarrow
\left\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\right\}.
$$

If sampling is selected, a valid function evaluation is performed.

If splitting is selected, the region is structurally refined.

If the region is stable, another eligible region must be considered.

After the action, the global state is updated and region priorities are
recomputed.

Thus,

$$
\text{Global Selection}
\rightarrow
\text{Local Refinement}
\rightarrow
\text{Global State Update}
\rightarrow
\text{Global Reselection}.
$$

### No-Pruning Principle

Global selection does not remove unselected regions from the hierarchy.

A region that is not selected remains stored.

A region that becomes stable remains stored.

A parent region remains stored after splitting.

Therefore,

$$
\text{Global Selection}
\neq
\text{Region Deletion}.
$$

This preserves the complete search history and allows previously inactive
regions to become relevant again.

### Global Region Selection Principle

The central principle is:

$$
\boxed{
\text{Select the eligible region whose current information state is most
relevant to resolving the global optimization objective.}
}
$$

The selection must preserve fairness, avoid arbitrary scalarization, respect the
distinction between empirical evidence and certified bounds, and maintain the
persistent region hierarchy.

The next section defines the information used to establish a region's global
priority before the final selection is made.

## Global Region Priority

Global region selection determines which eligible region receives the next
refinement opportunity.

To perform this selection consistently, ARRGO requires a representation of the
current priority of each eligible region.

Global priority is not a permanent numerical property of a region.

It is a dynamic decision state that depends on the current information inside
the region and the current global optimization state.

### Priority State

For an eligible region \(R\), the global priority state can be represented
conceptually as

$$
\Pi_R
=
\left(
\Pi_{\mathrm{coverage}},
\Pi_{\mathrm{behavior}},
\Pi_{\mathrm{uncertainty}},
\Pi_{\mathrm{potential}},
\Pi_{\mathrm{feasibility}}
\right).
$$

These components describe different reasons why a region may deserve the next
refinement opportunity.

They are not assumed to have a common numerical scale.

### Coverage Priority

Coverage priority represents the importance of resolving insufficient spatial
sampling information.

A primary indicator is the largest unresolved gap:

$$
G_{\max}(R)
=
\max G(R).
$$

A large unresolved gap may increase the region's coverage priority when spatial
information is important for the current global decision.

However, a large gap alone does not establish that the region contains a better
solution.

Therefore,

$$
\text{Coverage Priority}
\neq
\text{Optimization Priority}.
$$

Coverage is one source of priority information rather than a complete global
ranking criterion.

### Behavioral Priority

Behavioral priority represents the importance of resolving observed local
behavior.

Relevant evidence may include:

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}
$$

and

$$
\Delta s_i=s_{i+1}-s_i.
$$

Directional reversals, strong slope variation, and insufficient observations
may increase the behavioral priority of a region.

However, observed behavioral complexity does not prove that the region contains
a global optimum.

It indicates that additional information may be useful for understanding the
region.

### Certified Uncertainty Priority

In Certified Mode, a region may receive priority because its certified
uncertainty remains unresolved.

The regional uncertainty summary is

$$
u_{\max}(R)
=
\max_{x\in R}
\left(
U_R(x)-L_R(x)
\right).
$$

A large value indicates that the current certified representation leaves a
wide range of possible function values.

Such a region may require additional refinement if reducing this uncertainty is
relevant to the global optimization certificate.

The interpretation remains deterministic rather than probabilistic.

### Certified Potential Priority

In Certified Mode, optimization potential provides a direct global relevance
measure.

The regional potential is

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

A region is competitive at tolerance \(\varepsilon\) when

$$
P(R)
>
f_{\mathrm{best}}+\varepsilon.
$$

The strongest remaining certified potential across the domain is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P(R).
$$

Regions contributing to this maximum are directly relevant to the remaining
certified optimization gap.

### Refinement Feasibility

A region cannot receive an actual refinement action if no valid action exists.

Therefore, global priority must also consider action feasibility.

For region \(R\), define the feasible action set

$$
F_R
\subseteq
\left\{
\mathrm{Sample},
\mathrm{Split}
\right\}.
$$

A region is refinement-eligible only when

$$
F_R\neq\varnothing.
$$

For sampling, at least one valid non-duplicate candidate must exist.

For splitting, at least one valid interior split satisfying the contraction
condition must exist.

Therefore,

$$
\text{Priority}
\quad\text{cannot replace}\quad
\text{Feasibility}.
$$

A highly relevant region with no valid refinement action must trigger a
reconsideration of candidate generation or refinement representation rather
than an invalid operation.

### Objective-Dependent Priority

Global priority depends on the currently active unresolved objectives.

Let

$$
O_R^*
\subseteq
\left\{
\mathrm{coverage},
\mathrm{behavior},
\mathrm{uncertainty},
\mathrm{potential}
\right\}.
$$

Only objectives relevant to the current region state should influence its
priority comparison.

For example, if

$$
O_R^*
=
\left\{
\mathrm{coverage}
\right\},
$$

coverage information has primary relevance.

If

$$
O_R^*
=
\left\{
\mathrm{behavior},
\mathrm{potential}
\right\},
$$

both behavioral evidence and optimization potential participate in the
comparison.

This prevents irrelevant criteria from dominating the decision.

### Relative Global Relevance

Priority is determined relative to other eligible regions.

Suppose two regions \(R_1\) and \(R_2\) have unresolved coverage.

The fact that

$$
G_{\max}(R_1)>G_{\max}(R_2)
$$

provides evidence that \(R_1\) has a larger unresolved coverage gap.

It does not automatically establish that \(R_1\) should always be selected.

Other active objectives may make \(R_2\) more relevant to the global optimization
decision.

Therefore, ARRGO compares regions using their active objective profiles rather
than a single universally dominant quantity.

### Multi-Criteria Global Priority

The complete priority representation remains multi-dimensional:

$$
\Pi_R
=
\left(
\Pi_{\mathrm{coverage}},
\Pi_{\mathrm{behavior}},
\Pi_{\mathrm{uncertainty}},
\Pi_{\mathrm{potential}},
\Pi_{\mathrm{feasibility}}
\right).
$$

ARRGO does not assume a fixed scalarization such as

$$
w_1\Pi_{\mathrm{coverage}}
+
w_2\Pi_{\mathrm{behavior}}
+
w_3\Pi_{\mathrm{uncertainty}}
+
w_4\Pi_{\mathrm{potential}}.
$$

Such a scalar would introduce arbitrary preferences between different types of
information.

Instead, comparable regions are evaluated using objective-specific comparison
and dominance where appropriate.

### Non-Dominated Global Regions

Let

$$
\mathcal{G}_t
$$

be the set of globally eligible regions.

The non-dominated subset is

$$
\mathcal{G}_{\mathrm{ND}}
=
\left\{
R\in\mathcal{G}_t:
\text{no eligible region dominates }R
\right\}.
$$

The non-dominated set preserves regions representing different trade-offs
between unresolved objectives.

For example, one region may have stronger certified potential while another
has substantially greater behavioral uncertainty.

Neither should be discarded merely because the objectives cannot be reduced to a
justified common scalar.

### Deterministic Global Tie-Breaking

After objective-based comparison, several regions may remain equally relevant.

ARRGO then applies a deterministic tie-breaking rule.

Possible rules include:

- smaller region diameter,
- greater refinement depth,
- deterministic spatial ordering,
- creation order,
- or another explicitly defined deterministic rule.

The exact tie-breaker must be fixed by the implementation.

Its role is only to ensure reproducibility:

$$
\text{Tie-Breaking}
\rightarrow
\text{Deterministic Choice}.
$$

It does not imply that the selected region is mathematically superior to the
other tied regions.

### Dynamic Priority

Global priority must be recomputed after every meaningful global update.

For example, after a new evaluation,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

This can change the competitive status of regions because their potentials are
now compared against a different incumbent.

Similarly, splitting can change the spatial representation and the regional
information profiles.

Therefore,

$$
\Pi_R^{(t)}
\neq
\Pi_R^{(t+1)}
$$

in general.

Priority is therefore a current-state property, not a permanent label.

### Fairness and Priority

A deterministic priority mechanism must still preserve the global fairness
condition.

If a region remains persistently relevant, it must eventually receive a
refinement opportunity.

Conceptually,

$$
\text{Persistent Relevance}
\Rightarrow
\text{Eventual Selection Opportunity}.
$$

This prevents a region from being permanently ignored because other regions
happen to receive slightly higher priority at every individual iteration.

Fairness therefore constrains the global selection mechanism beyond local
priority comparison.

### Global Priority Principle

The central principle is:

$$
\boxed{
\text{Represent region priority as dynamic, multi-dimensional relevance under
the current global information state.}
}
$$

Priority must respect feasibility, active objectives, certified relevance when
available, deterministic comparison, and fairness.

It must not be interpreted as a prediction of where the global optimizer is
located.

The next section defines the complete global optimization objective that ARRGO
uses to coordinate regional refinement toward the overall solution.

## Global Optimization Objective

The global optimization objective defines what ARRGO attempts to achieve across
the entire search domain.

The objective is not simply to maximize the observed function value as quickly
as possible.

ARRGO must simultaneously maintain sufficient spatial resolution, acquire
optimization-relevant information, and progressively reduce the set of
plausible locations that may still contain a better solution.

The global objective can therefore be viewed as the progressive resolution of
the remaining global optimization uncertainty.

### Global State

At iteration \(t\), ARRGO maintains a global state

$$
G_t
=
\left(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t
\right).
$$

In Certified Mode, the state additionally contains the certified regional
bounds and global potential:

$$
G_t^{\mathrm{cert}}
=
\left(
G_t,
\{U_R,L_R,P_R\}_{R\in\mathcal{R}_t},
P_{\mathrm{global}}^{(t)},
\Delta_{\mathrm{global}}^{(t)}
\right).
$$

The global optimization objective is evaluated from this evolving state.

### Primary Objective

The primary objective of ARRGO is to progressively obtain information that
allows the algorithm to identify a globally competitive solution over the
entire domain.

Let

$$
f^*
=
\max_{x\in\Omega}f(x).
$$

The best observed value is

$$
f_{\mathrm{best}}^{(t)}
=
\max_{(x_i,f(x_i))\in D_t}f(x_i).
$$

ARRGO seeks to make

$$
f_{\mathrm{best}}^{(t)}
\rightarrow
f^*
$$

as refinement and information acquisition continue under the theoretical
conditions established previously.

This is an objective-value convergence goal.

It does not by itself require convergence of the reported optimizer location.

### Certified Global Objective

In Certified Mode, the global objective can be expressed more precisely through
the remaining certified optimality gap.

The global potential is

$$
P_{\mathrm{global}}^{(t)}
=
\max_{R\in\mathcal{R}_t}P_R.
$$

The certified global gap is

$$
\Delta_{\mathrm{global}}^{(t)}
=
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}.
$$

Because the regional potentials are valid upper bounds,

$$
f_{\mathrm{best}}^{(t)}
\le
f^*
\le
P_{\mathrm{global}}^{(t)}.
$$

Therefore,

$$
0
\le
f^*-f_{\mathrm{best}}^{(t)}
\le
\Delta_{\mathrm{global}}^{(t)}.
$$

The certified global objective is consequently to reduce this remaining gap.

### Global Information Resolution

The global optimization problem is not represented by one unresolved quantity.

Different regions may require different types of information.

For the complete region hierarchy, define the collection of unresolved
information states as

$$
\mathcal{Q}_t
=
\left\{
Q_R:
R\in\mathcal{R}_t
\right\}.
$$

Each region may contain unresolved:

- coverage information,
- behavioral information,
- certified uncertainty,
- optimization potential.

The global objective is therefore to resolve the information that most affects
the remaining global optimization decision.

### Global Coverage Objective

ARRGO must preserve sufficient spatial coverage over the entire search domain.

A region with large unresolved gaps may require additional sampling or structural
refinement even when its currently observed function values are not among the
best observed values.

This prevents the algorithm from reducing the search to already well-sampled
locations.

Coverage therefore supports global exploration.

However,

$$
\text{Coverage}
\neq
\text{Proof of Global Optimality}.
$$

Coverage is a necessary component of a reliable global search mechanism, not a
standalone optimality criterion.

### Global Behavioral Objective

ARRGO also seeks to resolve important changes in observed local behavior.

Directional reversals, strong slope variation, and spatially separated
behavioral patterns may indicate that the current representation is too coarse.

The purpose is not to construct an exact global model of \(f\).

Instead, behavioral information is used to determine where additional sampling
or spatial refinement may be informative.

Thus,

$$
\text{Observed Behavior}
\rightarrow
\text{Refinement Evidence},
$$

rather than

$$
\text{Observed Behavior}
\rightarrow
\text{Unverified Global Prediction}.
$$

### Global Uncertainty Objective

In Certified Mode, unresolved uncertainty provides another global refinement
objective.

For each region,

$$
u_{\max}(R)
=
\max_{x\in R}
\left(
U_R(x)-L_R(x)
\right).
$$

Regions with unresolved certified uncertainty may require additional
information before their global relevance can be determined accurately.

Sampling can tighten the regional envelopes.

Therefore, the global refinement process can progressively reduce the
uncertainty associated with unresolved regions.

### Global Potential Objective

Certified optimization potential determines whether a region can still
contain a solution competitive with the incumbent.

For tolerance \(\varepsilon\), define the competitive set

$$
\mathcal{C}_t(\varepsilon)
=
\left\{
R\in\mathcal{R}_t:
P_R
>
f_{\mathrm{best}}^{(t)}+\varepsilon
\right\}.
$$

Regions outside this set cannot improve the certified solution by more than
the selected tolerance.

However, ARRGO does not delete such regions.

Instead, their information and hierarchy remain persistent.

This preserves the no-pruning principle while allowing the active refinement
process to focus on currently relevant regions.

### Exploration and Exploitation

ARRGO does not treat exploration and exploitation as two manually weighted
objectives.

Instead, both emerge from the current information state.

Exploration is represented by unresolved spatial or behavioral information.

Exploitation is represented by optimization relevance, particularly certified
potential when available.

The balance between them is therefore determined by the information currently
available rather than by a fixed exploration coefficient.

Conceptually,

$$
\text{Global Refinement}
=
\text{Information Acquisition}
+
\text{Optimization Relevance}.
$$

The two components are coordinated through region analysis and action
selection.

### Global Objective and Refinement Actions

At every iteration, ARRGO must determine which refinement action contributes
most appropriately to the current global objective.

The available actions are

$$
A_R
=
\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\}.
$$

Sampling is appropriate when new function evaluations are required.

Splitting is appropriate when the current spatial representation is too coarse.

Stable is appropriate when the currently relevant information is sufficiently
resolved for the present decision.

The selected action is therefore a consequence of the unresolved information
state rather than an independent optimization rule.

### Global Objective as a Decision Process

The global optimization process can be represented conceptually as

$$
\boxed{
\text{Global State}
\rightarrow
\text{Region Analysis}
\rightarrow
\text{Priority}
\rightarrow
\text{Refinement Action}
\rightarrow
\text{New Information}
\rightarrow
\text{Updated Global State}
}
$$

Each iteration should therefore improve the representation of the optimization
problem rather than merely increase the number of function evaluations.

### Empirical and Certified Objectives

The exact interpretation of the global objective depends on the operating mode.

In Empirical Mode, ARRGO seeks strong optimization performance using observed
function information, adaptive refinement, and deterministic decision rules.

No finite empirical state is interpreted as a rigorous proof of global
optimality without additional assumptions.

In Certified Mode, valid Lipschitz-based bounds allow ARRGO to formulate a
finite-time optimality certificate through

$$
\Delta_{\mathrm{global}}^{(t)}
\le
\varepsilon.
$$

Thus, the same refinement architecture supports both practical optimization
and mathematically certified optimization, while keeping their claims
separate.

### Global Objective and No-Pruning

The global objective does not permit deleting regions from the persistent
search structure.

A region may become:

- currently low priority,
- temporarily stable,
- certified non-competitive,
- or structurally refined.

Nevertheless, its representation remains part of the global hierarchy.

This distinction is fundamental:

$$
\boxed{
\text{Low Current Priority}
\neq
\text{Removal From the Search Structure}
}
$$

Persistent representation allows global information to remain available if
future updates change regional relevance.

### Global Objective Principle

The central principle of ARRGO is:

$$
\boxed{
\text{Progressively resolve the information that limits the global optimization
decision while preserving complete spatial and informational history.}
}
$$

This objective connects local region refinement to the global convergence
conditions established in the theoretical foundations.

The next section defines how this global objective is used to update the
incumbent solution and maintain the best globally observed value throughout
the execution.

## Global Best Solution and Incumbent Update

ARRGO continuously maintains the best solution observed among all function
evaluations performed up to the current iteration.

This solution is called the **global incumbent**.

The incumbent is an observed solution and must be distinguished from the unknown
true global optimizer.

### Evaluated Point Set

At iteration \(t\), let

$$
D_t
=
\left\{
(x_i,f(x_i))
\right\}_{i=1}^{N_t}
$$

denote the complete persistent evaluation history.

The history is cumulative:

$$
D_t
\subseteq
D_{t+1}.
$$

Previously evaluated points are therefore never removed from the global
information history.

### Global Incumbent

The global incumbent is defined as

$$
x_{\mathrm{best}}^{(t)}
\in
\operatorname*{arg\,max}_{x_i\in D_t}
f(x_i).
$$

Its corresponding objective value is

$$
f_{\mathrm{best}}^{(t)}
=
f\left(x_{\mathrm{best}}^{(t)}\right)
=
\max_{x_i\in D_t}f(x_i).
$$

If multiple evaluated points have the same maximum value, ARRGO may retain one
according to a deterministic tie-breaking rule.

The existence of the incumbent is guaranteed whenever at least one valid
function evaluation has been performed.

### Incumbent Update After a New Evaluation

Suppose ARRGO evaluates a new candidate \(x_c\).

The new objective value is

$$
y_c=f(x_c).
$$

The candidate is compared with the current incumbent.

If

$$
y_c>f_{\mathrm{best}}^{(t)},
$$

then the incumbent is updated:

$$
x_{\mathrm{best}}^{(t+1)}=x_c,
$$

and

$$
f_{\mathrm{best}}^{(t+1)}=y_c.
$$

Otherwise, the incumbent remains unchanged.

Conceptually,

$$
f_{\mathrm{best}}^{(t+1)}
=
\max
\left(
f_{\mathrm{best}}^{(t)},
f(x_c)
\right).
$$

### Monotonicity of the Incumbent

Because ARRGO retains all previous evaluations,

$$
D_t\subseteq D_{t+1}.
$$

Therefore,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

Consequently, the sequence of incumbent objective values is monotone
non-decreasing:

$$
f_{\mathrm{best}}^{(0)}
\le
f_{\mathrm{best}}^{(1)}
\le
f_{\mathrm{best}}^{(2)}
\le
\cdots.
$$

This is an important invariant of ARRGO.

A new evaluation can improve the incumbent or leave it unchanged, but it can
never make the best observed value worse.

### Incumbent and the True Global Optimum

Let

$$
f^*
=
\max_{x\in\Omega}f(x).
$$

Because the incumbent is obtained from evaluated points inside the search
domain,

$$
f_{\mathrm{best}}^{(t)}
\le
f^*.
$$

The inequality may be strict because the true global optimizer may not yet have
been sufficiently explored or evaluated.

Therefore,

$$
x_{\mathrm{best}}^{(t)}
\neq
\text{necessarily a global optimizer}.
$$

The incumbent should always be interpreted as

> the best solution found so far.

It should not be described as the exact global optimum unless a valid
optimality argument establishes that conclusion.

### Incumbent Improvement and Global Search

An improved incumbent changes the global optimization state.

Suppose

$$
f_{\mathrm{best}}^{(t+1)}
>
f_{\mathrm{best}}^{(t)}.
$$

Then regions whose current information was only competitive against the old
incumbent may become less relevant.

In Certified Mode, this is reflected directly through the competitive
condition

$$
P_R
>
f_{\mathrm{best}}^{(t)}+\varepsilon.
$$

After the incumbent improves, the corresponding condition becomes

$$
P_R
>
f_{\mathrm{best}}^{(t+1)}+\varepsilon.
$$

Thus, incumbent improvement automatically updates the global competitive
landscape.

### Incumbent and Certified Global Gap

In Certified Mode, the incumbent is one endpoint of the certified global
optimality gap.

The global potential is

$$
P_{\mathrm{global}}^{(t)}
=
\max_{R\in\mathcal{R}_t}P_R.
$$

The certified gap is

$$
\Delta_{\mathrm{global}}^{(t)}
=
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}.
$$

When the incumbent improves while the global potential remains unchanged,

$$
f_{\mathrm{best}}^{(t+1)}
>
f_{\mathrm{best}}^{(t)}
$$

implies

$$
\Delta_{\mathrm{global}}^{(t+1)}
<
\Delta_{\mathrm{global}}^{(t)}.
$$

More generally, both incumbent improvement and valid tightening of regional
potentials can reduce the certified gap.

### Incumbent and Sampling

Sampling is the primary action that can directly produce a new observed
objective value.

After sampling candidate \(x_c\),

$$
D_{t+1}
=
D_t
\cup
\left\{
(x_c,f(x_c))
\right\}.
$$

The new observation may:

- improve the incumbent,
- leave the incumbent unchanged,
- tighten certified bounds,
- alter behavioral information,
- modify regional priority,
- or change the next refinement decision.

Therefore, one evaluation can affect multiple parts of the global state.

### Incumbent and Splitting

Splitting alone does not evaluate the objective function.

Therefore, if

$$
R
\rightarrow
\{R_L,R_R\}
$$

is performed without a new function evaluation, then the incumbent value does
not change merely because the region hierarchy changed.

Instead,

$$
f_{\mathrm{best}}^{(t+1)}
=
f_{\mathrm{best}}^{(t)}
$$

unless the split operation is accompanied by a new evaluation.

The split changes spatial structure and regional representation rather than
directly producing new objective information.

### Incumbent and Persistent History

The incumbent is derived from the complete evaluation history rather than only
from currently active regions.

Therefore, a previously evaluated point remains eligible to define the
incumbent even if its containing region later becomes:

- STABLE,
- REFINED,
- low priority,
- or otherwise not selected for the next refinement.

This follows directly from the persistent-history principle.

### Duplicate Evaluations

ARRGO should avoid evaluating the same point repeatedly within the numerical
duplicate tolerance.

For candidate \(x_c\), a duplicate exists if there is an evaluated point
\(x_i\) such that

$$
|x_c-x_i|
\le
\tau_x.
$$

Duplicate candidates should normally be rejected before an objective
evaluation is performed.

This prevents unnecessary function evaluations and preserves a meaningful
interpretation of the evaluation history.

### Deterministic Tie Handling

If multiple evaluated points have the same objective value,

$$
f(x_i)=f(x_j)=f_{\mathrm{best}},
$$

the incumbent location is not uniquely determined by the objective value.

ARRGO therefore uses a deterministic tie-breaking rule when a unique
representative is required.

Possible rules include:

- smallest coordinate,
- earliest evaluation,
- or another explicitly fixed deterministic ordering.

The tie-breaker affects representation, not objective quality.

### Incumbent as a Global Reference

The incumbent provides a global reference for several ARRGO decisions.

It is used to determine:

- whether a newly sampled point improves the best known solution,
- whether a certified region remains competitive,
- whether the global certified gap has decreased,
- whether the current information state is sufficient for termination,
- and how the relevance of regions changes over time.

Thus, the incumbent connects local evaluations to global optimization.

### Incumbent Invariant

After every valid evaluation, ARRGO must preserve the invariant

$$
\boxed{
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in D_t}f(x_i)
}
$$

and therefore

$$
\boxed{
f_{\mathrm{best}}^{(t)}
\le
f^*.
}
$$

In Certified Mode, this combines with valid regional potentials to give

$$
f_{\mathrm{best}}^{(t)}
\le
f^*
\le
P_{\mathrm{global}}^{(t)}.
$$

This inequality is one of the central correctness invariants of the algorithm.

### Incumbent Update Principle

The central principle is:

$$
\boxed{
\text{Every new evaluation updates the global incumbent if and only if it
improves the best observed objective value.}
}
$$

The incumbent is persistent, monotone, globally defined, and independent of
which region is currently selected for refinement.

The next section defines how the complete global state is updated after
sampling, splitting, or other changes to the region hierarchy.

## Global State Update

ARRGO is an iterative information-refinement algorithm.

After every refinement action, the global state must be updated consistently so
that all previously established invariants remain valid.

A global update does not simply record the latest action.

It reconstructs the relevant optimization state from the accumulated function
evaluations, persistent region hierarchy, local information, and, when
available, certified bounds.

### Global State Representation

At iteration \(t\), the basic global state is

$$
G_t
=
\left(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t
\right).
$$

In Certified Mode, the state additionally contains

$$
G_t^{\mathrm{cert}}
=
\left(
G_t,
\{L_R,U_R,P_R\}_{R\in\mathcal{R}_t},
P_{\mathrm{global}}^{(t)},
\Delta_{\mathrm{global}}^{(t)}
\right).
$$

The state is updated after every valid refinement operation.

### Persistent Region Hierarchy

The region hierarchy is persistent:

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

A refinement operation may introduce new child regions, but existing regions are
not deleted from the hierarchy.

If

$$
R
\rightarrow
\{R_L,R_R\},
$$

then

$$
\mathcal{R}_{t+1}
=
\mathcal{R}_t
\cup
\{R_L,R_R\}.
$$

The parent \(R\) remains represented in the hierarchy.

This preserves the complete structural history of the optimization process.

### Persistent Evaluation History

The global evaluation history is also persistent.

After a new valid evaluation at \(x_c\),

$$
D_{t+1}
=
D_t
\cup
\left\{
(x_c,f(x_c))
\right\}.
$$

If no new function evaluation occurs, then

$$
D_{t+1}=D_t.
$$

Previously observed function values are never discarded.

This allows the incumbent, regional analysis, and certified bounds to remain
consistent with the complete available information.

### Evaluation Counter

The evaluation counter is defined as

$$
N_t=|D_t|.
$$

When a new non-duplicate evaluation is performed,

$$
N_{t+1}=N_t+1.
$$

For a structural operation such as splitting without evaluation,

$$
N_{t+1}=N_t.
$$

Therefore, structural refinement and function evaluation are tracked
separately.

### Incumbent Update

The incumbent is recomputed or incrementally updated from the persistent
evaluation history:

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in D_t}f(x_i).
$$

The corresponding point is

$$
x_{\mathrm{best}}^{(t)}
\in
\operatorname*{arg\,max}_{x_i\in D_t}f(x_i).
$$

Consequently,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

The incumbent therefore remains globally valid regardless of which region was
refined.

### Region Information Update

After every action affecting a region, ARRGO updates its local information.

The local information state can be represented as

$$
S_R
=
(D_R,B_R,Q_R,U_R,P_R,\mathrm{state}).
$$

The update may modify:

- local samples \(D_R\),
- observed behavior \(B_R\),
- unresolved objectives \(Q_R\),
- certified envelopes \(L_R,U_R\),
- certified potential \(P_R\),
- lifecycle state.

The region must therefore be reanalyzed whenever new information changes its
representation.

### Update After Sampling

Suppose a candidate \(x_c\in R\) is sampled.

The following sequence occurs:

$$
x_c
\rightarrow
f(x_c)
\rightarrow
D_{t+1}
\rightarrow
D_R^{\,new}
\rightarrow
B_R^{\,new}
\rightarrow
Q_R^{\,new}.
$$

In Certified Mode, the new observation also updates the envelopes:

$$
L_R^{\,new}(x)
=
\max
\left(
L_R(x),
f(x_c)-L|x-x_c|
\right),
$$

and

$$
U_R^{\,new}(x)
=
\min
\left(
U_R(x),
f(x_c)+L|x-x_c|
\right).
$$

Therefore,

$$
U_R^{\,new}(x)
\le
U_R(x)
$$

and

$$
L_R^{\,new}(x)
\ge
L_R(x).
$$

The regional potential cannot increase as a consequence of adding a valid
sample:

$$
P_R^{\,new}
\le
P_R.
$$

### Update After Splitting

Suppose

$$
R=[l,r]
$$

is split at \(s\), producing

$$
R_L=[l,s],
\qquad
R_R=[s,r].
$$

The parent remains in the hierarchy.

The children inherit all relevant information associated with observations
inside their domains.

Conceptually,

$$
D_{R_L}
=
\{(x_i,f(x_i))\in D_R:x_i\in R_L\},
$$

and

$$
D_{R_R}
=
\{(x_i,f(x_i))\in D_R:x_i\in R_R\}.
$$

No new function value is created merely by splitting.

Therefore, splitting updates spatial structure and local representation while
leaving the global evaluation history unchanged.

### Reanalysis After Splitting

After a split, both children must be analyzed independently.

For each child \(R_c\), ARRGO recomputes or updates:

$$
S_{R_c}
=
(D_{R_c},B_{R_c},Q_{R_c},U_{R_c},P_{R_c},\mathrm{state}).
$$

This is necessary because the same inherited observations may have different
meaning inside the smaller child regions.

A parent-level behavioral pattern may, for example, become a directional
difference between the two children.

Thus,

$$
\text{Split}
\rightarrow
\text{New Spatial Representation}
\rightarrow
\text{Child Reanalysis}.
$$

### Certified Global Potential Update

In Certified Mode, after local bounds are updated, ARRGO recomputes the global
potential:

$$
P_{\mathrm{global}}^{(t+1)}
=
\max_{R\in\mathcal{R}_{t+1}}P_R.
$$

The certified global gap is then

$$
\Delta_{\mathrm{global}}^{(t+1)}
=
P_{\mathrm{global}}^{(t+1)}
-
f_{\mathrm{best}}^{(t+1)}.
$$

The validity invariant must remain:

$$
f_{\mathrm{best}}^{(t+1)}
\le
f^*
\le
P_{\mathrm{global}}^{(t+1)}.
$$

### Global Priority Recalculation

After updating local and global information, ARRGO recomputes the relevance of
eligible regions.

For each eligible region,

$$
\Pi_R^{(t+1)}
=
\operatorname{Priority}
\left(
S_R^{(t+1)},
G_{t+1}
\right).
$$

This is necessary because one local update can affect the global decision.

For example, an improved incumbent may cause previously competitive regions to
become non-competitive in Certified Mode.

Similarly, splitting one region can change the relative potential and
information resolution of multiple regions.

### State Transition

A region may change lifecycle state after reanalysis.

The conceptual state transitions are

$$
\mathrm{ACTIVE}
\rightarrow
\mathrm{STABLE},
$$

or

$$
\mathrm{ACTIVE}
\rightarrow
\mathrm{REFINED}.
$$

A stable region may later become active again if the global information state
changes.

The hierarchy itself remains persistent.

Therefore, lifecycle state controls current refinement activity rather than
existence in the search structure.

### Global Update Sequence

A complete global update can be represented as

$$
\boxed{
\text{Execute Action}
\rightarrow
\text{Update History}
\rightarrow
\text{Update Regions}
\rightarrow
\text{Update Behavior}
\rightarrow
\text{Update Bounds}
\rightarrow
\text{Update Incumbent}
\rightarrow
\text{Update Global Potential}
\rightarrow
\text{Recompute Priorities}
}
$$

The exact steps executed depend on whether the action was sampling or
splitting.

For example, splitting does not require a new function evaluation, while
sampling does.

### Information Monotonicity

ARRGO must preserve the cumulative nature of available information.

The evaluation history satisfies

$$
D_t
\subseteq
D_{t+1}.
$$

The region hierarchy satisfies

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

The incumbent satisfies

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

In Certified Mode, valid additional samples cannot weaken an existing regional
envelope:

$$
U_R^{\,new}(x)
\le
U_R(x).
$$

These properties form important implementation invariants.

### No-Pruning Invariant

Global state updates must never remove regions merely because they currently
have low relevance.

Therefore,

$$
\boxed{
\text{Global Update}
\neq
\text{Region Deletion}
}
$$

A region can become temporarily irrelevant without disappearing from the
hierarchy.

This preserves information required for analysis, reproducibility, and future
relevance changes.

### Consistency of the Global State

After every update, ARRGO should satisfy the following consistency conditions:

$$
x_{\mathrm{best}}^{(t)}
\in
D_t,
$$

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in D_t}f(x_i),
$$

and, in Certified Mode,

$$
f_{\mathrm{best}}^{(t)}
\le
f^*
\le
P_{\mathrm{global}}^{(t)}.
$$

The region hierarchy must continue to represent the complete search domain,
and every active region must have a valid refinement decision or be explicitly
marked as temporarily stable.

### Global Update Principle

The central principle is:

$$
\boxed{
\text{Every refinement action must produce a consistent new global state
without destroying previously acquired information.}
}
$$

Global state updates therefore connect local refinement operations with the
persistent optimization history and the global convergence mechanism.

The next section defines how information acquired in one region can be used by
other regions without violating the locality of regional analysis.

## Information Sharing Between Regions

ARRGO maintains both local regional information and global information.

Local information describes what has been observed inside a specific region,
while global information coordinates refinement decisions across the entire
search domain.

Information sharing allows regions to benefit from the global state without
violating the locality of their observations.

### Local and Global Information

For a region \(R\), the local information state is represented as

$$
S_R
=
\left(
D_R,
B_R,
Q_R,
U_R,
P_R,
\mathrm{state}
\right).
$$

The global state contains the complete evaluation history and region hierarchy:

$$
G_t
=
\left(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t
\right).
$$

The relationship between regional and global evaluation data is

$$
D_R
=
\left\{
(x_i,f(x_i))\in D_t:
x_i\in R
\right\}.
$$

Therefore, regional information is derived from the global evaluation history
according to spatial membership.

### Global Incumbent Sharing

The global incumbent is available to every region.

For every region \(R\),

$$
f_{\mathrm{best}}^{(t)}
$$

provides a common global reference.

This allows regional decisions to be interpreted relative to the best solution
currently observed anywhere in the domain.

In Certified Mode, the incumbent is also used to determine whether a region
remains globally competitive:

$$
P_R
>
f_{\mathrm{best}}^{(t)}+\varepsilon.
$$

Thus, an improvement discovered in one region can change the relevance of
other regions.

### Global Evaluation History

The complete evaluation history

$$
D_t
$$

is persistent and globally available.

A region can use observations from the global history when those observations
belong to its spatial domain.

However, a regional analysis must not treat observations outside the region as
local observations.

Therefore,

$$
x_i\notin R
\Rightarrow
(x_i,f(x_i))\notin D_R.
$$

This preserves the spatial meaning of regional information.

### Neighboring Information

Regions may also use information about neighboring regions when making global
decisions.

For adjacent regions

$$
R_1=[a,s],
\qquad
R_2=[s,b],
$$

observations near the shared boundary may provide useful contextual information.

Such information can help ARRGO identify:

- changes in observed behavior,
- differences in sampling density,
- possible transitions near region boundaries,
- and differences in certified potential.

However, neighboring observations remain external to the local dataset unless
they actually belong to the region under consideration.

### Boundary Information

Suppose a sample exists at a shared boundary \(s\).

The same point may belong to both child regions under a closed-interval
representation.

To avoid ambiguity, ARRGO must define a consistent boundary ownership or
shared-boundary convention.

For example, the geometric relation may be represented as

$$
R_L=[l,s],
\qquad
R_R=(s,r].
$$

Alternatively, the boundary point may be stored once globally and referenced by
both regions.

The implementation must use one explicit convention consistently.

The purpose is to prevent duplicate evaluations and inconsistent regional
datasets.

### Information Inheritance

When a region is split,

$$
R
\rightarrow
\{R_L,R_R\},
$$

the children inherit relevant information from the parent.

For a child \(R_c\),

$$
D_{R_c}
=
D_R
\cap
R_c.
$$

This means that previously evaluated points remain available after structural
refinement.

Behavioral information may also be inherited as an initial representation,
but it should be recomputed or restricted according to the child geometry.

This distinction is important because a behavioral pattern observed across the
parent may not remain meaningful inside a smaller child.

### Certified Information Sharing

In Certified Mode, a valid Lipschitz constant provides a global structural
constraint:

$$
|f(x)-f(y)|
\le
L|x-y|.
$$

This allows an observation at \(x_i\) to contribute to a valid bound at another
point \(x\), provided the same valid Lipschitz assumption applies.

Therefore,

$$
f(x_i)-L|x-x_i|
\le
f(x)
\le
f(x_i)+L|x-x_i|.
$$

This is a mathematically valid form of information sharing.

The resulting envelopes can be restricted to individual regions.

### Regional Bounds from Shared Observations

For a region \(R\), the certified upper envelope can use observations belonging
to the regional dataset:

$$
U_R(x)
=
\min_{x_i\in D_R}
\left[
f(x_i)+L|x-x_i|
\right].
$$

When a global observation belongs to \(R\), it can therefore contribute directly
to the regional certificate.

Observations outside \(R\) may also provide valid Lipschitz bounds over \(R\) in
principle, but using them requires an explicitly defined global-bound
construction.

ARRGO should therefore distinguish between:

- local regional bounds,
- globally shared certified bounds,
- and purely observational local behavior.

This prevents accidental mixing of different information semantics.

### Information Sharing Is Not Function Prediction

Information sharing must not be interpreted as estimating unobserved function
values without justification.

For example, observing a high value in \(R_1\) does not imply that a neighboring
region \(R_2\) also contains high function values.

Similarly,

$$
f(x_1)>f(x_2)
$$

does not imply a corresponding ordering at unobserved points.

ARRGO uses shared information to improve decision-making, not to invent
unobserved objective values.

### Global Priority Sharing

The global state allows every region to be compared against the same incumbent
and, in Certified Mode, the same global optimization gap.

Therefore, a region's priority depends partly on information outside the
region.

Conceptually,

$$
\Pi_R^{(t)}
=
\operatorname{Priority}
\left(
S_R^{(t)},
G_t
\right).
$$

The local state determines what remains unresolved inside \(R\).

The global state determines how important that unresolved information is to the
overall optimization problem.

### Information Sharing After a New Evaluation

Suppose a new point \(x_c\) is evaluated in region \(R_c\).

The update sequence is

$$
(x_c,f(x_c))
\rightarrow
D_{t+1}
\rightarrow
S_{R_c}^{\,new}
\rightarrow
G_{t+1}.
$$

The updated global incumbent may then affect every region.

For example,

$$
f_{\mathrm{best}}^{(t+1)}
>
f_{\mathrm{best}}^{(t)}
$$

can cause some previously competitive certified regions to become
non-competitive.

Thus, information discovered locally can produce a global state transition.

### Information Sharing After Splitting

When a region is split, the new child structure becomes globally visible:

$$
\mathcal{R}_{t+1}
=
\mathcal{R}_t
\cup
\{R_L,R_R\}.
$$

The children receive inherited observations and are analyzed independently.

Their resulting priorities may differ even though they originate from the same
parent.

This allows the global selection mechanism to distinguish the two spatial
subregions.

### Persistent Information

Information acquired by ARRGO should be persistent unless it is explicitly
invalidated by a numerical or logical consistency rule.

The main persistent components are:

$$
D_t,
\qquad
\mathcal{R}_t,
\qquad
x_{\mathrm{best}}^{(t)},
\qquad
f_{\mathrm{best}}^{(t)}.
$$

Persistence ensures that structural refinement does not erase previously
acquired knowledge.

It also makes the algorithm reproducible because the complete refinement
history remains represented.

### Information Sharing and No-Pruning

Because regions are never deleted, information remains associated with the
region hierarchy even when a region is temporarily low priority.

Therefore,

$$
\boxed{
\text{Low Priority}
\neq
\text{Information Loss}
}
$$

A region that becomes relevant again can be reconsidered using its accumulated
information.

This is particularly important when improvements to the incumbent change the
relative relevance of regions.

### Information Consistency

Information sharing must preserve the distinction between three categories:

1. **Observed information:** directly obtained from function evaluations.
2. **Derived information:** computed from observations, such as secant slopes.
3. **Certified information:** mathematically valid consequences of explicit
   assumptions, such as Lipschitz envelopes.

These categories must not be treated as equivalent.

For example,

$$
\text{Observed Slope}
\neq
\text{Certified Derivative Bound}.
$$

Likewise,

$$
\text{Heuristic Estimate}
\neq
\text{Certified Upper Bound}.
$$

### Global Information Principle

The central principle is:

$$
\boxed{
\text{Share global context and valid structural information across regions,
while preserving the locality and provenance of regional observations.}
}
$$

Information sharing therefore enables coordinated global optimization without
turning local observations into unsupported predictions.

The next section defines the persistent region hierarchy and the exact
parent-child relationship used to organize all structural refinements.

## Region Hierarchy and Parent-Child Relationship

ARRGO represents spatial refinement through a persistent hierarchy of regions.

Each structural split creates child regions while preserving the parent region
in the global hierarchy.

This structure allows ARRGO to maintain the complete history of spatial
refinement and inherited information.

### Region Tree

Let the initial search domain be

$$
R_0=\Omega=[a,b].
$$

The complete collection of regions generated during execution forms a
hierarchical structure

$$
\mathcal{T}
=
\left(
\mathcal{R},
\mathcal{E}
\right),
$$

where \(\mathcal{R}\) is the set of regions and \(\mathcal{E}\) represents
parent-child relationships.

The initial domain is the root of the hierarchy.

### Parent-Child Relationship

Suppose a region

$$
R_p=[l,r]
$$

is split at an interior point \(s\), where

$$
l<s<r.
$$

The resulting children are

$$
R_L=[l,s],
$$

and

$$
R_R=(s,r].
$$

The exact boundary convention is implementation-defined but must be globally
consistent.

The parent-child relationships are then

$$
R_L\rightarrow R_p,
\qquad
R_R\rightarrow R_p.
$$

Equivalently,

$$
R_L\subset R_p,
\qquad
R_R\subset R_p.
$$

### Spatial Coverage

The children must cover the parent region.

Under the chosen boundary convention,

$$
R_p
=
R_L\cup R_R.
$$

Their interiors must not overlap:

$$
\operatorname{int}(R_L)
\cap
\operatorname{int}(R_R)
=
\varnothing.
$$

This guarantees that structural refinement does not leave uncovered portions
of the search domain.

### Contraction of Children

Every valid split must satisfy the contraction condition

$$
\max
\left(
\operatorname{diam}(R_L),
\operatorname{diam}(R_R)
\right)
\le
\rho\operatorname{diam}(R_p),
$$

where

$$
0<\rho<1.
$$

Since

$$
\operatorname{diam}(R_p)=r-l,
$$

the condition becomes

$$
\max
\left(
s-l,
r-s
\right)
\le
\rho(r-l).
$$

This ensures that both children are strictly smaller than their parent.

### Recursive Refinement

A child can later become a parent itself.

For example,

$$
R_0
\rightarrow
\{R_1,R_2\},
$$

and then

$$
R_1
\rightarrow
\{R_3,R_4\}.
$$

The resulting hierarchy is therefore a recursive refinement tree.

For a nested sequence of regions,

$$
R_0
\supset
R_1
\supset
R_2
\supset
\cdots,
$$

the contraction property gives

$$
\operatorname{diam}(R_k)
\le
\rho^k\operatorname{diam}(R_0).
$$

Consequently,

$$
\lim_{k\rightarrow\infty}
\operatorname{diam}(R_k)
=
0.
$$

### Region Identity

Each region must have a persistent identity independent of its current
priority.

Conceptually, a region record contains:

$$
R
=
\left(
\mathrm{Id},
[l,r],
\mathrm{ParentId},
\mathrm{ChildrenIds},
D_R,
B_R,
Q_R,
U_R,
P_R,
\mathrm{state}
\right).
$$

The identifier remains associated with the region throughout execution.

This allows the algorithm to preserve structural history and distinguish
different regions even when their numerical properties become similar.

### Parent Persistence

When a region is split, the parent is not deleted.

If

$$
R_p
\rightarrow
\{R_L,R_R\},
$$

then the hierarchy becomes

$$
\mathcal{R}_{t+1}
=
\mathcal{R}_t
\cup
\{R_L,R_R\}.
$$

The parent \(R_p\) remains part of \(\mathcal{R}_{t+1}\).

Its state may change to indicate that it has been structurally refined, but its
historical representation remains available.

### Parent and Child States

A parent region may transition to

$$
\mathrm{REFINED}
$$

after a successful split.

Its children become candidates for subsequent analysis and refinement.

A child may later become:

$$
\mathrm{ACTIVE},
\qquad
\mathrm{STABLE},
\qquad
\mathrm{REFINED}.
$$

The lifecycle state describes the current role of a region.

It does not determine whether the region exists in the hierarchy.

### Information Inheritance

When a parent is split, observations belonging to each child are inherited.

For child \(R_c\),

$$
D_{R_c}
=
\left\{
(x_i,f(x_i))\in D_{R_p}
:
x_i\in R_c
\right\}.
$$

The child therefore begins with all relevant previously acquired observations.

No previously evaluated function value is lost.

### Behavioral Information Inheritance

Behavioral information can be inherited as an initial reference.

However, it must be reinterpreted or recomputed inside the child.

For example, a slope transition observed across the parent may disappear from one
child after splitting.

Therefore,

$$
B_{R_c}^{\,new}
\neq
B_{R_p}
$$

in general.

The parent behavior remains historical information, while the child behavior
must represent observations in the child geometry.

### Certified Information Inheritance

In Certified Mode, valid Lipschitz-based information remains valid after
subdivision.

If

$$
|f(x)-f(y)|
\le
L|x-y|,
$$

then the same assumption applies to every child region.

Inherited observations can therefore be used to construct valid child
envelopes:

$$
U_{R_c}(x)
=
\min_{x_i\in D_{R_c}}
\left[
f(x_i)+L|x-x_i|
\right].
$$

The resulting bound is valid over the child whenever the Lipschitz assumption
is valid.

### Parent-Child Potential Relationship

In Certified Mode, the parent and children satisfy a consistency relationship.

Since

$$
R_p
=
R_L\cup R_R,
$$

the true regional optimum satisfies

$$
\max_{x\in R_p}f(x)
=
\max
\left(
\max_{x\in R_L}f(x),
\max_{x\in R_R}f(x)
\right).
$$

Because the child potentials are valid upper bounds,

$$
\max_{x\in R_p}f(x)
\le
\max
\left(
P_{R_L},
P_{R_R}
\right).
$$

Therefore, structural refinement preserves the validity of the global
certificate.

### Region Depth

The root region has depth zero.

Each child has depth one greater than its parent:

$$
\mathrm{depth}(R_c)
=
\mathrm{depth}(R_p)+1.
$$

For a nested refinement path,

$$
\mathrm{depth}(R_k)=k.
$$

Depth is structural metadata.

It may be used as a deterministic tie-breaker, but it must not automatically
be interpreted as optimization relevance.

### Region Adjacency

Two regions are adjacent when their boundaries meet without overlapping
interiors.

For example,

$$
R_1=[a,s],
\qquad
R_2=(s,b]
$$

are adjacent at \(s\).

Adjacency may be useful for:

- contextual behavioral comparison,
- detecting spatial transitions,
- evaluating coverage balance,
- and coordinating global refinement.

However, adjacency does not imply similarity of function values.

### Region Containment

Every non-root region has exactly one direct parent.

For a child \(R_c\),

$$
R_c\subset R_p.
$$

A region may have descendants at multiple levels:

$$
R_c
\subset
R_p
\subset
R_{\mathrm{ancestor}}.
$$

This containment relation provides the spatial structure required for recursive
refinement.

### Persistent History

The hierarchy stores the sequence of structural decisions.

A complete refinement history can therefore be reconstructed from:

$$
\mathcal{T}
=
(\mathcal{R},\mathcal{E}).
$$

This makes it possible to determine:

- which regions were split,
- where splits occurred,
- which regions descended from which parents,
- which regions remain stable,
- and how the spatial representation evolved.

### No-Pruning Principle

The region tree is persistent.

Therefore,

$$
\boxed{
\text{Split}
\rightarrow
\text{Add Children}
\neq
\text{Delete Parent}
}
$$

Even when a parent is no longer directly refined, it remains part of the
hierarchical representation.

Likewise, a region with low current priority is not removed from the tree.

This preserves complete structural information.

### Hierarchy and Global Selection

Global selection operates over eligible regions rather than over the entire
tree indiscriminately.

Thus,

$$
\mathcal{E}_t
\subseteq
\mathcal{R}_t.
$$

A region may exist in the hierarchy while not currently being eligible for
refinement.

This separates:

- structural existence,
- lifecycle state,
- current eligibility,
- and global priority.

### Hierarchy Invariant

After every valid split, ARRGO must preserve:

$$
\boxed{
R_c\subset R_p,
\qquad
R_L\cup R_R=R_p,
\qquad
\operatorname{int}(R_L)\cap
\operatorname{int}(R_R)=\varnothing
}
$$

together with the contraction condition

$$
\boxed{
\max
\left(
\operatorname{diam}(R_L),
\operatorname{diam}(R_R)
\right)
\le
\rho\operatorname{diam}(R_p).
}
$$

These properties guarantee that the region hierarchy remains a valid spatial
representation of the original search domain.

### Region Hierarchy Principle

The central principle is:

$$
\boxed{
\text{Every structural refinement adds finer spatial representation while
preserving the complete parent-child history.}
}
$$

The hierarchy therefore provides the persistent structural backbone of ARRGO,
connecting local refinement decisions with global search coverage and
convergence.

The next section defines the complete lifecycle of a region and how its state
changes between ACTIVE, STABLE, and REFINED.

## Region Lifecycle

Each ARRGO region has a lifecycle that describes its current structural and
refinement role.

The lifecycle state is separate from the persistent existence of the region in
the hierarchy.

A region may change its state during execution, but its historical identity and
structural representation remain preserved.

### Region Lifecycle States

ARRGO uses three primary lifecycle states:

$$
\mathrm{ACTIVE},
\qquad
\mathrm{STABLE},
\qquad
\mathrm{REFINED}.
$$

These states describe the current role of a region rather than its
optimization quality.

### ACTIVE State

An ACTIVE region is currently eligible for further analysis and potentially for
a refinement action.

Conceptually,

$$
\mathrm{ACTIVE}
\Rightarrow
\text{Region may receive a refinement opportunity}.
$$

An active region may require:

- additional sampling,
- structural splitting,
- behavioral resolution,
- uncertainty reduction,
- or optimization-potential resolution.

Being ACTIVE does not imply that the region is likely to contain the global
optimizer.

It only means that further refinement is currently considered possible and
relevant.

### STABLE State

A region is STABLE when its currently relevant information is sufficiently
resolved for the present decision state.

Conceptually,

$$
\mathrm{STABLE}
\Rightarrow
\text{No immediate refinement is required under the current criteria}.
$$

Stability may arise because:

- spatial coverage is sufficiently resolved,
- observed behavior is sufficiently represented,
- certified uncertainty is sufficiently small,
- certified potential is no longer competitive,
- or the current combination of objectives does not justify another action.

Stability is therefore a decision state rather than a mathematical statement
that the unknown function is completely known.

### REFINED State

A region becomes REFINED after it has undergone structural subdivision.

Suppose

$$
R_p
\rightarrow
\{R_L,R_R\}.
$$

The parent region may then enter

$$
\mathrm{REFINED}.
$$

This indicates that its spatial representation has been structurally expanded
through child regions.

The parent remains in the hierarchy.

### Lifecycle and Region Existence

Lifecycle state does not determine whether a region exists.

For every region \(R\),

$$
R\in\mathcal{R}_t
$$

remains true after the region changes state.

Therefore,

$$
\mathrm{STABLE}
\neq
\mathrm{DELETED},
$$

and

$$
\mathrm{REFINED}
\neq
\mathrm{DELETED}.
$$

This distinction is fundamental to the no-pruning architecture.

### Initial State

The root region

$$
R_0=\Omega
$$

is initially created as an ACTIVE region.

It can then undergo the standard refinement cycle:

$$
\mathrm{ACTIVE}
\rightarrow
\mathrm{STABLE},
$$

or

$$
\mathrm{ACTIVE}
\rightarrow
\mathrm{REFINED}.
$$

The children created by a split are initially analyzed and may become ACTIVE or
STABLE depending on their resulting information state.

### Activation

A STABLE region may become relevant again when the global information state
changes.

Therefore, ARRGO allows the conceptual transition

$$
\mathrm{STABLE}
\rightarrow
\mathrm{ACTIVE}.
$$

This may occur when:

- the global incumbent improves,
- new global information changes regional relevance,
- previously unresolved neighboring structure becomes important,
- certified competition changes,
- or the region requires additional information under updated criteria.

Thus, STABLE is not necessarily permanent.

### Reanalysis Before Activation

A region should not be activated merely because time has passed.

Its current state should first be re-evaluated using the available global and
regional information.

Conceptually,

$$
\text{Global Update}
\rightarrow
\text{Region Reanalysis}
\rightarrow
\text{Eligibility Decision}.
$$

This prevents unnecessary refinement and keeps lifecycle transitions
information-driven.

### Refinement Transition

When an ACTIVE region is selected for structural refinement,

$$
R
\rightarrow
\{R_L,R_R\}.
$$

The parent transitions to

$$
\mathrm{REFINED},
$$

while the newly created children enter the region analysis process.

The transition therefore represents a change in spatial representation rather
than removal of the parent.

### Sampling and Lifecycle State

Sampling does not automatically change a region to REFINED because sampling
does not modify the structural hierarchy.

Instead, after sampling, the region is reanalyzed.

It may remain ACTIVE if important unresolved information remains:

$$
\mathrm{ACTIVE}
\xrightarrow{\mathrm{Sample}}
\mathrm{ACTIVE}.
$$

It may become STABLE if the relevant objectives are sufficiently resolved:

$$
\mathrm{ACTIVE}
\xrightarrow{\mathrm{Sample}}
\mathrm{STABLE}.
$$

Thus, the state transition depends on the resulting information state.

### Splitting and Lifecycle State

Splitting creates structural refinement.

Therefore,

$$
\mathrm{ACTIVE}
\xrightarrow{\mathrm{Split}}
\mathrm{REFINED}.
$$

The children are then analyzed independently.

Their states may differ:

$$
R_L\rightarrow\mathrm{ACTIVE},
\qquad
R_R\rightarrow\mathrm{STABLE},
$$

or both may remain ACTIVE.

This allows the global algorithm to allocate future refinement opportunities
according to the information state of each child.

### Stability Is Local

A region may be stable even while other regions remain highly unresolved.

Therefore,

$$
\mathrm{Stable}(R_1)
\not\Rightarrow
\mathrm{Stable}(\mathcal{R}_t).
$$

Likewise,

$$
\mathrm{Stable}(R)
\not\Rightarrow
\text{Global Optimality}.
$$

Global termination requires a global criterion, especially in Certified Mode.

### Certified Stability

In Certified Mode, stability can be connected to explicit mathematical
criteria.

For example, a region whose certified potential satisfies

$$
P_R
\le
f_{\mathrm{best}}+\varepsilon
$$

cannot contain a solution that exceeds the incumbent by more than the selected
tolerance.

Such a region may therefore be considered locally stable with respect to the
current certified objective.

However, the region remains represented in the hierarchy.

### Global Relevance Can Change Stability

Suppose a region is currently STABLE because its certified potential is below
the relevant threshold.

A change in the global state may alter its status.

For example, if the incumbent or tolerance changes, the competitive condition
must be reevaluated.

Therefore,

$$
\mathrm{STABLE}
\xrightarrow{\text{Reanalysis}}
\mathrm{ACTIVE}
$$

is permitted whenever the updated information state requires further
refinement.

### Lifecycle and Fairness

The lifecycle mechanism must preserve global selection fairness.

A region that remains persistently relevant cannot be prevented indefinitely
from receiving refinement opportunities simply because its state was
temporarily marked STABLE.

Therefore, the lifecycle system must support:

$$
\text{Persistent Relevance}
\Rightarrow
\text{Eventual Reanalysis and Refinement Opportunity}.
$$

This is particularly important in Certified Mode when multiple regions remain
competitive.

### Lifecycle and History

Every lifecycle transition should be recorded as part of the persistent
algorithmic history.

Conceptually, ARRGO may retain transitions such as

$$
R:
\mathrm{ACTIVE}
\rightarrow
\mathrm{STABLE}
\rightarrow
\mathrm{ACTIVE}
\rightarrow
\mathrm{REFINED}.
$$

This history helps reconstruct the evolution of the search process and
distinguish temporary stability from structural refinement.

### Lifecycle and No-Pruning

The lifecycle model explicitly separates state from existence.

Therefore,

$$
\boxed{
\text{Lifecycle Transition}
\neq
\text{Region Deletion}
}
$$

A region may stop being active without being removed from the hierarchy.

Its information remains available for future analysis.

### Lifecycle Invariant

At every iteration, each region has exactly one current lifecycle state:

$$
\mathrm{state}(R)
\in
\left\{
\mathrm{ACTIVE},
\mathrm{STABLE},
\mathrm{REFINED}
\right\}.
$$

The state must be consistent with the current region representation and global
information.

For a structurally refined parent,

$$
\mathrm{state}(R_p)=\mathrm{REFINED}.
$$

For a region currently requiring additional refinement,

$$
\mathrm{state}(R)=\mathrm{ACTIVE}.
$$

For a region requiring no immediate refinement under the current criteria,

$$
\mathrm{state}(R)=\mathrm{STABLE}.
$$

### Region Lifecycle Principle

The central principle is:

$$
\boxed{
\text{Lifecycle state controls current refinement activity, while the region
hierarchy preserves the complete structural history.}
}
$$

This allows ARRGO to adaptively activate, stabilize, and structurally refine
regions without ever losing previously acquired information.

The next section defines the termination criteria that determine when ARRGO
should stop refinement and return its current solution.

## Termination Criteria

ARRGO must distinguish between stopping the execution and establishing that the
optimization objective has been sufficiently resolved.

Termination is therefore defined according to the available mathematical
information and the operating mode.

ARRGO supports two fundamentally different termination mechanisms:

- **Certified Termination:** stopping because a valid mathematical optimality
  certificate has reached the requested tolerance.
- **Budget-Limited Termination:** stopping because the allowed computational
  budget has been exhausted.

These two conditions must never be interpreted as equivalent.

### Certified Termination

In Certified Mode, ARRGO can terminate when the global certified optimality gap
satisfies

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
\le
\varepsilon.
$$

Because

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}},
$$

we obtain

$$
0
\le
f^*-f_{\mathrm{best}}
\le
\varepsilon.
$$

Therefore, the current incumbent is guaranteed to be
\(\varepsilon\)-optimal in objective value.

### Global Certificate Requirement

The certified stopping condition is valid only when the global potential is a
valid upper bound over the complete search domain.

ARRGO must therefore maintain:

$$
f^*
\le
P_{\mathrm{global}}.
$$

This requires:

- a valid Lipschitz constant,
- valid regional envelopes,
- complete region coverage,
- valid regional potentials,
- and a correctly maintained incumbent.

If any of these conditions fail, the gap cannot be interpreted as a valid
optimality certificate.

### Regional Certificate Interpretation

A region \(R\) is certified non-competitive at tolerance
\(\varepsilon\) when

$$
P_R
\le
f_{\mathrm{best}}+\varepsilon.
$$

Such a region cannot contain a point whose objective value exceeds the current
incumbent by more than \(\varepsilon\).

However, the region remains represented in the persistent hierarchy.

Thus,

$$
\boxed{
\text{Certified Non-Competitive}
\neq
\text{Deleted}
}
$$

The certificate affects refinement relevance, not structural existence.

### Global Certificate Condition

Global certified termination requires that no region can still violate the
requested tolerance.

Equivalently,

$$
\forall R\in\mathcal{R}_t,
\qquad
P_R
\le
f_{\mathrm{best}}+\varepsilon.
$$

This is equivalent to

$$
P_{\mathrm{global}}
\le
f_{\mathrm{best}}+\varepsilon.
$$

Therefore,

$$
\boxed{
\Delta_{\mathrm{global}}\le\varepsilon
}
$$

is the central certified termination criterion.

### Budget-Limited Termination

ARRGO may also terminate when the available evaluation budget is exhausted.

Let

$$
N_t
$$

denote the number of objective evaluations and let

$$
N_{\max}
$$

be the maximum permitted number of evaluations.

The budget condition is

$$
N_t\ge N_{\max}.
$$

When this condition is reached, ARRGO stops acquiring new function evaluations.

The returned solution is then

$$
x_{\mathrm{best}}^{(t)}
$$

with objective value

$$
f_{\mathrm{best}}^{(t)}.
$$

This represents the best solution found within the available budget.

### Budget Exhaustion Is Not Optimality

Budget-limited termination does not establish

$$
f_{\mathrm{best}}=f^*.
$$

In general,

$$
f_{\mathrm{best}}
\le
f^*.
$$

Even in Certified Mode, if the budget expires while

$$
\Delta_{\mathrm{global}}>\varepsilon,
$$

the requested certificate has not been established.

Therefore,

$$
\boxed{
N_t\ge N_{\max}
\not\Rightarrow
\Delta_{\mathrm{global}}\le\varepsilon
}
$$

### Empirical Mode Termination

Empirical Mode does not possess a universal finite-time mathematical
optimality certificate.

It may therefore use practical stopping conditions such as:

- evaluation budget,
- maximum number of refinement iterations,
- sufficiently small region diameters,
- lack of meaningful candidate generation,
- or implementation-specific numerical limits.

These conditions indicate computational stopping rather than proof of global
optimality.

Therefore, empirical termination should be reported explicitly as a
best-found result.

### Multiple Termination Conditions

ARRGO may monitor several stopping conditions simultaneously.

Conceptually,

$$
\mathrm{Terminate}
=
\mathrm{CertifiedTolerance}
\lor
\mathrm{BudgetExhausted}
\lor
\mathrm{NumericalLimit}.
$$

However, the reason for termination must be preserved.

For example:

$$
\mathrm{TerminationReason}
=
\begin{cases}
\mathrm{CertifiedTolerance},
\\
\mathrm{BudgetExhausted},
\\
\mathrm{NumericalLimit}.
\end{cases}
$$

This prevents a budget-limited result from being incorrectly reported as a
certified optimum.

### Evaluation Budget as a Global Constraint

The evaluation budget applies to the entire algorithm rather than to an
individual region.

Therefore,

$$
\sum_R N_R
$$

must be interpreted carefully because regional datasets may share boundary
observations or inherited information.

The authoritative evaluation count is the size of the unique global history:

$$
N_t=|D_t|.
$$

This prevents the same objective evaluation from being counted multiple times
merely because it is relevant to multiple regions.

### Termination After Sampling

After a new sample, ARRGO should update the global state before checking
termination.

The sequence is

$$
\boxed{
\text{Sample}
\rightarrow
\text{Update Information}
\rightarrow
\text{Update Incumbent}
\rightarrow
\text{Update Bounds}
\rightarrow
\text{Compute Global Gap}
\rightarrow
\text{Check Termination}
}
$$

The new evaluation may improve the incumbent and simultaneously tighten
certified bounds.

### Termination After Splitting

Splitting does not necessarily create a new function evaluation.

Therefore, a split may change the regional representation without changing
\(f_{\mathrm{best}}\).

Nevertheless, the regional potentials may need to be recomputed or restricted
to the new child regions.

The termination condition must then be evaluated using the updated global
representation.

Conceptually,

$$
\boxed{
\text{Split}
\rightarrow
\text{Update Hierarchy}
\rightarrow
\text{Reanalyze Children}
\rightarrow
\text{Update Bounds}
\rightarrow
\text{Compute Global Gap}
\rightarrow
\text{Check Termination}
}
$$

### Termination and Stable Regions

A region being STABLE is not itself a global termination condition.

It is possible that

$$
\mathrm{state}(R)=\mathrm{STABLE}
$$

for many regions while another region remains globally competitive.

Therefore,

$$
\boxed{
\text{All Local Decisions Stable}
\neq
\text{Certified Global Termination}
}
$$

Global termination must be determined from the global state.

### Termination and Fairness

The algorithm must not terminate merely because currently selected regions
appear resolved while persistently relevant regions remain insufficiently
refined.

In Certified Mode, this is naturally controlled by the global potential:

$$
P_{\mathrm{global}}
=
\max_R P_R.
$$

Any region capable of violating the requested tolerance keeps the global gap
above the termination threshold.

### Exact Optimality

If exact real-valued arithmetic and exact valid bounds were available, then
taking

$$
\varepsilon=0
$$

would give

$$
P_{\mathrm{global}}
\le
f_{\mathrm{best}}.
$$

Together with

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}},
$$

this implies

$$
f_{\mathrm{best}}
=
f^*
=
P_{\mathrm{global}}.
$$

In practical floating-point computation, exact equality should not be expected
and numerical tolerances must be considered separately.

### Termination and Numerical Tolerance

The optimization tolerance

$$
\varepsilon
$$

must be distinguished from numerical tolerances such as:

$$
\tau_x
$$

for duplicate-point detection and other floating-point consistency
thresholds.

A numerical tolerance is not automatically an optimization guarantee.

Therefore,

$$
\varepsilon_{\mathrm{optimization}}
\neq
\tau_{\mathrm{numerical}}.
$$

The implementation must keep these concepts separate.

### Termination Output

When ARRGO terminates, the output should contain at least:

- best point found,
- best objective value,
- termination reason,
- number of evaluations,
- final region hierarchy,
- and, in Certified Mode, the final certified global gap.

Conceptually,

$$
\mathrm{Result}
=
\left(
x_{\mathrm{best}},
f_{\mathrm{best}},
\mathrm{TerminationReason},
N,
\mathcal{R},
\Delta_{\mathrm{global}}
\right).
$$

The certified gap is included only when a valid certified mode is active.

### Termination Principle

The central principle is:

$$
\boxed{
\text{Stop with a certificate only when the global certified gap satisfies the
requested tolerance; otherwise report the result according to the actual
reason for stopping.}
}
$$

This distinction ensures that ARRGO never confuses computational exhaustion
with mathematical optimality.

The next section establishes the contraction guarantee required for the
spatial refinement process to support global convergence.

## Contraction Guarantee

Spatial contraction is the fundamental structural property that ensures
successive refinements produce progressively smaller regions.

Without contraction, repeated splitting would not necessarily increase spatial
resolution.

ARRGO therefore requires every valid structural split to satisfy an explicit
contraction condition.

### Parent Region

Consider a parent region

$$
R_p=[l,r],
\qquad
l<r.
$$

Its diameter is

$$
\operatorname{diam}(R_p)=r-l.
$$

Suppose the region is split at an interior point

$$
s\in(l,r).
$$

The resulting child regions are

$$
R_L=[l,s],
$$

and

$$
R_R=(s,r].
$$

The boundary convention does not affect the contraction argument as long as
the children cover the parent without overlapping interiors.

### Child Diameters

The diameters of the two children are

$$
\operatorname{diam}(R_L)=s-l,
$$

and

$$
\operatorname{diam}(R_R)=r-s.
$$

The largest child diameter is therefore

$$
d_{\max}
=
\max
\left(
s-l,
r-s
\right).
$$

### Contraction Condition

ARRGO requires the split to satisfy

$$
\boxed{
\max
\left(
s-l,
r-s
\right)
\le
\rho(r-l)
}
$$

for some fixed contraction factor

$$
0<\rho<1.
$$

This guarantees that every child region is strictly smaller than its parent.

In particular,

$$
\operatorname{diam}(R_c)
\le
\rho\operatorname{diam}(R_p)
<
\operatorname{diam}(R_p).
$$

### Why the Condition Is Necessary

A split alone does not mathematically guarantee useful spatial refinement.

For example, repeatedly creating highly unbalanced regions without controlling
their diameter could leave one branch of the hierarchy insufficiently refined.

The contraction condition prevents this by requiring every child to shrink by a
uniform factor.

Therefore,

$$
\text{Valid Split}
\Rightarrow
\text{Guaranteed Spatial Contraction}.
$$

### Repeated Contraction

Consider a nested sequence of regions

$$
R_0
\supset
R_1
\supset
R_2
\supset
\cdots
$$

generated by repeated valid refinement.

Applying the contraction condition recursively gives

$$
\operatorname{diam}(R_1)
\le
\rho\operatorname{diam}(R_0),
$$

then

$$
\operatorname{diam}(R_2)
\le
\rho\operatorname{diam}(R_1)
\le
\rho^2\operatorname{diam}(R_0).
$$

By induction,

$$
\boxed{
\operatorname{diam}(R_k)
\le
\rho^k\operatorname{diam}(R_0)
}
$$

for every refinement depth \(k\).

Since

$$
0<\rho<1,
$$

we have

$$
\lim_{k\rightarrow\infty}\rho^k=0.
$$

Consequently,

$$
\boxed{
\lim_{k\rightarrow\infty}
\operatorname{diam}(R_k)
=
0
}
$$

along every infinitely refined path.

### Spatial Resolution

The contraction property means that repeated structural refinement can
eventually localize a region to an arbitrarily small spatial scale.

For every

$$
\delta>0,
$$

there exists a sufficiently large \(k\) such that

$$
\operatorname{diam}(R_k)<\delta.
$$

Therefore, contraction provides the spatial-resolution mechanism required by
ARRGO.

### Contraction and Continuity

Suppose \(f\) is continuous on the search domain.

As the diameter of a nested region tends to zero, points inside that region
become arbitrarily close to one another.

Continuity therefore implies that sufficiently small spatial regions exhibit
correspondingly small function-value variation in a local sense.

Conceptually,

$$
\operatorname{diam}(R_k)\rightarrow0
\quad\text{and}\quad
f\in C(\Omega)
$$

imply increasingly localized objective behavior.

However, continuity alone does not provide a numerical rate for that
variation.

Thus,

$$
\text{Contraction + Continuity}
\neq
\text{Explicit Numerical Error Bound}.
$$

### Contraction Under a Lipschitz Assumption

If \(f\) additionally satisfies a valid Lipschitz condition

$$
|f(x)-f(y)|
\le
L|x-y|,
$$

then every region satisfies

$$
\max_{x,y\in R}|f(x)-f(y)|
\le
L\operatorname{diam}(R).
$$

For a nested refinement sequence,

$$
\max_{x,y\in R_k}|f(x)-f(y)|
\le
L\rho^k\operatorname{diam}(R_0).
$$

Therefore,

$$
\boxed{
L\rho^k\operatorname{diam}(R_0)
\rightarrow
0
}
$$

as

$$
k\rightarrow\infty.
$$

This provides an explicit relationship between spatial contraction and
possible function-value variation.

### Contraction Does Not Prove Optimality

A crucial distinction is that contraction only controls spatial resolution.

It does not determine which region contains the global optimizer.

Therefore,

$$
\boxed{
\text{Spatial Contraction}
\neq
\text{Global Optimality}
}
$$

A theoretically perfect sequence of shrinking regions could still converge to
a non-optimal location if global selection repeatedly ignored the region
containing a global optimizer.

This is why contraction must be combined with global coverage and fair
selection.

### Contraction and Global Selection

ARRGO requires the global selection mechanism to provide refinement
opportunities to persistently relevant regions.

Conceptually,

$$
\text{Fair Global Selection}
+
\text{Contraction}
$$

ensures that relevant regions can receive progressively finer spatial
resolution.

The convergence mechanism therefore depends on both:

$$
\boxed{
\text{Spatial Contraction}
+
\text{Global Selection Fairness}
}
$$

rather than either property alone.

### Contraction and Information Acquisition

Splitting reduces region diameter but does not automatically create a new
function evaluation.

Therefore,

$$
\text{Split}
\rightarrow
\text{Spatial Resolution},
$$

while

$$
\text{Sample}
\rightarrow
\text{Function Information}.
$$

ARRGO requires both mechanisms because spatial resolution and information
resolution are different properties.

A region can become geometrically small while still having insufficient
observational information.

Thus,

$$
\boxed{
\text{Small Region}
\neq
\text{Sufficient Information}
}
$$

without additional assumptions or information.

### Contraction and Evaluation Density

For global objective-value convergence, spatial contraction must be connected
to actual function evaluations.

In particular, ARRGO requires that relevant regions eventually receive
evaluations sufficiently close to global maximizers.

For a global maximizer \(x^*\), the desired condition is

$$
\inf_{x\in D_\infty}|x-x^*|=0,
$$

where \(D_\infty\) denotes the set of all evaluated points over the complete
execution.

This is the evaluation-density condition established in the theoretical
foundations.

Contraction makes increasingly precise localization possible, while evaluation
density ensures that this localization is actually accompanied by objective
information.

### Contraction and Certified Termination

In Certified Mode, contraction contributes to reducing the spatial scale of
remaining uncertainty.

Under a valid Lipschitz constant,

$$
\operatorname{osc}_R(f)
\le
L\operatorname{diam}(R).
$$

As relevant regions contract,

$$
\operatorname{diam}(R)\rightarrow0,
$$

and therefore

$$
\operatorname{osc}_R(f)\rightarrow0.
$$

This supports the eventual tightening of certified reasoning, but contraction
alone does not guarantee that the global certified gap reaches zero.

Certified convergence additionally requires valid bounds, sufficient
information acquisition, global coverage, and fair selection.

### Uniform Refinement Is Not Required

ARRGO does not require every region to be refined at the same rate.

Different regions may have different information states.

Therefore, refinement depth may vary:

$$
\operatorname{depth}(R_i)
\neq
\operatorname{depth}(R_j).
$$

The contraction requirement applies to each individual valid split, not to a
uniform global refinement schedule.

This enables adaptive allocation of computational effort.

### Contraction and No-Pruning

Contraction changes the spatial representation but does not delete previous
regions.

When

$$
R_p
\rightarrow
\{R_L,R_R\},
$$

the parent remains in the hierarchy.

Therefore,

$$
\boxed{
\text{Contraction}
\neq
\text{Pruning}
}
$$

Spatial refinement and structural persistence are independent principles.

### Contraction Invariant

Every executed split must satisfy

$$
\boxed{
0<\rho<1
}
$$

and

$$
\boxed{
\max
\left(
\operatorname{diam}(R_L),
\operatorname{diam}(R_R)
\right)
\le
\rho\operatorname{diam}(R_p).
}
$$

This is a structural invariant of ARRGO.

If a candidate split does not satisfy the condition, it must not be executed as a
valid ARRGO split.

### Contraction Principle

The central principle is:

$$
\boxed{
\text{Every valid structural refinement must reduce the diameter of every
resulting child by a uniform contraction factor.}
}
$$

Contraction provides the spatial mechanism that allows ARRGO to progressively
resolve the search domain.

It becomes meaningful for global convergence only when combined with
information acquisition, evaluation density, global coverage, and fair
selection.

The next section combines these ingredients into the complete global
convergence mechanism of ARRGO.

## Global Convergence Mechanism

The convergence of ARRGO is not caused by a single mechanism.

It results from the interaction of several structural and informational
properties established throughout the theoretical foundations and algorithm
design.

The central convergence mechanism can be summarized as

$$
\boxed{
\text{Global Coverage}
+
\text{Fair Selection}
+
\text{Spatial Contraction}
+
\text{Information Refinement}
+
\text{Evaluation Density}
}
$$

Under the assumptions established previously, these components provide the
mechanism through which ARRGO can progressively resolve the global optimization
problem.

### Global Coverage

Let the search domain be

$$
\Omega=[a,b].
$$

ARRGO maintains a persistent region hierarchy whose regions collectively
represent the search domain.

Structural refinement must preserve the coverage invariant.

For every valid split,

$$
R_p=R_L\cup R_R,
$$

with non-overlapping interiors.

Therefore, structural refinement changes the resolution of the representation
without removing parts of the original search domain.

This property is essential because the global optimizer may lie anywhere in
\(\Omega\).

Consequently,

$$
\boxed{
\text{Refinement}
\neq
\text{Removal of Search Space}
}
$$

ARRGO refines the representation of the domain rather than permanently
discarding regions.

### Fair Global Selection

Coverage alone is not sufficient.

A region containing a global optimizer could remain unresolved forever if the
global selection mechanism permanently ignored it.

ARRGO therefore requires a fairness condition:

$$
\boxed{
\text{Persistently Relevant Region}
\Rightarrow
\text{Eventual Refinement Opportunity}
}
$$

This does not require uniform refinement.

Different regions may receive different numbers of refinement operations.

The requirement is only that a persistently relevant region cannot be
permanently starved of refinement opportunities.

### Spatial Contraction

Whenever a region is structurally refined, the contraction condition requires

$$
\max
\left(
\operatorname{diam}(R_L),
\operatorname{diam}(R_R)
\right)
\le
\rho\operatorname{diam}(R_p),
\qquad
0<\rho<1.
$$

Therefore, along any nested sequence of regions receiving repeated valid
refinements,

$$
\operatorname{diam}(R_k)
\le
\rho^k\operatorname{diam}(R_0).
$$

Hence,

$$
\lim_{k\rightarrow\infty}
\operatorname{diam}(R_k)=0.
$$

Spatial contraction provides the localization mechanism of ARRGO.

### Information Refinement

Spatial localization alone is not enough.

ARRGO also requires the information state of relevant regions to become
progressively more informative.

Let the regional information state be

$$
S_R=(D_R,B_R,Q_R),
$$

where:

- \(D_R\) represents observed function evaluations,
- \(B_R\) represents observed local behavior,
- \(Q_R\) represents unresolved optimization-relevant information.

Refinement should reduce unresolved information whenever further information is
necessary for the optimization decision.

This can occur through two different mechanisms:

$$
\text{Sampling}
\rightarrow
\text{New Function Information},
$$

and

$$
\text{Splitting}
\rightarrow
\text{Finer Spatial Representation}.
$$

The two mechanisms complement each other but are not interchangeable.

### Evaluation Density Near Global Maximizers

Let

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x)
$$

denote the set of global maximizers.

For objective-value convergence, ARRGO requires that evaluated points become
arbitrarily close to the relevant global maximizers.

Let \(D_\infty\) denote the complete set of evaluated points.

The evaluation-density condition is

$$
\boxed{
\forall x^*\in X^*,
\qquad
\inf_{x\in D_\infty}|x-x^*|=0.
}
$$

Equivalently, for every

$$
\delta>0,
$$

there exists an evaluated point \(x\in D_\infty\) such that

$$
|x-x^*|<\delta.
$$

This condition connects spatial refinement to actual objective evaluations.

### From Evaluation Density to Objective-Value Convergence

Assume \(f\) is continuous.

Consider a sequence of evaluated points

$$
x_k\rightarrow x^*,
$$

where \(x^*\in X^*\).

By continuity,

$$
f(x_k)\rightarrow f(x^*).
$$

Since

$$
f(x^*)=f^*,
$$

we obtain

$$
f(x_k)\rightarrow f^*.
$$

The ARRGO incumbent is defined as

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in D_t}f(x_i).
$$

Because evaluated points are retained,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

Thus, once evaluations approach a global maximizer sufficiently closely, the
incumbent approaches the global optimal value.

Therefore,

$$
\boxed{
f_{\mathrm{best}}^{(t)}
\rightarrow
f^*
}
$$

under the stated convergence conditions.

### Why Continuity Matters

The evaluation-density condition alone is not enough.

If evaluated points approach \(x^*\) but the objective function is not continuous,
function values need not approach \(f(x^*)\).

Continuity provides the connection

$$
x_k\rightarrow x^*
\quad\Rightarrow\quad
f(x_k)\rightarrow f(x^*).
$$

Therefore, continuity is the bridge between spatial localization and
objective-value convergence.

### Combined Convergence Chain

The complete conceptual chain is

$$
\boxed{
\text{Global Coverage}
\rightarrow
\text{Fair Selection}
\rightarrow
\text{Relevant Refinement}
\rightarrow
\text{Spatial Contraction}
\rightarrow
\text{Evaluation Density}
\rightarrow
\text{Continuity}
\rightarrow
f_{\mathrm{best}}\rightarrow f^*
}
$$

The arrows represent logical dependence rather than a claim that every step
automatically produces the next one.

In particular, the actual ARRGO candidate-generation and selection mechanisms
must be designed so that the required evaluation-density condition is achieved.

### Certified Convergence Extension

In Certified Mode, ARRGO additionally assumes a valid Lipschitz constant \(L\).

For every region,

$$
|f(x)-f(y)|
\le
L|x-y|.
$$

Therefore,

$$
\operatorname{osc}_R(f)
\le
L\operatorname{diam}(R).
$$

For a contracting sequence,

$$
\operatorname{diam}(R_k)
\le
\rho^k\operatorname{diam}(R_0),
$$

and consequently,

$$
\operatorname{osc}_{R_k}(f)
\le
L\rho^k\operatorname{diam}(R_0)
\rightarrow0.
$$

This gives an explicit spatial contribution to the tightening of certified
regional bounds.

### Certified Global Gap

Recall the certified global potential

$$
P_{\mathrm{global}}
=
\max_R P(R),
$$

and the certified gap

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

Valid bounds give

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}}.
$$

Therefore,

$$
0
\le
f^*-f_{\mathrm{best}}
\le
\Delta_{\mathrm{global}}.
$$

If the combined refinement process causes

$$
\Delta_{\mathrm{global}}
\rightarrow0,
$$

then

$$
f_{\mathrm{best}}
\rightarrow
f^*.
$$

Moreover, whenever

$$
\Delta_{\mathrm{global}}
\le
\epsilon,
$$

ARRGO has a finite-time certificate that

$$
f_{\mathrm{best}}
\ge
f^*-\epsilon.
$$

### Role of Sampling

Sampling is responsible for acquiring new objective information.

A new evaluation may:

- improve the incumbent,
- reveal previously unresolved behavior,
- reduce certified envelope uncertainty,
- reduce certified regional potential,
- change the dominant unresolved objective,
- change global region relevance.

Therefore, sampling contributes to convergence by improving the informational
state of the optimization process.

### Role of Splitting

Splitting provides finer spatial organization.

A split does not create a new function evaluation.

Instead, it transforms

$$
R
\rightarrow
\{R_L,R_R\},
$$

allowing ARRGO to distinguish information that was previously represented by a
single region.

Repeated splitting provides the contraction mechanism required for spatial
localization.

### Interaction Between Sampling and Splitting

The two operations form a complementary refinement cycle:

$$
\boxed{
\text{Analyze}
\rightarrow
\text{Sample or Split}
\rightarrow
\text{Update Information}
\rightarrow
\text{Reanalyze}
}
$$

Sampling may reveal a behavioral transition that motivates a split.

Splitting may create smaller regions in which additional sampling becomes more
informative.

Thus, the two operations can influence one another without being treated as
the same operation.

### No-Pruning and Convergence

ARRGO does not require pruning to obtain convergence.

All regions remain represented in the persistent hierarchy.

A region may become:

$$
\text{STABLE}
$$

or

$$
\text{REFINED},
$$

but its historical representation is not deleted.

This preserves the global search history and allows region relevance to be
reconsidered when new information changes the global state.

### What the Mechanism Does Not Claim

The convergence mechanism does not claim that every finite execution finds the
exact global optimizer.

In particular,

$$
\text{Finite Evaluation Budget}
\not\Rightarrow
\text{Exact Global Optimality}.
$$

Likewise,

$$
\text{Contraction Alone}
\not\Rightarrow
\text{Global Convergence},
$$

and

$$
\text{Continuity Alone}
\not\Rightarrow
\text{Finite-Time Certificate}.
$$

The strongest finite-time statement available in ARRGO requires valid
certified bounds and a sufficiently small global certified gap.

### Convergence Levels

ARRGO distinguishes three levels of convergence claims.

#### Empirical Level

The algorithm may perform well numerically and produce a high-quality solution
on tested benchmark functions.

This is an experimental observation, not a theorem.

#### Asymptotic Level

Under the stated convergence assumptions,

$$
f_{\mathrm{best}}^{(t)}
\rightarrow
f^*.
$$

This concerns the limiting objective value as the number of refinement
operations and evaluations grows.

#### Certified Finite-Time Level

Under a valid Lipschitz assumption and valid regional bounds,

$$
\Delta_{\mathrm{global}}
\le
\epsilon
$$

provides the finite-time guarantee

$$
f_{\mathrm{best}}
\ge
f^*-\epsilon.
$$

These three levels must not be conflated.

### Global Convergence Principle

The central convergence principle of ARRGO is:

$$
\boxed{
\text{Preserve Global Coverage}
+
\text{Refine Relevant Regions Fairly}
+
\text{Contract Spatially}
+
\text{Acquire Informative Evaluations}
}
$$

Together with continuity, these properties provide the mechanism for
objective-value convergence.

In Certified Mode, valid Lipschitz bounds additionally convert the refinement
process into a quantitative uncertainty and optimality-certification
mechanism.

The next section formalizes the exact computation of certified uncertainty
within each region.

## Exact Uncertainty Computation

Certified Mode requires an explicit and mathematically valid representation of
the uncertainty that remains inside each region.

The uncertainty used by ARRGO is not probabilistic.

It is derived directly from the observed function evaluations and a valid
Lipschitz constant.

Therefore, the uncertainty computation provides a deterministic enclosure of
the unknown objective values.

### Regional Data

Consider a region

$$
R=[l,r].
$$

Suppose ARRGO has evaluated the objective at points

$$
D_R=
\left\{
(x_i,f(x_i))
\right\}_{i=1}^{n},
$$

where

$$
x_i\in R.
$$

Assume the objective satisfies the valid Lipschitz condition

$$
|f(x)-f(y)|
\le
L|x-y|
$$

for every

$$
x,y\in R.
$$

The constant \(L\) must be a valid upper bound.

An empirically estimated value of \(L\) is not automatically sufficient for
certification.

### Pointwise Lower Bounds

For every observed point \(x_i\), the Lipschitz condition gives

$$
f(x)
\ge
f(x_i)-L|x-x_i|.
$$

Thus, every observation provides a valid lower bound over the entire region.

Taking the strongest available lower bound gives

$$
\boxed{
L_R(x)
=
\max_{1\le i\le n}
\left[
f(x_i)-L|x-x_i|
\right]
}
$$

for every

$$
x\in R.
$$

The notation \(L_R(x)\) denotes a lower envelope and should not be confused
with the Lipschitz constant \(L\).

### Pointwise Upper Bounds

Similarly, the Lipschitz condition gives

$$
f(x)
\le
f(x_i)+L|x-x_i|.
$$

Therefore, every observed point provides a valid upper bound.

Taking the strongest upper bound gives

$$
\boxed{
U_R(x)
=
\min_{1\le i\le n}
\left[
f(x_i)+L|x-x_i|
\right]
}
$$

for every

$$
x\in R.
$$

### Certified Enclosure

Combining the lower and upper envelopes gives

$$
\boxed{
L_R(x)
\le
f(x)
\le
U_R(x)
}
$$

for every

$$
x\in R.
$$

Therefore, the true objective value at every point in the region is enclosed
by a deterministic interval

$$
f(x)\in
[L_R(x),U_R(x)].
$$

This enclosure is valid regardless of the unknown behavior of \(f\) between
the sampled points, provided the Lipschitz assumption is valid.

### Pointwise Certified Uncertainty

The remaining uncertainty at a point \(x\) is defined as the width of the
certified enclosure:

$$
\boxed{
u_R(x)
=
U_R(x)-L_R(x)
}
$$

Since

$$
L_R(x)\le U_R(x),
$$

we have

$$
u_R(x)\ge0.
$$

A smaller value means that the available observations provide a tighter
certified range for the objective value at that point.

This quantity is a deterministic uncertainty width rather than a probability,
variance, or confidence interval.

### Regional Worst-Case Uncertainty

ARRGO can summarize the uncertainty of the entire region using its maximum
pointwise uncertainty:

$$
\boxed{
u_{\max}(R)
=
\max_{x\in R}
u_R(x)
}
$$

This quantity represents the largest remaining certified uncertainty width
inside the region.

A region with a large \(u_{\max}(R)\) contains locations whose objective values
remain weakly constrained by the current observations.

### Regional Average Uncertainty

A secondary descriptive quantity may also be defined as

$$
\bar{u}(R)
=
\frac{1}{r-l}
\int_l^r u_R(x)\,dx.
$$

This quantity describes the average certified enclosure width across the
region.

However, \(u_{\max}(R)\) is more directly relevant to worst-case certification
because it does not allow a small highly uncertain area to be hidden by a large
well-resolved area.

ARRGO should therefore retain the complete uncertainty profile rather than
reducing the region to a single scalar whenever more detailed information is
useful.

### Effect of Adding a New Evaluation

Suppose ARRGO evaluates a new point

$$
x_c\in R
$$

and obtains

$$
y_c=f(x_c).
$$

The new lower envelope becomes

$$
L_R^{\mathrm{new}}(x)
=
\max
\left\{
L_R(x),
y_c-L|x-x_c|
\right\}.
$$

Therefore,

$$
L_R^{\mathrm{new}}(x)
\ge
L_R(x).
$$

Similarly, the new upper envelope becomes

$$
U_R^{\mathrm{new}}(x)
=
\min
\left\{
U_R(x),
y_c+L|x-x_c|
\right\}.
$$

Therefore,

$$
U_R^{\mathrm{new}}(x)
\le
U_R(x).
$$

Consequently, the certified enclosure can only become tighter:

$$
\boxed{
u_R^{\mathrm{new}}(x)
\le
u_R(x)
}
$$

for every

$$
x\in R.
$$

Thus, additional valid observations cannot make the certified pointwise
uncertainty larger.

### Effect on Regional Uncertainty

Because the pointwise uncertainty cannot increase,

$$
u_{\max}^{\mathrm{new}}(R)
\le
u_{\max}(R).
$$

Therefore, valid additional evaluations produce a monotone tightening of the
regional certified uncertainty.

This property is one of the key invariants of Certified ARRGO.

### Uncertainty and Sampling

The uncertainty profile directly informs the sampling mechanism.

If a region contains locations with large

$$
u_R(x),
$$

those locations may represent valuable candidates for additional evaluation.

However, high uncertainty alone does not imply that the region contains a
global optimizer.

Therefore, ARRGO should consider uncertainty together with optimization
relevance.

In particular,

$$
\boxed{
\text{High Uncertainty}
\neq
\text{High Optimization Potential}
}
$$

A region may be highly uncertain while its known objective values are already
far below the global incumbent.

### Uncertainty and Optimization Potential

The certified upper envelope defines the regional optimization potential:

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

The potential answers a different question from uncertainty.

Uncertainty asks:

> How wide is the remaining certified range?

Potential asks:

> How large could the objective value still be within this region?

Thus,

$$
\boxed{
\text{Uncertainty}
\neq
\text{Potential}
}
$$

Both quantities are retained because they support different refinement
decisions.

### Uncertainty After Splitting

Suppose

$$
R_p
\rightarrow
\{R_L,R_R\}.
$$

The children inherit all observations that lie inside their respective
domains.

Their uncertainty profiles are then computed independently:

$$
u_{R_L}(x)
=
U_{R_L}(x)-L_{R_L}(x),
$$

and

$$
u_{R_R}(x)
=
U_{R_R}(x)-L_{R_R}(x).
$$

Splitting itself does not generate a new objective evaluation.

Therefore, it does not automatically provide new empirical function
information.

Its role is to reorganize the existing information into smaller spatial
regions.

### Effect of Spatial Contraction

Under the valid Lipschitz assumption,

$$
|f(x)-f(y)|
\le
L\operatorname{diam}(R)
$$

for all

$$
x,y\in R.
$$

Therefore, as a region contracts,

$$
\operatorname{diam}(R)\rightarrow0,
$$

the maximum possible variation of the objective inside that region also tends
to zero:

$$
L\operatorname{diam}(R)\rightarrow0.
$$

This does not mean that the exact envelope width automatically decreases at
the same rate for every sampling configuration.

It means that the structural range of possible objective variation becomes
increasingly constrained.

### Sparse Sampling

The uncertainty computation remains conceptually valid with sparse
observations.

However, sparse sampling can produce a wide certified enclosure.

In particular:

- zero observations do not provide a function-value anchor;
- one observation provides only one Lipschitz cone;
- multiple observations provide intersecting upper and lower constraints;
- additional spatially distributed observations can tighten the enclosure.

Therefore,

$$
\text{Few Samples}
\rightarrow
\text{Potentially Large Certified Uncertainty}.
$$

Sparse data should not be interpreted as evidence that the objective is
constant or simple.

### Duplicate Evaluations

ARRGO avoids redundant evaluations.

A candidate point \(x_c\) is considered a duplicate of an existing observation
\(x_i\) when

$$
|x_c-x_i|
\le
\tau_x,
$$

where \(\tau_x\) is the numerical point tolerance.

Such candidates should not be treated as new information.

This prevents artificial inflation of the evaluation count and avoids
recomputing identical information.

### Boundary Points

The region boundaries

$$
l
\quad\text{and}\quad
r
$$

are valid points of the closed optimization domain.

If a boundary has not been evaluated and boundary information is relevant to the
current refinement decision, ARRGO may generate it as a sampling candidate.

Boundary handling must remain consistent with the global duplicate tolerance.

### Numerical Computation of the Envelope

The mathematical definitions

$$
L_R(x)
=
\max_i
\left[
f(x_i)-L|x-x_i|
\right]
$$

and

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right]
$$

define piecewise-linear envelopes in one dimension.

This structure allows their extrema and intersections to be analyzed
numerically without assuming a parametric model for \(f\).

The implementation may evaluate these envelopes at candidate locations or
derive their relevant breakpoints analytically.

The mathematical definition remains authoritative regardless of the numerical
strategy used to compute it.

### Certified Versus Estimated Uncertainty

The distinction between certified and empirical quantities is essential.

When \(L\) is valid,

$$
u_R(x)
=
U_R(x)-L_R(x)
$$

is a certified uncertainty width.

If \(L\) is estimated from observations without a valid guarantee, the same
calculation may still be useful as a heuristic diagnostic, but it must not be
called a certificate.

Therefore,

$$
\boxed{
\text{Estimated Lipschitz Constant}
\neq
\text{Certified Lipschitz Bound}
}
$$

unless the validity of the bound has been independently established.

### Uncertainty Computation Principle

The central principle is:

$$
\boxed{
\text{Use Valid Lipschitz Constraints and All Available Evaluations to
Construct the Tightest Deterministic Enclosure Supported by the Current
Information.}
}
$$

The resulting uncertainty profile provides ARRGO with a mathematically
interpretable measure of what remains unresolved inside each region.

The next section uses the same certified envelopes to compute the exact
optimization potential of each region.

## Exact Optimization Potential

In Certified Mode, ARRGO requires a mathematically valid measure of how
competitive a region can still be with respect to the global optimization
objective.

This quantity must not be interpreted as a prediction of the unknown
objective.

Instead, it represents a certified upper limit on the objective value that
could still occur inside the region.

### Regional Upper Envelope

Consider a region

$$
R=[l,r]
$$

with observed data

$$
D_R=
\left\{
(x_i,f(x_i))
\right\}_{i=1}^{n}.
$$

Assume that the objective satisfies a valid Lipschitz condition

$$
|f(x)-f(y)|
\le
L|x-y|.
$$

The corresponding certified upper envelope is

$$
U_R(x)
=
\min_{1\le i\le n}
\left[
f(x_i)+L|x-x_i|
\right].
$$

For every point inside the region,

$$
f(x)\le U_R(x).
$$

Therefore, the upper envelope provides a valid pointwise upper bound on the
unknown objective.

### Definition of Regional Optimization Potential

The optimization potential of region \(R\) is defined as

$$
\boxed{
P(R)
=
\max_{x\in R}U_R(x)
}
$$

This quantity represents the largest objective value that remains compatible
with the current certified information inside the region.

It is therefore an **upper potential**, not a predicted objective value.

### Fundamental Upper-Bound Property

Since

$$
f(x)\le U_R(x)
$$

for every

$$
x\in R,
$$

we have

$$
\max_{x\in R}f(x)
\le
\max_{x\in R}U_R(x).
$$

Therefore,

$$
\boxed{
\max_{x\in R}f(x)
\le
P(R)
}
$$

This is the fundamental validity property of regional optimization
potential.

### Interpretation

The regional potential answers the following question:

> Based on all currently available certified information, how large could the
> objective value still be somewhere inside this region?

It does not answer:

> Where is the optimizer?

and it does not answer:

> What value will the objective actually attain?

Therefore,

$$
\boxed{
P(R)
\neq
\text{Prediction of the Regional Optimum}
}
$$

It is instead a certified upper limit.

### Relationship to the Regional Uncertainty

The optimization potential and uncertainty describe different properties.

Pointwise uncertainty is

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

Regional uncertainty summarizes how weakly constrained the objective remains.

Regional potential is

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

Potential measures the highest remaining competitive possibility.

Therefore,

$$
\boxed{
\text{Uncertainty}
\neq
\text{Optimization Potential}
}
$$

A region can have high uncertainty but low potential, or relatively low
uncertainty but high potential.

### Relationship to the Global Incumbent

Let the current global incumbent be

$$
f_{\mathrm{best}}
=
\max_{x_i\in D_t}f(x_i).
$$

A region is potentially competitive when

$$
P(R)>f_{\mathrm{best}}.
$$

In tolerance-based Certified Mode, define the competitive condition as

$$
\boxed{
P(R)>f_{\mathrm{best}}+\epsilon
}
$$

where \(\epsilon\) is the requested objective-value tolerance.

A region satisfying this condition still has certified potential to improve
the incumbent by more than the requested tolerance.

### Non-Competitive Regions

If

$$
P(R)
\le
f_{\mathrm{best}}+\epsilon,
$$

then every point in the region satisfies

$$
f(x)
\le
f_{\mathrm{best}}+\epsilon.
$$

Therefore, the region cannot contain a point whose objective value exceeds
the incumbent by more than \(\epsilon\).

This makes the region non-competitive with respect to the requested certified
tolerance.

Importantly, ARRGO does not delete such a region.

It remains in the persistent hierarchy and may be re-evaluated if global
information or the optimization objective changes.

### Global Optimization Potential

Let the persistent region hierarchy be

$$
\mathcal{R}_t.
$$

The global certified potential is

$$
\boxed{
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P(R)
}
$$

provided that the represented regions maintain the required global coverage
for the certification procedure.

Because every region provides a valid upper bound,

$$
\max_{x\in R}f(x)
\le
P(R).
$$

Taking the maximum over all represented regions gives

$$
f^*
\le
P_{\mathrm{global}},
$$

where

$$
f^*
=
\max_{x\in\Omega}f(x).
$$

### Certified Global Chain

The global incumbent satisfies

$$
f_{\mathrm{best}}
\le
f^*.
$$

The global certified potential satisfies

$$
f^*
\le
P_{\mathrm{global}}.
$$

Therefore,

$$
\boxed{
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}}
}
$$

This chain is the central certified invariant of ARRGO.

### Certified Global Gap

The difference between the global potential and the incumbent is

$$
\boxed{
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
}
$$

Since

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}},
$$

we obtain

$$
0
\le
f^*-f_{\mathrm{best}}
\le
\Delta_{\mathrm{global}}.
$$

Thus, the global potential provides an upper bound on the remaining
objective-value error.

### Effect of Adding a New Evaluation

Suppose a new point

$$
x_c\in R
$$

is evaluated and produces

$$
y_c=f(x_c).
$$

The updated upper envelope is

$$
U_R^{\mathrm{new}}(x)
=
\min
\left\{
U_R(x),
y_c+L|x-x_c|
\right\}.
$$

Therefore,

$$
U_R^{\mathrm{new}}(x)
\le
U_R(x).
$$

Taking the maximum over the region gives

$$
\boxed{
P_{\mathrm{new}}(R)
\le
P(R)
}
$$

Thus, a valid additional observation cannot increase the certified regional
potential.

### Important Distinction

The fact that

$$
P_{\mathrm{new}}(R)\le P(R)
$$

does not mean that the actual optimum inside the region decreases.

The true function does not change.

Only the amount of uncertainty about the function changes.

Therefore,

$$
\boxed{
\text{Potential Decrease}
=
\text{Information Tightening}
}
$$

not a decrease in the underlying objective function.

### Effect of Splitting

Suppose the parent region is split:

$$
R_p
\rightarrow
\{R_L,R_R\}.
$$

The child regions inherit the relevant observations from the parent.

Their potentials are then computed independently:

$$
P(R_L)
=
\max_{x\in R_L}U_{R_L}(x),
$$

and

$$
P(R_R)
=
\max_{x\in R_R}U_{R_R}(x).
$$

Splitting itself does not introduce a new function evaluation.

Therefore, it does not automatically add new empirical information.

Instead, it reorganizes the existing certified information according to a finer
spatial structure.

### Parent-Child Consistency

Because

$$
R_p=R_L\cup R_R,
$$

the true optimum over the parent satisfies

$$
\max_{x\in R_p}f(x)
=
\max
\left\{
\max_{x\in R_L}f(x),
\max_{x\in R_R}f(x)
\right\}.
$$

Using the child upper bounds,

$$
\max_{x\in R_L}f(x)\le P(R_L),
$$

and

$$
\max_{x\in R_R}f(x)\le P(R_R).
$$

Therefore,

$$
\boxed{
\max_{x\in R_p}f(x)
\le
\max
\left\{
P(R_L),P(R_R)
\right\}
}
$$

This preserves certified correctness after structural refinement.

### Potential and Region Priority

The potential provides one of the most important signals for global region
selection in Certified Mode.

A region with

$$
P(R)
\le
f_{\mathrm{best}}+\epsilon
$$

cannot improve the incumbent by more than the requested tolerance.

A region with

$$
P(R)>f_{\mathrm{best}}+\epsilon
$$

remains potentially relevant to the global optimization objective.

Therefore, potential contributes directly to the definition of the certified
competitive region set:

$$
\boxed{
\mathcal{C}_t(\epsilon)
=
\left\{
R\in\mathcal{R}_t:
P(R)>f_{\mathrm{best}}+\epsilon
\right\}
}
$$

### Potential Is Not the Only Refinement Signal

A region with high potential may still have different unresolved information
states.

For example, its main unresolved issue may be:

- insufficient sampling coverage,
- unresolved local behavior,
- wide certified uncertainty,
- or insufficient spatial resolution.

Therefore, potential should not replace the complete regional information
profile.

Instead,

$$
P(R)
$$

is integrated with

$$
Q_R
=
(Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}).
$$

### Potential and Sampling

Sampling can simultaneously affect several quantities.

A new evaluation may:

$$
\text{increase } f_{\mathrm{best}},
$$

or

$$
\text{decrease }P(R),
$$

or change the behavioral and uncertainty profiles.

Consequently, one evaluation can alter both local and global priorities.

This is why ARRGO recomputes the relevant information state after each new
evaluation.

### Potential and Splitting

Splitting can change how certified potential is distributed across the region
hierarchy.

Before splitting,

$$
R_p
$$

may contain a broad range of possible high-value locations.

After splitting, this possibility is represented separately by

$$
R_L
\quad\text{and}\quad
R_R.
$$

This may reveal which child remains more competitive under the current
certified information.

However, splitting does not itself prove that the more competitive child
contains the global optimizer.

It only provides a more localized certified representation.

### Potential Monotonicity

For valid additional evaluations, regional potential cannot increase.

For a valid refinement that preserves coverage and recomputes valid regional
bounds, the global certified potential should likewise not increase under the
certified representation.

Thus,

$$
P_{\mathrm{global}}^{(t+1)}
\le
P_{\mathrm{global}}^{(t)}
$$

when the update consists of valid information tightening and consistent
recomputation of the certified bounds.

At the same time,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

Therefore,

$$
\Delta_{\mathrm{global}}^{(t+1)}
\le
\Delta_{\mathrm{global}}^{(t)}.
$$

The certified global gap is consequently non-increasing.

### Exact Potential Computation in One Dimension

Because the upper envelope is defined by

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right],
$$

it is a piecewise-linear function in one dimension.

Therefore, its maximum over a bounded interval can be determined from its
piecewise-linear structure.

The implementation may use:

- relevant envelope breakpoints,
- intersections between Lipschitz cones,
- region boundaries,
- or a numerically robust equivalent procedure.

The implementation strategy must preserve the mathematical definition

$$
P(R)=\max_{x\in R}U_R(x).
$$

### Certified Versus Empirical Potential

The quantity \(P(R)\) is a certified optimization potential only when the
underlying upper envelope is valid.

This requires a valid Lipschitz bound.

If ARRGO uses an estimated Lipschitz constant without an established guarantee,
the resulting quantity may still be useful as a heuristic ranking signal, but
it must not be presented as a certified upper bound.

Therefore,

$$
\boxed{
\text{Certified Potential}
\Rightarrow
\text{Valid Upper Bound}
\Rightarrow
\text{Valid Lipschitz Assumption}
}
$$

### Exact Optimization Potential Principle

The central principle is:

$$
\boxed{
\text{Represent the Remaining Optimization Possibility of Each Region by the
Maximum of Its Valid Certified Upper Envelope.}
}
$$

This allows ARRGO to distinguish regions that are no longer competitive from
regions that still require global optimization attention.

The next section formalizes how ARRGO generates candidate evaluation points
from the exact certified potential and the other unresolved information
sources.

## Exact Candidate Generation

Candidate generation defines the set of locations at which ARRGO may acquire
new objective information.

The purpose of candidate generation is not to predict the unknown objective
function.

Instead, it constructs a finite and deterministic set of locations that are
justified by the current information state.

Candidate generation is therefore separated from candidate evaluation and
candidate selection.

### Candidate Generation Pipeline

For a selected region \(R\), the sampling process follows

$$
\boxed{
\text{Region Analysis}
\rightarrow
\text{Candidate Generation}
\rightarrow
\text{Candidate Evaluation}
\rightarrow
\text{Candidate Selection}
\rightarrow
\text{Function Evaluation}
}
$$

Each stage has a different responsibility.

- **Region Analysis** identifies unresolved information.
- **Candidate Generation** identifies locations capable of addressing that
  information.
- **Candidate Evaluation** measures the structural or informational value of
  those locations using only currently available information.
- **Candidate Selection** chooses a valid candidate.
- **Function Evaluation** obtains the new black-box objective value.

### Sampling Candidate Set

Let the selected region be

$$
R=[l,r].
$$

ARRGO constructs a finite sampling candidate set

$$
\boxed{
C_{\mathrm{sample}}(R)
=
\left\{
x_1^c,x_2^c,\ldots,x_m^c
\right\}
}
$$

with every candidate satisfying

$$
x_j^c\in[l,r].
$$

Candidates that violate the region boundaries are invalid.

Candidates that duplicate existing evaluations within numerical tolerance are
also invalid.

### Candidate Validity

Let the existing regional evaluation set be

$$
D_R=
\left\{
x_1,\ldots,x_n
\right\}.
$$

A candidate \(x^c\) is valid only if

$$
x^c\in[l,r]
$$

and

$$
|x^c-x_i|>\tau_x
\qquad
\forall x_i\in D_R.
$$

Here \(\tau_x\) is the numerical point tolerance.

Therefore,

$$
\boxed{
C_{\mathrm{valid}}(R)
=
\left\{
x^c\in C_{\mathrm{sample}}(R):
|x^c-x_i|>\tau_x
\ \forall x_i\in D_R
\right\}
}
$$

Only valid candidates may proceed to candidate evaluation.

### Deterministic Candidate Sources

ARRGO generates candidates from observable properties of the current
information state.

The main sources are:

1. spatial coverage gaps,
2. observed behavioral transitions,
3. observed slope variation,
4. certified uncertainty,
5. certified optimization potential,
6. unevaluated region boundaries.

These sources are not assumed to be equally important.

Their relevance depends on the current unresolved objective profile.

### Coverage-Based Candidates

Suppose the regional observations are ordered as

$$
x_1<x_2<\cdots<x_n.
$$

For each internal gap,

$$
g_i=x_{i+1}-x_i.
$$

A deterministic coverage candidate can be generated at the midpoint:

$$
\boxed{
x_i^c
=
\frac{x_i+x_{i+1}}{2}
}
$$

This candidate divides the existing gap into two smaller gaps.

Before sampling,

$$
g_i=x_{i+1}-x_i.
$$

After inserting the midpoint,

$$
g_i^{(L)}
=
\frac{g_i}{2},
\qquad
g_i^{(R)}
=
\frac{g_i}{2}.
$$

Thus, the largest gap inside that local interval is reduced from \(g_i\) to

$$
\frac{g_i}{2}.
$$

Coverage-based candidates are especially relevant when the dominant unresolved
objective is spatial resolution.

### Boundary-Coverage Candidates

The region boundaries

$$
l
\quad\text{and}\quad
r
$$

may also be generated as candidates when they have not already been evaluated.

Boundary evaluations are important because a global or local optimum may occur
at the boundary.

Therefore, boundary points must not be excluded merely because they are not
interior points.

### Behavior-Based Candidates

ARRGO can also generate candidates near observed changes in local behavior.

For consecutive secant slopes,

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i},
$$

a directional reversal occurs when

$$
s_i s_{i+1}<0.
$$

For example,

$$
s_i>0,
\qquad
s_{i+1}<0
$$

is consistent with a possible local maximum between the corresponding
observations.

The region around such a transition can generate additional sampling
candidates.

These candidates are intended to resolve an observed transition.

They do not constitute a proof that an extremum exists.

### Slope-Variation Candidates

The observed change in secant slope is

$$
\Delta s_i
=
s_{i+1}-s_i.
$$

Large observed values of

$$
|\Delta s_i|
$$

indicate that the observed directional behavior is changing rapidly.

A candidate can therefore be generated inside or near the interval where the
change occurs.

This helps ARRGO distinguish regions with simple observed behavior from regions
where the current samples indicate more complex local behavior.

However,

$$
\Delta s_i
\neq
f''(x)
$$

in general.

It is an observed finite-difference quantity rather than an exact derivative.

### Certified-Uncertainty Candidates

In Certified Mode, ARRGO has the pointwise uncertainty profile

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

Locations where

$$
u_R(x)
$$

is large represent areas where the current certified information is relatively
weak.

Candidate locations can therefore be generated near local maxima or other
structurally relevant points of the uncertainty profile.

The purpose is to acquire additional information where the certified enclosure
is wide.

This mechanism is available only when the underlying certified bounds are
valid.

### Certified-Potential Candidates

The certified upper envelope is

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

The regional optimization potential is

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

Locations where the upper envelope is high can be used as candidate sampling
locations because they represent points where the current information permits
relatively large objective values.

However,

$$
U_R(x)
\neq
f(x).
$$

Therefore, a high upper-envelope value does not imply that the actual function
value at that point will be high.

It only identifies a location that remains potentially important under the
certified model.

### Candidate Source Provenance

Every generated candidate should retain information about why it was
generated.

Conceptually, a candidate can be represented as

$$
c=
\left(
x^c,
S_c
\right),
$$

where \(x^c\) is the candidate location and \(S_c\) is its source set.

For example,

$$
S_c
=
\{
\text{coverage},
\text{behavior}
\}.
$$

A candidate may therefore be supported by more than one information source.

This provenance is useful for candidate comparison and debugging.

### Candidate Deduplication

Different generation mechanisms may produce the same or nearly identical
locations.

For example, a coverage midpoint may coincide with a behavioral candidate.

ARRGO therefore performs deterministic deduplication using

$$
|x_i^c-x_j^c|
\le
\tau_x.
$$

Candidates satisfying this condition are treated as the same numerical
location.

The associated source sets should be merged rather than discarded.

Thus, a deduplicated candidate can preserve all known reasons for its
generation.

### Candidate Feasibility

Candidate generation must also respect the geometry of the selected region.

For

$$
R=[l,r],
$$

every candidate must satisfy

$$
l\le x^c\le r.
$$

Candidates outside the region are removed.

Candidates within the numerical tolerance of an already evaluated point are
removed.

Candidates that do not provide a meaningful new location are therefore
excluded before candidate evaluation.

### Candidate Generation Is Deterministic

For a fixed regional information state and fixed numerical tolerances,
candidate generation should produce the same candidate set.

Therefore,

$$
\boxed{
\text{Same Information State}
\Rightarrow
\text{Same Candidate Set}
}
$$

This deterministic property is important for reproducibility and scientific
comparison.

Random candidate generation is not required by ARRGO.

### Candidate Generation Does Not Evaluate the Function

A candidate is only a location.

Before function evaluation, ARRGO does not know

$$
f(x^c).
$$

Therefore, candidate generation must never use the unknown value

$$
f(x^c)
$$

as an input.

Instead, it relies exclusively on already available information.

This preserves the black-box nature of the optimization problem.

### Candidate Generation and the Active Objective Set

Recall the regional active unresolved objective set

$$
O_R^*
\subseteq
\{
\text{coverage},
\text{behavior},
\text{uncertainty},
\text{potential}
\}.
$$

Candidate generation should emphasize sources associated with the currently
active objectives.

For example, if coverage is unresolved, coverage candidates should be
present.

If observed behavior is unresolved, behavior-based candidates should be
included.

If Certified Mode identifies wide uncertainty, uncertainty-based candidates
should be represented.

If a region remains strongly competitive under its certified potential,
potential-based candidates may be included.

Thus,

$$
\boxed{
O_R^*
\rightarrow
\text{Relevant Candidate Sources}
}
$$

The active objective set determines which information sources deserve
representation in the candidate set.

### Candidate Completeness

Candidate generation should represent every refinement mechanism that is
currently relevant.

For the active objectives, let

$$
\mathcal{S}(O_R^*)
$$

denote the set of candidate-generation sources associated with those
objectives.

A candidate-generation procedure is considered structurally complete when
every currently active source contributes its valid candidate locations to the
candidate set, subject to boundary, duplicate, and numerical feasibility
constraints.

This does not mean that every possible point in the region must be generated.

It means that the deterministic candidate mechanisms required by the current
refinement state are represented.

### Candidate Set Versus Candidate Choice

It is important to distinguish

$$
C_{\mathrm{sample}}(R)
$$

from the selected point

$$
x_{\mathrm{selected}}^c.
$$

Candidate generation answers:

> Which locations are worth considering?

Candidate selection answers:

> Which one of the valid candidates should be evaluated next?

These are separate decisions.

Therefore,

$$
\boxed{
\text{Candidate Generation}
\neq
\text{Candidate Selection}
}
$$

This separation prevents the algorithm from hiding selection preferences inside
the candidate-generation mechanism.

### Candidate Generation and Multi-Criteria Evaluation

Each candidate may support several unresolved objectives.

Conceptually, its information profile is

$$
I(x^c)
=
\left(
I_{\mathrm{coverage}},
I_{\mathrm{behavior}},
I_{\mathrm{uncertainty}},
I_{\mathrm{potential}}
\right).
$$

These components generally have different meanings and scales.

ARRGO therefore does not require them to be combined into an arbitrary weighted
scalar.

Instead, candidate profiles can be compared using the active objectives and
non-dominance.

### Candidate Generation After Splitting

When a region is structurally split,

$$
R_p
\rightarrow
\{R_L,R_R\},
$$

candidate generation is re-run independently for the relevant child regions.

The inherited observations are used as the initial information state.

This allows the same deterministic candidate mechanisms to operate at the
finer spatial resolution.

### Candidate Generation and No-Pruning

Candidate generation operates on the currently selected region.

It does not delete other regions from the hierarchy.

Thus,

$$
\boxed{
\text{Candidate Generation}
\neq
\text{Global Region Pruning}
}
$$

All regions remain represented, even when only one region receives the next
sampling opportunity.

### Failure to Generate a Valid Candidate

It is possible that all generated candidates are invalid because of:

- duplicate evaluations,
- numerical tolerance,
- insufficient interior space,
- or already resolved information.

In this case, ARRGO must not invent an arbitrary new point solely to continue
execution.

Instead, the refinement decision should be reconsidered.

Possible alternatives include:

$$
\text{Sampling}
\rightarrow
\text{Splitting},
$$

or

$$
\text{Sampling}
\rightarrow
\text{Stable}.
$$

The appropriate transition depends on the current information state and
feasibility conditions.

### Candidate Generation Principle

The central principle is:

$$
\boxed{
\text{Generate Deterministic Candidate Locations That Are Justified by
Currently Unresolved Optimization-Relevant Information, Without Using
Unknown Function Values.}
}
$$

Candidate generation therefore forms the bridge between regional analysis and
actual black-box function evaluation.

The next section formalizes the exact decision rule used to choose the
appropriate refinement action after candidate generation.

## Exact Action Selection

After analyzing a region and generating the corresponding refinement
candidates, ARRGO must determine which refinement action should be executed.

The available actions are

$$
\mathcal{A}_R =
\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\}.
$$

The action-selection mechanism must remain consistent with the theoretical
properties established previously.

In particular, it must:

- respect the current unresolved information state;
- distinguish sampling from structural splitting;
- avoid arbitrary weighted objective scores;
- preserve the region hierarchy;
- preserve previously acquired information;
- remain deterministic;
- distinguish empirical decisions from certified decisions.

### Inputs to Action Selection

For a region \(R\), the action-selection procedure receives

$$
\mathcal{I}_R =
\left(
R,
D_R,
B_R,
Q_R,
O_R^*,
C_{\mathrm{sample}}(R),
C_{\mathrm{split}}(R),
G
\right).
$$

Here:

- \(D_R\) is the regional evaluation history;
- \(B_R\) is the observed behavioral information;
- \(Q_R\) is the unresolved information profile;
- \(O_R^*\) is the active unresolved objective set;
- \(C_{\mathrm{sample}}(R)\) is the valid sampling candidate set;
- \(C_{\mathrm{split}}(R)\) is the valid splitting candidate set;
- \(G\) is the current global state.

The decision must use only information available at the current iteration.

### Action Feasibility

Before comparing actions, ARRGO determines whether each refinement action is
feasible.

Define

$$
F_R
\subseteq
\{
\mathrm{Sample},
\mathrm{Split}
\}
$$

as the set of currently feasible refinement actions.

Sampling is feasible when at least one valid sampling candidate exists:

$$
\mathrm{Sample}\in F_R
\iff
C_{\mathrm{sample}}(R)\neq\varnothing.
$$

Splitting is feasible when at least one valid split candidate exists:

$$
\mathrm{Split}\in F_R
\iff
C_{\mathrm{split}}(R)\neq\varnothing.
$$

Every split candidate must additionally satisfy the contraction condition.

The feasibility test is performed before any preference comparison.

Therefore,

$$
\boxed{
\text{Infeasible Action}
\Rightarrow
\text{Cannot Be Selected}
}
$$

### Stable Action

The stable action is not a refinement operation.

It represents the conclusion that the currently available information does not
require immediate refinement of the region.

A region may therefore be considered stable when its currently active unresolved
objectives satisfy the applicable resolution criteria and no refinement action
is currently required.

This does not imply that the objective function is completely known inside
the region.

Instead,

$$
\boxed{
\mathrm{Stable}
\neq
\mathrm{Known}
}
$$

Stability means only that the current information state does not justify
another immediate refinement action.

### Resolution Before Action Preference

ARRGO first determines whether the region is sufficiently resolved.

The decision therefore follows the hierarchy

$$
\boxed{
\text{Resolution Requirement}
\rightarrow
\text{Action Feasibility}
\rightarrow
\text{Action Preference}
}
$$

If no active unresolved objective remains, the region may become STABLE.

If unresolved information remains, ARRGO determines whether sampling or
splitting is the more appropriate refinement mechanism.

### Sampling Action

Sampling is preferred when the dominant unresolved information requires a new
function evaluation.

Typical examples include:

- insufficient sampling coverage;
- unresolved directional behavior;
- unresolved slope variation;
- wide certified uncertainty;
- a potentially important certified upper envelope requiring further
  evaluation.

The defining property is

$$
\boxed{
\mathrm{Sample}
\rightarrow
\text{New Objective Information}
}
$$

A sampling action selects a valid candidate

$$
x^c\in C_{\mathrm{sample}}(R)
$$

and evaluates

$$
y^c=f(x^c).
$$

The new observation is then incorporated into the global and regional
information states.

### Splitting Action

Splitting is preferred when the main unresolved issue is structural rather than
purely observational.

Typical examples include:

- the region contains a meaningful observed behavioral transition;
- different parts of the region exhibit substantially different observed
  behavior;
- spatial resolution is too coarse;
- certified potential is distributed across structurally different
  subregions;
- the current region is too large to represent the relevant information
  adequately.

The defining property is

$$
\boxed{
\mathrm{Split}
\rightarrow
\text{Finer Spatial Representation}
}
$$

A split selects

$$
s^c\in C_{\mathrm{split}}(R)
$$

and creates the child regions

$$
R_L=[l,s^c],
$$

and

$$
R_R=(s^c,r].
$$

No new function value is obtained merely by splitting.

### Sampling and Splitting Are Not Equivalent

The distinction is fundamental.

Sampling changes the regional evaluation set:

$$
D_R^{(t+1)}
=
D_R^{(t)}
\cup
\left\{
(x^c,f(x^c))
\right\}.
$$

Splitting changes the region structure.

Thus,

$$
\boxed{
\mathrm{Sample}
\neq
\mathrm{Split}
}
$$

A sample can reveal new behavior without changing the region hierarchy.

A split can increase spatial resolution without adding a new objective
evaluation.

### Action Support

Each refinement action supports a subset of the active unresolved objectives.

Define

$$
S_{\mathrm{sample}}(R)
\subseteq
O_R^*
$$

and

$$
S_{\mathrm{split}}(R)
\subseteq
O_R^*.
$$

The preferred action should be one whose support addresses the currently
important unresolved information.

For example, when coverage is the dominant unresolved objective and new
function information is required, sampling can address the unresolved
coverage.

When observed behavior reveals a meaningful structural transition, splitting
can provide a finer spatial representation.

These mappings are contextual rather than universal.

### Multi-Objective Action Comparison

More than one unresolved objective may remain active.

For example,

$$
O_R^*
=
\{
\mathrm{Coverage},
\mathrm{Behavior}
\}.
$$

ARRGO does not collapse these objectives into an arbitrary weighted scalar.

Instead, each feasible action is evaluated according to the currently active
objectives that it can address.

Conceptually, an action profile is

$$
V(a)
=
\left(
V_{\mathrm{coverage}}(a),
V_{\mathrm{behavior}}(a),
V_{\mathrm{uncertainty}}(a),
V_{\mathrm{potential}}(a)
\right).
$$

Only components corresponding to active objectives are considered.

### Action Dominance

Let \(a_i\) and \(a_j\) be two feasible actions.

Action \(a_i\) dominates action \(a_j\) when \(a_i\) is at least as effective as
\(a_j\) on every currently active objective and strictly more effective on at
least one.

For every active objective \(k\),

$$
V_k(a_i)
\ge
V_k(a_j).
$$

For at least one active objective \(m\),

$$
V_m(a_i)
>
V_m(a_j).
$$

The non-dominated feasible action set is

$$
\boxed{
\mathcal{A}_{\mathrm{ND}}(R)
=
\left\{
a\in F_R
\;:\;
\nexists b\in F_R
\text{ such that } b \text{ dominates } a
\right\}.
}
$$

This preserves legitimate trade-offs between sampling and splitting.

### Objective-Specific Preference

When one objective clearly dominates the current unresolved state, ARRGO can
use that objective as the primary decision criterion.

For example, if coverage is the only active unresolved objective and sampling
is the only action that can materially improve it, then

$$
\mathrm{Decision}(R)
=
\mathrm{Sample}.
$$

Similarly, if structural resolution is the dominant unresolved objective and a
valid split is available, then

$$
\mathrm{Decision}(R)
=
\mathrm{Split}.
$$

The preference is derived from the current information state rather than from
a permanent action priority.

### Action Comparison Is Dynamic

The preferred action can change after every new evaluation or structural
refinement.

For example,

$$
\mathrm{Sample}
\rightarrow
\mathrm{Split}
$$

may occur when a new sample reveals a behavioral transition.

Conversely,

$$
\mathrm{Split}
\rightarrow
\mathrm{Sample}
$$

may occur when structural refinement produces smaller child regions whose
remaining issue is insufficient function information.

Therefore, action preference is recomputed after each refinement.

### Certified Potential Constraint

In Certified Mode, a region may remain globally relevant when

$$
P(R)
>
f_{\mathrm{best}}+\epsilon.
$$

Such a region cannot be declared globally irrelevant merely because one local
criterion appears resolved.

Its certified optimization potential must remain part of the action-selection
context.

In particular,

$$
\boxed{
P(R)
>
f_{\mathrm{best}}+\epsilon
\Rightarrow
\text{Certified Relevance Is Preserved}
}
$$

This does not force a specific action.

It only prevents an unjustified loss of global relevance.

### Certified Uncertainty Constraint

Similarly, if

$$
u_{\max}(R)
$$

remains large under the current certified resolution criterion, ARRGO may
require additional sampling or structural refinement.

The choice depends on the information structure.

A split reorganizes uncertainty spatially.

A sample introduces a new Lipschitz constraint and can directly tighten the
certified envelopes.

Therefore,

$$
\text{Uncertainty}
\rightarrow
\begin{cases}
\mathrm{Sample}, & \text{when new information is required},\\
\mathrm{Split}, & \text{when spatial separation is required}.
\end{cases}
$$

### Deterministic Tie-Breaking

It is possible that multiple actions remain non-dominated.

In that case, ARRGO uses deterministic tie-breaking only to guarantee
reproducibility.

The decision hierarchy is

$$
\boxed{
\text{Resolution Requirement}
\rightarrow
\text{Feasibility}
\rightarrow
\text{Objective Relevance}
\rightarrow
\text{Dominance}
\rightarrow
\text{Deterministic Tie-Breaking}
}
$$

Possible deterministic tie-breakers include:

- stronger support for the currently dominant objective;
- greater coverage of currently unresolved information;
- stronger structural separation;
- smaller numerical ambiguity;
- fixed action ordering when all meaningful criteria remain equivalent.

A tie-breaker does not establish mathematical superiority.

Its purpose is reproducibility.

### Stable-State Transition

If the currently active unresolved objectives are sufficiently resolved and no
refinement is required, ARRGO may transition

$$
\mathrm{ACTIVE}
\rightarrow
\mathrm{STABLE}.
$$

The region remains part of the persistent hierarchy.

If later global information changes its relevance, it may transition back:

$$
\mathrm{STABLE}
\rightarrow
\mathrm{ACTIVE}.
$$

Therefore, stability is reversible.

### Action Execution

Once an action has been selected, ARRGO executes exactly one refinement
operation.

For sampling,

$$
R
\xrightarrow{\mathrm{Sample}}
R.
$$

The regional evaluation set is expanded:

$$
D_R^{(t+1)}
=
D_R^{(t)}
\cup
\left\{
(x^c,f(x^c))
\right\}.
$$

For splitting,

$$
R
\xrightarrow{\mathrm{Split}}
\{R_L,R_R\}.
$$

The parent remains in the hierarchy.

For stability,

$$
R
\xrightarrow{\mathrm{Stable}}
R.
$$

Its state changes to STABLE and no new function evaluation is performed.

### Post-Action Reanalysis

After every refinement action, ARRGO updates the information state.

The complete local cycle is

$$
\boxed{
\text{Analyze}
\rightarrow
\text{Identify Unresolved Objectives}
\rightarrow
\text{Generate Candidates}
\rightarrow
\text{Evaluate Action Profiles}
\rightarrow
\text{Select Action}
\rightarrow
\text{Execute}
\rightarrow
\text{Update}
\rightarrow
\text{Reanalyze}
}
$$

This prevents the algorithm from relying on stale information.

### Global Context

Action selection is local to the selected region but is not independent of the
global state.

The global incumbent

$$
f_{\mathrm{best}}
$$

can change the optimization relevance of the region.

In Certified Mode, the global potential

$$
P_{\mathrm{global}}
$$

and global gap

$$
\Delta_{\mathrm{global}}
$$

also influence whether further refinement is necessary.

Therefore,

$$
\boxed{
\text{Local Action}
=
\text{Local Information}
+
\text{Global Context}
}
$$

### No-Pruning Requirement

Action selection never deletes a region.

Selecting any of the actions

$$
\mathrm{Sample},
\qquad
\mathrm{Split},
\qquad
\mathrm{Stable}
$$

only changes the current refinement state.

The complete region hierarchy remains persistent.

Thus,

$$
\boxed{
\text{Action Selection}
\neq
\text{Region Deletion}
}
$$

### Exact Action-Selection Rule

The complete deterministic action-selection procedure is

$$
\boxed{
\begin{aligned}
&\text{Analyze the current regional information state.}\\
&\text{Identify the active unresolved objectives.}\\
&\text{Determine feasible sampling and splitting actions.}\\
&\text{Check whether the region is sufficiently resolved.}\\
&\text{Compare feasible actions on the active objectives.}\\
&\text{Remove dominated actions.}\\
&\text{Preserve non-dominated trade-offs.}\\
&\text{Apply deterministic tie-breaking if necessary.}\\
&\text{Execute exactly one selected action.}\\
&\text{Update the global and regional information states.}
\end{aligned}
}
$$

The decision therefore remains adaptive rather than following a fixed
sample-first or split-first policy.

### Action-Selection Principle

The central principle is

$$
\boxed{
\text{Choose the Feasible Refinement Action That Most Directly Addresses the
Currently Unresolved Optimization-Relevant Information, While Preserving
Multi-Criteria Trade-Offs and Deterministic Reproducibility.}
}
$$

This completes the local decision mechanism of ARRGO.

The next stage integrates local action selection with the global region
selection process to define the complete iteration of the algorithm.

## Complete ARRGO Iteration

A complete ARRGO iteration combines the local refinement mechanism with the
global region-selection mechanism.

The purpose of one iteration is to transform the current global information
state into a new state by executing exactly one selected refinement action.

The iteration must preserve the theoretical properties established in the
previous sections, including:

- persistent evaluation history;
- persistent region hierarchy;
- global coverage;
- deterministic decision making;
- adaptive refinement;
- separation between sampling and splitting;
- monotonicity of the incumbent;
- and, in Certified Mode, validity of the global optimality certificate.

### Iteration State

Let the global state at iteration \(t\) be

$$
G_t
=
\left(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t
\right).
$$

In Certified Mode, the state additionally contains the certified regional
potentials and the global optimality gap:

$$
G_t^{\mathrm{cert}}
=
\left(
G_t,
\{P_R^{(t)}\}_{R\in\mathcal{R}_t},
P_{\mathrm{global}}^{(t)},
\Delta_{\mathrm{global}}^{(t)}
\right).
$$

Here:

- \(\mathcal{R}_t\) is the persistent region hierarchy;
- \(D_t\) is the complete evaluation history;
- \(x_{\mathrm{best}}^{(t)}\) is the current incumbent point;
- \(f_{\mathrm{best}}^{(t)}\) is the best observed objective value;
- \(N_t=|D_t|\) is the number of objective evaluations.

The state at iteration \(t+1\) must contain all information from iteration
\(t\) together with any newly acquired structural or objective information.

### Iteration Overview

A complete ARRGO iteration follows the conceptual sequence

$$
G_t
\rightarrow
\text{Global Reanalysis}
\rightarrow
\text{Region Selection}
\rightarrow
\text{Local Action Selection}
\rightarrow
\text{Action Execution}
\rightarrow
\text{State Update}
\rightarrow
\text{Termination Check}
\rightarrow
G_{t+1}.
$$

The iteration therefore connects two levels of decision making.

The global level determines **which region should receive the next refinement
opportunity**.

The local level determines **how that region should be refined**.

### Global Reanalysis

At the beginning of an iteration, ARRGO updates the relevant information used
for global region selection.

For each eligible region \(R\), the algorithm considers its current state

$$
S_R
=
\left(
D_R,
B_R,
Q_R,
U_R,
P_R,
\mathrm{state}_R
\right).
$$

The regional analysis determines:

- current sampling coverage;
- observed behavioral structure;
- unresolved information;
- certified uncertainty when Certified Mode is active;
- certified optimization potential when available;
- and current refinement feasibility.

The analysis uses the most recent global information.

Therefore, a region is not selected using stale information from an earlier
iteration.

### Determine Active Regions

The persistent hierarchy contains all regions created during the search.

Only a subset may currently require refinement.

Define the eligible region set as

$$
\mathcal{E}_t
\subseteq
\mathcal{R}_t.
$$

Eligibility depends on the current lifecycle state and global optimization
context.

An ACTIVE region can normally receive a refinement opportunity.

A STABLE region does not require immediate refinement under its current local
criteria, but it remains in the hierarchy.

A REFINED region has already been structurally subdivided, while its children
become the relevant finer-scale regions for subsequent refinement.

The hierarchy itself is never reduced.

### Determine Local Decisions

For each eligible region, ARRGO determines its current local decision.

The decision is

$$
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\}.
$$

The decision is obtained from the previously defined action-selection
mechanism.

Conceptually,

$$
\mathrm{Decision}_t(R)
=
\mathrm{ActionSelection}
\left(
S_R,
O_R^*,
C_{\mathrm{sample}}(R),
C_{\mathrm{split}}(R),
G_t
\right).
$$

The local decision is therefore conditional on the current information state.

It is not a permanent property of the region.

### Global Region Selection

After local decisions have been determined, ARRGO compares the eligible
regions.

Define the globally actionable region set as

$$
\mathcal{G}_t
=
\left\{
R\in\mathcal{E}_t
:
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split}
\}
\right\}.
$$

If \(\mathcal{G}_t\) is non-empty, ARRGO selects one region for the next
refinement operation.

The selected region is

$$
R_t^*
\in
\mathcal{G}_t.
$$

The selection is based on the current global relevance of the regions.

In Certified Mode, certified potential provides an important source of global
relevance:

$$
P_R^{(t)}
>
f_{\mathrm{best}}^{(t)}
+
\epsilon.
$$

A region satisfying this condition remains potentially capable of improving
the current certified result.

### Local Action Selection

Once the region \(R_t^*\) has been selected, its local action is executed.

Define

$$
a_t
=
\mathrm{Decision}_t(R_t^*).
$$

Therefore,

$$
a_t
\in
\{
\mathrm{Sample},
\mathrm{Split}
\}.
$$

The Stable action does not normally become the executed global refinement
operation because a stable region does not require immediate refinement.

If no actionable region exists, ARRGO must instead reconsider global relevance
or evaluate the applicable termination condition rather than repeatedly
executing a no-op Stable action.

### Sampling Iteration

If

$$
a_t
=
\mathrm{Sample},
$$

ARRGO selects one valid sampling candidate

$$
x_t^c
\in
C_{\mathrm{sample}}(R_t^*)
$$

and evaluates the objective function:

$$
y_t^c
=
f(x_t^c).
$$

The global evaluation history becomes

$$
D_{t+1}
=
D_t
\cup
\left\{
(x_t^c,y_t^c)
\right\}.
$$

The evaluation count therefore increases by one:

$$
N_{t+1}
=
N_t+1.
$$

The new observation can modify:

- regional behavior information;
- sampling coverage;
- certified uncertainty;
- certified potential;
- the global incumbent;
- and the relevance of other regions.

### Splitting Iteration

If

$$
a_t
=
\mathrm{Split},
$$

ARRGO selects a valid split point

$$
s_t^c
\in
C_{\mathrm{split}}(R_t^*).
$$

The selected region

$$
R_t^*
=
[l,r]
$$

is divided into two children:

$$
R_L
=
[l,s_t^c]
$$

and

$$
R_R
=
(s_t^c,r].
$$

The split must satisfy the contraction condition

$$
\max
\left(
s_t^c-l,
r-s_t^c
\right)
\le
\rho(r-l),
\qquad
0<\rho<1.
$$

No new objective evaluation is performed merely because the region is split.

Therefore,

$$
N_{t+1}
=
N_t.
$$

The parent region remains in the persistent hierarchy, while the child regions
become available for subsequent analysis and refinement.

### State Update After Sampling

After a sampling operation, ARRGO updates the global and regional state.

The new observation is incorporated into the relevant regional dataset:

$$
D_{R_t^*}^{(t+1)}
=
D_{R_t^*}^{(t)}
\cup
\left\{
(x_t^c,y_t^c)
\right\}.
$$

The regional behavior profile and unresolved information state are then
recomputed.

In Certified Mode, the new observation updates the regional envelopes:

$$
L_R^{\mathrm{new}}(x)
=
\max
\left\{
L_R(x),
y_t^c-L|x-x_t^c|
\right\},
$$

and

$$
U_R^{\mathrm{new}}(x)
=
\min
\left\{
U_R(x),
y_t^c+L|x-x_t^c|
\right\}.
$$

Consequently, the certified uncertainty cannot increase solely because a valid
new sample was added.

### State Update After Splitting

After a split operation, ARRGO creates the two child regions and assigns their
initial regional information.

The child regions inherit all relevant observations located inside them.

The parent remains available as part of the persistent hierarchy.

The child states are then independently analyzed.

Thus, splitting changes the structural representation without destroying the
information accumulated at the parent level.

The hierarchy evolves according to

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

### Incumbent Update

After every new objective evaluation, ARRGO updates the incumbent.

The invariant is

$$
f_{\mathrm{best}}^{(t)}
=
\max_{(x_i,y_i)\in D_t}
y_i.
$$

After sampling,

$$
f_{\mathrm{best}}^{(t+1)}
=
\max
\left(
f_{\mathrm{best}}^{(t)},
y_t^c
\right).
$$

Therefore,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

A split does not directly change the incumbent because it does not introduce a
new objective evaluation.

### Certified Global Update

In Certified Mode, ARRGO recomputes the regional certified potentials after
the refinement.

The global potential is

$$
P_{\mathrm{global}}^{(t+1)}
=
\max_{R\in\mathcal{R}_{t+1}}
P_R^{(t+1)}.
$$

The certified global gap becomes

$$
\Delta_{\mathrm{global}}^{(t+1)}
=
P_{\mathrm{global}}^{(t+1)}
-
f_{\mathrm{best}}^{(t+1)}.
$$

When the regional bounds remain valid, the fundamental enclosure is preserved:

$$
f_{\mathrm{best}}^{(t+1)}
\le
f^*
\le
P_{\mathrm{global}}^{(t+1)}.
$$

Therefore,

$$
0
\le
f^*
-
f_{\mathrm{best}}^{(t+1)}
\le
\Delta_{\mathrm{global}}^{(t+1)}.
$$

### Reanalysis After the Action

The action is not considered complete until the resulting information state
has been reanalyzed.

The post-action sequence is

$$
\text{Execute}
\rightarrow
\text{Update History}
\rightarrow
\text{Update Region Information}
\rightarrow
\text{Update Incumbent}
\rightarrow
\text{Update Certified Bounds}
\rightarrow
\text{Recompute Global Relevance}.
$$

This ensures that the next iteration begins with an internally consistent
state.

### Termination Check

Termination is checked only after the state update is complete.

In Certified Mode, ARRGO can terminate with an optimization certificate when

$$
\Delta_{\mathrm{global}}^{(t+1)}
\le
\epsilon.
$$

This implies

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f^*
-
\epsilon.
$$

If the evaluation budget is exhausted instead,

$$
N_{t+1}
\ge
N_{\max},
$$

ARRGO terminates with the best solution found under the available budget.

Budget exhaustion alone does not establish global optimality.

### Iteration Invariants

A correct ARRGO iteration must preserve several invariants.

The evaluation history is persistent:

$$
D_t
\subseteq
D_{t+1}.
$$

The region hierarchy is persistent:

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

The evaluation count is non-decreasing:

$$
N_{t+1}
\ge
N_t.
$$

The incumbent value is non-decreasing:

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

In Certified Mode, the global enclosure remains valid:

$$
f_{\mathrm{best}}^{(t+1)}
\le
f^*
\le
P_{\mathrm{global}}^{(t+1)}.
$$

These invariants form the consistency requirements of the iteration.

### Complete Iteration Mapping

The complete conceptual iteration can therefore be represented as

$$
\boxed{
G_t
\rightarrow
\text{Analyze Regions}
\rightarrow
\text{Determine Local Decisions}
\rightarrow
\text{Select Global Region}
\rightarrow
\text{Execute Sample or Split}
\rightarrow
\text{Update Information}
\rightarrow
\text{Update Incumbent}
\rightarrow
\text{Update Certified State}
\rightarrow
\text{Check Termination}
\rightarrow
G_{t+1}
}
$$

The order of these operations is important.

Global selection must use current regional information.

Local action selection must use the current global context.

State updates must occur before the next global decision.

Termination must be evaluated using the updated state.

### Determinism of the Complete Iteration

ARRGO is deterministic when:

- the objective function is deterministic;
- candidate generation is deterministic;
- candidate comparison is deterministic;
- action selection is deterministic;
- global region selection is deterministic;
- tie-breaking is deterministic;
- numerical tolerances are fixed;
- and the same initial state and evaluation budget are used.

Under these conditions, the same initial problem state produces the same sequence
of refinement decisions and objective evaluations, up to explicitly defined
floating-point behavior.

### Sampling and Splitting Within One Iteration

A single ARRGO iteration performs exactly one selected refinement action.

Therefore:

$$
\mathrm{Sample}
\Rightarrow
N_{t+1}=N_t+1
$$

while

$$
\mathrm{Split}
\Rightarrow
N_{t+1}=N_t.
$$

This distinction is important for evaluation-budget management.

An iteration therefore does not necessarily correspond to one function
evaluation.

It corresponds to one refinement decision.

### No-Pruning Across Iterations

No region is deleted as a consequence of the iteration.

After a split,

$$
R_t^*
\in
\mathcal{R}_{t+1}
$$

remains part of the hierarchy even though its children provide a finer spatial
representation.

Likewise, a STABLE region remains represented and may become relevant again if
the global incumbent or other information changes its relative importance.

Thus, the complete search history remains available throughout the algorithm.

### Iteration Principle

The central principle of a complete ARRGO iteration is

$$
\boxed{
\text{Use the Current Global Information State to Select One Relevant Region,
Apply the Most Appropriate Feasible Refinement Action, Preserve All Previous
Information, and Recompute the Global State Before the Next Iteration.}
}
$$

This completes the conceptual definition of one ARRGO iteration.

The next stage converts this iteration into an explicit algorithm execution
cycle and defines the repeated control flow used until a valid termination
condition is reached.

## Algorithm Execution Cycle

The complete ARRGO iteration defines one refinement decision and its resulting
state transition.

The algorithm execution cycle extends this concept by repeatedly applying
complete iterations until a valid termination condition is reached.

The execution cycle must preserve the deterministic and information-driven
nature of ARRGO while ensuring that every refinement step contributes to the
progressive resolution of the global optimization problem.

### Initial State

ARRGO begins with a bounded search domain

$$
\Omega=[a,b],
\qquad
a<b.
$$

The initial region is the root region

$$
R_0=\Omega.
$$

The initial region hierarchy is

$$
\mathcal{R}_0=\{R_0\}.
$$

Initially, the evaluation history may be empty:

$$
D_0=\varnothing.
$$

The evaluation counter is

$$
N_0=0.
$$

If initial evaluations are required before region analysis, ARRGO evaluates a
deterministically defined initial set of points.

The resulting observations become part of the persistent evaluation history.

### Initialization of the Incumbent

When at least one valid objective evaluation exists, the initial incumbent is
defined by

$$
x_{\mathrm{best}}^{(0)}
\in
\operatorname*{arg\,max}_{(x_i,y_i)\in D_0} y_i
$$

with

$$
f_{\mathrm{best}}^{(0)}
=
\max_{(x_i,y_i)\in D_0} y_i.
$$

If the initial evaluation history is empty, the incumbent is undefined until
the first objective evaluation is performed.

The algorithm must therefore ensure that a valid evaluation exists before
requiring an incumbent value.

### Initial Region Analysis

After initialization, ARRGO analyzes the available information in the root
region.

The initial analysis constructs the regional state

$$
S_{R_0}
=
\left(
D_{R_0},
B_{R_0},
Q_{R_0},
U_{R_0},
P_{R_0},
\mathrm{state}_{R_0}
\right).
$$

The analysis identifies:

- current spatial coverage;
- observed directional behavior;
- observed slope variation;
- candidate extrema;
- unresolved information;
- feasible sampling actions;
- feasible splitting actions;
- and, in Certified Mode, certified uncertainty and optimization potential.

The initial analysis establishes the information state from which the first
global refinement decision is made.

### Main Execution Loop

After initialization, ARRGO repeatedly performs complete iterations.

Conceptually, the execution loop is

$$
G_0
\rightarrow
G_1
\rightarrow
G_2
\rightarrow
\cdots
\rightarrow
G_T.
$$

Each transition

$$
G_t
\rightarrow
G_{t+1}
$$

corresponds to one complete ARRGO iteration.

The process continues while no valid termination condition has been satisfied.

### Global Reanalysis

At the beginning of each iteration, ARRGO updates the relevant global and
regional information.

For each relevant region \(R\), the algorithm recomputes or refreshes the
information required for decision making.

This includes

$$
S_R
=
\left(
D_R,
B_R,
Q_R,
U_R,
P_R,
\mathrm{state}_R
\right).
$$

The reanalysis is necessary because new information acquired in one region can
change the relevance of other regions.

For example, a new incumbent can make previously competitive regions
non-competitive in Certified Mode.

Therefore, global relevance cannot be treated as static.

### Eligibility Determination

ARRGO determines which regions can currently receive a refinement opportunity.

Define

$$
\mathcal{E}_t
\subseteq
\mathcal{R}_t
$$

as the set of eligible regions at iteration \(t\).

Eligibility depends on:

- lifecycle state;
- current unresolved objectives;
- refinement feasibility;
- global optimization relevance;
- and termination status.

A region may be STABLE while remaining part of the hierarchy.

If later information changes its relevance, the region can become ACTIVE again.

Therefore, eligibility is recomputed dynamically.

### Local Action Determination

For every eligible region, ARRGO determines the locally appropriate action.

The local decision is

$$
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\}.
$$

The decision uses:

$$
\mathrm{Decision}_t(R)
=
\mathrm{ActionSelection}
\left(
S_R,
O_R^*,
C_{\mathrm{sample}}(R),
C_{\mathrm{split}}(R),
G_t
\right).
$$

The result is therefore dependent on the current information state rather than
on a fixed action priority.

### Global Region Selection

After local decisions have been determined, ARRGO compares the eligible
regions globally.

The globally actionable set is

$$
\mathcal{G}_t
=
\left\{
R\in\mathcal{E}_t
:
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split}
\}
\right\}.
$$

If

$$
\mathcal{G}_t
\neq
\varnothing,
$$

ARRGO selects one region

$$
R_t^*
\in
\mathcal{G}_t.
$$

The selection is based on current global relevance and the active unresolved
information.

In Certified Mode, the regional potential provides a rigorous measure of
remaining optimization relevance:

$$
P_R^{(t)}
>
f_{\mathrm{best}}^{(t)}
+
\epsilon.
$$

Regions satisfying this condition remain candidates for further refinement.

### Refinement Action Execution

Once the region \(R_t^*\) has been selected, ARRGO executes its previously
determined local action.

The action is

$$
a_t
=
\mathrm{Decision}_t(R_t^*).
$$

There are two refinement operations:

$$
a_t
\in
\{
\mathrm{Sample},
\mathrm{Split}
\}.
$$

A Stable state does not constitute a refinement operation.

### Sampling Branch

If

$$
a_t=\mathrm{Sample},
$$

ARRGO selects a valid candidate

$$
x_t^c
\in
C_{\mathrm{sample}}(R_t^*)
$$

and evaluates

$$
y_t^c=f(x_t^c).
$$

The global history is updated:

$$
D_{t+1}
=
D_t
\cup
\left\{
(x_t^c,y_t^c)
\right\}.
$$

The evaluation count becomes

$$
N_{t+1}
=
N_t+1.
$$

The regional information state is then recomputed using the new observation.

### Splitting Branch

If

$$
a_t=\mathrm{Split},
$$

ARRGO selects a valid split point

$$
s_t^c
\in
C_{\mathrm{split}}(R_t^*).
$$

The selected region

$$
R_t^*
=
[l,r]
$$

is divided into

$$
R_L=[l,s_t^c]
$$

and

$$
R_R=(s_t^c,r].
$$

The contraction condition must hold:

$$
\max
\left(
s_t^c-l,
r-s_t^c
\right)
\le
\rho(r-l),
\qquad
0<\rho<1.
$$

The parent remains in the hierarchy.

No new objective evaluation is performed solely because of the split.

Therefore,

$$
N_{t+1}=N_t.
$$

### Information Preservation

Every execution cycle preserves previously acquired information.

The evaluation history satisfies

$$
D_t
\subseteq
D_{t+1}.
$$

The region hierarchy satisfies

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

Consequently, refinement adds information or structure without deleting
previously acquired information.

This property is fundamental to the no-pruning design of ARRGO.

### Incumbent Update

Whenever a new function evaluation is obtained, ARRGO updates the incumbent.

The updated value is

$$
f_{\mathrm{best}}^{(t+1)}
=
\max
\left(
f_{\mathrm{best}}^{(t)},
y_t^c
\right).
$$

Therefore,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

A splitting operation does not directly modify the incumbent because it does
not evaluate the objective function.

### Certified State Update

In Certified Mode, the regional bounds and potentials are recomputed after
each refinement.

For every relevant region,

$$
L_R(x)
\le
f(x)
\le
U_R(x).
$$

The regional optimization potential is

$$
P_R
=
\max_{x\in R}U_R(x).
$$

The global potential is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P_R.
$$

The global certified gap is

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

The fundamental certified relation remains

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}}.
$$

### Reanalysis After Every Refinement

After the selected action is executed, ARRGO reanalyzes the affected
information.

For sampling, this means updating the sampled data, behavior, uncertainty,
potential, and unresolved objectives.

For splitting, this means creating and analyzing the child regions.

The post-action process is therefore

$$
\text{Action}
\rightarrow
\text{Information Update}
\rightarrow
\text{Regional Reanalysis}
\rightarrow
\text{Global Reanalysis}.
$$

This ensures that the next iteration operates on current information.

### Global Priority Recalculation

After reanalysis, ARRGO recalculates the global relevance of the regions.

A region's priority may change because:

- its information resolution has improved;
- its certified uncertainty has changed;
- its potential has changed;
- a different region produced a better incumbent;
- a structural split created more relevant child regions;
- or the global optimization gap has decreased.

Therefore,

$$
\Pi_R^{(t+1)}
\neq
\Pi_R^{(t)}
$$

in general.

Global priority is a dynamic property rather than a permanent ranking.

### Termination Evaluation

After all state updates are complete, ARRGO checks the applicable termination
criteria.

In Certified Mode, the primary accuracy condition is

$$
\Delta_{\mathrm{global}}
\le
\epsilon.
$$

If this condition holds, ARRGO can terminate with an
\(\epsilon\)-optimality certificate.

The certificate follows from

$$
f_{\mathrm{best}}
\ge
f^*
-
\epsilon.
$$

If the evaluation budget is the limiting condition, ARRGO may terminate when

$$
N_t
\ge
N_{\max}.
$$

This produces a best-found result but does not by itself establish global
optimality.

### No-Action Situation

It is possible that no eligible region currently has a feasible refinement
action.

That situation must not produce an infinite sequence of Stable iterations.

Instead, ARRGO performs the following logical check:

$$
\mathcal{G}_t
=
\varnothing.
$$

The algorithm then determines whether:

- the certified termination condition is satisfied;
- the evaluation budget is exhausted;
- numerical limits have been reached;
- or the region eligibility and resolution criteria require reconsideration.

If none of these conditions provides a valid continuation, the algorithm
terminates with an explicit termination reason.

The algorithm must never silently interpret the absence of an action as proof
of global optimality.

### Iteration Counter

The iteration counter measures refinement decisions rather than objective
evaluations.

Therefore,

$$
t
\neq
N_t
$$

in general.

For a sampling iteration,

$$
t\rightarrow t+1,
\qquad
N_{t+1}=N_t+1.
$$

For a splitting iteration,

$$
t\rightarrow t+1,
\qquad
N_{t+1}=N_t.
$$

This distinction is essential for correct budget management.

### Complete Execution Cycle

The complete ARRGO execution cycle can be summarized as

$$
\boxed{
\begin{aligned}
&G_t
\rightarrow
\text{Reanalyze Global State}\\
&\rightarrow
\text{Determine Eligible Regions}\\
&\rightarrow
\text{Determine Local Actions}\\
&\rightarrow
\text{Select Global Region}\\
&\rightarrow
\text{Execute Sample or Split}\\
&\rightarrow
\text{Update Evaluation History}\\
&\rightarrow
\text{Update Region Hierarchy}\\
&\rightarrow
\text{Update Incumbent}\\
&\rightarrow
\text{Update Certified Bounds}\\
&\rightarrow
\text{Recompute Global Priorities}\\
&\rightarrow
\text{Check Termination}\\
&\rightarrow
G_{t+1}.
\end{aligned}
}
$$

The cycle is repeated while a valid continuation exists.

### Deterministic Execution

The complete execution cycle is deterministic when all decision components are
deterministic.

This requires deterministic:

- initialization;
- candidate generation;
- candidate comparison;
- action selection;
- global region selection;
- tie-breaking;
- boundary handling;
- duplicate handling;
- numerical tolerances;
- and objective evaluation.

Given the same initial problem, configuration, and deterministic objective
function, ARRGO therefore follows the same refinement trajectory, subject to
the defined numerical precision behavior.

### Execution Invariants

Throughout the execution cycle, ARRGO preserves the following invariants:

$$
D_t
\subseteq
D_{t+1},
$$

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1},
$$

$$
N_{t+1}
\ge
N_t,
$$

and

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

In Certified Mode, the enclosure invariant is

$$
f_{\mathrm{best}}^{(t)}
\le
f^*
\le
P_{\mathrm{global}}^{(t)}.
$$

These invariants must hold after every completed iteration.

### Execution Principle

The execution cycle of ARRGO is therefore based on the following principle:

$$
\boxed{
\text{Repeatedly Select the Most Relevant Refinement Opportunity, Acquire or
Organize Optimization-Relevant Information, Preserve the Complete Search
History, and Recompute the Global State Until a Valid Termination Condition Is
Reached.}
}
$$

This defines the repeated control flow of ARRGO.

The next stage integrates this execution cycle with the explicit global
selection and refinement loop that determines how competing regions are
compared throughout the search.

## Global Selection and Refinement Loop

The global selection and refinement loop is the mechanism that coordinates
refinement across all regions in the persistent ARRGO hierarchy.

The local action-selection mechanism determines how an individual region should
be refined.

The global loop determines which region should receive the next refinement
opportunity.

Therefore, ARRGO separates the two decisions:

$$
\text{Global Decision}
=
\text{Which Region?}
$$

and

$$
\text{Local Decision}
=
\text{Which Action?}
$$

This separation allows ARRGO to adapt both the spatial location of refinement
and the type of refinement operation.

### Global Refinement State

At iteration \(t\), ARRGO maintains the global state

$$
G_t
=
\left(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t
\right).
$$

In Certified Mode, the state additionally contains regional certified
potentials and the global certified gap.

The persistent hierarchy

$$
\mathcal{R}_t
$$

contains every region created during the search.

No region is removed when another region becomes more relevant.

### Global Eligibility

The first stage of the global loop is to determine which regions are currently
eligible for refinement.

Define

$$
\mathcal{E}_t
\subseteq
\mathcal{R}_t.
$$

Eligibility is determined from the current region state, unresolved objectives,
action feasibility, and global optimization context.

A region may be excluded from immediate refinement when it is currently
STABLE, but it remains part of the persistent hierarchy.

If its relevance changes later, it may become eligible again.

### Globally Actionable Regions

After local action selection, ARRGO constructs the set

$$
\mathcal{G}_t
=
\left\{
R\in\mathcal{E}_t
:
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split}
\}
\right\}.
$$

Only regions in \(\mathcal{G}_t\) can receive the next refinement opportunity.

A STABLE region is therefore not treated as an active refinement candidate
unless its state changes after reanalysis.

### Global Relevance

The relevance of a region is determined relative to the current global
optimization state.

A region can be relevant because it contains:

- unresolved spatial coverage;
- unresolved behavioral structure;
- substantial certified uncertainty;
- substantial certified optimization potential;
- or information that may affect the global optimization decision.

These sources of relevance are not combined into an arbitrary fixed weighted
score.

Instead, ARRGO preserves their multi-dimensional structure.

### Certified Global Relevance

In Certified Mode, a regional potential provides a rigorous criterion for
remaining optimization relevance.

For a region \(R\),

$$
P_R
>
f_{\mathrm{best}}
+
\epsilon
$$

means that the region cannot yet be excluded from the set of regions that
could contain an improvement of at least the requested tolerance.

Define the certified competitive set

$$
\mathcal{C}_t(\epsilon)
=
\left\{
R\in\mathcal{R}_t
:
P_R^{(t)}
>
f_{\mathrm{best}}^{(t)}
+
\epsilon
\right\}.
$$

Every region in this set remains globally competitive under the current
certificate.

A region outside this set may still remain in the hierarchy, but its current
certified potential does not require further refinement for an
\(\epsilon\)-optimal certificate.

### Empirical Global Relevance

In Empirical Mode, no rigorous upper bound on unseen function values is
available.

Therefore, global relevance must be based on observed information.

Possible evidence includes:

- promising observed objective values;
- unresolved behavioral transitions;
- insufficient coverage near promising observations;
- unresolved spatial structure;
- and other deterministic information derived from evaluated points.

Such evidence supports adaptive search but does not constitute a global
optimality certificate.

### Region Comparison

When multiple regions are globally actionable, ARRGO compares them using their
current information profiles.

A conceptual global profile is

$$
V_R
=
\left(
V_{\mathrm{coverage}},
V_{\mathrm{behavior}},
V_{\mathrm{uncertainty}},
V_{\mathrm{potential}}
\right).
$$

Only objectives relevant to the current global state are considered.

The comparison therefore depends on what remains unresolved and what can
affect the global optimization decision.

### Region Dominance

Let \(R_i\) and \(R_j\) be two globally actionable regions.

Region \(R_i\) dominates \(R_j\) when it is at least as relevant as \(R_j\) on
every currently active global criterion and strictly more relevant on at least
one.

For every active criterion \(k\),

$$
V_k(R_i)
\ge
V_k(R_j).
$$

For at least one active criterion \(m\),

$$
V_m(R_i)
>
V_m(R_j).
$$

The globally non-dominated region set is

$$
\boxed{
\mathcal{G}_{\mathrm{ND}}^{(t)}
=
\left\{
R\in\mathcal{G}_t
:
\nexists Q\in\mathcal{G}_t
\text{ such that }Q\text{ dominates }R
\right\}.
}
$$

This prevents ARRGO from discarding a region merely because another region is
stronger under one criterion.

### Certified Potential as a Global Criterion

In Certified Mode, potential has a special role because it is connected
directly to the global optimality certificate.

The global potential is

$$
P_{\mathrm{global}}^{(t)}
=
\max_{R\in\mathcal{R}_t}
P_R^{(t)}.
$$

Therefore, the regions attaining or approaching this global potential are
directly relevant to reducing the certified gap.

The global gap is

$$
\Delta_{\mathrm{global}}^{(t)}
=
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}.
$$

Reducing this gap is the primary certified global objective.

### Global Selection After Dominance

If multiple regions remain non-dominated, ARRGO applies objective-specific
preference.

For example, if the dominant global unresolved objective is certified
potential, regions with larger certified potential receive stronger priority.

If the dominant objective is spatial resolution, regions with more important
unresolved spatial structure receive stronger priority.

The preference is determined dynamically from the current global state.

It is not a permanent ranking of regions.

### Deterministic Global Tie-Breaking

Multiple regions may remain equivalent after the meaningful criteria have been
considered.

ARRGO then applies deterministic tie-breaking.

Possible tie-breakers include:

- stronger support for the current dominant objective;
- larger unresolved spatial gap;
- stronger observed behavioral transition;
- larger certified potential;
- smaller region diameter when finer localization is relevant;
- region depth;
- deterministic region identifier.

Tie-breaking is applied only after meaningful information-based comparison.

Its purpose is reproducibility rather than mathematical superiority.

### Global Region Selection

The selected region is

$$
R_t^*
\in
\mathcal{G}_{\mathrm{ND}}^{(t)}.
$$

The corresponding local action is

$$
a_t
=
\mathrm{Decision}_t(R_t^*).
$$

Therefore, the global selection produces the pair

$$
\left(
R_t^*,
a_t
\right).
$$

This pair completely determines the next refinement operation.

### Refinement Execution

If

$$
a_t=\mathrm{Sample},
$$

ARRGO selects the appropriate sampling candidate inside \(R_t^*\) and
evaluates the objective function.

If

$$
a_t=\mathrm{Split},
$$

ARRGO selects the appropriate split point inside \(R_t^*\) and creates the
corresponding child regions.

In both cases, the selected region remains represented in the persistent
hierarchy.

### Sampling Branch

For sampling,

$$
x_t^c
\in
C_{\mathrm{sample}}(R_t^*)
$$

is selected and evaluated:

$$
y_t^c
=
f(x_t^c).
$$

The global evaluation history becomes

$$
D_{t+1}
=
D_t
\cup
\left\{
(x_t^c,y_t^c)
\right\}.
$$

The evaluation count becomes

$$
N_{t+1}
=
N_t+1.
$$

The new observation may change the relevance of every region because it may
change the global incumbent.

### Splitting Branch

For splitting,

$$
s_t^c
\in
C_{\mathrm{split}}(R_t^*)
$$

is selected.

The selected region

$$
R_t^*
=
[l,r]
$$

is divided into

$$
R_L=[l,s_t^c]
$$

and

$$
R_R=(s_t^c,r].
$$

The contraction requirement is

$$
\max
\left(
s_t^c-l,
r-s_t^c
\right)
\le
\rho(r-l),
\qquad
0<\rho<1.
$$

No new objective evaluation is performed solely by the split.

Therefore,

$$
N_{t+1}
=
N_t.
$$

### Persistent Hierarchy Update

After splitting, the hierarchy is expanded:

$$
\mathcal{R}_{t+1}
=
\mathcal{R}_t
\cup
\left\{
R_L,R_R
\right\}.
$$

The parent region remains present.

Its lifecycle state becomes REFINED, while the child regions are analyzed
according to their newly inherited information.

Thus,

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

### Information Propagation

After either sampling or splitting, ARRGO propagates the appropriate
information through the global state.

Sampling adds new objective information.

Splitting adds new spatial structure.

The resulting state is then reanalyzed before the next global selection.

The update sequence is

$$
\text{Refinement}
\rightarrow
\text{Information Update}
\rightarrow
\text{Regional Reanalysis}
\rightarrow
\text{Global Relevance Update}.
$$

### Global Incumbent Update

If the refinement was a sampling operation, the incumbent is updated using
the new observation.

The invariant is

$$
f_{\mathrm{best}}^{(t+1)}
=
\max
\left(
f_{\mathrm{best}}^{(t)},
y_t^c
\right).
$$

Therefore,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

A split does not directly change the incumbent.

### Certified Global Gap Update

In Certified Mode, ARRGO recomputes all affected regional potentials.

The global potential becomes

$$
P_{\mathrm{global}}^{(t+1)}
=
\max_{R\in\mathcal{R}_{t+1}}
P_R^{(t+1)}.
$$

The certified gap becomes

$$
\Delta_{\mathrm{global}}^{(t+1)}
=
P_{\mathrm{global}}^{(t+1)}
-
f_{\mathrm{best}}^{(t+1)}.
$$

The certified enclosure remains

$$
f_{\mathrm{best}}^{(t+1)}
\le
f^*
\le
P_{\mathrm{global}}^{(t+1)}.
$$

Consequently,

$$
0
\le
f^*
-
f_{\mathrm{best}}^{(t+1)}
\le
\Delta_{\mathrm{global}}^{(t+1)}.
$$

### Fair Global Selection

Global selection must satisfy the fairness requirement established in the
theoretical foundations.

The central condition is

$$
\boxed{
\text{Persistent Global Relevance}
\Rightarrow
\text{Eventual Refinement Opportunity}
}
$$

A region that remains persistently relevant cannot be permanently ignored by
the global selection mechanism.

Fairness does not require uniform refinement.

Instead, it ensures that globally important regions continue to receive
refinement opportunities when their relevance persists.

### Fairness in Certified Mode

In Certified Mode, persistent competitiveness provides a rigorous relevance
signal.

If a region repeatedly satisfies

$$
P_R^{(t)}
>
f_{\mathrm{best}}^{(t)}
+
\epsilon,
$$

then the global selection mechanism must preserve its opportunity for eventual
refinement.

This condition prevents the algorithm from terminating refinement of a
potentially competitive region without justification.

### Reconsideration After Incumbent Improvement

An important consequence of global selection is that relevance can change
after a new incumbent is found.

Suppose a sampling operation produces

$$
f_{\mathrm{best}}^{(t+1)}
>
f_{\mathrm{best}}^{(t)}.
$$

Then a region that previously satisfied

$$
P_R
>
f_{\mathrm{best}}^{(t)}
+
\epsilon
$$

may later satisfy

$$
P_R
\le
f_{\mathrm{best}}^{(t+1)}
+
\epsilon.
$$

Such a region may become non-competitive under the updated certificate.

Therefore, global selection must be recomputed after every incumbent update.

### Reconsideration After Structural Refinement

Splitting can also change global relevance.

A parent region may contain information that becomes represented more precisely
by its children.

The children can have different:

- coverage states;
- behavioral profiles;
- uncertainty levels;
- certified potentials;
- and refinement requirements.

Therefore, after a split, global selection is performed using the new child
states rather than assuming that the parent's previous priority remains valid.

### No-Pruning Principle

Global selection never removes non-selected regions.

If

$$
R_i
\notin
\mathcal{G}_{\mathrm{ND}}^{(t)},
$$

this means only that another region currently has stronger or more relevant
information under the active comparison criteria.

It does not mean that \(R_i\) is deleted.

Thus,

$$
\boxed{
\text{Not Selected}
\neq
\text{Discarded}
}
$$

A previously non-selected region can become globally relevant again after new
information changes the global state.

### Complete Global Refinement Loop

The complete global loop is

$$
\boxed{
\begin{aligned}
&\text{Reanalyze the global state}\\
&\rightarrow
\text{Determine eligible regions}\\
&\rightarrow
\text{Determine local refinement actions}\\
&\rightarrow
\text{Construct globally actionable regions}\\
&\rightarrow
\text{Compare global relevance}\\
&\rightarrow
\text{Remove dominated candidates}\\
&\rightarrow
\text{Preserve non-dominated regions}\\
&\rightarrow
\text{Select one region deterministically}\\
&\rightarrow
\text{Execute its local refinement action}\\
&\rightarrow
\text{Update global information}\\
&\rightarrow
\text{Recompute global relevance}\\
&\rightarrow
\text{Check termination}.
\end{aligned}
}
$$

The process then continues from the updated global state.

### Global Selection and Refinement Invariant

After every completed global refinement step, ARRGO must preserve

$$
D_t
\subseteq
D_{t+1},
$$

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1},
$$

and

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

In Certified Mode, the enclosure must remain valid:

$$
f_{\mathrm{best}}^{(t+1)}
\le
f^*
\le
P_{\mathrm{global}}^{(t+1)}.
$$

These invariants ensure that global selection does not compromise the
theoretical properties of the refinement process.

### Global Selection Principle

The central principle of the global selection and refinement loop is

$$
\boxed{
\text{Select the Globally Relevant Region That Best Addresses the Current
Optimization Information, Apply Its Appropriate Local Refinement Action, and
Recompute Global Relevance After the New Information Is Incorporated.}
}
$$

This completes the integration of local action selection with global region
selection.

The next stage addresses how ARRGO manages the available evaluation budget
while preserving its refinement and convergence properties.

## Budget Management

ARRGO operates under a finite computational budget in practical numerical
optimization.

Because objective-function evaluations may be significantly more expensive
than structural operations, the evaluation budget must be explicitly separated
from the iteration counter.

The budget-management mechanism therefore controls the number of objective
evaluations without treating every structural refinement as an objective
evaluation.

### Evaluation Budget

Let the maximum allowed number of objective evaluations be

$$
N_{\max}.
$$

The current number of objective evaluations is

$$
N_t
=
|D_t|.
$$

The evaluation budget condition is therefore

$$
N_t
\le
N_{\max}.
$$

The remaining evaluation budget is

$$
B_t
=
N_{\max}
-
N_t.
$$

When

$$
B_t=0,
$$

no additional objective evaluation may be performed.

### Iteration Count Versus Evaluation Count

The ARRGO iteration counter \(t\) measures refinement decisions.

The evaluation counter \(N_t\) measures actual objective-function
evaluations.

Therefore, in general,

$$
t\neq N_t.
$$

A sampling iteration performs one new objective evaluation:

$$
\mathrm{Sample}
\Rightarrow
N_{t+1}=N_t+1.
$$

A splitting iteration introduces spatial structure without evaluating the
objective:

$$
\mathrm{Split}
\Rightarrow
N_{t+1}=N_t.
$$

Consequently, several iterations may occur without increasing the evaluation
count.

### Budget Consumption

Only an action that evaluates the objective consumes evaluation budget.

For sampling,

$$
B_{t+1}
=
B_t-1.
$$

For splitting,

$$
B_{t+1}
=
B_t.
$$

For a Stable state transition, no objective evaluation is performed:

$$
B_{t+1}
=
B_t.
$$

Therefore, the evaluation budget is directly coupled to the sampling branch
rather than to the total number of refinement iterations.

### Budget Feasibility

Before executing a sampling action, ARRGO checks whether sufficient
evaluation budget remains.

Sampling is budget-feasible when

$$
B_t>0.
$$

Equivalently,

$$
N_t<N_{\max}.
$$

Therefore,

$$
\boxed{
N_t=N_{\max}
\Rightarrow
\mathrm{Sample}
\text{ is not feasible}
}
$$

A splitting operation may remain feasible even when the evaluation budget has
been exhausted because splitting does not require a new objective evaluation.

However, continuing to split after the evaluation budget is exhausted must be
consistent with the selected termination policy.

### Budget-Aware Feasible Action Set

The local feasible action set must therefore incorporate the remaining
evaluation budget.

Define

$$
F_R^{(t)}
\subseteq
\{
\mathrm{Sample},
\mathrm{Split}
\}.
$$

Sampling belongs to the feasible set only when

$$
C_{\mathrm{sample}}(R)\neq\varnothing
$$

and

$$
N_t<N_{\max}.
$$

Thus,

$$
\mathrm{Sample}\in F_R^{(t)}
\iff
C_{\mathrm{sample}}(R)\neq\varnothing
\land
N_t<N_{\max}.
$$

Splitting remains feasible when a valid contraction-preserving split exists:

$$
\mathrm{Split}\in F_R^{(t)}
\iff
C_{\mathrm{split}}(R)\neq\varnothing.
$$

The budget therefore becomes part of action feasibility rather than an
after-the-fact execution check.

### Budget Does Not Override Mathematical Validity

The evaluation budget controls computational resources.

It does not change the mathematical meaning of the information already
acquired.

For example, if a region has a valid certified upper potential

$$
P_R,
$$

that potential remains valid regardless of how much evaluation budget remains.

Similarly, the global enclosure

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}}
$$

does not become weaker merely because the remaining budget is small.

Budget limitations affect what ARRGO can discover or certify next, not the
validity of previously established bounds.

### Budget Exhaustion

Budget exhaustion occurs when

$$
N_t=N_{\max}.
$$

At this point, ARRGO cannot perform another sampling operation.

The algorithm then evaluates the applicable termination policy.

In a budget-limited execution mode, ARRGO terminates and returns the best
solution found so far.

The result is

$$
x_{\mathrm{best}}^{(t)},
\qquad
f_{\mathrm{best}}^{(t)}.
$$

Budget exhaustion alone does not imply

$$
f_{\mathrm{best}}^{(t)}=f^*.
$$

Therefore,

$$
\boxed{
\text{Budget Exhaustion}
\neq
\text{Global Optimality}
}
$$

### Certified Termination Before Budget Exhaustion

Certified Mode may terminate before the evaluation budget is exhausted.

If

$$
\Delta_{\mathrm{global}}^{(t)}
\le
\epsilon,
$$

then ARRGO has obtained a valid \(\epsilon\)-optimality certificate.

In this case, further objective evaluations are unnecessary for the requested
accuracy.

Therefore,

$$
\boxed{
\Delta_{\mathrm{global}}\le\epsilon
\Rightarrow
\text{Certified Termination}
}
$$

provided that all assumptions required for the certificate remain valid.

### Budget Exhaustion Before Certified Termination

The opposite situation may also occur.

ARRGO may reach

$$
N_t=N_{\max}
$$

while

$$
\Delta_{\mathrm{global}}^{(t)}
>
\epsilon.
$$

In that case, the requested certified tolerance has not been achieved.

The algorithm may still return the best observed solution, but it must report
that the evaluation budget was exhausted before the requested certificate was
obtained.

Thus,

$$
\boxed{
N_t=N_{\max}
\land
\Delta_{\mathrm{global}}>\epsilon
\Rightarrow
\text{No Certified }\epsilon\text{-Optimality Claim}
}
$$

### Budget and Global Selection

The remaining evaluation budget affects global region selection.

Suppose several regions are globally relevant.

If the budget is positive, a region whose preferred action is sampling may
remain globally actionable.

When

$$
B_t=0,
$$

such a sampling action becomes infeasible.

The global selection mechanism must therefore recompute the globally
actionable set using the updated feasibility information.

Conceptually,

$$
\mathcal{G}_t
=
\left\{
R\in\mathcal{E}_t:
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split}
\}
\text{ and is budget-feasible}
\right\}.
$$

This prevents ARRGO from selecting an action that cannot actually be
executed.

### Budget and Splitting

Because splitting does not require an objective evaluation, a split may remain
computationally possible after the evaluation budget is exhausted.

However, budget-limited execution normally treats the evaluation budget as
the primary computational stopping resource.

Therefore, once

$$
N_t=N_{\max},
$$

ARRGO should terminate unless the configured execution policy explicitly
allows additional structural refinement without new evaluations.

If such structural continuation is allowed, it must not be interpreted as
additional objective information.

### Budget and Information Acquisition

The evaluation budget determines how many new objective observations ARRGO can
acquire.

The persistent evaluation history satisfies

$$
D_t
\subseteq
D_{t+1}.
$$

Every sampling action adds one new observation:

$$
|D_{t+1}|
=
|D_t|+1.
$$

Therefore,

$$
|D_t|
\le
N_{\max}
$$

throughout the execution.

The budget thus imposes an upper bound on the number of objective observations
available to the adaptive information-refinement mechanism.

### Budget and Adaptive Refinement

ARRGO does not distribute the evaluation budget uniformly across regions.

Instead, evaluations are allocated according to the current information state
and global relevance.

Therefore, two regions may receive different numbers of evaluations.

For regions \(R_i\) and \(R_j\),

$$
|D_{R_i}|
\neq
|D_{R_j}|
$$

is completely valid.

The difference reflects adaptive allocation rather than an implementation
error.

### Budget and No-Pruning

Budget management never deletes regions.

When the budget is exhausted, the complete hierarchy remains available in the
final state:

$$
\mathcal{R}_t.
$$

All previously created parent and child relationships remain preserved.

Therefore,

$$
\boxed{
\text{Budget Limitation}
\neq
\text{Region Deletion}
}
$$

The final hierarchy provides a complete record of the structural search
performed under the available budget.

### Budget and Determinism

For deterministic execution, the budget must be managed deterministically.

Given the same:

- initial state;
- evaluation budget;
- candidate-generation rules;
- objective function;
- numerical tolerances;
- action-selection rules;
- and global-selection rules;

ARRGO must produce the same sequence of budget-consuming sampling
operations.

The budget therefore acts as a deterministic constraint on the search
trajectory.

### Budget Accounting Invariant

The evaluation count must satisfy

$$
N_t
=
|D_t|.
$$

After sampling,

$$
N_{t+1}
=
N_t+1.
$$

After splitting,

$$
N_{t+1}
=
N_t.
$$

Therefore,

$$
N_t
\le
N_{\max}
$$

must hold for every valid state.

The remaining budget is always

$$
B_t
=
N_{\max}-N_t.
$$

These equations provide the basic accounting invariant of ARRGO.

### Budget-Aware Execution Rule

Before every refinement action, ARRGO checks whether the action is compatible
with the remaining evaluation budget.

The decision process is therefore

$$
\boxed{
\text{Determine Refinement Action}
\rightarrow
\text{Check Budget Feasibility}
\rightarrow
\text{Execute Action}
\rightarrow
\text{Update Evaluation Count}
}
$$

The budget check must occur before an objective evaluation is requested.

### Budget Termination Priority

When multiple stopping conditions are possible, ARRGO evaluates them in a
well-defined order.

A certified tolerance condition has priority when a valid certificate has
already been obtained:

$$
\Delta_{\mathrm{global}}
\le
\epsilon.
$$

Otherwise, if

$$
N_t
\ge
N_{\max},
$$

the evaluation budget is exhausted.

Additional termination conditions, such as numerical limitations, may then be
evaluated according to the configured execution policy.

The termination reason must be explicitly recorded.

### Budget-Aware Output

At termination, ARRGO should report at least:

- the best observed point;
- the best observed objective value;
- the total number of objective evaluations;
- the configured evaluation budget;
- the number of refinement iterations;
- the termination reason;
- the final region hierarchy;
- and, in Certified Mode, the final certified global gap when available.

The distinction between these quantities prevents the final result from being
misinterpreted.

### Budget Does Not Define Solution Quality by Itself

A larger evaluation budget does not automatically imply a better solution.

The quality of the result depends on how evaluations are allocated and what
information they reveal.

ARRGO therefore treats the budget as a constraint on information acquisition,
not as an optimization objective.

The central relationship is

$$
\text{Evaluation Budget}
\rightarrow
\text{Available Information}
\rightarrow
\text{Possible Refinement Progress}.
$$

This preserves the information-driven nature of ARRGO.

### Budget Management Principle

The central principle of budget management is

$$
\boxed{
\text{Use the Available Objective-Evaluation Budget as a Deterministic
Constraint on Information Acquisition, Allocate Evaluations According to
Current Global Relevance, and Never Interpret Budget Exhaustion Alone as
Evidence of Global Optimality.}
}
$$

This completes the budget-management mechanism of ARRGO.

The next stage addresses numerical robustness and specifies how duplicate
points, boundaries, tolerances, floating-point effects, and numerical
degeneracies are handled during the actual algorithm execution.

## Numerical Robustness in the Algorithm

ARRGO is defined mathematically over real-valued domains, but its numerical
implementation operates using finite-precision floating-point arithmetic.

Therefore, the implementation must distinguish between mathematical
properties and numerical representations.

Numerical robustness ensures that finite-precision effects do not introduce
invalid regions, duplicate evaluations, inconsistent boundaries, unstable
comparisons, or incorrect termination decisions.

### Mathematical Values and Floating-Point Values

The theoretical formulation assumes real numbers.

In implementation, points are represented using finite-precision numerical
values.

Therefore, two mathematically distinct values may become numerically
indistinguishable at the available precision.

Similarly, two values that should be mathematically equal may differ by a small
floating-point error.

ARRGO must therefore use explicitly defined numerical tolerances wherever
exact equality is inappropriate.

### Numerical Tolerances

ARRGO distinguishes between different types of tolerance.

The main numerical tolerances are:

- point tolerance \(\tau_x\);
- boundary tolerance \(\tau_b\);
- numerical comparison tolerance \(\tau_f\);
- optimization tolerance \(\epsilon\).

These quantities have different meanings and must not be treated as
interchangeable.

The optimization tolerance determines the requested solution accuracy.

The numerical tolerances control the stability of the implementation.

Thus,

$$
\boxed{
\text{Numerical Tolerance}
\neq
\text{Optimization Tolerance}
}
$$

### Point Equality

A candidate point \(x_c\) is considered numerically identical to an existing
evaluation \(x_i\) when

$$
|x_c-x_i|
\le
\tau_x.
$$

Such a candidate must not be evaluated again.

Therefore, the duplicate condition is

$$
\boxed{
|x_c-x_i|
\le
\tau_x
\Rightarrow
x_c
\text{ is a duplicate}
}
$$

This prevents repeated objective evaluations caused by floating-point
rounding.

### Duplicate Candidate Handling

Candidate generation may produce the same numerical location from different
information sources.

For example, a candidate may simultaneously arise from:

- a sampling gap;
- a behavioral transition;
- a certified uncertainty maximum;
- and a certified potential maximum.

ARRGO merges candidates that satisfy

$$
|x_i^c-x_j^c|
\le
\tau_x.
$$

The merged candidate retains the combined provenance of its sources.

Thus, numerical deduplication does not destroy information about why a
candidate was generated.

### Boundary Handling

The search domain is

$$
\Omega=[a,b].
$$

Candidate points must satisfy

$$
a\le x_c\le b.
$$

Because of floating-point arithmetic, a computed candidate may lie slightly
outside the mathematical domain.

ARRGO may therefore project a numerically close candidate onto the boundary
when the deviation is within the configured boundary tolerance.

Conceptually,

$$
x_c<a
\quad\text{and}\quad
|x_c-a|\le\tau_b
$$

is treated as the left boundary.

Similarly,

$$
x_c>b
\quad\text{and}\quad
|x_c-b|\le\tau_b
$$

is treated as the right boundary.

A candidate outside the domain by more than the boundary tolerance is invalid.

### Boundary Evaluation

The domain boundaries may be valid sampling locations.

Therefore,

$$
x=a
\qquad\text{and}\qquad
x=b
$$

may be evaluated when they provide relevant information and have not already
been evaluated.

Boundary handling must be consistent throughout candidate generation,
duplicate detection, and region assignment.

### Split-Point Validity

A split point must lie strictly inside the parent region.

For

$$
R=[l,r],
$$

a valid split satisfies

$$
l<s<r.
$$

In numerical implementation, ARRGO additionally requires sufficient numerical
separation from the boundaries.

Therefore, a practical split must satisfy

$$
s-l>\tau_x
$$

and

$$
r-s>\tau_x.
$$

This prevents the creation of numerically degenerate child regions.

### Contraction Verification

Every structural split must satisfy the contraction requirement

$$
\max
\left(
s-l,
r-s
\right)
\le
\rho(r-l),
\qquad
0<\rho<1.
$$

Because floating-point arithmetic can slightly perturb this comparison,
implementation may use a conservative numerical margin.

A split that fails the contraction requirement must not be executed.

Thus,

$$
\boxed{
\text{Invalid Contraction}
\Rightarrow
\text{Invalid Split}
}
$$

The numerical implementation must never silently relax the mathematical
contraction condition.

### Minimum Region Width

Repeated refinement can eventually produce regions whose width approaches
the limits of floating-point resolution.

Define the numerical region width as

$$
d_R=r-l.
$$

If

$$
d_R
\le
\tau_x,
$$

the region cannot safely support another meaningful interior split.

Such a region should therefore be treated as numerically non-splittable.

This does not imply that the objective function is known throughout the
region.

It means only that the numerical representation cannot safely introduce a
new interior structural boundary.

### Degenerate Regions

A region is numerically degenerate when its boundaries cannot be represented
as sufficiently distinct values.

Conceptually,

$$
|r-l|
\le
\tau_x.
$$

A degenerate region must not be split.

The region remains part of the hierarchy and can retain its existing
information.

### Secant Slope Stability

Observed behavioral analysis uses secant slopes

$$
s_i
=
\frac{f(x_{i+1})-f(x_i)}
{x_{i+1}-x_i}.
$$

When

$$
|x_{i+1}-x_i|
\le
\tau_x,
$$

the denominator is numerically too small for a reliable secant slope.

ARRGO must therefore avoid computing such a slope directly.

Instead, the corresponding pair is treated as numerically duplicated or
insufficiently separated.

This prevents unstable behavioral analysis.

### Nearly Equal Objective Values

Two objective values may differ only because of floating-point effects.

For numerical comparisons, ARRGO may use a function-value tolerance
\(\tau_f\).

Conceptually, two values are numerically indistinguishable when

$$
|f(x_i)-f(x_j)|
\le
\tau_f.
$$

However, \(\tau_f\) must not replace the optimization tolerance
\(\epsilon\).

The former controls numerical comparison stability.

The latter defines the requested optimization accuracy.

### Incumbent Tie Handling

If two evaluated points have numerically indistinguishable objective values,
both may represent equivalent incumbent candidates under the configured
numerical tolerance.

ARRGO must then use deterministic point ordering to select a unique
representative.

For example, a fixed numerical ordering of candidate coordinates may be used.

This ensures reproducibility without claiming that one point is mathematically
better than the other.

### Stable-State Numerical Criteria

A region must not be declared STABLE merely because numerical changes become
small.

For example,

$$
|f(x_{i+1})-f(x_i)|
\le
\tau_f
$$

does not prove that the function is constant between the observations.

Similarly,

$$
|s_i|
\le
\tau_f
$$

does not prove that the derivative is zero throughout the region.

Numerical smallness is therefore evidence of limited observed variation, not a
proof of global behavior.

### Numerical Uncertainty in Certified Mode

Certified Mode requires valid mathematical bounds.

Floating-point implementation can introduce small numerical errors when
evaluating expressions such as

$$
f(x_i)\pm L|x-x_i|.
$$

Therefore, a numerical implementation must ensure that floating-point
rounding does not invalidate the intended enclosure.

A conservative implementation may introduce a small numerical safety margin
when constructing or comparing certified bounds.

The safety margin must be documented and must preserve the direction of the
inequality.

The mathematical requirement remains

$$
L_R(x)
\le
f(x)
\le
U_R(x).
$$

Numerical approximation must never be presented as an exact certificate
unless the implementation guarantees that the bound remains valid.

### Certified Gap Comparison

In Certified Mode, termination depends on

$$
\Delta_{\mathrm{global}}
\le
\epsilon.
$$

Because \(\Delta_{\mathrm{global}}\) is numerically computed, the comparison
must account for floating-point effects.

A conservative implementation should avoid declaring certification merely
because a rounded value appears slightly below \(\epsilon\).

The implementation must therefore use a clearly defined numerical policy for
certified comparisons.

The mathematical certificate remains valid only when the implemented bound is
itself valid.

### Envelope Evaluation

In one dimension, the certified lower and upper envelopes are constructed from
terms of the form

$$
f(x_i)-L|x-x_i|
$$

and

$$
f(x_i)+L|x-x_i|.
$$

The absolute-value functions create piecewise-linear envelopes.

Numerical implementation must handle intersections and breakpoints
consistently.

Two nearly coincident breakpoints should be treated according to the same
numerical tolerance policy used elsewhere in the algorithm.

This prevents small floating-point discrepancies from producing artificial
structural distinctions.

### Candidate Ordering

Candidate sets must be ordered deterministically before final selection.

A deterministic ordering may use:

1. primary information-based comparison;
2. secondary objective relevance;
3. numerical coordinate ordering;
4. fixed provenance ordering when required.

The exact ordering must be fixed by the implementation.

Candidate ordering must never depend on non-deterministic container iteration
or unspecified external ordering.

### Region Ordering

The same principle applies to global region selection.

If two regions remain equivalent under all meaningful comparison criteria,
ARRGO must use a deterministic secondary ordering.

Possible identifiers include:

- region creation index;
- deterministic region identifier;
- depth followed by identifier;
- or another explicitly defined ordering.

The chosen rule must remain fixed throughout one execution.

### Numerical Consistency Across Regions

The same tolerance definitions must be applied consistently across the region
hierarchy.

For example, duplicate detection should not use one point tolerance in one
region and another point tolerance elsewhere unless this difference is
explicitly part of the algorithm.

Consistent numerical policy is necessary for reproducible region boundaries,
candidate sets, and evaluation histories.

### Numerical Limits and Refinement

There may be situations where the mathematical refinement mechanism remains
conceptually valid but the numerical representation can no longer safely
perform the operation.

Examples include:

- region width approaching floating-point resolution;
- candidate points becoming numerically duplicated;
- split points collapsing onto boundaries;
- secant denominators becoming too small;
- or envelope breakpoints becoming numerically indistinguishable.

Such situations should be represented explicitly as numerical limitations.

They must not be silently interpreted as evidence of global optimality.

### Numerical Termination

ARRGO may therefore have a numerical termination reason.

Conceptually,

$$
\mathrm{TerminationReason}
=
\mathrm{NumericalLimit}
$$

when no valid numerically stable refinement operation can be performed and the
configured execution policy requires termination.

Numerical termination returns the best result available at that point.

It does not automatically imply

$$
f_{\mathrm{best}}=f^*.
$$

### Numerical Robustness and Determinism

Numerical robustness and determinism are closely connected.

The same numerical tolerances, comparison rules, boundary conventions, and
ordering rules must produce the same decisions for the same numerical state.

Therefore,

$$
\boxed{
\text{Fixed Numerical Policy}
+
\text{Deterministic Decision Rules}
\Rightarrow
\text{Reproducible Execution}
}
$$

subject to the behavior of the underlying floating-point environment.

### Numerical Safety Principle

Numerical safeguards must never silently change the mathematical meaning of
the algorithm.

Their purpose is to prevent invalid numerical operations and unstable
decisions.

They must not be used to:

- create unsupported optimality claims;
- treat sparse observations as complete information;
- replace certified bounds with estimates;
- ignore the contraction requirement;
- or delete regions from the hierarchy.

### Numerical Robustness Invariants

A correct implementation should preserve the following conditions:

$$
a
\le
x_i
\le
b
$$

for every evaluated point \(x_i\).

Every valid region must satisfy

$$
l<r
$$

up to the configured numerical representation.

Every executed split must satisfy the contraction condition.

Every objective evaluation must correspond to a non-duplicate point under the
configured duplicate tolerance.

The evaluation count must satisfy

$$
N_t
=
|D_t|
\le
N_{\max}.
$$

In Certified Mode, the implemented enclosure must preserve the intended
inequality direction.

### Separation of Numerical and Theoretical Guarantees

The mathematical theory establishes properties under exact assumptions.

The implementation must separately establish that its numerical procedures
faithfully approximate those mathematical operations.

Therefore, ARRGO distinguishes:

$$
\text{Mathematical Validity}
$$

from

$$
\text{Numerical Implementation Validity}.
$$

A numerical safeguard is correct only when it preserves the intended
mathematical property.

### Numerical Robustness Principle

The central principle is

$$
\boxed{
\text{Handle Floating-Point Effects Explicitly Through Fixed Tolerances,
Consistent Boundary and Duplicate Rules, Conservative Certified Comparisons,
and Deterministic Ordering, Without Converting Numerical Limitations Into
Unsupported Optimization Claims.}
}
$$

This completes the numerical robustness requirements of ARRGO.

The next stage provides the complete formal specification of ARRGO by
integrating initialization, analysis, global selection, action selection,
refinement, budget management, state updates, and termination into one
algorithmic specification.

## Complete ARRGO Algorithm Specification

The previous sections defined the mathematical foundations, regional analysis,
candidate generation, action selection, global selection, execution cycle,
budget management, and numerical safeguards of ARRGO.

This section integrates these components into one complete algorithmic
specification.

The purpose is to define ARRGO as a deterministic adaptive refinement
framework before introducing implementation-specific code.

### Algorithm Objective

ARRGO solves the bounded global optimization problem

$$
x^*
\in
\operatorname*{arg\,max}_{x\in\Omega}
f(x),
$$

where

$$
\Omega=[a,b],
\qquad
a<b.
$$

The corresponding optimal objective value is

$$
f^*
=
\max_{x\in\Omega}f(x).
$$

ARRGO does not assume access to derivatives, symbolic expressions, or an
explicit analytical model of \(f\).

The objective is accessed through function evaluations.

### Algorithm Inputs

The core ARRGO procedure receives:

- a bounded domain \(\Omega=[a,b]\);
- a deterministic objective function \(f\);
- an evaluation budget \(N_{\max}\);
- numerical tolerances;
- refinement parameters;
- and the selected execution mode.

The execution mode is either:

$$
\mathrm{Mode}
\in
\{
\mathrm{Empirical},
\mathrm{Certified}
\}.
$$

Certified Mode additionally requires a valid Lipschitz constant \(L\).

### Algorithm Outputs

At termination, ARRGO returns at least:

$$
\left(
x_{\mathrm{best}},
f_{\mathrm{best}}
\right).
$$

It also returns the final:

- evaluation history;
- region hierarchy;
- evaluation count;
- iteration count;
- termination reason;
- final regional states;
- and, in Certified Mode, the final global potential and certified gap.

The result must therefore contain both the optimization result and the
information required to interpret how that result was obtained.

### Initial Global State

The initial region is

$$
R_0=\Omega.
$$

The initial hierarchy is

$$
\mathcal{R}_0
=
\{R_0\}.
$$

The initial evaluation history is

$$
D_0
=
\varnothing
$$

before any objective evaluation is performed.

The evaluation counter is

$$
N_0=0.
$$

The iteration counter is

$$
t=0.
$$

The root region is initially assigned the ACTIVE state.

### Initial Sampling

ARRGO establishes an initial set of valid objective evaluations before
performing information-driven refinement.

Let the deterministic initial candidate set be

$$
C_{\mathrm{init}}
\subseteq
\Omega.
$$

Every initial candidate must satisfy the numerical validity rules.

The objective is evaluated at the selected initial points and the resulting
observations are added to the global evaluation history.

The resulting dataset is

$$
D_0
=
\left\{
(x_i,f(x_i))
\right\}_{i=1}^{N_0}.
$$

The exact initial sampling policy is an implementation-level configuration,
but it must be deterministic and must respect the evaluation budget.

### Initial Incumbent

Once at least one valid evaluation exists, the initial incumbent is

$$
x_{\mathrm{best}}^{(0)}
\in
\operatorname*{arg\,max}_{(x_i,y_i)\in D_0}y_i.
$$

The best observed value is

$$
f_{\mathrm{best}}^{(0)}
=
\max_{(x_i,y_i)\in D_0}y_i.
$$

If multiple points have numerically equivalent objective values, deterministic
tie-breaking selects one representative.

### Initial Region Analysis

After initialization, ARRGO analyzes the root region.

The regional state is

$$
S_{R_0}
=
\left(
D_{R_0},
B_{R_0},
Q_{R_0},
U_{R_0},
P_{R_0},
\mathrm{state}_{R_0}
\right).
$$

The analysis identifies:

- spatial coverage;
- observed behavior;
- candidate extrema;
- unresolved information;
- feasible sampling actions;
- feasible splitting actions;
- and, in Certified Mode, certified uncertainty and potential.

### Main Algorithm Loop

ARRGO then enters its main refinement loop.

At iteration \(t\), the current state is

$$
G_t
=
\left(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t
\right).
$$

The iteration begins by reanalyzing the relevant regions using the current
global information.

### Step 1: Global Reanalysis

ARRGO analyzes the current regional states.

For every relevant region \(R\), the algorithm updates

$$
S_R
=
\left(
D_R,
B_R,
Q_R,
U_R,
P_R,
\mathrm{state}_R
\right).
$$

The analysis uses only currently available observations and valid derived
information.

No unknown function values are assumed.

### Step 2: Determine Active Objectives

For each eligible region, ARRGO identifies the currently unresolved
optimization objectives.

Define

$$
O_R^*
\subseteq
\{
\mathrm{Coverage},
\mathrm{Behavior},
\mathrm{Uncertainty},
\mathrm{Potential}
\}.
$$

The active objective set is dynamic.

It may change after every sampling operation, structural split, incumbent
improvement, or global state update.

### Step 3: Generate Refinement Candidates

For every eligible region, ARRGO generates candidate sampling and splitting
locations.

The sampling candidate set is

$$
C_{\mathrm{sample}}(R).
$$

The splitting candidate set is

$$
C_{\mathrm{split}}(R).
$$

Candidate generation may use:

- spatial gaps;
- observed behavioral transitions;
- observed slope variation;
- certified uncertainty;
- certified potential;
- and valid boundary locations.

Every candidate must satisfy the applicable numerical validity conditions.

### Step 4: Determine Feasible Actions

ARRGO determines the currently feasible actions.

The feasible action set is

$$
F_R
\subseteq
\{
\mathrm{Sample},
\mathrm{Split}
\}.
$$

Sampling requires both a valid candidate and remaining evaluation budget:

$$
\mathrm{Sample}\in F_R
\iff
C_{\mathrm{sample}}(R)\neq\varnothing
\land
N_t<N_{\max}.
$$

Splitting requires at least one valid contraction-preserving split candidate:

$$
\mathrm{Split}\in F_R
\iff
C_{\mathrm{split}}(R)\neq\varnothing.
$$

### Step 5: Select the Local Action

ARRGO evaluates the feasible actions using the active unresolved objectives.

The local decision is

$$
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\}.
$$

The decision hierarchy is

$$
\boxed{
\text{Resolution Requirement}
\rightarrow
\text{Feasibility}
\rightarrow
\text{Objective Relevance}
\rightarrow
\text{Dominance}
\rightarrow
\text{Deterministic Tie-Breaking}
}
$$

No arbitrary fixed weighted objective score is required.

### Step 6: Construct the Global Actionable Set

After local action decisions are determined, ARRGO constructs

$$
\mathcal{G}_t
=
\left\{
R:
R\in\mathcal{E}_t,
\;
\mathrm{Decision}_t(R)
\in
\{
\mathrm{Sample},
\mathrm{Split}
\}
\right\}.
$$

Only these regions can receive the next refinement opportunity.

### Step 7: Global Region Selection

If

$$
\mathcal{G}_t
\neq
\varnothing,
$$

ARRGO compares the globally actionable regions.

The comparison uses their current optimization-relevant information.

In Certified Mode, regional potential is directly connected to global
relevance.

A region satisfying

$$
P_R
>
f_{\mathrm{best}}
+
\epsilon
$$

remains competitive for an \(\epsilon\)-optimal certificate.

After dominance and objective-specific comparison, ARRGO selects one region

$$
R_t^*
\in
\mathcal{G}_t.
$$

Deterministic tie-breaking is applied when necessary.

### Step 8: Select the Refinement Candidate

After selecting \(R_t^*\), ARRGO uses its local decision

$$
a_t
=
\mathrm{Decision}_t(R_t^*).
$$

If

$$
a_t
=
\mathrm{Sample},
$$

ARRGO selects one valid candidate

$$
x_t^c
\in
C_{\mathrm{sample}}(R_t^*).
$$

If

$$
a_t
=
\mathrm{Split},
$$

ARRGO selects one valid split point

$$
s_t^c
\in
C_{\mathrm{split}}(R_t^*).
$$

Candidate selection remains deterministic.

### Step 9: Execute Sampling

If

$$
a_t
=
\mathrm{Sample},
$$

ARRGO evaluates

$$
y_t^c
=
f(x_t^c).
$$

The global evaluation history becomes

$$
D_{t+1}
=
D_t
\cup
\left\{
(x_t^c,y_t^c)
\right\}.
$$

The evaluation count becomes

$$
N_{t+1}
=
N_t+1.
$$

The new observation is incorporated into the regional information state.

### Step 10: Execute Splitting

If

$$
a_t
=
\mathrm{Split},
$$

the selected region

$$
R_t^*
=
[l,r]
$$

is divided at

$$
s_t^c
\in
(l,r).
$$

The resulting child regions are

$$
R_L
=
[l,s_t^c]
$$

and

$$
R_R
=
(s_t^c,r].
$$

The contraction requirement is

$$
\max
\left(
s_t^c-l,
r-s_t^c
\right)
\le
\rho(r-l),
\qquad
0<\rho<1.
$$

No new objective evaluation is performed solely by the split.

Therefore,

$$
N_{t+1}
=
N_t.
$$

### Step 11: Preserve the Parent Region

The parent region remains part of the persistent hierarchy.

The updated hierarchy satisfies

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

The parent region becomes REFINED.

The child regions are inserted into the hierarchy and analyzed using the
information inherited from the parent.

No region is deleted as a consequence of splitting.

### Step 12: Update the Incumbent

After a sampling operation, ARRGO updates the incumbent according to

$$
f_{\mathrm{best}}^{(t+1)}
=
\max
\left(
f_{\mathrm{best}}^{(t)},
y_t^c
\right).
$$

Therefore,

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

If the refinement operation is a split, the incumbent remains unchanged.

### Step 13: Update Regional Information

After refinement, ARRGO recomputes the affected regional states.

The state includes

$$
S_R
=
\left(
D_R,
B_R,
Q_R,
U_R,
P_R,
\mathrm{state}_R
\right).
$$

Sampling modifies the observation set and therefore may modify behavior,
coverage, uncertainty, and potential.

Splitting modifies the spatial representation and produces independently
analyzable child regions.

### Step 14: Update Certified Bounds

In Certified Mode, ARRGO recomputes the valid regional bounds.

The lower envelope is

$$
L_R(x)
=
\max_i
\left[
f(x_i)-L|x-x_i|
\right].
$$

The upper envelope is

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

The regional potential is

$$
P_R
=
\max_{x\in R}U_R(x).
$$

The global potential is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P_R.
$$

The certified gap is

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

The certified invariant is

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}}.
$$

### Step 15: Recompute Global Relevance

After the regional and certified states have been updated, ARRGO recomputes
global region relevance.

This is necessary because one refinement can change the relative importance
of many regions.

For example, an improved incumbent may reduce the competitive status of
previously promising regions.

Therefore, global relevance is always evaluated from the current state.

### Step 16: Check Termination

ARRGO checks termination only after all relevant state updates are complete.

In Certified Mode, the primary certificate condition is

$$
\Delta_{\mathrm{global}}
\le
\epsilon.
$$

If this condition holds, then

$$
f_{\mathrm{best}}
\ge
f^*
-
\epsilon.
$$

The algorithm may therefore terminate with an \(\epsilon\)-optimality
certificate.

If the evaluation budget is exhausted,

$$
N_t
\ge
N_{\max},
$$

the algorithm terminates according to the configured budget policy.

Budget exhaustion alone does not establish global optimality.

### Step 17: Continue or Terminate

If no termination condition is satisfied and at least one valid refinement
opportunity remains, ARRGO advances to the next iteration:

$$
t
\leftarrow
t+1.
$$

The new state becomes

$$
G_{t+1}.
$$

The complete cycle then repeats.

### No-Action Condition

If no globally actionable region exists,

$$
\mathcal{G}_t
=
\varnothing,
$$

ARRGO does not repeatedly execute Stable actions.

Instead, it checks whether:

- the certified termination condition is satisfied;
- the evaluation budget is exhausted;
- a numerical limit has been reached;
- or the current state requires eligibility or resolution reconsideration.

If none of these conditions provides a valid continuation, ARRGO terminates
with an explicit termination reason.

The absence of a refinement action is never silently interpreted as proof of
global optimality.

### Core State Invariants

Throughout the complete algorithm, ARRGO preserves

$$
D_t
\subseteq
D_{t+1},
$$

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1},
$$

$$
N_{t+1}
\ge
N_t,
$$

and

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

In Certified Mode,

$$
f_{\mathrm{best}}^{(t)}
\le
f^*
\le
P_{\mathrm{global}}^{(t)}.
$$

These invariants provide the consistency foundation of the complete
algorithm.

### Complete ARRGO Control Flow

The complete control flow is

$$
\boxed{
\begin{aligned}
&\text{Initialize Domain and Root Region}\\
&\rightarrow
\text{Perform Deterministic Initial Sampling}\\
&\rightarrow
\text{Initialize Incumbent}\\
&\rightarrow
\text{Analyze Regions}\\
&\rightarrow
\text{Determine Active Objectives}\\
&\rightarrow
\text{Generate Candidates}\\
&\rightarrow
\text{Determine Feasible Actions}\\
&\rightarrow
\text{Select Local Actions}\\
&\rightarrow
\text{Select Global Region}\\
&\rightarrow
\text{Select Refinement Candidate}\\
&\rightarrow
\text{Execute Sample or Split}\\
&\rightarrow
\text{Update Information}\\
&\rightarrow
\text{Update Incumbent}\\
&\rightarrow
\text{Update Certified Bounds}\\
&\rightarrow
\text{Recompute Global Relevance}\\
&\rightarrow
\text{Check Termination}\\
&\rightarrow
\text{Repeat or Return Result}.
\end{aligned}
}
$$

### Formal Algorithm Definition

ARRGO can therefore be characterized as the deterministic mapping

$$
G_{t+1}
=
\mathcal{T}
\left(
G_t,
f,
\Theta
\right),
$$

where \(\mathcal{T}\) is the complete refinement transition and
\(\Theta\) denotes the fixed algorithm configuration.

The configuration includes:

- numerical tolerances;
- refinement parameters;
- evaluation budget;
- execution mode;
- candidate-generation rules;
- candidate-comparison rules;
- action-selection rules;
- global-selection rules;
- and deterministic tie-breaking rules.

The transition \(\mathcal{T}\) must use only information available in \(G_t\)
and the objective evaluations explicitly requested by the algorithm.

### Determinism Requirement

For a deterministic objective function and fixed configuration \(\Theta\),
ARRGO must produce a reproducible refinement trajectory.

Therefore, given the same initial problem state,

$$
G_0,
$$

and the same configuration,

$$
\Theta,
$$

the sequence

$$
G_0,
G_1,
G_2,
\ldots
$$

is deterministic up to explicitly defined floating-point behavior.

### Certified and Empirical Interpretation

ARRGO has two distinct interpretation levels.

In Empirical Mode, the algorithm produces an adaptive best-found solution based
on observed function information.

The result is not automatically accompanied by a rigorous global optimality
certificate.

In Certified Mode, valid Lipschitz information enables deterministic regional
upper bounds and a global optimality gap.

The certified chain is

$$
f_{\mathrm{best}}
\le
f^*
\le
P_{\mathrm{global}},
$$

followed by

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

If

$$
\Delta_{\mathrm{global}}
\le
\epsilon,
$$

then the returned incumbent is guaranteed to be \(\epsilon\)-optimal under
the validity assumptions of Certified Mode.

### No-Pruning Requirement

The complete algorithm never deletes previously created regions.

The persistent hierarchy satisfies

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

A region can be:

- refined;
- stable;
- temporarily non-competitive;
- or globally non-selected;

without being removed from the historical hierarchy.

This preserves complete structural and informational history.

### Conceptual Identity of ARRGO

ARRGO can be summarized by the following transformation:

$$
\boxed{
\text{Objective Evaluations}
\rightarrow
\text{Regional Information}
\rightarrow
\text{Unresolved Information}
\rightarrow
\text{Adaptive Refinement}
\rightarrow
\text{Improved Global Information}
}
$$

The algorithm repeatedly applies this transformation until the configured
termination condition is satisfied.

### Final Algorithm Principle

The complete ARRGO principle is

$$
\boxed{
\text{Adaptively Allocate Refinement to the Most Relevant Regions, Use Sampling
to Acquire Missing Objective Information, Use Splitting to Increase Spatial
Resolution, Preserve the Complete Search History, and Terminate Only When the
Configured Empirical or Certified Stopping Condition Is Satisfied.}
}
$$

This completes the formal algorithm design of ARRGO.

The next notebook can translate this specification into an executable
implementation while preserving the mathematical assumptions, invariants,
refinement logic, and separation between Empirical and Certified modes.